# Setup

In [ ]:
import distinctipy
from matplotlib.dates import DayLocator, DateFormatter, HourLocator, MinuteLocator, ConciseDateFormatter, DateFormatter, AutoDateLocator, AutoDateFormatter, num2date
from matplotlib.ticker import FuncFormatter,Formatter,ScalarFormatter
from typing import Literal,Union,Any
from matplotlib.ticker import MaxNLocator
import tensorflow.keras as keras
import pandas as pd
from time import time
import datetime
from datetime import timedelta
import os
import copy
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
import math
import seaborn as sns
from IPython.display import display
from collections import namedtuple
import sys
import pickle
import glob
import optuna
from time import perf_counter
from typing import Literal
import matplotlib.axes

In [ ]:
os.chdir(Path(r""))
print("Current working directory:",os.getcwd())
data = "./Inputs/multivariate.csv"
dirpath_results = "./Results"

In [ ]:
def save_variable(variable, path_to_file):
    with open(path_to_file, 'wb') as file:
        pickle.dump(variable, file)
def load_variable(path_to_file):
    with open(path_to_file, 'rb') as file:
        return pickle.load(file)

In [ ]:
def manipulate_features(
        df_to_be_manipulated:pd.DataFrame,
        to_be_removed:list[str]|None=None,
        to_be_added:pd.DataFrame|pd.Series|None=None,
        concat_kwargs:dict={},
        return_id_manipulation:bool=True,
        print_info:bool=True,
        id_addition:str|None=None,
        id_removal:str|None=None,
        forced_id:str|None=None,
)->tuple[pd.DataFrame,str]|pd.DataFrame:
    df=copy.deepcopy(df_to_be_manipulated)
    if print_info:
        print("----df_to_be_manipulated----")
        print("Original column(s):",df.columns)
    add=copy.deepcopy(to_be_added)
    if all([to_be_removed is None,add is None]):
        if all([id_removal is None, id_addition is None, forced_id is None]):
            raise AssertionError("When both to_be_removed and to_be_added are None, one among id_removal, id_addition, and forced_id has to be not None.")
    if to_be_removed is not None:
        id_removal="R_"
        for label in to_be_removed:
            list_substrings=label.split("_")
            feature_id=""
            for substring in list_substrings:
                feature_id+=substring[0]
            id_removal+=feature_id
        assert all([label in df.columns for label in to_be_removed]),"Some features to be removed don't exist in df_to_be_manipulated."
        df.drop(columns=to_be_removed, inplace=True)
        if print_info:
            print("\nColumn(s) removed:",to_be_removed)
            print("Column(s) after drop:",df.columns)
    if add is not None:
        assert len(add)==df.shape[0],"Shape mismatch, no matching number of rows between df_to_be_manipulated and to_be_added."           
        if isinstance(add,pd.Series):
            assert add.name is not None,"You must provide a named Series. Set its name property using Series.name=..."
            assert add.name not in df.columns,"Added features must have different names from the existing ones.\n Select a name not in:\n{}".format(df.columns)
        else:
            assert all([nm not in df.columns for nm in add.columns]),"Added features must have different names from the existing ones.\n Select names not in:\n{}".format(df.columns)
        if any(add.index!=df.index):
            reply=input("to_be_added has index mismatch with df_to_be_manipulated.\nThis will lead to NaN values unless you specify proper concat_kwargs.\nDo you want to impose to_be_added.index=df_to_be_manipulated.index before concatenation (type 'ok' if yes)?")
            if reply=='ok':
                add.index=df.index
        df=pd.concat([df,add],axis="columns",**concat_kwargs)
        if print_info:
            labels_added=[add.name] if isinstance(add,pd.Series) else add.columns
            print("\nColumn(s) added:",labels_added)        
            print("Column(s) after concatenation:",df.columns)
        id_addition="A_"
        for label in labels_added:
            list_substrings=label.split("_")
            feature_id=""
            for substring in list_substrings:
                feature_id+=substring[0]
            id_addition+=feature_id
    id_manipulation=""
    if forced_id is not None:
        id_manipulation=forced_id
    else:
        if id_removal is not None:
            id_manipulation+=id_removal
            if id_addition is not None:
                id_manipulation+="_"+id_addition
        else:
            id_manipulation+=id_addition
    if print_info:
        print("\n----id_manipulation----")
        print(id_manipulation)
    if return_id_manipulation:
        return df,id_manipulation
    else:
        return df

## Postprocessing functions for Optuna studies

In [ ]:
def find_trial_in_study(
        study:optuna.study.Study,pick_best:bool=False,
        trial_number:int=0,
        return_only_val_losses_df:bool=False,
        best_as:Literal["lowest_intermediate","lowest_objective_return"]="lowest_intermediate",
        filter_on:list[tuple[str,Any]]|None=None
        )->tuple[optuna.trial.FrozenTrial,int,int]|pd.DataFrame:
    trials=study.trials
    if filter_on is not None:
        for filter in filter_on:
            trials=[trial for trial in trials if trial.params[filter[0]]==filter[1]]
    val_losses_dict={
        "trial number":[],
        "trial best epoch number":[],
        "trial best validation loss":[]
    }
    num_to_idx={}
    for idx,trial in enumerate(trials):
        num_to_idx.update({trial.number:idx})
        val_losses_dict["trial number"].append(trial.number)
        val_losses_trial_dict=trial.intermediate_values
        if len(val_losses_trial_dict.keys())>0:
            val_losses_trial_for_df={"epoch number":[],"val loss":[]}
            for epoch_num,val_loss in val_losses_trial_dict.items():
                val_losses_trial_for_df["epoch number"].append(epoch_num)
                val_losses_trial_for_df["val loss"].append(val_loss)
            val_losses_trial_Series=pd.Series(val_losses_trial_for_df["val loss"],index=val_losses_trial_for_df["epoch number"])
            if val_losses_trial_Series.isna().sum()==len(val_losses_trial_Series):
                val_losses_dict["trial best epoch number"].append(0)
                val_losses_dict["trial best validation loss"].append(np.nan)
            else:
                val_losses_dict["trial best epoch number"].append(val_losses_trial_Series.idxmin(skipna=True))
                val_losses_dict["trial best validation loss"].append(val_losses_trial_Series.min(skipna=True))
        else:
            val_losses_dict["trial best epoch number"].append(0)
            val_losses_dict["trial best validation loss"].append(np.inf)
    val_losses_df=pd.DataFrame(val_losses_dict)
    val_losses_df=val_losses_df.set_index("trial number")
    if best_as=="lowest_objective_return":
        best_trial_number=study.best_trial.number
    else:
        best_trial_number=val_losses_df["trial best validation loss"].idxmin(skipna=True)
    selected_trial_number=best_trial_number if pick_best else trial_number
    selected_trial=trials[num_to_idx[selected_trial_number]]
    selected_trial_best_epoch_number=val_losses_df.loc[selected_trial_number,"trial best epoch number"]
    if not return_only_val_losses_df:
        return selected_trial,selected_trial_number,selected_trial_best_epoch_number
    else:
        return val_losses_df

In [ ]:
from matplotlib.figure import Figure
import matplotlib.offsetbox
from matplotlib.patheffects import withStroke
import matplotlib.pyplot
def get_hyperparameters_exploration(
        study:optuna.study.Study,n_rows:int|None=None,hps_in_log_scale:list[str]|None=None,
        hps_in_discrete_scale:list[str]|None=None,hps_categorical:list[str]=[],
        hps_to_ignore:list[str]=[],
        to_shrink:list[str]=["attention_head_size"],
        figsize:tuple[float,float]=(16,16),
        subplots:bool=False,
        overlap_best:bool=True,hue:str="Trial State",detail_epochs:bool=False,
        categoricals:list[str]=["Trial State"],
        custom_ticks:list[str]=['hidden_size','hidden_continuous_size'],
        n_custom_ticks:list[int]=[10,10],
        xlabel_as_hp_name:list[str]=[],
        rename_map:dict[str,str]={},
        capital_title:list[str]=[],
        markersize:int=12,
        markerfacecolor:str="red",
        markeredgecolor:str="red",
        sns_histplot_kwargs:dict={},
        fontsize:int=14,
        scores:dict[str,float]|None=None,
        metric_name:Literal["MAE","MSE","RMSE","R2","MAPE"]="R2",
        study_number:int|None=None,
        hps_to_rotate_ticks:list[tuple[str,float]]=[],
        step_is:Literal["epoch","prediction day index"]="epoch",
        force_hps_non_const_to_be:list[str]|None=None,
        best_as:Literal["lowest_intermediate","lowest_objective_return"]="lowest_intermediate",
        filter_on:list[tuple[str,Any]]|None=None,
        shrink_factor:float=0.5,
        xaxis_integer:list[str]=[],
        ylims:dict[str,tuple[float]]={},
        marker=".",
        dpi:int|None=None,
        to_approx:dict[str,int]={}
        )->None:
    from matplotlib.ticker import MaxNLocator
    with plt.rc_context({'font.size': fontsize}):
        if scores is not None:
            scores_upper={key.upper():value for key,value in scores.items()}
            scores=scores_upper
            names_to_udm={
                "MAE":" kWh",
                "MSE":r" kWh$^2$",
                "RMSE":" kWh",
                "MAPE":"%",
                "R2":"",
            } 
        states={
            0: "Running",
            1: "Complete",
            2: "Pruned",
            3: "Failed",
            4: "Waiting"
        }
        states_colors={
            "Running":"tab:olive",
            "Complete":"tab:blue",
            "Pruned":"tab:orange",
            "Failed":"tab:red",
            "Waiting":"tab:gray"    
        }
        def plot_hist(data,ax,hp_name,discrete=None,shrink=None,binrange=None,log_scale=None,hue=None,sns_histplot_kwargs={}):
            if discrete is None:
                discrete=True if hp_name in hps_in_discrete_scale else False
            if binrange is not None:
                binrange=binrange[hp_name] if hp_name not in categoricals and hp_name not in hps_categorical else None
            if shrink is None:
                shrink=shrink_factor if hp_name in to_shrink else 1
            if log_scale is None:
                log_scale=True if hp_name in hps_in_log_scale else False
            if hue is not None:
                hue=hue
                hue_order=[states[i] for i in range(5) if states[i] in data[hue].unique().tolist()]
            else:
                hue_order=None
            sns.histplot(data=data,x=hp_name,hue=hue,
                        hue_order=hue_order,
                        ax=ax,log_scale=log_scale,
                        palette=sns.color_palette([states_colors[st] for st in hue_order]) if hue_order is not None else None,
                        discrete=discrete,
                        binrange=binrange,
                        shrink=shrink,
                        **sns_histplot_kwargs)
            if hp_name not in rename_map.keys():
                if hp_name not in xlabel_as_hp_name:
                    ax.set_xlabel(hp_name.replace("_"," ").capitalize())
            else:
                ax.set_xlabel(rename_map[hp_name])
            if hp_name in [hpn for hpn,_ in hps_to_rotate_ticks]:
                rt=[rot for hpn,rot in hps_to_rotate_ticks if hpn==hp_name][0]
                ax.set_xticks(ax.get_xticks(),ax.get_xticklabels(),rotation=rt)
        def customize_ticks(hp_name,binrange,ax,choices=None):
            if hp_name in custom_ticks:
                if choices is None:
                    idx_in_custom_ticks=0
                    for name in custom_ticks:
                        if hp_name==name:
                            break
                        else:
                            idx_in_custom_ticks+=1
                    if len(n_custom_ticks)<len(custom_ticks):
                        raise Exception("You need to provide the number of custom ticks to be visualized\nfor each hyperparameter in custom_ticks!")
                    min,max=binrange[hp_name]
                    step=math.floor((max-min)/(n_custom_ticks[idx_in_custom_ticks]-1))
                    ticks=np.arange(min,max+1,step=step)
                    ax.set_xticks(ticks,[str(value) for value in ticks])
                else:
                    ax.set_xticks(choices,[str(choice) for choice in choices])
        def plot_best(ax,best_trial,hp_name,marker=marker,add_ax_title=True):
            ax.plot(
                best_trial.params[hp_name] if hp_name!="model_dimension" else round(best_trial.params[hp_name]/best_trial.params["number_of_heads"])*best_trial.params["number_of_heads"],
                0,marker=marker,
                markerfacecolor=markerfacecolor,markeredgecolor=markeredgecolor,
                markersize=markersize,markeredgewidth=0.5
            )
            if add_ax_title:
                title="Study #{}:".format(study_number) if study_number is not None else ""
                if hp_name not in rename_map.keys():
                    hpn_to_show=hp_name.replace("_"," ")
                else:
                    if hp_name not in capital_title:
                        hpn_to_show=rename_map[hp_name].lower()
                    else:
                        hpn_to_show=rename_map[hp_name]
                btp_val=best_trial.params[hp_name] if hp_name not in list(to_approx.keys()) else round(best_trial.params[hp_name],to_approx[hp_name])
                title+=" {}est trial's {}{}{}".format(
                    "B" if study_number is None else "b",
                    hpn_to_show,
                    "=" if hp_name not in list(to_approx.keys()) else r"$\approx$",
                    btp_val if hp_name!="model_dimension" else round(best_trial.params[hp_name]/best_trial.params["number_of_heads"])*best_trial.params["number_of_heads"]
                )
                text=ax.set_title(
                    title if scores is None else title+", {}={:.2f}{}".format(metric_name,scores[metric_name],names_to_udm[metric_name]),
                    color=markerfacecolor
                )
                text.set_path_effects([withStroke(linewidth=0.5, foreground="black")])
        trials=study.get_trials()
        if filter_on is not None:
            for filter in filter_on:
                trials=[trial for trial in trials if trial.params[filter[0]]==filter[1]]
        if overlap_best:
            best_trial,_,_=find_trial_in_study(study,pick_best=True,best_as=best_as,filter_on=filter_on)
        hps_names=[key for key in trials[0].params]
        if not force_hps_non_const_to_be:
            hps_non_const=[hp_name for hp_name in hps_names if not isinstance(trials[0].distributions[hp_name],optuna.distributions.CategoricalDistribution) and trials[0].distributions[hp_name].low!=trials[0].distributions[hp_name].high]
        else:
            hps_non_const=[hp_name for hp_name in force_hps_non_const_to_be if not isinstance(trials[0].distributions[hp_name],optuna.distributions.CategoricalDistribution)]
        if hps_in_log_scale is None:
            hps_in_log_scale=[hp_name for hp_name in hps_names if not isinstance(trials[0].distributions[hp_name],optuna.distributions.CategoricalDistribution) and trials[0].distributions[hp_name].log==True]
        if hps_in_discrete_scale is None:
            hps_in_discrete_scale=[hp_name for hp_name in hps_names if not isinstance(trials[0].distributions[hp_name],optuna.distributions.CategoricalDistribution) and trials[0].distributions[hp_name].step!=None]
        try:
            assert len(hps_categorical)>0
            binrange={hp_name:[[],[]] for hp_name in hps_non_const+hps_categorical}
            hps_dict={hp_name:[] for hp_name in hps_non_const+hps_categorical}
        except AssertionError:
            binrange={hp_name:[[],[]] for hp_name in hps_non_const}
            hps_dict={hp_name:[] for hp_name in hps_non_const}
        hps_dict["Trial State"]=[]
        hps_dict["Last "+step_is]=[]
        hps_dict["Trial Number"]=[]
        for trial in trials:
            if "duplicate" not in trial.user_attrs.keys():
                trial_params=trial.params
                for hp_name in hps_non_const:
                    hps_dict[hp_name].append(trial_params[hp_name])
                    binrange[hp_name][0].append(math.log10(trial.distributions[hp_name].low) if hp_name in hps_in_log_scale else trial.distributions[hp_name].low)
                    binrange[hp_name][1].append(math.log10(trial.distributions[hp_name].high) if hp_name in hps_in_log_scale else trial.distributions[hp_name].high)
                if len(hps_categorical)>0:
                    for hp_name in hps_categorical:
                        if hp_name=="model_dimension":
                            hps_dict[hp_name].append(round(trial_params[hp_name]/trial_params["number_of_heads"])*trial_params["number_of_heads"])
                        else:
                            hps_dict[hp_name].append(trial_params[hp_name])
                        if isinstance(trial.distributions[hp_name].choices[0],str):
                            binrange[hp_name][0].append(None)
                            binrange[hp_name][1].append(None)
                        else:
                            binrange[hp_name][0].append(math.log10(min(trial.distributions[hp_name].choices)) if hp_name in hps_in_log_scale else min(trial.distributions[hp_name].choices))
                            binrange[hp_name][1].append(math.log10(max(trial.distributions[hp_name].choices)) if hp_name in hps_in_log_scale else max(trial.distributions[hp_name].choices))
                hps_dict["Trial Number"].append(trial.number) 
                if any(
                    [
                        "check_finite_triggered" in trial.user_attrs.keys(),
                        "max_lag_exceeding_lkb" in trial.user_attrs.keys(),
                        "LinAlgError" in trial.user_attrs.keys(),
                        "ValueError" in trial.user_attrs.keys()
                    ]
                ):
                    hps_dict["Trial State"].append(states[3])
                else:
                    hps_dict["Trial State"].append(states[trial.state])
                hps_dict["Last "+step_is].append(trial.last_step if trial.last_step is not None else 0)
        for hp_name in binrange:
            binrange[hp_name][0]=min(binrange[hp_name][0]) if all([val is not None for val in binrange[hp_name][0]]) else None
            binrange[hp_name][1]=max(binrange[hp_name][1]) if all([val is not None for val in binrange[hp_name][1]]) else None
        for name in categoricals:
            converted_values=[str(value) for value in hps_dict[name]]
            hps_dict[name]=converted_values
        hps_df=pd.DataFrame(hps_dict)
        if n_rows is None:
            n_rows=2
        if len(hps_categorical)>0:
            hps_non_const=hps_non_const+hps_categorical
        if len(hps_to_ignore)>0:
            hps_non_const=[hp_name for hp_name in hps_non_const if hp_name not in hps_to_ignore]
        n_cols=math.ceil(len(hps_non_const)/n_rows)
        if subplots:
            fig,axs=plt.subplots(n_rows,n_cols,squeeze=False,figsize=figsize,dpi=dpi)
            axs=axs.reshape((axs.shape[0]*axs.shape[1]))
            for ax_idx,hp_name in enumerate(hps_non_const):
                plot_hist(data=hps_df,ax=axs[ax_idx],hp_name=hp_name,binrange=binrange,hue=hue,sns_histplot_kwargs=sns_histplot_kwargs)
                if overlap_best:
                    plot_best(ax=axs[ax_idx],best_trial=best_trial,hp_name=hp_name)
                customize_ticks(
                    hp_name=hp_name,binrange=binrange,ax=axs[ax_idx],
                    choices=trials[0].distributions[hp_name].choices if isinstance(trials[0].distributions[hp_name],optuna.distributions.CategoricalDistribution) else None
                )
                if hp_name in ylims.keys():
                    axs[ax_idx].set_ylim(ylims[hp_name])
                axs[ax_idx].yaxis.set_major_locator(MaxNLocator(integer=True))
                if hp_name in xaxis_integer:
                    axs[ax_idx].xaxis.set_major_locator(MaxNLocator(integer=True,min_n_ticks=1))
            for idx in range(len(hps_non_const),n_rows*n_cols):
                axs[idx].axis("off")
        else:
            figlist=[]
            if not detail_epochs:
                for hp_name in hps_non_const:
                    fig=Figure(figsize=figsize,dpi=dpi)
                    ax=fig.gca()
                    plot_hist(data=hps_df,ax=ax,hp_name=hp_name,binrange=binrange,hue=hue,sns_histplot_kwargs=sns_histplot_kwargs)
                    if overlap_best:
                        plot_best(ax=ax,best_trial=best_trial,hp_name=hp_name)
                    customize_ticks(
                        hp_name=hp_name,binrange=binrange,ax=ax,
                        choices=trials[0].distributions[hp_name].choices if isinstance(trials[0].distributions[hp_name],optuna.distributions.CategoricalDistribution) else None
                    )
                    if hp_name in ylims.keys():
                        ax.set_ylim(ylims[hp_name])
                    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
                    if hp_name in xaxis_integer:
                        ax.xaxis.set_major_locator(MaxNLocator(integer=True,min_n_ticks=1))
                    figlist.append(fig)
            else:
                categories=np.sort(hps_df["Last "+step_is].unique())
                num_categories=len(categories)
                if n_rows is None:
                    n_rows=num_categories//2
                n_cols=math.ceil(num_categories/n_rows)
                cmap=plt.get_cmap()
                colors=[cmap(i) for i in np.linspace(0,1,num_categories)]
                for hp_name in hps_non_const:
                    fig,axs=plt.subplots(n_rows,n_cols,squeeze=False,figsize=figsize,dpi=dpi)
                    axs=axs.reshape((axs.shape[0]*axs.shape[1]))
                    for ax_idx,category in enumerate(categories):
                        sns_histplot_kwargs.update(dict(color=colors[ax_idx],alpha=0.5))
                        plot_hist(data=hps_df.loc[hps_df["Last "+step_is]==category,:],
                                ax=axs[ax_idx],
                                hp_name=hp_name,
                                binrange=binrange,
                                sns_histplot_kwargs=sns_histplot_kwargs)
                        if overlap_best:
                            if category==best_trial.last_step:
                                plot_best(ax=axs[ax_idx],best_trial=best_trial,hp_name=hp_name,add_ax_title=False)
                        customize_ticks(
                            hp_name=hp_name,binrange=binrange,ax=axs[ax_idx],
                            choices=trials[0].distributions[hp_name].choices if isinstance(trials[0].distributions[hp_name],optuna.distributions.CategoricalDistribution) else None
                        )
                        axs[ax_idx].set_title("Last {} = {}".format(step_is,category))
                        if hp_name in ylims.keys():
                            axs[ax_idx].set_ylim(ylims[hp_name])
                        axs[ax_idx].yaxis.set_major_locator(MaxNLocator(integer=True))
                        if hp_name in xaxis_integer:
                            axs[ax_idx].xaxis.set_major_locator(MaxNLocator(integer=True,min_n_ticks=1))
                    for idx in range(num_categories,n_rows*n_cols):
                        axs[idx].axis("off")
                    fig.tight_layout()
                    figlist.append(fig)
        plt.close("all")            
        if subplots:
            fig.tight_layout()
            display(fig)
        else:
            for fig in figlist:
                display(fig)

In [ ]:
def replace_capitalize_space(
        label:str,
        udms:dict[str,str]={}
        )->str:
    lbl=copy.deepcopy(label)
    if lbl in udms.keys():
        if "(" in lbl:
            lbl=lbl.replace(lbl[lbl.find("("):lbl.find(")")+1],"("+udms[lbl]+")",1)
        else:
            lbl+="("+udms[lbl]+")"
    lbl=lbl.replace(lbl[0],lbl[0].upper(),1).replace("_"," ",-1)
    pieces=lbl.split("(")
    if pieces[0][-1]!=" ":
        lbl=" (".join(pieces)
    return lbl

In [ ]:
og_labels=[
    "Time",
    "Season",
    "Month_of_year",
    "Day_of_week",
    "Day_of_month",
    "Hour_of_day",
    "Agriculture_building_load",
    "Mean_air_T(°C)",
    "Precipitation_(mm)",
    "Air_pressure_(hPa)",
    "Mean_air_humidity(%)",
    "Sol_Rad_(MJ/mq)",
    "Sol_Irrad(W/m2)",
    "Mean_wind_speed_2m(m/s)",
    "Max_wind_speed_2m(m/s)",
    "Direction_wind_2m",
    "Frequency_no_speed_2m(%)",
    "Mean_wind_speed_10m(m/s)",
    "Max_wind_speed_10m(m/s)",
    "Direction_wind_10m",
    "Frequency_no_speed_10m",
    "Architecture_building",
    "Economics",
    "Day_of_year",
]
manually_customized={
    "Agriculture_building_load":"Agriculture building load (kWh)",
    "Mean_air_T(°C)":"Mean air temperature (°C)",
    "Sol_Rad_(MJ/mq)":"Solar irradiation "+r"(MJ m$^{-2}$)",
    "Sol_Irrad(W/m2)":"Solar irradiance "+r"(W m$^{-2}$)",
    "Mean_wind_speed_2m(m/s)":"Mean wind speed at 2m (m"+r" s$^{-1}$)",
    "Max_wind_speed_2m(m/s)":"Max wind speed at 2m (m"+r" s$^{-1}$)",
    "Direction_wind_2m":"Direction of wind at 2m",
    "Frequency_no_speed_2m(%)":"Frequency of null wind speed at 2m (%)",
    "Mean_wind_speed_10m(m/s)":"Mean wind speed at 10m (m"+r" s$^{-1}$)",
    "Max_wind_speed_10m(m/s)":"Max wind speed at 10m (m"+r" s$^{-1}$)",
    "Direction_wind_10m":"Direction of wind at 10m",
    "Frequency_no_speed_10m":"Frequency of null wind speed at 10m",
    "Architecture_building":"Architecture building load (kWh)",
    "Economics":"Economics building load (kWh)",
}
og_to_new_labels={lbl:replace_capitalize_space(lbl) for lbl in og_labels if lbl not in manually_customized.keys()}
og_to_new_labels.update(manually_customized)
print("og_to_new_labels:")
display(og_to_new_labels)
og_to_new_labels_no_udms={
    "Time":"Time",
    "Season":"Season",
    "Month_of_year":"Month of year",
    "Day_of_week":"Day of week",
    "Day_of_month":"Day of month",
    "Hour_of_day":"Hour of day",
    "Precipitation_(mm)":"Precipitation",
    "Air_pressure_(hPa)":"Air pressure",
    "Mean_air_humidity(%)":"Mean air humidity",
    "Day_of_year":"Day of year",
    "Agriculture_building_load":"Agriculture building load",
    "Mean_air_T(°C)":"Mean air temperature",
    "Sol_Rad_(MJ/mq)":"Solar irradiation",
    "Sol_Irrad(W/m2)":"Solar irradiance",
    "Mean_wind_speed_2m(m/s)":"Mean wind speed at 2m",
    "Max_wind_speed_2m(m/s)":"Max wind speed at 2m",
    "Direction_wind_2m":"Direction of wind at 2m",
    "Frequency_no_speed_2m(%)":"Frequency of null wind speed at 2m",
    "Mean_wind_speed_10m(m/s)":"Mean wind speed at 10m",
    "Max_wind_speed_10m(m/s)":"Max wind speed at 10m",
    "Direction_wind_10m":"Direction of wind at 10m",
    "Frequency_no_speed_10m":"Frequency of null wind speed at 10m",
    "Architecture_building":"Architecture building load",
    "Economics":"Economics building load",
}
og_to_udms={
    "Agriculture_building_load":" kWh",
    "Mean_air_T(°C)":"°C",
    "Precipitation_(mm)":" mm",
    "Air_pressure_(hPa)":" hPa",
    "Mean_air_humidity(%)":"%",
    "Sol_Rad_(MJ/mq)":r" MJ m$^{-2}$",
    "Sol_Irrad(W/m2)":r" W m$^{-2}$",
    "Mean_wind_speed_2m(m/s)":" m"+r" s$^{-1}$",
    "Max_wind_speed_2m(m/s)":" m"+r" s$^{-1}$",
    "Frequency_no_speed_2m(%)":"%",
    "Mean_wind_speed_10m(m/s)":" m"+r" s$^{-1}$",
    "Max_wind_speed_10m(m/s)":" m"+r" s$^{-1}$",
    "Architecture_building":" kWh",
    "Economics":" kWh",
}
og_to_udms.update({ft:"" for ft in og_labels if not ft in og_to_udms.keys()})
print("og_to_new_labels_no_udms:")
display(og_to_new_labels_no_udms)

In [ ]:
def match_data_area(fig_ref,fig_tgt):
    fig_ref.canvas.draw()
    fig_tgt.canvas.draw()
    ax_ref=fig_ref.get_axes()
    assert len(ax_ref)<=1,"The reference Figure has more than one Axes."
    ax_tgt=fig_tgt.get_axes()
    pos_ref=ax_ref[0].get_position()
    w_ref,h_ref=fig_ref.get_size_inches()
    axes_w=w_ref*(pos_ref.x1-pos_ref.x0)
    axes_h=h_ref*(pos_ref.y1-pos_ref.y0)
    if not isinstance(ax_tgt,list):
        ax_tgt=[ax_tgt]
    for ax in ax_tgt:
        pos_tgt=ax.get_position()
        new_w=axes_w/(pos_tgt.x1-pos_tgt.x0)
        new_h=axes_h/(pos_tgt.y1-pos_tgt.y0)
    fig_tgt.set_size_inches(new_w,new_h)
    for ax in ax_tgt:
        ax.set_position(pos_tgt)

## Extraction from Tensorboard logs

In [ ]:
from tensorboard.backend.event_processing import event_accumulator
def load_scalars_from_tensorboard(path:str, tag:str)->tuple[list]:
    ea=event_accumulator.EventAccumulator(path)
    ea.Reload()
    events=ea.Scalars(tag)
    steps=[e.step for e in events]
    vals=[e.value for e in events]
    return steps, vals

In [ ]:
def learning_curves_from_tensorboard(
        path:str, tags:dict[str,str],
        figsize:tuple[float]=(16,9),
        fontsize:float=12,
        xlabel:str="Number of batches seen",
        ylabel:str="Epoch loss (average across batches) (-)",
        colors=dict[str,Any],
        ylims:dict[str,Any]|None=None,
        xlims:dict[str,Any]|None=None,
        linewidth:float=1,
        nbins:int|str="auto",
        mcd_bins:list[int]|None=None,
        exclude_first_epoch:bool=False,
        title:str|None=None,
        fig_ref:Figure|None=None,
        return_fig:bool=False,
        hgl_selected:bool=True,
        dpi:int|None=None,
        USA:bool=False,
        )->None:
    with plt.rc_context({"font.size":fontsize}):
        fig=plt.figure(figsize=figsize,dpi=dpi)
        ax=fig.gca()
        ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins,integer=True,steps=mcd_bins))
        if USA:
                ax.xaxis.set_major_formatter(
                        matplotlib.ticker.StrMethodFormatter("{x:,.0f}")
                )
        min_x=np.inf
        max_x=-np.inf
        lines=[]
        for is_train,(label,tag) in enumerate(tags.items()):
                steps,vals=load_scalars_from_tensorboard(path,tag)
                if exclude_first_epoch:
                       steps=steps[1:]
                       vals=vals[1:]
                if min(steps)<min_x:
                      min_x=min(steps)
                if max(steps)>max_x:
                      max_x=max(steps)
                line,=ax.plot(steps,vals,label=label,color=colors[label],linewidth=linewidth)
                lines.append(line)
                if hgl_selected:
                       if not is_train:
                               best_step_index=vals.index(min(vals))
                       ax.plot(
                              steps[best_step_index],vals[best_step_index],marker=".",
                              markerfacecolor="lime",markeredgecolor="k",markersize=16,
                              label=""
                       )
        if ylims is not None:
                ax.set_ylim(**ylims)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        if xlims is None:
                ax.set_xlim((min_x,max_x))
        else:
                ax.set_xlim(**xlims)
        lines.reverse()
        ax.legend(loc="upper right",handles=lines)
        ax.grid(visible=True)
        if title is not None:
               ax.set_title(title)
        if fig_ref is not None:
                match_data_area(fig_ref,fig)
        else:
                fig.tight_layout()
    plt.close("all")
    display(fig)
    if return_fig:
           return fig

In [ ]:
etsf_lc_fig=learning_curves_from_tensorboard(
    path="./etsf_logs/optuna_HyperbandPruner_with_EarlyStopping/trial_384/events.out.tfevents",
    tags={
        "Validation":"Epoch (average across batches) loss: validation",
        "Training":"Epoch (average across batches) loss: training",
    },
    fontsize=15,
    xlabel="Number of batches seen [-]",
    ylabel="Epoch loss [-]",
    colors={
        "Training":"tab:blue",
        "Validation":"tab:orange"
    },
    linewidth=1.5,
    mcd_bins=[1],
    figsize=(16,4.5),
    return_fig=True,
    dpi=300,
    USA=True,
)

In [ ]:
learning_curves_from_tensorboard(
    path="./lightning_logs/optuna_HyperbandPruner/version_63/events.out.tfevents",
    tags={
        "Validation":"val_loss",
        "Training":"train_loss_epoch",
    },
    fontsize=15,
    xlabel="Number of batches seen [-]",
    ylabel="Epoch loss [-]",
    colors={
        "Training":"tab:blue",
        "Validation":"tab:orange"
    },
    ylims=dict(top=0.019),
    linewidth=1.5,
    mcd_bins=[1],
    figsize=(16,4.5),
    fig_ref=etsf_lc_fig,
    dpi=300,
    USA=True
)

# Preprocessing

In [ ]:
df = pd.read_csv(data, encoding='latin-1', low_memory=False)
print(df.shape[0])
print(df.shape[1])
print(df["Frequency_no_speed_10m"].max())
display(df.head())
display(df.dtypes)

In [ ]:
import charset_normalizer
with open(data, 'rb') as file:
    result = charset_normalizer.detect(file.read(1001))
print(result)

In [ ]:
print((df=="#NUM!").sum(axis=0).sum())
df[df=="#NUM!"]=None
print((df=="#NUM!").sum(axis=0).sum())
df.dtypes

In [ ]:
df['Mean_air_T(°C)'] = pd.to_numeric(df['Mean_air_T(°C)'], errors='raise')
df['Air_pressure_(hPa)'] = pd.to_numeric(df['Air_pressure_(hPa)'], errors='raise')
df['Sol_Rad_(MJ/mq)'] = pd.to_numeric(df['Sol_Rad_(MJ/mq)'], errors='raise')
df['Mean_wind_speed_2m(m/s)'] = pd.to_numeric(df['Mean_wind_speed_2m(m/s)'], errors='raise')
df['Max_wind_speed_2m(m/s)'] = pd.to_numeric(df['Max_wind_speed_2m(m/s)'], errors='raise')
df['Mean_wind_speed_10m(m/s)'] = pd.to_numeric(df['Mean_wind_speed_10m(m/s)'], errors='raise')
df['Max_wind_speed_10m(m/s)'] = pd.to_numeric(df['Max_wind_speed_10m(m/s)'], errors='raise')
df.dtypes

In [ ]:
missing_values = df.isnull().sum()
display(missing_values)
print(missing_values.sum())

In [ ]:
df['Agriculture_building_load'].fillna(df['Agriculture_building_load'].mean(), inplace=True)
df['Architecture_building'].fillna(df['Architecture_building'].mean(), inplace=True)
df['Economics'].fillna(df['Economics'].mean(), inplace=True)
df['Mean_air_T(°C)'].fillna(df['Mean_air_T(°C)'].mean(), inplace=True)
df['Precipitation_(mm)'].fillna(df['Precipitation_(mm)'].mean(), inplace=True)
df['Air_pressure_(hPa)'].fillna(df['Air_pressure_(hPa)'].mean(), inplace=True)
df['Mean_air_humidity(%)'].fillna(df['Mean_air_humidity(%)'].mean(), inplace=True)
df['Sol_Rad_(MJ/mq)'].fillna(df['Sol_Rad_(MJ/mq)'].mean(), inplace=True)
df['Sol_Irrad(W/m2)'] = pd.to_numeric(df['Sol_Irrad(W/m2)'], errors='coerce')
df['Sol_Irrad(W/m2)'].fillna(df['Sol_Irrad(W/m2)'].mean(), inplace=True)
df['Mean_wind_speed_2m(m/s)'].fillna(df['Mean_wind_speed_2m(m/s)'].mean(), inplace=True)
df['Max_wind_speed_2m(m/s)'].fillna(df['Max_wind_speed_2m(m/s)'].mean(), inplace=True)
df['Direction_wind_2m'].fillna(df['Direction_wind_2m'].mean(), inplace=True)
df['Frequency_no_speed_2m(%)'].fillna(df['Frequency_no_speed_2m(%)'].mean(), inplace=True)
df['Mean_wind_speed_10m(m/s)'].fillna(df['Mean_wind_speed_10m(m/s)'].mean(), inplace=True)
df['Max_wind_speed_10m(m/s)'].fillna(df['Max_wind_speed_10m(m/s)'].mean(), inplace=True)
df['Direction_wind_10m'].fillna(df['Direction_wind_10m'].mean(), inplace=True)
df['Frequency_no_speed_10m'].fillna(df['Frequency_no_speed_10m'].mean(), inplace=True)

In [ ]:
missing_values = df.isnull().sum()
missing_values

In [ ]:
df.head()

In [ ]:
df['Time'] = df['Time'].str.replace('/2019', '/19')
df['Time'] = df['Time'].str.replace('/2023', '/23')
df['Time'] = df['Time'].str.replace(':00', '.00')
df['Day_of_year'] = pd.to_datetime(df['Time'], format='%d/%m/%y %H.%M', utc=True).dt.dayofyear

In [ ]:
df.head()

In [ ]:
label_encoder = LabelEncoder()
categorical_features = ['Season']
for feature in categorical_features:
     df[feature] = label_encoder.fit_transform(df[feature])
display(df[df["Season"]==0])

In [ ]:
df.head()

In [ ]:
corr = df.drop(columns=['Time']).corr()
display(
    copy.deepcopy(corr).abs().loc[
        ["Sol_Irrad(W/m2)","Mean_air_T(°C)","Mean_wind_speed_2m(m/s)",
         "Air_pressure_(hPa)","Precipitation_(mm)","Mean_air_humidity(%)"],
        "Agriculture_building_load"
        ].sort_values(ascending=False)
    )
square_brackets=False
with plt.rc_context({"font.size":12}):
    plt.figure(
        figsize=(13, 13),
        dpi=600
        )
    sns.heatmap(
        corr[abs(corr) > 0.50],
        annot=True, 
        cmap='coolwarm', 
        cbar=False,
        linewidths=1, 
        linecolor='lightgrey',
        annot_kws={
            "fontsize":9.5,
            "fontweight":"bold"
        },
        square=True
    )
    ax=plt.gca()
    for i in range(2):
        if i==0:         
            ticks=ax.get_yticks()
            ticks_labels=ax.get_yticklabels()
        else:
            ticks=ax.get_xticks()
            ticks_labels=ax.get_xticklabels()
        ticks_labels=[label.get_text() for label in ticks_labels]
        ticks_labels=[og_to_new_labels[label] for label in ticks_labels]
        if square_brackets:
            ticks_labels=[label.replace("(","[").replace(")","]") for label in ticks_labels]
        if i==0:
            ax.set_yticks(ticks=ticks,labels=ticks_labels)
        else:
            ax.set_xticks(ticks=ticks,labels=ticks_labels)
    plt.tight_layout()
    plt.show()

In [ ]:
df = df.drop(columns=[
    'Sol_Rad_(MJ/mq)',
    "Max_wind_speed_2m(m/s)",
    "Max_wind_speed_10m(m/s)",
    "Direction_wind_10m",
    "Direction_wind_2m",
    "Frequency_no_speed_2m(%)",
    "Frequency_no_speed_10m",
    "Mean_wind_speed_10m(m/s)",
    "Day_of_month",
    "Day_of_year",
    "Hour_of_day",
    ])
df = df.drop(columns=["Season"])
df

In [ ]:
filepath_new="./Preprocessing_results/df.csv"

In [ ]:
# df.to_csv(filepath_new, index=False)

# Data preparation

In [ ]:
n_future = 24
n_past = 7*24
total_period = n_past + n_future
resolution = 60
df = pd.read_csv(filepath_new)
display(df.columns)
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%y %H.%M', utc=True)
datetime_series=copy.deepcopy(df["Time"])

In [ ]:
labels_common_plot={
    "Agri":copy.deepcopy(df.loc[:,["Time","Agriculture_building_load"]].set_index("Time")),
    "Archi":copy.deepcopy(df.loc[:,["Time","Architecture_building"]].set_index("Time")),
    "Econ":copy.deepcopy(df.loc[:,["Time","Economics"]].set_index("Time")),
}

In [ ]:
festivities=[
    dict(month=1,day=1),
    dict(month=1,day=6),
    dict(month=4,day=25),
    dict(month=5,day=1),
    dict(month=6,day=2),
    dict(month=8,day=15),
    dict(month=11,day=1),
    dict(month=12,day=8),
    dict(month=12,day=25),
    dict(month=12,day=26),
    dict(year=2020,month=4,day=12),
    dict(year=2021,month=4,day=4),
    dict(year=2022,month=4,day=17),
    dict(year=2023,month=4,day=9),
    dict(year=2020,month=4,day=13),
    dict(year=2021,month=4,day=5),
    dict(year=2022,month=4,day=18),
    dict(year=2023,month=4,day=10),
    dict(month=7,day=15),
]
def is_in_festivities(datetime:pd.Timestamp,festivities:list[dict[str,int]]=festivities):
    if datetime.day_name()=="Saturday" or datetime.day_name()=="Sunday":
        return 1
    for date_specs in festivities:
        if all([datetime.__getattribute__(spec)==date_specs[spec] for spec in date_specs]):
            return 1
    return 0
add_features=False
if add_features:
    is_festivities=datetime_series.apply(is_in_festivities)
    is_festivities.name='Holidays_indicator'
    df,id_addition=manipulate_features(df,to_be_added=is_festivities)
    print("\nNON-DEFAULT FEATURES ADDED. YOU ARE SIMULATING A NON-DEFAULT SET OF FEATURES.")
else:
    print("NO NON-DEFAULT FEATURES ADDED. YOU ARE SIMULATING A DEFAULT SET OF FEATURES.")

In [ ]:
def demand_profiles(
        search_df:pd.DataFrame,features:str|list[str],per_dow:bool=False,
        fontsize:float=14,figsize=(16,9),title:bool=True,legend:bool=True,
        interval:int=2,subplots:bool=False
        ):
    fts=[feature] if isinstance(features,str) else features
    cols=search_df.columns
    assert all(
        [ft in cols for ft in fts]
        ),"{} not in search df.".format(features)
    df=copy.deepcopy(search_df)
    df["Hour"]=df["Time"].dt.hour
    df["Dow_name"]=df["Time"].dt.day_name()
    assert "Holidays_indicator" in cols,"Holidays indicator must be in search df, run the previous cell with add_features=True"
    grouping_vars=["Holidays_indicator","Hour","Dow_name"] if per_dow else ["Holidays_indicator","Hour"]
    stats=df.groupby(grouping_vars)[fts].agg(["mean","std"])
    hour_to_fmt_hour=pd.Series(
        pd.date_range(
            start=pd.Timestamp(year=2025,month=9,day=27,hour=0,minute=0),
            freq="h",
            periods=24
        ),index=range(0,24)
    )
    def get_fig(stats,ft,hi,dow=None,ax=None):
        if ax is None:
            fig=plt.figure(figsize=figsize)
            ax=fig.gca()
        else:
            fig=None
        means=stats["mean"]
        stds=stats["std"]
        ax.plot(hour_to_fmt_hour.loc[means.index],means.values)
        ax.fill_between(
            hour_to_fmt_hour.loc[means.index],
            means.values-stds.values,
            means.values+stds.values,
            alpha=0.3,
            )
        ax.set_xlim(
            left=pd.Timestamp(year=2025,month=9,day=27,hour=0,minute=0),
            right=pd.Timestamp(year=2025,month=9,day=27,hour=23,minute=0)
        )
        ax.xaxis.set_major_locator(HourLocator(interval=interval))
        ax.xaxis.set_major_formatter(DateFormatter(fmt="%H:%M"))
        ax.set_xlabel('Time')
        if fig is not None:
            title="Holidays" if hi else "Working days"
            if dow is not None:
                title+=": "+dow
        else:
            title=dow
        if title:
            ax.set_title(title)
        if fig is not None:
            ax.set_ylabel(og_to_new_labels[ft])
        ax.grid(True)
        if fig is not None:
            fig.tight_layout()
            return fig
    figlist=[]
    with plt.rc_context({"font.size":fontsize}):
        for ft in fts:
            for hi in [0,1]:
                ft_stats=stats.xs(key=ft,axis=1,level=0)
                hour_dow_ft_stats=ft_stats.xs(key=hi,axis=0,level=0)
                if per_dow:
                    dows=hour_dow_ft_stats.index.get_level_values(level=1).unique()
                    if subplots:
                        fig,axs=plt.subplots(nrows=math.ceil(len(dows)/3),ncols=3,squeeze=True,figsize=figsize,sharey=True)
                        spt="Holidays" if hi else "Working days"
                        fig.suptitle(spt+": "+og_to_new_labels[ft])
                        axs=axs.flatten()
                    for i,dow in enumerate(dows):
                        hour_ft_stats=hour_dow_ft_stats.xs(key=dow,axis=0,level=1)
                        if not subplots:
                            fig=get_fig(hour_ft_stats,ft,hi,dow)
                        else:
                            get_fig(hour_ft_stats,ft,hi,dow,ax=axs[i])
                    if subplots:
                        if i<=len(dows):
                            for j in range(i+1,math.ceil(len(dows)/3)*3):
                                axs[j].axis("off")
                        fig.tight_layout()
                    figlist.append(fig)
                else:
                    fig=get_fig(hour_dow_ft_stats,ft,hi)
                    figlist.append(fig)
    plt.close("all")
    for fig in figlist:
        display(fig)
if add_features:
    demand_profiles(
        search_df=df,
        features=['Agriculture_building_load'],
        per_dow=True,
        subplots=True,
        interval=4
    )

In [ ]:
def weekly_profiles(
        search_df:pd.DataFrame,features:str|list[str],
        fontsize:float=14,figsize=(16,9),
        subplots:bool=False,
        plotting_time_interval:int=2,
        dpi:int|None=None,
        ylabels:list[str]|None=None,
        formatter:bool=True,
        stairplot:bool=False
        ):
    fts=[features] if isinstance(features,str) else features
    cols=search_df.columns
    assert all(
        [ft in cols for ft in fts]
        ),"{} not in search df.".format(features)
    df=copy.deepcopy(search_df)
    df["Hour"]=df["Time"].dt.hour
    df["Dow_name"]=df["Time"].dt.day_name()
    grouping_vars=["Hour","Dow_name"]
    stats=df.groupby(grouping_vars)[fts].agg(["mean","std"])
    if not subplots:
        figlist=[]
    else:
        fig=plt.figure(figsize=figsize,dpi=dpi)
        nrows=len(features)
    with plt.rc_context({"font.size":fontsize}):
        for ift,ft in enumerate(fts):
            means={}
            stds={}
            weekly_mean=np.array([])
            weekly_index=np.array([])
            weekly_std=np.array([])
            ft_stats=stats.xs(key=ft,axis=1,level=0)
            for dow in ft_stats.index.get_level_values(level=1).unique():
                hour_ft_stats=ft_stats.xs(key=dow,axis=0,level=1)
                means[dow]=hour_ft_stats["mean"]
                stds[dow]=hour_ft_stats["std"]
            for dow in ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]:
                weekly_mean=np.concatenate((weekly_mean,means[dow].values),axis=0)
                weekly_index=np.concatenate((weekly_index,means[dow].index),axis=0)
                weekly_std=np.concatenate((weekly_std,stds[dow].values),axis=0)
            if not subplots:
                fig=plt.figure(figsize=figsize,dpi=dpi)
                ax=fig.gca()
            else:
                if not ift:
                    ax=fig.add_subplot(nrows,1,ift+1)
                else:
                    ax=fig.add_subplot(nrows,1,ift+1,sharex=ax)
            if not stairplot:
                ax.plot(list(range(len(weekly_mean))),weekly_mean)
            else:
                ax.step(list(range(len(weekly_mean))),weekly_mean,where="post")
            ax.fill_between(
                list(range(len(weekly_mean))),
                weekly_mean-weekly_std,
                weekly_mean+weekly_std,
                alpha=0.3,
                step=None if not stairplot else "post"
            )
            ax.set_xlim(
                left=0,
                right=len(weekly_mean),
            )
            ax.set_xlabel('Time')
            if ylabels is None:
                if not subplots:
                    ax.set_ylabel("{}".format(og_to_new_labels_no_udms[ft]))
                else:
                    ax.set_title("{}".format(og_to_new_labels_no_udms[ft]))
            else:
                ax.set_ylabel(ylabels[ift])
            ax.grid(True)
            if formatter:
                ax.yaxis.set_major_formatter("{x}"+og_to_udms[ft])
            if not subplots:
                fig.tight_layout()
                figlist.append(fig)
            ticks=np.arange(
                0,
                len(weekly_mean),
                plotting_time_interval
            )
            dummy_week=pd.Series(
                pd.date_range(
                    start=pd.Timestamp(year=2025,month=10,day=6,hour=0,minute=0),
                    end=pd.Timestamp(year=2025,month=10,day=12,hour=23,minute=0),
                    freq="h",
                )
            )
            if plotting_time_interval<24 and 24%plotting_time_interval==0:
                labels=[]
                for date_idx,date in enumerate(dummy_week.values):
                    date=pd.to_datetime(date)
                    if date_idx%plotting_time_interval==0:
                        if date_idx%24==0:
                            labels.append(date.strftime("%a %H:%M"))
                        else:
                            labels.append(date.strftime("%H:%M"))
            else:
                labels=[
                    pd.to_datetime(date).strftime("%a %H:%M") for date in dummy_week.values[slice(0,None,plotting_time_interval)]
                ]
            ax.set_xticks(ticks=ticks,labels=labels,rotation=90)
    plt.close("all")
    if not subplots:
        for fig in figlist:
            display(fig)
    else:
        fig.tight_layout()
        display(fig)
weekly_profiles(
    search_df=df,
    features=[
        'Agriculture_building_load',
        'Architecture_building', 
        'Economics',
    ], 
    figsize=(16,3),
    subplots=False,
    plotting_time_interval=6,
    dpi=300,
    ylabels=[
        "Electricity\nconsumption [kWh]",
        "Electricity\nconsumption [kWh]",
        "Electricity\nconsumption [kWh]"
    ],
    formatter=False,
    stairplot=False,
    fontsize=16,
    )

In [ ]:
def month_by_month_weekly_profiles(
        search_df:pd.DataFrame,features:str|list[str],
        fontsize:float=14,figsize=(16,9),
        plotting_time_interval:int=2,
        dpi:int|None=None,
        stairplot:bool=False
        ):
    fts=[features] if isinstance(features,str) else features
    cols=search_df.columns
    assert all(
        [ft in cols for ft in fts]
        ),"{} not in search df.".format(features)
    df=copy.deepcopy(search_df)
    df["Hour"]=df["Time"].dt.hour
    df["Dow_name"]=df["Time"].dt.day_name()
    df["Month_name"]=df["Time"].dt.month_name()
    grouping_vars=["Hour","Dow_name","Month_name"]
    stats=df.groupby(grouping_vars)[fts].agg(["mean","std"])
    nrows=len(df["Month_name"].unique())
    with plt.rc_context({"font.size":fontsize}):
        for ft in fts:
            fig=plt.figure(figsize=figsize,dpi=dpi)
            for imt,mt in enumerate(
                [
                    "January","February","March",
                    "April","May","June",
                    "July","August","September",
                    "October","November","December"
                ]):
                means={}
                stds={}
                weekly_mean=np.array([])
                weekly_index=np.array([])
                weekly_std=np.array([])
                ft_stats=stats.xs(key=ft,axis=1,level=0)
                ft_stats_in_month=ft_stats.xs(key=mt,axis=0,level=2)
                for dow in ft_stats_in_month.index.get_level_values(level=1).unique():
                    hour_ft_stats=ft_stats_in_month.xs(key=dow,axis=0,level=1)
                    means[dow]=hour_ft_stats["mean"]
                    stds[dow]=hour_ft_stats["std"]
                for dow in ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]:
                    weekly_mean=np.concatenate((weekly_mean,means[dow].values),axis=0)
                    weekly_index=np.concatenate((weekly_index,means[dow].index),axis=0)
                    weekly_std=np.concatenate((weekly_std,stds[dow].values),axis=0)
                if not imt:
                    ax=fig.add_subplot(nrows,1,imt+1)
                else:
                    ax=fig.add_subplot(nrows,1,imt+1,sharex=ax,sharey=ax)
                if not stairplot:
                    ax.plot(list(range(len(weekly_mean))),weekly_mean)
                else:
                    ax.step(list(range(len(weekly_mean))),weekly_mean,where="post")
                ax.fill_between(
                    list(range(len(weekly_mean))),
                    weekly_mean-weekly_std,
                    weekly_mean+weekly_std,
                    alpha=0.3,
                    step=None if not stairplot else "post"
                )
                ax.set_xlim(
                    left=0,
                    right=len(weekly_mean),
                )
                ax.set_xlabel('Time')
                ax.set_title(mt)
                ax.grid(True)
                ax.yaxis.set_major_formatter("{x}"+og_to_udms[ft])
                ticks=np.arange(
                    0,
                    len(weekly_mean),
                    plotting_time_interval
                )
                dummy_week=pd.Series(
                    pd.date_range(
                        start=pd.Timestamp(year=2025,month=10,day=6,hour=0,minute=0),
                        end=pd.Timestamp(year=2025,month=10,day=12,hour=23,minute=0),
                        freq="h",
                    )
                )
                if plotting_time_interval<24 and 24%plotting_time_interval==0:
                    labels=[]
                    for date_idx,date in enumerate(dummy_week.values):
                        date=pd.to_datetime(date)
                        if date_idx%plotting_time_interval==0:
                            if date_idx%24==0:
                                labels.append(date.strftime("%a %H:%M"))
                            else:
                                labels.append(date.strftime("%H:%M"))
                else:
                    labels=[
                        pd.to_datetime(date).strftime("%a %H:%M") for date in dummy_week.values[slice(0,None,plotting_time_interval)]
                    ]
                ax.set_xticks(ticks=ticks,labels=labels,rotation=90)
            fig.tight_layout()
            plt.close("all")
            display(fig)
month_by_month_weekly_profiles(
    search_df=df,
    features=[
        'Agriculture_building_load',
    ],
    figsize=(16,48),
    plotting_time_interval=6,
    dpi=600,
    stairplot=True
    )

In [ ]:
def investigate_outliers(
        df,cols:list[str]=[
            "Agriculture_building_load",
            "Architecture_building",
            "Economics",
            "Mean_air_T(°C)",
            "Precipitation_(mm)",
            "Air_pressure_(hPa)",
            "Mean_air_humidity(%)",
            "Sol_Irrad(W/m2)",
            "Mean_wind_speed_2m(m/s)",
            ],
        nrows:int=3,figsize:tuple[float]=(16,9),fontsize:float=14,
        kind:Literal["box","violin","boxen"]="boxen",
        kwargs:dict[Any,Any]={},
        square_brackets_units:bool=False,
        dpi:int|None=None
        ):
    assert all([col in df.columns for col in cols]),"Some of cols is not in df"
    bxn_cols=cols
    n_bxn=len(bxn_cols)
    with plt.rc_context({"font.size":fontsize}):
        bxn_fig,bxn_axs=plt.subplots(nrows=nrows,ncols=math.ceil(n_bxn/nrows),figsize=figsize,dpi=dpi)
        bxn_axs=bxn_axs.reshape(-1)
        for idx,ax in enumerate(bxn_axs):
            if idx>=n_bxn:
                ax.axis("off")
            else:
                if kind=="box":
                    sns.boxplot(data=df,y=bxn_cols[idx],ax=ax,**kwargs)
                elif kind=="violin":
                    sns.violinplot(data=df,y=bxn_cols[idx],ax=ax,**kwargs)
                else:
                    sns.boxenplot(data=df,y=bxn_cols[idx],ax=ax,**kwargs)
                ax.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=6))
                ylabel=ax.get_ylabel()
                ax.set_ylabel("")
                ylabel=og_to_new_labels[ylabel]
                if square_brackets_units:
                    ylabel=ylabel.replace("(","[").replace(")","]")
                ax.set_xlabel(ylabel)
        bxn_fig.tight_layout()
        plt.close("all")
        display(bxn_fig)
investigate_outliers(
    df,
    kind="boxen",
    kwargs={"line_kws":dict(linewidth=1.5, color="tab:orange")},
    fontsize=16,
    dpi=600,
)

In [ ]:
datetime_list = copy.deepcopy(df['Time'].to_numpy().reshape(-1,1))
df_Architecture_building = df.drop(columns=['Time', "Agriculture_building_load", "Economics"])
display(df_Architecture_building)
df_Economics_building = df.drop(columns=['Time', "Architecture_building", "Agriculture_building_load"])
display(df_Economics_building)
df = df.drop(columns=['Time', "Architecture_building", "Economics"])
display(df)


In [ ]:
test_size = math.floor(len(df) / 4)
timestamp = pd.Timestamp('2022-07-01 00:00:00+0000')
index = next((i for i, t in enumerate(datetime_list) if any(t == timestamp)), None)
if index is None:
    raise Exception("Timestamp not found in datetime_list")
train_index = [i for i in range(index)]
test_index = [i for i in range(index, len(df))]
df_train, df_test = df.iloc[train_index, :], df.iloc[test_index,:]
prova_train, prova_test=train_test_split(df,test_size=(len(df)-index),shuffle=False)
scaler = MinMaxScaler()
scaler_y = MinMaxScaler()
display(df_train.shape[0]+df_test.shape[0])
df_train_x = pd.DataFrame(scaler.fit_transform(df_train), columns=df_train.columns)
arr_train_y = scaler_y.fit_transform(np.asarray(df_train["Agriculture_building_load"]).reshape(-1, 1))
df_test_x = pd.DataFrame(scaler.transform(df_test), columns=df_test.columns)
arr_test_y = scaler_y.transform(np.asarray(df_test["Agriculture_building_load"]).reshape(-1, 1))

In [ ]:
arr_Architecture_building_test_y = scaler_y.transform(np.asarray(df_Architecture_building["Architecture_building"]).reshape(-1, 1))
df_Architecture_building = df_Architecture_building.rename(columns={"Architecture_building": "Agriculture_building_load"})
df_Architecture_building = df_Architecture_building[df_train.columns]
df_Architecture_building_test_x = pd.DataFrame(scaler.transform(df_Architecture_building), columns=df_Architecture_building.columns)
arr_Economics_building_test_y = scaler_y.transform(np.asarray(df_Economics_building["Economics"]).reshape(-1, 1))
df_Economics_building = df_Economics_building.rename(columns={"Economics": "Agriculture_building_load"}) 
df_Economics_building = df_Economics_building[df_train.columns]
df_Economics_building_test_x = pd.DataFrame(scaler.transform(df_Economics_building), columns=df_Economics_building.columns)
display(df_train.shape)
display(df_test.shape)
display(df_Architecture_building_test_x.shape)
display(df_Economics_building_test_x.shape)

In [ ]:
def partition_dataset_into_seqs(df_x, arr_y, overlapped_allowed:bool=False):
    idx_end = len(df_x)
    idx_start = 0
    X_new = []
    y_new = []
    while idx_start+total_period-1<idx_end:
        x_line=df_x.iloc[idx_start:idx_start+n_past]
        y_line=arr_y[idx_start+n_past:idx_start+total_period]
        X_new.append(x_line)
        y_new.append(y_line)
        if not overlapped_allowed:
            idx_start = idx_start + n_future
        else:
            idx_start = idx_start + 1
    print(math.floor((df_x.shape[0]-1-total_period)/n_future)+1)
    X_train = np.asarray(X_new)
    y_train = np.asarray(y_new).squeeze()
    return X_train, y_train

X_train, y_train = partition_dataset_into_seqs(df_train_x, arr_train_y)
X_train_all_seqs, y_train_all_seqs = partition_dataset_into_seqs(df_train_x, arr_train_y, overlapped_allowed=True)
n_features = X_train.shape[2]
X_test, y_test = partition_dataset_into_seqs(df_test_x, arr_test_y)
X_Agri_extended,y_Agri_extended=partition_dataset_into_seqs(pd.concat([df_train_x,df_test_x]),np.concat([arr_train_y,arr_test_y]))
X_Architecture_building_test, y_Architecture_building_test = partition_dataset_into_seqs(df_Architecture_building_test_x, arr_Architecture_building_test_y)
X_Economics_building_test, y_Economics = partition_dataset_into_seqs(df_Economics_building_test_x, arr_Economics_building_test_y)

In [ ]:
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_train_all_seqs shape: {X_train_all_seqs.shape}")
print(f"y_train_all_seqs shape: {y_train_all_seqs.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"X_Architecture_building_test shape: {X_Architecture_building_test.shape}")
print(f"y_Architecture_building_test shape: {y_Architecture_building_test.shape}")
print(f"X_Economics_building_test shape: {X_Economics_building_test.shape}")
print(f"y_Economics shape: {y_Economics.shape}")

In [ ]:
from sklearn.metrics import root_mean_squared_error
def mape(y_test, preds):
    y_test = np.asarray(y_test)
    preds = np.asarray(preds)
    mask = y_test != 0
    return np.mean(np.abs(y_test[mask] - preds[mask]) / y_test[mask])
def rmse(y_test, preds):
    return np.sqrt(((preds - y_test) ** 2).mean())
def evaluate_forecast(y_test, preds, datetime_list=None, to_string=False):
    scores = {
        'mae': mean_absolute_error(y_test, preds),
        'mse': mean_squared_error(y_test, preds),
        'rmse': root_mean_squared_error(y_test, preds),
        'mape': mape(y_test, preds) * 100,
        'r2': r2_score(y_test, preds),
    }
    if to_string:
        return f"{'Daytime scores: ' if datetime_list else 'Scores: '}MAE={'%.2f' % scores['mae']}, RMSE={'%.2f' % scores['mse']}, MAPE={'%.2f' % scores['mape']}%"
    return scores
def print_scores(results, unit, filepath=None, title=""):
    str = ''
    str += f"{title}\n"
    for k, v in results.items():
        if k == 'mse':
            unit_str = f"{unit}^2"
        elif k == 'mae' or k == 'rmse':
            unit_str = f"{unit}"
        elif k == 'mape':
            unit_str = '%'
        else:
            unit_str = ''
        str += f"{k} = {'%.2f' % results[k]} {unit_str}\n"
    str += "\n"
    print(str)

# LSTM Keras (evaluation, CPU)

## Functions

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
import torch
def lstm_above_and_below(
        output_preds:np.ndarray,
        input_target:np.ndarray,
        input_starting_date:pd.Timestamp,
        sklearn_metric_name:Literal["MAE","MSE","RMSE","MAPE","R2"],
        upper_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.95),
        lower_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.05),
        show_rescaled:bool=True,
        scaler_y=scaler_y,
        figsize:tuple[float,float]=(16,9),
        plot_kwargs:dict[str,]={"linewidth":1},
        return_above_and_below_dfs:bool=False,
        test_period:pd.DatetimeIndex|None=None,
        thresholds_wrt_non_overlapping_in_test:bool=True,
        return_scores:bool=False,
        building_name:str="Agriculture",
        fontsize:float=14,
        xlabel:str|None="Test set dates.",
        legend_only:bool=False,
        return_only_figure:bool=False,
        stairplot:bool=False,
        dpi:int|None=None,
        target_name:str="Electricity consumption (kWh)"
        )->None|list[pd.DataFrame,pd.DataFrame]|dict[str,float]|tuple[list[pd.DataFrame,pd.DataFrame],dict[str,float]]:
    def compute_metrics(target,preds):
        metrics={
            "MAE":mean_absolute_error(target,preds),
            "MSE":mean_squared_error(target,preds),
            "RMSE":root_mean_squared_error(target,preds),
            "MAPE":torch.mean(
                torch.abs(
                    target[target!=0] - preds[target!=0]
                ) / target[target!=0]
            ).item()*100,             
            "R2":r2_score(target,preds),
        }
        return metrics
    predictions=torch.tensor(copy.deepcopy(output_preds))
    y=torch.tensor(copy.deepcopy(input_target))
    for seq_id in range(y.shape[0]):
        pred_time_idxs=np.arange(n_past+n_future*seq_id,n_past+n_future*(seq_id+1)).reshape(1,-1)
        if seq_id==0:
            predictions_time_dummy=pred_time_idxs.reshape(1,-1)
        else:
            predictions_time_dummy=np.concat(
                (predictions_time_dummy,pred_time_idxs),
                axis=0
            )
    time_dummy_to_date=pd.Series(
        pd.date_range(
            input_starting_date,freq="1h",
            periods=predictions_time_dummy.max()+1
        ),
        index=np.arange(0,predictions_time_dummy.max()+1)
    )
    if test_period is None:
        test_period=copy.deepcopy(time_dummy_to_date)
    else:
        assert test_period.min()>=time_dummy_to_date.min(),"test_period first date is before the input_starting_date"
        assert test_period.max()<=time_dummy_to_date.max(),"test_period last date is after the last date predicted"
        test_period=pd.Series(
            test_period,
            index=time_dummy_to_date.loc[time_dummy_to_date.isin(test_period)].index
        )
    all_seqs_metrics={"MAE":[],"MSE":[],"RMSE":[],"MAPE":[],"R2":[]}
    for seq_id in range(y.shape[0]):
        seq_metrics=compute_metrics(y[seq_id,:],predictions[seq_id,:])
        for mtc in all_seqs_metrics.keys():
            all_seqs_metrics[mtc].append(seq_metrics[mtc])
    all_seqs_metrics=pd.DataFrame(all_seqs_metrics,index=range(0,y.shape[0]))
    predictions_time_dummy=torch.tensor(predictions_time_dummy)
    if thresholds_wrt_non_overlapping_in_test:
        non_overlapping_seqs_in_test=[]
        already_seen_decoder_time_idxs=torch.tensor([])
        for seq_id in range(len(predictions_time_dummy)):
            decoder_time_idxs=predictions_time_dummy[seq_id,:]
            if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                if all(pd.Series(decoder_time_idxs).isin(test_period.index)):
                    already_seen_decoder_time_idxs=torch.cat(
                        (already_seen_decoder_time_idxs,decoder_time_idxs)
                    )
                    non_overlapping_seqs_in_test.append(seq_id)
        not_overlapped_metrics=copy.deepcopy(
            all_seqs_metrics.loc[non_overlapping_seqs_in_test,sklearn_metric_name]
        )
    metrics=np.array(all_seqs_metrics[sklearn_metric_name])
    if isinstance(upper_threshold,tuple):
        upper_label=" (P{:.1f})".format(upper_threshold[1]*100)
        upper_threshold=np.quantile(
            not_overlapped_metrics if thresholds_wrt_non_overlapping_in_test else metrics,
            upper_threshold[1]
        )
    else:
        upper_label=""
    if isinstance(lower_threshold,tuple):
        lower_label=" (P{:.1f})".format(lower_threshold[1]*100)
        lower_threshold=np.quantile(
            not_overlapped_metrics if thresholds_wrt_non_overlapping_in_test else metrics,
            lower_threshold[1]
        )
    else:
        lower_label=""
    start=True
    already_seen_decoder_time_idxs=torch.tensor([])
    seqs_idxs_significant={"above":[],"below":[]}
    metrics_significant={"above":[],"below":[]}
    decoder_time_idxs_significant={"above":[],"below":[]}
    for seq_id in range(y.shape[0]):       
        decoder_time_idxs=predictions_time_dummy[seq_id,:]
        if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
            if all(pd.Series(decoder_time_idxs).isin(test_period.index)):
                if metrics[seq_id]>upper_threshold:  
                    decoder_time_idxs_significant["above"].append(decoder_time_idxs)
                    seqs_idxs_significant["above"].append(seq_id)
                    metrics_significant["above"].append(metrics[seq_id])
                if metrics[seq_id]<lower_threshold:  
                    decoder_time_idxs_significant["below"].append(decoder_time_idxs)
                    seqs_idxs_significant["below"].append(seq_id)
                    metrics_significant["below"].append(metrics[seq_id])
                already_seen_decoder_time_idxs=torch.cat(
                    (already_seen_decoder_time_idxs,decoder_time_idxs)
                )
                predictions_sequence=predictions[seq_id,:]
                if start:
                    preds=predictions_sequence
                    target=y[seq_id,:]
                    start=False
                else:    
                    target=torch.cat((target,y[seq_id,:]))
                    preds=torch.cat((preds,predictions_sequence))               
    tau_max=n_future
    last_encoder_time_dummies=(already_seen_decoder_time_idxs-1).int().tolist()[:-(tau_max-1)]
    decoder_dates=time_dummy_to_date.loc[already_seen_decoder_time_idxs]

    if show_rescaled:
        target=torch.tensor(
            scaler_y.inverse_transform(target.cpu().reshape((-1,1))).reshape(-1)
        )
        preds=torch.tensor(
            scaler_y.inverse_transform(preds.cpu().reshape((-1,1))).reshape(-1)
        )
    else:
        target=torch.tensor(target).cpu()
        preds=torch.tensor(preds).cpu()
    overall_metrics={
        "MAE":mean_absolute_error(target,preds),
        "MSE":mean_squared_error(target,preds),
        "RMSE":root_mean_squared_error(target,preds),
        "MAPE":torch.mean(
            torch.abs(
                target[target!=0] - preds[target!=0]
            ) / target[target!=0]
        ).item()*100,             
        "R2":r2_score(target,preds),
    }
    names_to_udm={
        "MAE":" (-)" if not show_rescaled else " kWh",
        "MSE":r" (-)$^2$" if not show_rescaled else r" kWh$^2$",
        "RMSE":" (-)" if not show_rescaled else " kWh",
        "MAPE":"%",
        "R2":"",
    }  

    absolute_min=min([target.min(),preds.min()])
    absolute_max=max([target.max(),preds.max()])

    fig_preds=plt.figure(figsize=figsize,dpi=dpi)    
    target_color="r"
    preds_color="b"
    bad_color="#FFD700" 
    good_color = "g"
    with plt.rc_context({"font.size":fontsize}):
        ax_target=fig_preds.gca()
        if not stairplot:
            ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
        else:
            ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
        ax_target.set_ylim(
            bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
            top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
        )
        ax_target.set_ylabel(
            "Observed {}".format(target_name),
            color=target_color
        )
        ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
        if xlabel is not None:
            ax_target.set_xlabel(xlabel)
        better_if_lower=["MAE","MSE","RMSE","MAPE"]
        ymin,ymax=ax_target.get_ylim()
        for list_idx,above_indices in enumerate(decoder_time_idxs_significant["above"]):
            ax_target.fill_between(
                x=time_dummy_to_date.loc[above_indices],
                y1=ymin,
                y2=ymax,
                fc=bad_color if sklearn_metric_name in better_if_lower else good_color,
                alpha=0.35,
                label="{}>{:.4f}".format(sklearn_metric_name,upper_threshold)+names_to_udm[sklearn_metric_name]+upper_label if list_idx==0 else ""
            )

        for list_idx,below_indices in enumerate(decoder_time_idxs_significant["below"]):
            ax_target.fill_between(
                x=time_dummy_to_date.loc[below_indices],
                y1=ymin,
                y2=ymax,
                fc=good_color if sklearn_metric_name in better_if_lower else bad_color,
                alpha=0.35,
                label="{}<{:.4f}".format(sklearn_metric_name,lower_threshold)+names_to_udm[sklearn_metric_name]+lower_label if list_idx==0 else ""
            )

        ax_target.legend(loc="lower center",bbox_to_anchor=(0.5, 1.0),ncols=2)
        ax_target.set_xlim(
            (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
            max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
        )
        ax_preds=ax_target.twinx()
        if not stairplot:
            ax_preds.plot(decoder_dates,preds,color=preds_color)
        else:
            ax_preds.step(decoder_dates,preds,color=preds_color,where="post")
        ax_preds.set_ylim(
            bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
            top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
        )
        ax_preds.set_ylabel("Predicted {}".format(target_name),color=preds_color)
        ax_preds.tick_params(axis="y",color=preds_color,labelcolor=preds_color)
        if not legend_only:
            suptitle_strs=["{} building".format(building_name),"\nMetrics:"]
            for idx,(name,value) in enumerate(overall_metrics.items()):
                if idx!=(len(overall_metrics.items())-1):
                    suptitle_strs.append("{}={:.2f}".format(name,value)+names_to_udm[name]+",")
                else:
                    suptitle_strs.append("{}={:.3f}".format(name,value)+names_to_udm[name])
            fig_preds.suptitle(" ".join(suptitle_strs))
        fig_preds.tight_layout()
        plt.close("all")
        display(fig_preds)
    if return_only_figure:
        return fig_preds
    else:
        if return_above_and_below_dfs:
            above_df=pd.DataFrame(
                {"seq_idx":seqs_idxs_significant["above"],
                sklearn_metric_name:metrics_significant["above"]}
                ).sort_values(by=sklearn_metric_name,ascending=False)
            above_df.set_index(
                np.arange(1,len(seqs_idxs_significant["above"])+1,dtype=int),inplace=True
            )
            below_df=pd.DataFrame(
                {"seq_idx":seqs_idxs_significant["below"],
                sklearn_metric_name:metrics_significant["below"]}
                ).sort_values(by=sklearn_metric_name,ascending=True)
            below_df.set_index(
                np.arange(1,len(seqs_idxs_significant["below"])+1,dtype=int),inplace=True
            )
            if not return_scores:
                return [above_df,below_df]
            else:
                return [above_df,below_df],overall_metrics
        else:
            if return_scores:
                return overall_metrics
def lstm_visualize_seq(
        output_preds:np.ndarray,
        input_target:np.ndarray,
        input_features:np.ndarray,
        input_starting_date:pd.Timestamp,
        seq_idx:int,
        figsize:tuple[float,float]=(16,8),
        linewidth:float=1.5,
        show_rescaled:bool=True,
        scaler_y=scaler_y,
        custom_ticks:bool=True,
        plotting_time_interval:int=24,
        add_loss_to_title:bool=True,
        show_future_observed:bool=True,
        use_dates:bool=False,
        )->None:
    assert seq_idx<len(output_preds),"seq_idx provided falls out of its limits ([0,...,{}]).".format(len(output_preds)-1)
    for seq_id in range(output_preds.shape[0]):
        pred_time_idxs=np.arange(n_past+n_future*seq_id,n_past+n_future*(seq_id+1)).reshape(1,-1)
        if seq_id==0:
            predictions_time_dummy=pred_time_idxs.reshape(1,-1)
        else:
            predictions_time_dummy=np.concat(
                (predictions_time_dummy,pred_time_idxs),
                axis=0
            )
    for seq_id in range(input_features.shape[0]):
        features_time_idxs=np.arange(n_future*seq_id,n_future*seq_id+n_past).reshape(1,-1)
        if seq_id==0:
            features_time_dummy=features_time_idxs.reshape(1,-1)
        else:
            features_time_dummy=np.concat(
                (features_time_dummy,features_time_idxs),
                axis=0
            )
    time_dummy_to_date=pd.Series(
        pd.date_range(
            input_starting_date,freq="1h",
            periods=predictions_time_dummy.max()+1
        ),
        index=np.arange(features_time_dummy.min(),predictions_time_dummy.max()+1)
    )
    predictions=copy.deepcopy(output_preds)

    predictions_sequence=torch.tensor(predictions[seq_idx,:])
    encoder_target=torch.tensor(input_features[seq_idx,:,2])
    decoder_target=torch.tensor(input_target[seq_idx,:])
    if show_rescaled:
        predictions_sequence=torch.tensor(
            scaler_y.inverse_transform(predictions_sequence.reshape((-1,1))).reshape(-1)
        )
        encoder_target=torch.tensor(
            scaler_y.inverse_transform(encoder_target.reshape((-1,1))).reshape(-1)
        )
        decoder_target=torch.tensor(
            scaler_y.inverse_transform(decoder_target.reshape((-1,1))).reshape(-1)
        )
        
    fig=plt.figure(figsize=figsize)
    ax_predictions=fig.gca()
    encoder_positions=np.arange(
        start=-(n_past-1),
        stop=1
    )
    decoder_positions=np.arange(
        start=1,
        stop=n_future+1
    )
    prop_cycle = iter(plt.rcParams["axes.prop_cycle"])
    target_color = next(prop_cycle)["color"]
    predictions_color = next(prop_cycle)["color"]
    ax_predictions.plot(encoder_positions,encoder_target,color=target_color,linewidth=linewidth,label="Observed")
    if show_future_observed:
        ax_predictions.plot(decoder_positions,decoder_target,color=target_color,linewidth=linewidth)
    ax_predictions.plot(
        decoder_positions,
        predictions_sequence.cpu(),
        color=predictions_color,linewidth=linewidth,label="Predicted")
    from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score,mean_squared_error
    metrics_dict={
        "MAE":dict(
            value=mean_absolute_error(decoder_target,predictions_sequence.cpu()),
            udm=" (-)" if not show_rescaled else " kWh"
        ),
        "MSE":dict(
            value=mean_squared_error(decoder_target,predictions_sequence.cpu()),
            udm=r" (-)$^2$" if not show_rescaled else r" kWh$^2$"
        ),
        "RMSE":dict(
            value=root_mean_squared_error(decoder_target,predictions_sequence.cpu()),
            udm=" (-)" if not show_rescaled else " kWh"
        ),
        "MAPE":dict(
            value=torch.mean(
                torch.abs(
                    decoder_target[decoder_target!=0] - predictions_sequence.cpu()[decoder_target!=0]
                ) / decoder_target[decoder_target!=0]
            ).item()*100,
            udm="%"
        ),                
        "R2":dict(
            value=r2_score(decoder_target,predictions_sequence.cpu()),
            udm=""
        )
    }
    title_for_loss=" Loss computed on predictions: "
    metrics_str=", ".join(["{}={:.2f}{}".format(
        mtc_name,
        metrics_dict[mtc_name]["value"],
        metrics_dict[mtc_name]["udm"]
    ) for mtc_name in list(metrics_dict.keys())])
    title_for_loss+=metrics_str+"."
    ax_predictions.set_ylabel("Target (kWh)" if show_rescaled else "Target rescaled (-)")
    ax_predictions.set_xlabel("Position index (n)" if not use_dates else "") 
    title="Predictions for sequence index={}.".format(
        seq_idx
    )
    title=title+title_for_loss if add_loss_to_title else title
    ax_predictions.legend(
        loc="lower center",
        ncols=2,
        bbox_to_anchor=(0.5,1)
    )
    if custom_ticks:
        ticks=np.arange(
            encoder_positions.min(),
            decoder_positions.max(),
            plotting_time_interval
        )
        if not use_dates:
            labels=[str(tick) for tick in ticks]
        else:
            dates=time_dummy_to_date.loc[
                torch.cat(
                    (
                        torch.tensor(features_time_dummy[seq_idx,:]),
                        torch.tensor(predictions_time_dummy[seq_idx,:])
                    )
                )
            ]
            if plotting_time_interval<24 and 24%plotting_time_interval==0:
                labels=[]
                for date_idx,date in enumerate(dates):
                    if date_idx%plotting_time_interval==0:
                        if date_idx%24==0:
                            labels.append(date.strftime("%a %Y/%m/%d %H:%M"))
                        else:
                            labels.append(date.strftime("%H:%M"))
            else:
                labels=[
                    date.strftime("%a %Y/%m/%d %H:%M") for date in dates[slice(0,None,plotting_time_interval)]
                ] 
        ax_predictions.set_xticks(
            ticks=ticks,
            labels=labels,
            rotation=0 if not use_dates else 90
        )
        ax_predictions.set_xlim((encoder_positions.min(),decoder_positions.max()))
    fig.suptitle(title)
    fig.tight_layout()
    plt.close("all")
    display(fig)

In [ ]:
import optuna
import tensorflow as tf
class ScheduleLogger(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self._step=0
        self._LR_log = {}

    def on_batch_end(self, batch, logs=None):
        lr_tensor=self.model.optimizer.learning_rate
        lr_value = keras.backend.get_value(lr_tensor)
        self._LR_log[self._step]=lr_value
        self._step+=1

class lstm_LRLogger(keras.callbacks.Callback): 
    def __init__(self): 
        super().__init__()
        self._LR_log={} 
    def on_epoch_end(self,epoch,logs=None): 
        self._LR_log[epoch]=float(self.model.optimizer.learning_rate.numpy())
class lstm_OptunaPruningCallback(keras.callbacks.Callback):
    def __init__(self,trial:optuna.trial.Trial):
        super().__init__()
        self._trial=trial
    def on_epoch_end(self,epoch,logs=None):
        epoch_val_loss=logs["val_loss"]
        self._trial.report(epoch_val_loss,epoch)
        if np.isnan(epoch_val_loss) or np.isinf(epoch_val_loss):
            self._trial.set_user_attr("check_finite_triggered",True)
            print("Trial {} is going to be pruned at epoch {} end because of check_finite triggering...".format(self._trial.number,epoch),flush=True)
            raise optuna.exceptions.TrialPruned
        else:
            if self._trial.should_prune():
                print("Trial {} is going to be pruned at epoch {} end...".format(self._trial.number,epoch),flush=True)
                raise optuna.exceptions.TrialPruned
import gc
class lstm_ClearCacheOnEpochEnd(keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs=None):
        torch.cuda.empty_cache()
        gc.collect()
def tune_lstm(
        study_name:str,
        pruner:optuna.pruners.BasePruner,
        n_trials:int,
        X_train:np.ndarray,y_train:np.ndarray,
        max_epochs:int=200,
        X_val:np.ndarray|None=None,y_val:np.ndarray|None=None,
        validation_split:float|None=0.2,
        path_to_lstm_results:Path=Path("./Results/LSTM"),
        units_distribution:dict[Literal["low","high","step","log"],int|bool]|list[int]=[
            32,64,128
        ],
        batch_size_distribution:dict[Literal["low","high","step","log"],int|bool]|list[int]=[
            16,32,64,128
        ],
        dropout_distribution:dict[Literal["low","high","step","log"],float|bool]|list[float]={
            "low":0.1,"high":0.3,"step":None,"log":False
        },
        gradient_clipping_distribution:dict[Literal["low","high","step","log"],float|bool]|list[float]|None={
            "low":1.0,"high":1.0,"step":None,"log":False
        },
        LR_distribution:dict[Literal["low","high","step","log"],float|bool]|list[float]={
            "low":1e-4,"high":1e-1,"step":None,"log":True
        },
        LR_schedule:keras.optimizers.schedules.LearningRateSchedule|None=None,
        reduce_LR_on_plateau:keras.callbacks.ReduceLROnPlateau|None=None,
        loss:keras.losses.Loss|str='mean_squared_error',
        metrics:list[keras.metrics.Metric|str]=[
            'mean_absolute_error','mean_squared_error','mean_absolute_percentage_error',
            'root_mean_squared_error','r2_score'
        ],
        timeout:int|float=8*3600,
        verbose:int|None=1,
        direction:optuna.study.StudyDirection=optuna.study.StudyDirection.MINIMIZE,
        early_stopping:bool=False,
        clear_cache_on_epoch_end:bool=True,
        n_past:int=n_past,
        n_features:int=n_features
):
    if not all(
        [reduce_LR_on_plateau is None,LR_schedule is None]
    ):
        if reduce_LR_on_plateau is not None:
            assert LR_schedule is None,"Reduce on plateau and LR schedules may unwantedly interfere, provide either one of those or none of those."
        elif LR_schedule is not None:
            assert  reduce_LR_on_plateau is None,"Reduce on plateau and LR schedules may unwantedly interfere, provide either one of those or none of those."
    import optuna.logging
    logging_level={
        None:optuna.logging.get_verbosity(),
        0:optuna.logging.WARNING,
        1:optuna.logging.INFO,
        2:optuna.logging.DEBUG,
    }
    optuna.logging.set_verbosity(logging_level[verbose])
    def objective(trial:optuna.Trial):
        dropout=trial.suggest_float(
            "dropout",**dropout_distribution
        ) if isinstance(dropout_distribution,dict) else trial.suggest_categorical(
            "dropout",dropout_distribution
        )
        model=Sequential()
        model.add(keras.Input(shape=(n_past, n_features)))
        model.add(
            LSTM(
                trial.suggest_int(
                    "1st_LSTM_layer_units",**units_distribution
                ) if isinstance(units_distribution,dict) else trial.suggest_categorical(
                    "1st_LSTM_layer_units",units_distribution
                ), 
                activation='tanh', 
                return_sequences=True
            )
        )
        model.add(Dropout(dropout))
        model.add(
            LSTM(
                trial.suggest_int(
                    "2nd_LSTM_layer_units",**units_distribution
                ) if isinstance(units_distribution,dict) else trial.suggest_categorical(
                    "2nd_LSTM_layer_units",units_distribution
                ), 
                activation='tanh',
            )
        )
        model.add(Dropout(dropout))
        model.add(
            Dense(
                trial.suggest_int(
                    "Dense_layer_units",**units_distribution
                ) if isinstance(units_distribution,dict) else trial.suggest_categorical(
                    "Dense_layer_units",units_distribution
                )
            )
        )
        model.add(Dropout(dropout))
        model.add(Dense(n_future))
        
        if gradient_clipping_distribution is not None:
            gradient_clip_val=trial.suggest_float(
                "gradient_clip_val",**gradient_clipping_distribution
            ) if isinstance(gradient_clipping_distribution,dict) else trial.suggest_categorical(
                "gradient_clip_val",gradient_clipping_distribution
            )
        learning_rate_suggested=trial.suggest_float(
            "learning_rate",**LR_distribution
        ) if isinstance(LR_distribution,dict) else trial.suggest_categorical(
            "learning_rate",LR_distribution
        )
        if LR_schedule is not None:
            assert "initial_learning_rate" in LR_schedule.__dict__.keys(),"Only LR schedules which require an initial_learning_rate are accepted.\nIf it's custom made, make sure to define this attribute in the schedule __init__()."
            LR_schedule.initial_learning_rate=learning_rate_suggested

        model.compile(
            optimizer=keras.optimizers.Adam(
                learning_rate=learning_rate_suggested if LR_schedule is None else LR_schedule,
                global_clipnorm=None if gradient_clipping_distribution is None else gradient_clip_val
            ),
            loss=loss,
            metrics=metrics,
        )

        history_callback=keras.callbacks.History()
        checkpointing_callback=keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(
                path_to_lstm_results,
                "Optimization/{}/Checkpoints/trial_{}/best_model.keras".format(
                    study_name,trial.number
                )
            ),
            monitor="val_loss",
            save_best_only=True,
            mode="min" if study.direction==optuna.study.StudyDirection.MINIMIZE else "max",
            save_weights_only=False
        )
        pruning_callback=lstm_OptunaPruningCallback(trial)
        callbacks=[history_callback,checkpointing_callback]
        if LR_schedule is None:
            LR_logger_callback=lstm_LRLogger()
            callbacks.append(LR_logger_callback)
            if reduce_LR_on_plateau is not None:
                callbacks.append(reduce_LR_on_plateau)
        else:
            schedule_logger_callback=ScheduleLogger()
            callbacks.append(schedule_logger_callback)
        callbacks.append(pruning_callback)
        if early_stopping:
            earlystopping_callback=keras.callbacks.EarlyStopping(
                monitor='val_loss',
                min_delta=0,
                patience=8,
                verbose=1,
                mode="min" if study.direction==optuna.study.StudyDirection.MINIMIZE else "max",
                baseline=None,
                restore_best_weights=False,
                start_from_epoch=0
            )
            callbacks.append(earlystopping_callback)
        if clear_cache_on_epoch_end:
            clearing_callback=lstm_ClearCacheOnEpochEnd()
            callbacks.append(clearing_callback)
        try:
            if all([X_val is not None,y_val is not None]):
                history=model.fit(
                    X_train,y_train,
                    validation_data=(X_val,y_val),
                    epochs=max_epochs,
                    batch_size=trial.suggest_int(
                        "batch_size",**batch_size_distribution
                    ) if isinstance(batch_size_distribution,dict) else trial.suggest_categorical(
                        "batch_size",batch_size_distribution
                    ),
                    shuffle=True,
                    verbose=0,
                    callbacks=callbacks
                )
            else:
                assert validation_split is not None,"If any of X_val and y_val is not provided, validation_split must be not None."
                history=model.fit(
                    X_train,y_train,
                    validation_split=validation_split,
                    epochs=max_epochs,
                    batch_size=trial.suggest_int(
                        "batch_size",**batch_size_distribution
                    ) if isinstance(batch_size_distribution,dict) else trial.suggest_categorical(
                        "batch_size",batch_size_distribution
                    ),
                    shuffle=True,
                    verbose=0,
                    callbacks=callbacks
                )
            save_variable(
                history,
                os.path.join(
                    path_to_lstm_results,
                    "Optimization/{}/Checkpoints/trial_{}/history.pkl".format(
                        study_name,trial.number
                    )
                )
            )
            save_variable(
                schedule_logger_callback._LR_log if LR_schedule is not None else LR_logger_callback._LR_log,
                os.path.join(
                    path_to_lstm_results,
                    "Optimization/{}/Checkpoints/trial_{}/LR_log.pkl".format(
                        study_name,trial.number
                    )
                )
            )
        except optuna.exceptions.TrialPruned:
            save_variable(
                history_callback.history,
                os.path.join(
                    path_to_lstm_results,
                    "Optimization/{}/Checkpoints/trial_{}/history.pkl".format(
                        study_name,trial.number
                    )
                )
            )
            save_variable(
                schedule_logger_callback._LR_log if LR_schedule is not None else LR_logger_callback._LR_log,
                os.path.join(
                    path_to_lstm_results,
                    "Optimization/{}/Checkpoints/trial_{}/LR_log.pkl".format(
                        study_name,trial.number
                    )
                )
            )
            raise optuna.exceptions.TrialPruned
        return min(history.history["val_loss"])
    try:
        if any(pd.Series(
                os.listdir(
                    os.path.join(
                        path_to_lstm_results,
                        "Optimization/{}".format(study_name),
                    )
                )
            ).isin(["{}.pkl".format(study_name)])
        ):
            study=load_variable(os.path.join(path_to_lstm_results,"Optimization/{}/{}.pkl".format(study_name,study_name)))
            print("{} correctly loaded and resumed from its end.\nThe first new trial is trial n°{} (enumerated from 0)".format(
                os.path.join(path_to_lstm_results,"Optimization/{}/{}.pkl".format(study_name,study_name)),
                len(study.trials)),flush=True
            )
        else:
            print("'{}' folder exists in '{}' but there are no '{}.pkl' to resume, a new study will be created...".format(
                study_name,os.path.join(path_to_lstm_results,"Optimization"),study_name
                ),flush=True
            )
            study=optuna.study.create_study(pruner=pruner,direction=direction)
            study.set_user_attr("study_name",study_name)
    except FileNotFoundError as e:
        print(e,flush=True)
        print("Since no '{}' folder exists in '{}', a new study will be created...".format(
            study_name,os.path.join(path_to_lstm_results,"Optimization")
            ),flush=True
        )
        study=optuna.study.create_study(pruner=pruner,direction=direction)
        study.set_user_attr("study_name",study_name)
    if n_trials>0:
        study.optimize(
            func=objective,
            n_trials=n_trials,
            timeout=timeout,
            gc_after_trial=True,
        )
        save_variable(
            study,
            os.path.join(
                path_to_lstm_results,
                "Optimization/{}/{}.pkl".format(study_name,study_name),
            )
        )
        print("Study named '{}' correctly saved at '{}'.".format(
            study_name,os.path.join(
                path_to_lstm_results,
                "Optimization/{}/{}.pkl".format(study_name,study_name),
                )
            ),flush=True
        )
    else:
        print("n_trials<=0, thus no new trials are run. The study is returned as is.")
    return study

In [ ]:
def train_lstm_like_tuning(
        X_train:np.ndarray,y_train:np.ndarray,
        max_epochs:int=200,
        X_val:np.ndarray|None=None,y_val:np.ndarray|None=None,
        validation_split:float|None=0.2,
        path_to_lstm_results:Path=Path("./Results/LSTM"),
        units:list[int]=[
            64,32,128
        ],
        batch_size:int=16,
        dropout:float=0.12,
        gradient_clipping:float=1,
        LR:float=0.0021,
        reduce_LR_on_plateau:keras.callbacks.ReduceLROnPlateau|None=None,
        loss:keras.losses.Loss|str='mean_squared_error',
        metrics:list[keras.metrics.Metric|str]=[
            'mean_absolute_error','mean_squared_error','mean_absolute_percentage_error',
            'root_mean_squared_error','r2_score'
        ],
        early_stopping:bool=False,
        clear_cache_on_epoch_end:bool=True,
        n_past:int=n_past,
        n_features:int=n_features,
):
    model=Sequential()
    model.add(keras.Input(shape=(n_past, n_features)))
    model.add(
        LSTM(
            units[0], 
            activation='tanh', 
            return_sequences=True
        )
    )
    model.add(Dropout(dropout))
    model.add(
        LSTM(
            units[1], 
            activation='tanh',
        )
    )
    model.add(Dropout(dropout))
    model.add(Dense(units[2]))
    model.add(Dropout(dropout))
    model.add(Dense(n_future))
    
    gradient_clip_val=gradient_clipping
    learning_rate_suggested=LR

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate_suggested,
            global_clipnorm=gradient_clip_val
        ),
        loss=loss,
        metrics=metrics,
    )

    history_callback=keras.callbacks.History()
    from time import time
    exp_name="Experiment {}".format(datetime.datetime.fromtimestamp(time()).strftime("%d %m %Y %H_%M_%S"))
    checkpointing_callback=keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(
            path_to_lstm_results,
            "Statistical significance/{}/best_model.keras".format(
                exp_name
            )
        ),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        save_weights_only=False
    )
    callbacks=[history_callback,checkpointing_callback]
    LR_logger_callback=lstm_LRLogger()
    callbacks.append(LR_logger_callback)
    if reduce_LR_on_plateau is not None:
        callbacks.append(reduce_LR_on_plateau)
    if early_stopping:
        earlystopping_callback=keras.callbacks.EarlyStopping(
            monitor='val_loss',
            min_delta=0,
            patience=8,
            verbose=1,
            mode="min",
            baseline=None,
            restore_best_weights=False,
            start_from_epoch=0
        )
        callbacks.append(earlystopping_callback)
    if clear_cache_on_epoch_end:
        clearing_callback=lstm_ClearCacheOnEpochEnd()
        callbacks.append(clearing_callback)
    if all([X_val is not None,y_val is not None]):
        history=model.fit(
            X_train,y_train,
            validation_data=(X_val,y_val),
            epochs=max_epochs,
            batch_size=batch_size,
            shuffle=True,
            verbose=0,
            callbacks=callbacks
        )
    else:
        assert validation_split is not None,"If any of X_val and y_val is not provided, validation_split must be not None."
        history=model.fit(
            X_train,y_train,
            validation_split=validation_split,
            epochs=max_epochs,
            batch_size=batch_size,
            shuffle=True,
            verbose=0,
            callbacks=callbacks
        )
    save_variable(
        history,
        os.path.join(
            path_to_lstm_results,
            "Statistical significance/{}/history.pkl".format(
                exp_name
            )
        )
    )
    save_variable(
        LR_logger_callback._LR_log,
        os.path.join(
            path_to_lstm_results,
            "Statistical significance/{}/LR_log.pkl".format(
                exp_name
            )
        )
    )
    print("Training completed.")

## Tuning

In [ ]:
max_epochs_study=200
study=tune_lstm(
    study_name="HPBP_no_ExpDec_ES_yes_GC_ROP",
    pruner=optuna.pruners.HyperbandPruner(
        max_resource=max_epochs_study,
        min_resource=2,
        reduction_factor=2,
        bootstrap_count=0
    ),
    n_trials=0,
    X_train=X_train,
    y_train=y_train,
    max_epochs=max_epochs_study,
    validation_split=0.2,
    LR_schedule=None,
    reduce_LR_on_plateau=keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.1,
        patience=4, 
        mode="min",
        min_delta=0
    ),
    timeout=16*3600,
)

In [ ]:
trial_number=None
if trial_number is None:
  _,trial_number,lstm_best_epoch=find_trial_in_study(study,pick_best=True)
path_to_trial_folder=os.path.join(
  Path("./Results/LSTM"),
  "Optimization/{}/Checkpoints/trial_{}".format(
    study.user_attrs["study_name"],trial_number
  )
)
print("Trial selected in study (enumerated from 0): {}".format(trial_number))
print("Its best epoch (enumerated from 0): {}".format(lstm_best_epoch))
print("----Hyperparams----")
display(study.trials[trial_number].params)
model=keras.models.load_model(os.path.join(path_to_trial_folder,"best_model.keras"))
history=load_variable(os.path.join(path_to_trial_folder,"history.pkl"))
model.summary()
fig=plt.figure(
  figsize=(16,4.5),
  dpi=300
  )
exclude_first_epoch=False
with plt.rc_context({"font.size":15}):
  plt.plot(np.arange(exclude_first_epoch,len(history.history['loss'])),history.history['loss'][exclude_first_epoch:],linewidth=1.5,label="Training")
  plt.plot(np.arange(exclude_first_epoch,len(history.history['loss'])),history.history['val_loss'][exclude_first_epoch:],linewidth=1.5,label="Validation")
  plt.plot(lstm_best_epoch,history.history['val_loss'][lstm_best_epoch],marker=".",markersize=16,markerfacecolor="lime",markeredgecolor="k",label="")
  plt.plot(lstm_best_epoch,history.history['loss'][lstm_best_epoch],marker=".",markersize=16,markerfacecolor="lime",markeredgecolor="k",label="")
  plt.ylabel("Epoch loss [-]")
  plt.xlabel('Epoch number [-]')
  plt.xlim(left=exclude_first_epoch,right=200)
  plt.legend(loc="upper right")
  plt.grid()
  plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True,nbins=10))
  match_data_area(fig_ref=etsf_lc_fig,fig_tgt=fig)
  fig.savefig(os.path.join(path_to_trial_folder,'Learning_curves.png'))
  plt.close("all")
  display(fig)
fig=plt.figure(figsize=(16,9))
lr_history=load_variable(os.path.join(path_to_trial_folder,"LR_log.pkl"))
with plt.rc_context({"font.size":16}):
  plt.plot(lr_history.keys(),lr_history.values())
  plt.title('Learning rate history')
  plt.ylabel('Learning rate (-)')
  plt.xlabel('Training epoch' if len(lr_history.keys())<=max_epochs_study else 'Training step')
  plt.xlim(left=0)
  plt.grid()
  plt.tight_layout()
  fig.savefig(os.path.join(path_to_trial_folder,'LR_history.png'))
  plt.show()

In [ ]:
building="Agriculture"
preds=model.predict(X_test)

In [ ]:
_,scores=lstm_above_and_below(
    output_preds=preds,
    input_target=y_test,
    input_starting_date=pd.Timestamp(year=2022,month=7,day=1,hour=0,minute=0),
    sklearn_metric_name="MAPE",
    upper_threshold=("quantile",0.975),
    lower_threshold=("quantile",0.025),
    return_above_and_below_dfs=True,
    return_scores=True,
    thresholds_wrt_non_overlapping_in_test=True,
    building_name=building,
)

In [ ]:
val_losses_df=find_trial_in_study(study,return_only_val_losses_df=True)
display(val_losses_df.sort_values(by="trial best validation loss"))

In [ ]:
get_hyperparameters_exploration(
    study,
    hps_to_ignore=["gradient_clip_val"],
    hps_in_log_scale=["learning_rate"],
    hps_in_discrete_scale=[
        "1st_LSTM_layer_units","2nd_LSTM_layer_units","Dense_layer_units",
        "batch_size"
    ],
    hps_categorical=[
        "1st_LSTM_layer_units","2nd_LSTM_layer_units","Dense_layer_units",
        "batch_size"
    ],
    to_shrink=[
        "1st_LSTM_layer_units","2nd_LSTM_layer_units","Dense_layer_units",
        "batch_size"
    ],
    custom_ticks=[
        "1st_LSTM_layer_units","2nd_LSTM_layer_units","Dense_layer_units",
        "batch_size"
    ],
    rename_map={
        "dropout":"Dropout rate",
        "1st_LSTM_layer_units":r"$n^{[1]}$",
        "2nd_LSTM_layer_units":r"$n^{[2]}$",
        "Dense_layer_units":r"$n^{[3]}$",
        "batch_size":r"$N^{\{i\}}$",
    },
    capital_title=["batch_size"],
    figsize=(16,9),
    sns_histplot_kwargs=dict(multiple="stack"),
    markerfacecolor="lime",
    markeredgecolor="black",    
    fontsize=24,
    metric_name="MAPE",
    markersize=15,
    marker="o",
    dpi=600,
    to_approx={"learning_rate":4,"dropout":2}
)

## Generalization

In [ ]:
building="Agriculture_extended"
lstm_bdg_to_test_df={
    "Agriculture":{
        "features":X_test,"target":y_test,
        "starting_date":pd.Timestamp(year=2022,month=7,day=1,hour=0,minute=0)
    },
    "Agriculture_extended":{
        "features":X_Agri_extended,"target":y_Agri_extended,
        "starting_date":pd.Timestamp(year=2019,month=7,day=1,hour=0,minute=0)
    },
    "Architecture":{
        "features":X_Architecture_building_test,"target":y_Architecture_building_test,
        "starting_date":pd.Timestamp(year=2019,month=7,day=1,hour=0,minute=0)
    },
    "Economics":{
        "features":X_Economics_building_test,"target":y_Economics,
        "starting_date":pd.Timestamp(year=2019,month=7,day=1,hour=0,minute=0)
    }
}
assert building in lstm_bdg_to_test_df.keys(),"The selected building is not implemented."
preds=model.predict(lstm_bdg_to_test_df[building]["features"])

In [ ]:
lstm_in_test_fig=lstm_above_and_below(
    output_preds=preds,
    input_target=lstm_bdg_to_test_df[building]["target"],
    input_starting_date=lstm_bdg_to_test_df[building]["starting_date"],
    sklearn_metric_name="MAPE",
    upper_threshold=("quantile",0.975),
    lower_threshold=("quantile",0.025),
    thresholds_wrt_non_overlapping_in_test=True,
    building_name=building,
    fontsize=13,
    figsize=(16,4.5),
    legend_only=True,
    xlabel="",
    return_only_figure=True,
    stairplot=True,
    dpi=600,
    target_name="electric load (kW)",
)

## Training significance

In [ ]:
train_lstm_like_tuning(
    X_train=X_train,
    y_train=y_train,
    max_epochs=200,
    validation_split=0.2,
    units=[64, 32, 128],
    batch_size=16,
    dropout=0.12105448968706753,
    gradient_clipping=1.0,
    LR=0.0021096806409865447,
    reduce_LR_on_plateau=keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.1,
        patience=4,
        mode="min",
        min_delta=0
    ),
    clear_cache_on_epoch_end=False,
)

In [ ]:
trainings_available=[
  item for item in os.listdir(os.path.join(Path("./Results/LSTM"),"Statistical significance"))
]
print(trainings_available,flush=True)
index_training=int(input("Type the index corresponding to the training you want to load:"))
training_name=trainings_available[index_training]
path_to_training_folder=os.path.join(Path("./Results/LSTM"),"Statistical significance",training_name)
model=keras.models.load_model(os.path.join(path_to_training_folder,"best_model.keras"))
history=load_variable(os.path.join(path_to_training_folder,"history.pkl"))
model.summary()
fig=plt.figure(
  figsize=(16,4.5),
  dpi=600
  )
exclude_first_epoch=False
with plt.rc_context({"font.size":13}):
  plt.plot(np.arange(exclude_first_epoch,len(history.history['loss'])),history.history['loss'][exclude_first_epoch:],linewidth=1.5,label="Training")
  plt.plot(np.arange(exclude_first_epoch,len(history.history['loss'])),history.history['val_loss'][exclude_first_epoch:],linewidth=1.5,label="Validation")
  plt.ylabel("Epoch loss [-]")
  plt.xlabel('Epoch number [-]')
  plt.xlim(left=exclude_first_epoch,right=200)
  plt.legend(loc="upper right")
  plt.grid()
  plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True,nbins=10))
  match_data_area(fig_ref=etsf_lc_fig,fig_tgt=fig)
  fig.savefig(os.path.join(path_to_training_folder,'Learning_curves.png'))
  plt.close("all")
  display(fig)
fig=plt.figure(figsize=(16,9))
lr_history=load_variable(os.path.join(path_to_training_folder,"LR_log.pkl"))
with plt.rc_context({"font.size":16}):
  plt.plot(lr_history.keys(),lr_history.values())
  plt.title('Learning rate history')
  plt.ylabel('Learning rate (-)')
  plt.xlabel('Training epoch' if len(lr_history.keys())<=200 else 'Training step')
  plt.xlim(left=0)
  plt.grid()
  plt.tight_layout()
  fig.savefig(os.path.join(path_to_training_folder,'LR_history.png'))
  plt.show()
if "batch_times.pkl" in os.listdir(path_to_training_folder):
  batch_times=pd.Series(load_variable(os.path.join(path_to_training_folder,"batch_times.pkl")),name="Elapsed times per training batch")
  display(batch_times.describe())
  batch_per_training_epoch=(len(batch_times)//200)
  epoch_times=[]
  for i in range(200):
    epoch_times.append(
      batch_times.iloc[
        i*batch_per_training_epoch:(i+1)*batch_per_training_epoch
        ].sum()
    )
  epoch_times=pd.Series(epoch_times,name="Elapsed times per training epoch [s]")
  display(epoch_times.describe())

In [ ]:
for nitem,item in enumerate(os.listdir(os.path.join(Path("./Results/LSTM"),"Statistical significance"))):
    path_to_training_folder=os.path.join(Path("./Results/LSTM"),"Statistical significance",item)
    history=load_variable(os.path.join(path_to_training_folder,"history.pkl"))
    loss=history.history["loss"]
    val_loss=history.history["val_loss"]
    if not nitem:
        stat_loss=np.array(loss).reshape(1,-1)
        stat_val_loss=np.array(val_loss).reshape(1,-1)
    else:
        stat_loss=np.concat((stat_loss,np.array(loss).reshape(1,-1)),axis=0)
        stat_val_loss=np.concat((stat_val_loss,np.array(val_loss).reshape(1,-1)),axis=0)
stat_final_loss={
    "min":stat_loss[:,-1].min().item(),
    "max":stat_loss[:,-1].max().item(),
    "mean":stat_loss[:,-1].mean().item(),
    "std":stat_loss[:,-1].std().item(),
}
display(stat_final_loss)
ave_loss=stat_loss.mean(axis=0)
ave_val_loss=stat_val_loss.mean(axis=0)
fig=plt.figure(
  figsize=(5,4),
  dpi=300
  )
exclude_first_epoch=False
with plt.rc_context({"font.size":12}):
  plt.plot(np.arange(exclude_first_epoch,len(ave_loss)),ave_loss[exclude_first_epoch:],linewidth=1.5,label="Training")
  plt.plot(np.arange(exclude_first_epoch,len(ave_val_loss)),ave_val_loss[exclude_first_epoch:],linewidth=1.5,label="Validation")
  plt.fill_between(
    np.arange(exclude_first_epoch,len(ave_loss)),
    ave_loss[exclude_first_epoch:]-stat_loss.std(axis=0),
    ave_loss[exclude_first_epoch:]+stat_loss.std(axis=0),
    label="",
    fc="tab:blue",
    step=None,
    alpha=0.3
    )
  plt.fill_between(
    np.arange(exclude_first_epoch,len(ave_val_loss)),
    ave_val_loss[exclude_first_epoch:]-stat_val_loss.std(axis=0),
    ave_val_loss[exclude_first_epoch:]+stat_val_loss.std(axis=0),
    label="",
    fc="tab:orange",
    step=None,
    alpha=0.3
    )
  plt.ylabel("Average epoch MSE loss [-]")
  plt.xlabel('Epoch number [-]')
  plt.xlim(left=exclude_first_epoch,right=200)
  plt.legend(loc="upper right")
  plt.grid()
  plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True,nbins=10))
  plt.close("all")
  display(fig)

# TFT DFs/DSs/DLs

In [ ]:
import torch
import lightning.pytorch as pl
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

In [ ]:
import pytorch_forecasting
from pytorch_forecasting.models.baseline import Baseline
from pytorch_forecasting.models.temporal_fusion_transformer import TemporalFusionTransformer
from pytorch_forecasting.data.timeseries import TimeSeriesDataSet
from pytorch_forecasting.data.encoders import GroupNormalizer
from pytorch_forecasting.metrics.quantile import QuantileLoss

df_train_x_tft = copy.deepcopy(df_train_x)
df_train_x_tft['target'] = arr_train_y
df_train_x_tft = df_train_x_tft.drop(columns=["Agriculture_building_load"])

df_train_x_tft['time_idx'] = [i for i in range(len(df_train_x_tft))]

df_train_x_tft['group_ids'] = [0 for _ in range(len(df_train_x_tft))]

df_test_x_tft = copy.deepcopy(df_test_x)
df_test_x_tft = df_test_x_tft[[c for c in df_train_x_tft.columns if c not in ['target', 'time_idx', 'group_ids']]]
df_test_x_tft['target'] = arr_test_y
df_test_x_tft['time_idx'] = [i for i in range(len(df_train_x_tft), len(df_train_x_tft)+len(df_test_x_tft))]
df_test_x_tft['group_ids'] = [0 for _ in range(len(df_test_x_tft))]

df_Architecture_building_test_x_tft = copy.deepcopy(df_Architecture_building_test_x)
df_Architecture_building_test_x_tft = df_Architecture_building_test_x_tft[[c for c in df_train_x_tft.columns if c not in ['target', 'time_idx', 'group_ids']]]
df_Architecture_building_test_x_tft['target'] = arr_Architecture_building_test_y
df_Architecture_building_test_x_tft['time_idx'] = [i for i in range(len(df_Architecture_building_test_x_tft))]
df_Architecture_building_test_x_tft['group_ids'] = [0 for _ in range(len(df_Architecture_building_test_x_tft))]
df_Economics_building_test_x_tft = copy.deepcopy(df_Economics_building_test_x)
df_Economics_building_test_x_tft = df_Economics_building_test_x_tft[[c for c in df_train_x_tft.columns if c not in ['target', 'time_idx', 'group_ids']]]
df_Economics_building_test_x_tft['target'] = arr_Economics_building_test_y
df_Economics_building_test_x_tft['time_idx'] = [i for i in range(len(df_Economics_building_test_x_tft))]
df_Economics_building_test_x_tft['group_ids'] = [0 for _ in range(len(df_Economics_building_test_x_tft))]

max_encoder_length = n_past
max_prediction_length = n_future

train_df_tft, val_df_tft = train_test_split(df_train_x_tft, test_size=0.2, shuffle=False, stratify=None, random_state=0)

train_data = TimeSeriesDataSet(
    train_df_tft,
    time_idx="time_idx",
    target="target",
    group_ids=["group_ids"], 
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_known_reals=['Month_of_year', 'Day_of_week'],
    time_varying_unknown_reals=["target", 'Mean_air_T(°C)', 'Precipitation_(mm)', 'Air_pressure_(hPa)', 'Mean_air_humidity(%)', 'Sol_Irrad(W/m2)', 'Mean_wind_speed_2m(m/s)'],
)
display(train_data.get_parameters())

val_data=TimeSeriesDataSet.from_dataset(train_data,val_df_tft,stop_randomization=True,predict=False)
test_data=TimeSeriesDataSet.from_dataset(train_data,df_test_x_tft,stop_randomization=True,predict=False)
test_data_Architecture_building=TimeSeriesDataSet.from_dataset(train_data,df_Architecture_building_test_x_tft,stop_randomization=True,predict=False)
test_data_Economics_building=TimeSeriesDataSet.from_dataset(train_data,df_Economics_building_test_x_tft,stop_randomization=True,predict=False)

train_dataloader = train_data.to_dataloader(train=True, batch_size=64, num_workers=0)
def get_params(dataloader):
  print(f"Batch size: {dataloader.batch_size}")
  print(f"Number of workers: {dataloader.num_workers}")
  print(f"Drop last: {dataloader.drop_last}")
val_dataloader = val_data.to_dataloader(train=False, batch_size=64, num_workers=1)
dict_first_x=next(iter(val_dataloader))[0]
for key,value in dict_first_x.items():
  print("{}.shape:{}".format(key,value.shape))
print("target.shape:",next(iter(val_dataloader))[1][0].shape)
print(next(iter(val_dataloader))[0]['encoder_lengths'].dtype)
test_dataloader = test_data.to_dataloader(train=False, batch_size=1, num_workers=1,drop_last=True)
test_Architecture_building_dataloader = test_data_Architecture_building.to_dataloader(train=False, batch_size=1, num_workers=1, drop_last=True)
test_Economics_building_dataloader = test_data_Economics_building.to_dataloader(train=False, batch_size=1, num_workers=1, drop_last=True)

In [ ]:
def print_n_seqs_tft_dls(des:str,df,dl):
    n=0
    for x_dict,_ in dl:
        n+=x_dict["encoder_target"].shape[0]
    print(des)
    print("# samples in df: {}".format(df.shape[0]))
    print("# sequences in dl: {}".format(n))
    print("# non overlapping sequences in dl: {}".format(math.floor((n-1)/n_future)+1))

In [ ]:
from lightning.pytorch.trainer.trainer import Trainer
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
import gc
class ClearCacheCallback(pl.callbacks.Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        torch.cuda.empty_cache()
        gc.collect()

class EpochProgressCallback(pl.callbacks.Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        print(f"Epoch {trainer.current_epoch}")

In [ ]:
def show_results(predictions, title,
                 saving_path=Path(dirpath_results),
                 start_timestep_preds:pd.Timestamp=pd.Timestamp(year=2022,month=7,day=8,hour=0,minute=0,second=0,tz="UTC"),
                 return_metrics:bool=False)->None|dict[str,float]:

    preds_seq_inv_1 = predictions.output.cpu()
    y_test_seq_inv_1 = predictions.y[0][0].reshape((preds_seq_inv_1.shape[0],preds_seq_inv_1.shape[1])).cpu()

    def sliding_window_selection(arr, n, m):
        result = []
        for i in range(0, len(arr), m):  
            if i + n <= len(arr):  
                result.append(arr[i:i+n])
        return np.array(result)

    n = 1
    m = n_future

    preds_seq_inv_1 = sliding_window_selection(preds_seq_inv_1, n, m).squeeze()
    y_test_seq_inv_1 = sliding_window_selection(y_test_seq_inv_1, n, m).squeeze()

    print(y_test_seq_inv_1.shape)
    y_test_seq_inv = y_test_seq_inv_1.reshape((y_test_seq_inv_1.shape[0]*y_test_seq_inv_1.shape[1], 1))
    print(y_test_seq_inv.shape)
    print(preds_seq_inv_1.shape)
    preds_seq_inv = preds_seq_inv_1.reshape((preds_seq_inv_1.shape[0]*preds_seq_inv_1.shape[1], 1))
    print(preds_seq_inv.shape)

    y_test_seq_inv = scaler_y.inverse_transform(y_test_seq_inv)
    preds_seq_inv = scaler_y.inverse_transform(preds_seq_inv)
    
    if start_timestep_preds.tz is None:
        start_timestep_preds=start_timestep_preds.tz_localize("UTC")
    Series_datetime_list=pd.DataFrame(datetime_list).squeeze()
    idx=(Series_datetime_list==start_timestep_preds).idxmax()
    fig_start = 2*7*24 + 3*24
    fig_end = 3*7*24 +3*24

    fig = plt.figure(figsize=(10, 3))
    plt.plot(datetime_list[idx+fig_start:idx+fig_end], y_test_seq_inv[fig_start:fig_end], color="red")
    plt.plot(datetime_list[idx+fig_start:idx+fig_end], preds_seq_inv[fig_start:fig_end], color="blue")
    plt.legend(['Actual Energy Consumption', 'Predicted Energy Consumption'], loc='upper right') 
    plt.xlabel('Time steps of test data')
    plt.xticks(rotation=90)
    plt.title(title)
    plt.tight_layout()
    fig.savefig(os.path.join(saving_path,f'TFT_{title}.png'))
    plt.show()

    fig_start = 0
    fig_end = math.floor(len(y_test_seq_inv))

    fig = plt.figure(figsize=(10, 3))
    plt.plot(datetime_list[idx+fig_start:idx+fig_end], y_test_seq_inv[fig_start:fig_end], color="red")
    plt.plot(datetime_list[idx+fig_start:idx+fig_end], preds_seq_inv[fig_start:fig_end], color="blue")
    plt.legend(['Actual Energy Consumption', 'Predicted Energy Consumption'], loc='upper right')
    plt.xlabel('Time steps of test data')
    plt.xticks(rotation=90)
    plt.title(title)
    plt.tight_layout()
    fig.savefig(os.path.join(saving_path,f'TFT_{title}_all.png'))
    plt.show()


    results = evaluate_forecast(y_test_seq_inv, preds_seq_inv)

    df_res = pd.DataFrame.from_dict(data = {"y": np.squeeze(y_test_seq_inv), "preds": np.squeeze(preds_seq_inv)}, orient="columns")
    df_res.to_csv(os.path.join(saving_path,"TFT_{}.csv".format(title)))

    title2 = f"----- Predictions of {title} with TFT -----"
    print_scores(results, "", title=title2)
    if return_metrics:
        return results


# TFT optimization

## Studies

In [ ]:
def rename_optuna_study_tensorboard_folder(
        study_name:str,
        path_to_search_into:str=".",
        parent_folder_name:str="lightning_logs",
        child_folder_name:str="optuna"
        )->None:
    import os
    from pathlib import Path
    new_folder_name=child_folder_name+"_{}".format(study_name)
    folder_found=False
    path_found=False
    parent_path=os.path.join(path_to_search_into,parent_folder_name)
    for path_explored,folder_names_in_path_explored,_ in os.walk(path_to_search_into):
        if path_explored==parent_path:
            for folder_name in folder_names_in_path_explored:
                if folder_name==child_folder_name:
                    path_to_rename=os.path.join(path_explored,folder_name)
                    path_renamed=os.path.join(path_explored,new_folder_name)
                    try:
                        os.rename(path_to_rename,path_renamed)
                        print("Searched '{}' folder was found in '{}' and was renamed to '{}'.".format(
                            child_folder_name,
                            parent_path,
                            new_folder_name
                            )
                        )
                    except FileExistsError:
                        import shutil
                        for filename in os.listdir(path_to_rename):
                            src_path = os.path.join(path_to_rename, filename)
                            dest_path = os.path.join(path_renamed, filename)
                            shutil.move(src_path, dest_path)
                        print("Searched '{}' folder was found in '{}' but '{}' folder already existed in '{}'.".format(
                            child_folder_name,
                            parent_path,
                            new_folder_name,
                            parent_path
                            )
                        )
                        print("Thus, '{}' folder's content was moved to '{}' folder.".format(
                            child_folder_name,
                            new_folder_name
                            )
                        )
                        shutil.rmtree(path_to_rename)
                        print("The original '{}' folder was eliminated.".format(
                            child_folder_name
                            )
                        )
                    folder_found=True
            if not folder_found:
                print("Searched '{}' folder was not found in '{}'. No renaming occured.".format(
                    child_folder_name,
                    parent_path
                    )
                )
            path_found=True
    if not path_found:
        print("Searched '{}' path was not found from '{}' path top-down. No renaming occured.".format(
            parent_path,
            path_to_search_into
            )
        )
import optuna
def hyperparameters_search(
        study_name:str,
        train_dataloaders:torch.utils.data.DataLoader,
        val_dataloaders:torch.utils.data.DataLoader,
        first_study:bool=False,latest_study_name:str=None,
        do_study:bool=True,resume_study:bool=False,
        optimize_hyperparameters_kwargs:dict=None,
        save_study:bool=True, return_paths:bool=False
        )->optuna.study.Study|dict[str,Path]|tuple[optuna.study.Study,dict[str,Path]]|None:
    from pathlib import Path
    path_to_ckpts_folder=Path("./Results/TFT/Optimization/{}/Checkpoints".format(study_name))
    path_to_uncomplete_study_file=Path("./Results/TFT/Optimization/{}/study_uncomplete.pkl".format(study_name))
    path_to_save_file=Path("./Results/TFT/Optimization/{}/study_resumed.pkl".format(study_name) if resume_study else "./Results/TFT/Optimization/{}/study.pkl".format(study_name))
    paths={
        "to_ckpts_folder":path_to_ckpts_folder,
        "to_uncomplete_study_file":path_to_uncomplete_study_file,
        "to_save_file":path_to_save_file
    }
    os.makedirs(path_to_ckpts_folder,exist_ok=True)
    if do_study:
        if not first_study:
            print("Checking for previous studies' not renamed tensorboard folder...")
            rename_optuna_study_tensorboard_folder(
                study_name=latest_study_name,
                parent_folder_name=optimize_hyperparameters_kwargs["log_dir"],
            )
        if resume_study:
            study_reload=load_variable(path_to_uncomplete_study_file)
            print("Optuna study object correctly reloaded.")
        from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
        study=optimize_hyperparameters(
            train_dataloaders=train_dataloaders,
            val_dataloaders=val_dataloaders,
            model_path=path_to_ckpts_folder,
            study=study_reload if resume_study else None,
            **optimize_hyperparameters_kwargs
        )
        if save_study:
            save_variable(study,path_to_save_file) 
            print("Study correctly saved at '{}'.".format(path_to_save_file))       
        print("Renaming the current study's tensorboard folder...")
        rename_optuna_study_tensorboard_folder(
            study_name=study_name,
            parent_folder_name=optimize_hyperparameters_kwargs["log_dir"],
        )
        if return_paths:
            return (study,paths)
        return study
    if return_paths:
        return paths

In [ ]:
def TFT_from_trial(
        path_to_checkpoints:Path|str,study:optuna.study.Study,
        trial_number:int=0,pick_best:bool=False,
        print_info:bool=True,return_path:bool=True,
        best_as:Literal['lowest_intermediate', 'lowest_objective_return']="lowest_intermediate",
        )->tuple[Path,TemporalFusionTransformer]|TemporalFusionTransformer:
    selected_trial,selected_trial_number,selected_trial_best_epoch_number=find_trial_in_study(study,pick_best=pick_best,trial_number=trial_number,best_as=best_as)
    path_to_trial_ckpt=os.path.join(
        path_to_checkpoints,
        "trial_{}".format(selected_trial_number),
        "epoch={}.ckpt".format(selected_trial_best_epoch_number)
    )
    if print_info:
        print('Trial selected (enumerated from 0): {}/{}\n'.format(selected_trial_number,len(study.trials)-1))
        print('Distributions to sample tentative hyperparameters from:')
        for key,value in selected_trial.distributions.items():
            print('->{}:{}'.format(key,value))
        print('\nHyperparameters:')
        for key,value in selected_trial.params.items():
            print('->{}:{}'.format(key,value))
        print('\nBest epoch (enumerated from 0): {}/{}\n'.format(selected_trial_best_epoch_number,selected_trial.last_step))
    tft=TemporalFusionTransformer.load_from_checkpoint(path_to_trial_ckpt)
    path_to_trial=os.path.join(
        path_to_checkpoints,
        "trial_{}".format(selected_trial_number)
    )
    if return_path:
        return (path_to_trial,tft)
    else:
        return tft

In [ ]:
study_name="HyperbandPruner"
latest_study_name="PatientPruner_wrapping_PercentilePruner"
paths=hyperparameters_search(
    study_name,train_dataloader,val_dataloader,do_study=False,resume_study=True,
    first_study=False,latest_study_name=latest_study_name,
    optimize_hyperparameters_kwargs=dict(
        n_trials=1,
        max_epochs=20,
        timeout=3600*1,
        gradient_clip_val_range=(0.1,10),
        pruner=optuna.pruners.HyperbandPruner(
            max_resource=20,
            min_resource=2,
            reduction_factor=2,
            bootstrap_count=0
        ),
        reduce_on_plateau_patience=4,
        use_learning_rate_finder=False,
        log_dir="lightning_logs",
    ),
    save_study=True,return_paths=True
)

In [ ]:
building_name="Agriculture"
completed_study=load_variable(paths["to_save_file"])
do_predictions=False
path_to_trial,tft_optuna_best=TFT_from_trial(paths["to_ckpts_folder"],study=completed_study,pick_best=True)
if do_predictions:
    predictions=tft_optuna_best.predict(test_dataloader, return_y=True, trainer_kwargs=dict(accelerator="gpu" if torch.cuda.is_available() else "cpu"))
else:
    predictions=load_variable(os.path.join(path_to_trial,"predictions_{}.pkl".format(building_name)))
scores=show_results(
    predictions, "{} building".format(building_name),saving_path=path_to_trial,
    return_metrics=True
)

In [ ]:
val_losses_df=find_trial_in_study(completed_study,return_only_val_losses_df=True)
display(val_losses_df.sort_values(by="trial best validation loss"))

In [ ]:
get_hyperparameters_exploration(
    study=completed_study,
    hps_in_log_scale=["gradient_clip_val","learning_rate"],
    hps_in_discrete_scale=[
        "attention_head_size",
        ],
    custom_ticks=['hidden_size','hidden_continuous_size','attention_head_size'],
    n_custom_ticks=[10,10,4],    
    figsize=(16,9),
    sns_histplot_kwargs=dict(multiple="stack"),
    markerfacecolor="lime",
    markeredgecolor="black",
    fontsize=24,
    metric_name="MAPE",
    markersize=15,
    marker="o",
    rename_map={"gradient_clip_val":"Gradient clip value","dropout":"Dropout rate"},
    dpi=600,
    to_approx={"gradient_clip_val":2,"dropout":3,"learning_rate":4}
)

## Variants of the best trial

### Change in features

In [ ]:
tft_dfs_to_be_manipulated={
    "train_df_tft":train_df_tft,
    "val_df_tft":val_df_tft,
    "df_test_x_tft":df_test_x_tft,
    "df_Architecture_building_test_x_tft":df_Architecture_building_test_x_tft,
    "df_Economics_building_test_x_tft":df_Economics_building_test_x_tft
}

# # ----CASE #1_OO----: 'R_MaT_OO' (removal as from Original Order of VSWs)
# to_be_removed=["Mean_air_T(°C)"]
# to_be_added=None
# forced_id='R_MaT_OO'
# id_removal=None
# id_addition=None

# # ----CASE #2_OO----: 'R_MaTAp(_OO' (removal as from Original Order of VSWs)
# to_be_removed=["Mean_air_T(°C)","Air_pressure_(hPa)"]
# to_be_added=None
# forced_id='R_MaTAp(_OO'
# id_removal=None
# id_addition=None

# # ----CASE #3_OO----: 'R_MaTAp(P(_OO' (removal as from Original Order of VSWs)
# to_be_removed=["Mean_air_T(°C)","Air_pressure_(hPa)","Precipitation_(mm)"] 
# to_be_added=None
# forced_id='R_MaTAp(P(_OO'
# id_removal=None
# id_addition=None

# # ----CASE #4_OO----: 'R_MaTAp(P(Mah_OO' (removal as from Original Order of VSWs)
# to_be_removed=["Mean_air_T(°C)","Air_pressure_(hPa)","Precipitation_(mm)","Mean_air_humidity(%)"] 
# to_be_added=None
# forced_id='R_MaTAp(P(Mah_OO'
# id_removal=None
# id_addition=None

# # ----CASE #5_OO----: 'R_MaTAp(P(MahMws2_OO' (removal as from Original Order of VSWs)
# to_be_removed=["Mean_air_T(°C)","Air_pressure_(hPa)","Precipitation_(mm)","Mean_air_humidity(%)","Mean_wind_speed_2m(m/s)"]
# to_be_added=None
# forced_id='R_MaTAp(P(MahMws2_OO'
# id_removal=None
# id_addition=None

# # ----CASE #6_OO----: 'R_MaTAp(P(MahMws2SI_OO' (removal as from Original Order of VSWs)
# to_be_removed=["Mean_air_T(°C)","Air_pressure_(hPa)","Precipitation_(mm)","Mean_air_humidity(%)","Mean_wind_speed_2m(m/s)","Sol_Irrad(W/m2)"] 
# to_be_added=None
# forced_id='R_MaTAp(P(MahMws2SI_OO'
# id_removal=None
# id_addition=None

# ----CASE #7_OO----: 'R_MaTAp(P(MahMws2SITarget_OO' (removal as from Original Order of VSWs)
to_be_removed=["Mean_air_T(°C)","Air_pressure_(hPa)","Precipitation_(mm)","Mean_air_humidity(%)","Mean_wind_speed_2m(m/s)","Sol_Irrad(W/m2)"]
to_be_added=None
forced_id='R_MaTAp(P(MahMws2SITarget_OO'
id_removal=None
id_addition=None

# # NOTE: PURE ADDITION
# assert add_features,"You must restart the kernel and run the 'Data preparation' section with\nadd_features=True to make sure all the features you wanted to add are in the dataframes."
# assert id_addition is not None,"You must restart the kernel and run the 'Data preparation' section with\nadd_features=True to make sure all the features you wanted to add are in the dataframes."
# to_be_added=None
# to_be_removed=None
# id_removal=None
# id_addition=id_addition

# # ----CASE #1----: 'A_Hi'. Add an holiday indicator, including Saturday and Sunday.
# forced_id=None

tft_dfs_manipulated={}
for idx,(df_name,dataframe) in enumerate(tft_dfs_to_be_manipulated.items()):
    if idx==0:
        tft_dfs_manipulated[df_name],id_manipulation=manipulate_features(
            dataframe,to_be_removed,to_be_added,print_info=True,
            return_id_manipulation=True,id_addition=id_addition,
            id_removal=id_removal,forced_id=forced_id
        )
    else:
        tft_dfs_manipulated[df_name]=manipulate_features(
            dataframe,to_be_removed,to_be_added,print_info=False,
            return_id_manipulation=False,id_addition=id_addition,
            id_removal=id_removal,forced_id=forced_id
        )

In [ ]:
tft_tsds_manipulated={}
tft_dataloaders={
    "train_dataloader":train_dataloader,
    "val_dataloader":val_dataloader,
    "test_dataloader":test_dataloader,
    "test_Architecture_building_dataloader":test_Architecture_building_dataloader,
    "test_Economics_building_dataloader":test_Economics_building_dataloader
}
df_name_to_dataloader_name={
    "train_df_tft":"train_dataloader",
    "val_df_tft":"val_dataloader",
    "df_test_x_tft":"test_dataloader",
    "df_Architecture_building_test_x_tft":"test_Architecture_building_dataloader",
    "df_Economics_building_test_x_tft":"test_Economics_building_dataloader"
}
tft_dataloaders_manipulated={}
for df_name,dataframe in tft_dfs_manipulated.items():
    if df_name=="train_df_tft":
        tft_tsds_manipulated[df_name]=TimeSeriesDataSet(
            dataframe,
            time_idx="time_idx",  
            target="target", 
            group_ids=["group_ids"], 
            max_encoder_length=max_encoder_length,
            max_prediction_length=max_prediction_length,
            # # NOTE: PURE ABLATION

            # # ----CASE #1_OO----: 'R_MaT_OO'
            # time_varying_known_reals=['Day_of_week','Month_of_year'],
            # time_varying_unknown_reals=['target','Precipitation_(mm)', 'Mean_air_humidity(%)', 'Sol_Irrad(W/m2)', 'Mean_wind_speed_2m(m/s)',"Air_pressure_(hPa)"],

            # # ----CASE #2_OO----: 'R_MaTAp(_OO'
            # time_varying_known_reals=['Day_of_week','Month_of_year'],
            # time_varying_unknown_reals=['target','Precipitation_(mm)', 'Mean_air_humidity(%)', 'Sol_Irrad(W/m2)', 'Mean_wind_speed_2m(m/s)'],

            # # ----CASE #3_OO----: 'R_MaTAp(P(_OO' (removal as from Original Order of VSWs)
            # time_varying_known_reals=['Day_of_week','Month_of_year'],
            # time_varying_unknown_reals=['target', 'Mean_air_humidity(%)', 'Sol_Irrad(W/m2)', 'Mean_wind_speed_2m(m/s)'],

            # # ----CASE #4_OO----: 'R_MaTAp(P(Mah_OO'
            # time_varying_known_reals=['Day_of_week','Month_of_year'],
            # time_varying_unknown_reals=['target', 'Sol_Irrad(W/m2)', 'Mean_wind_speed_2m(m/s)'],
            
            # # ----CASE #5_OO----: 'R_MaTAp(P(MahMws2_OO'
            # time_varying_known_reals=['Day_of_week','Month_of_year'],
            # time_varying_unknown_reals=['target', 'Sol_Irrad(W/m2)'],

            # # ----CASE #6_OO----: 'R_MaTAp(P(MahMws2SI_OO'
            # time_varying_known_reals=['Day_of_week','Month_of_year'],
            # time_varying_unknown_reals=['target'],

            # ----CASE #7_OO----: 'R_MaTAp(P(MahMws2SITarget_OO'
            time_varying_known_reals=['Day_of_week','Month_of_year'],
            time_varying_unknown_reals=[],

            # # NOTE: PURE ADDITION
            # # ----CASE #1----: 'A_Hi'. Add 'Holidays_indicator'
            # time_varying_known_reals=['Day_of_week','Month_of_year','Holidays_indicator'],
            # time_varying_unknown_reals=['target','Precipitation_(mm)', 'Mean_air_humidity(%)', 'Sol_Irrad(W/m2)', 'Mean_wind_speed_2m(m/s)',"Mean_air_T(°C)","Air_pressure_(hPa)"],

        )
        train=True
    else:
        tft_tsds_manipulated[df_name]=TimeSeriesDataSet.from_dataset(
            tft_tsds_manipulated["train_df_tft"],
            tft_dfs_manipulated[df_name],
            stop_randomization=True,
            predict=False
        )
        train=False
    tft_dataloaders_manipulated[df_name]=tft_tsds_manipulated[df_name].to_dataloader(
        train=train,
        batch_size=tft_dataloaders[df_name_to_dataloader_name[df_name]].batch_size,
        num_workers=tft_dataloaders[df_name_to_dataloader_name[df_name]].num_workers,
        drop_last=tft_dataloaders[df_name_to_dataloader_name[df_name]].drop_last
    )

In [ ]:
optimum_trial,_,_=find_trial_in_study(
    completed_study,pick_best=True
)
optimum_trial_hps_dict=optimum_trial.params
optimum_TFT_hps={}
TFT_hps_names=["hidden_size","dropout","attention_head_size","hidden_continuous_size",
               "learning_rate"]
optimum_Trainer_hps={}
Trainer_hps_names=["gradient_clip_val"]
for hp_name,hp_value in optimum_trial_hps_dict.items():
    if hp_name in TFT_hps_names:
        optimum_TFT_hps[hp_name]=hp_value
    if hp_name in Trainer_hps_names:
        optimum_Trainer_hps[hp_name]=hp_value

print("----TFT's hyperparameters recap----")
for hp_name,hp_value in optimum_TFT_hps.items():
    print("{} = {}".format(hp_name,hp_value))
print("----Trainer's hyperparameters recap----")
for hp_name,hp_value in optimum_Trainer_hps.items():
    print("{} = {}".format(hp_name,hp_value))
    
optimum_tft = TemporalFusionTransformer.from_dataset(
    tft_tsds_manipulated["train_df_tft"], 
    **optimum_TFT_hps,
    output_size=7, 
    loss=QuantileLoss(), 
    log_interval=10,
    reduce_on_plateau_patience=4
)

early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=15, verbose=False, mode="min"
)
version_name="Best_in_{}_{}".format(study_name,id_manipulation)
logger = TensorBoardLogger(".", version=version_name) 
clear_cache_callback = ClearCacheCallback()
epoch_progress_callback = EpochProgressCallback()

from lightning.pytorch.callbacks import ModelCheckpoint
ckpts_path = "./Results/TFT/Optimization/{}/Twists_on_best/Change_features/{}".format(study_name,id_manipulation)
best_model_callback=ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    dirpath=ckpts_path,
    filename="best_{epoch}",
    save_top_k=1
)

trainer = Trainer(
    deterministic="warn", 
    max_epochs=500,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    **optimum_Trainer_hps, 
    enable_model_summary=False, 
    enable_checkpointing=True,  
    enable_progress_bar=True,
    callbacks=[clear_cache_callback, early_stop_callback, epoch_progress_callback, best_model_callback],
    logger=logger,
)

In [ ]:
do_train=False
if do_train:
    print(f"CUDA available: {torch.cuda.is_available()}")
    trainer.fit(optimum_tft, train_dataloaders=tft_dataloaders_manipulated["train_df_tft"], val_dataloaders=tft_dataloaders_manipulated["val_df_tft"])  
    optimum_model_path=trainer.checkpoint_callback.best_model_path
    last_epoch_filename = "last_epoch={}".format(trainer.current_epoch-1)
    last_epoch_model_path = "{}/{}.ckpt".format(ckpts_path,last_epoch_filename)
    trainer.save_checkpoint(last_epoch_model_path) 
else:
    matching_path_list=glob.glob(pathname=os.path.join(Path(ckpts_path),"best_epoch=*.ckpt"))
    print("Matching paths found:",matching_path_list)
    confirm=input("Type 'ok' if the path to the best model is the expected one: ")
    if confirm=="ok":
        optimum_model_path=matching_path_list[0]
    else:
        print("Not found the proper best model path.")
print("Path selected:",optimum_model_path)    

In [ ]:
do_predictions=False
building_name="Agriculture"
optimum_tft = TemporalFusionTransformer.load_from_checkpoint(optimum_model_path)
if do_predictions:
    predictions=optimum_tft.predict(tft_dataloaders_manipulated["df_test_x_tft"], return_y=True, trainer_kwargs=dict(accelerator="gpu" if torch.cuda.is_available() else "cpu"))
    save_variable(predictions, os.path.join(ckpts_path,"predictions_{}.pkl".format(building_name)))
else:
    predictions=load_variable(os.path.join(ckpts_path,"predictions_{}.pkl".format(building_name)))
show_results(predictions, "{} building".format(building_name),saving_path=ckpts_path)

In [ ]:
ablation_results={
    9:7.40, # Original
    8:7.68, # 'R_MaT_OO'
    7:8.32, # 'R_MaTAp(_OO'
    6:8.52, # 'R_MaTAp(P(_OO'
    5:7.99, # 'R_MaTAp(P(Mah_OO'
    4:7.84, # 'R_MaTAp(P(MahMws2_OO'
    3:8.31, # 'R_MaTAp(P(MahMws2SI_OO'
    2:18.18, # 'R_MaTAp(P(MahMws2SITarget_OO'
}
def plot_ablation_summary(
        ablation_results:dict[int,float],many_to_few:bool=False,
        avoid_callout:bool=False,figsize:tuple[float,float]=(16,9),fontsize=12,
        def_ofs_x:float=0,def_ofs_y:float=-20,offsets:dict[int,dict[Literal["x","y"],float]]={},
        dpi:int|None=None,
        formatter:bool=True,
        ylims:tuple[float]|None=None
        )->None:
    with plt.rc_context({'font.size':fontsize}):
        index=list(ablation_results.keys())
        index.sort(reverse=many_to_few)
        ablation_summary=pd.Series(
            [ablation_results[n_enc_var] for n_enc_var in index],
            index=index,
        )
        fig=plt.figure(figsize=figsize,dpi=dpi)
        ax=fig.gca()
        if many_to_few:
            ax.invert_xaxis()
        ax.plot(ablation_summary.index,ablation_summary,linestyle="-")
        all_offsets={nvar:{"x":def_ofs_x,"y":def_ofs_y} for nvar in ablation_summary.index}
        for nvar,nvar_ofs in all_offsets.items():
            if nvar in offsets.keys():
                nvar_ofs.update(offsets[nvar])
        if not avoid_callout:
            ax.plot(ablation_summary.index,ablation_summary,'.',markersize=12,markerfacecolor="red",markeredgecolor="k")
            for idx in ablation_summary.index:
                offset_x=all_offsets[idx]["x"]
                offset_y=all_offsets[idx]["y"]
                ax.annotate(
                    "{:.2f}%".format(ablation_summary.loc[idx]),
                    xy=(idx,ablation_summary.loc[idx]),
                    xytext=(offset_x,offset_y),
                    color="red",
                    textcoords="offset points",
                    size="medium",
                    ha="center",
                )
        ax.set_xlabel(
            "Number of features available at the encoder (-)" if formatter else "Number of features available at the encoder [-]"
            )
        if formatter:
            yticks=ax.get_yticks()
            yticks_labels=["{}%".format(label) for label in yticks]
            ax.set_yticks(ticks=yticks,labels=yticks_labels)
            ax.set_ylabel("MAPE")
        else:
            ax.set_ylabel("MAPE [%]")
        ax.grid(visible=True,alpha=0.3)
        if ylims is not None:
            ax.set_ylim(ylims)
        fig.tight_layout()
        plt.close("all")
        display(fig)
plot_ablation_summary(
    ablation_results,many_to_few=True,
    fontsize=15,
    def_ofs_y=-18,
    offsets={
        2:{"y":10},
        9:{"y":10},
    },
    figsize=(16,4.5),
    dpi=300, 
    formatter=False,
    ylims=(6,20)
)

### Training significance

In [ ]:
from lightning.pytorch.callbacks import Callback
class BatchTimeLogger_pytorch_lightning(Callback):
    def __init__(self):
        super().__init__()
        self.batch_times = []
        self._is_cuda = False
        self._device = None
    
    def on_train_start(self, trainer, pl_module):
        self._device = pl_module.device
        self._is_cuda = (self._device.type == "cuda")
    
    def on_train_batch_start(
        self, trainer, pl_module, batch, batch_idx
    ):
        if self._is_cuda: 
            torch.cuda.synchronize(self._device)

        self._start_time = perf_counter()

    def on_train_batch_end(
        self, trainer, pl_module, outputs, batch, batch_idx
    ):
        if self._is_cuda:
            torch.cuda.synchronize(self._device)

        elapsed = perf_counter() - self._start_time
        self.batch_times.append(elapsed)

optimum_trial,_,_=find_trial_in_study(
    completed_study,pick_best=True
)
optimum_trial_hps_dict=optimum_trial.params
optimum_TFT_hps={}
TFT_hps_names=["hidden_size","dropout","attention_head_size","hidden_continuous_size",
            "learning_rate"]
optimum_Trainer_hps={}
Trainer_hps_names=["gradient_clip_val"]
for hp_name,hp_value in optimum_trial_hps_dict.items():
    if hp_name in TFT_hps_names:
        optimum_TFT_hps[hp_name]=hp_value
    if hp_name in Trainer_hps_names:
        optimum_Trainer_hps[hp_name]=hp_value

print("----TFT's hyperparameters recap----")
for hp_name,hp_value in optimum_TFT_hps.items():
    print("{} = {}".format(hp_name,hp_value))
print("----Trainer's hyperparameters recap----")
for hp_name,hp_value in optimum_Trainer_hps.items():
    print("{} = {}".format(hp_name,hp_value))
    
optimum_tft = TemporalFusionTransformer.from_dataset(
    train_data, 
    **optimum_TFT_hps,
    output_size=7,  
    loss=QuantileLoss(), 
    log_interval=10,
    reduce_on_plateau_patience=4
)

from time import time
training_name=datetime.datetime.fromtimestamp(time()).strftime("%d_%m_%Y_%H_%M_%S")
version_name="Best_in_{}_stats_{}".format(study_name,training_name)
logger = TensorBoardLogger(".", version=version_name) 
clear_cache_callback = ClearCacheCallback()
epoch_progress_callback = EpochProgressCallback()
learning_rate_callback = LearningRateMonitor()

from lightning.pytorch.callbacks import ModelCheckpoint
do_train=True
do_predictions=True
if any([not do_train,not do_predictions]):
    possible_trainings=os.listdir("./Results/TFT/Optimization/{}/Twists_on_best/Statistical_significance".format(study_name))
    print("Possible trainings:",possible_trainings)
    idx_training=int(input("Type the index of the selected training"))
    ckpts_path=os.path.join("./Results/TFT/Optimization/{}/Twists_on_best/Statistical_significance".format(study_name),possible_trainings[idx_training])
else:
    ckpts_path = "./Results/TFT/Optimization/{}/Twists_on_best/Statistical_significance/{}".format(study_name,training_name)
best_model_callback=ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    dirpath=ckpts_path,
    filename="best_{epoch}",
    save_top_k=1
)
batch_time_logger_tft=BatchTimeLogger_pytorch_lightning()

trainer = Trainer(
    deterministic=False,
    max_epochs=20,
    accelerator="gpu" if torch.cuda.is_available() else "cpu", 
    **optimum_Trainer_hps, 
    enable_model_summary=False,
    enable_checkpointing=True,  
    enable_progress_bar=False,
    callbacks=[
        best_model_callback,
        batch_time_logger_tft],
    logger=False,
)
if do_train:
    print(f"CUDA available: {torch.cuda.is_available()}")
    trainer.fit(optimum_tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)  
    optimum_model_path=trainer.checkpoint_callback.best_model_path
    last_epoch_filename = "last_epoch={}".format(trainer.current_epoch-1)
    last_epoch_model_path = "{}/{}.ckpt".format(ckpts_path,last_epoch_filename)
    trainer.save_checkpoint(last_epoch_model_path) 
    save_variable(
        batch_time_logger_tft.batch_times,
        os.path.join(
            ckpts_path,
            "batch_times.pkl"
        )
    )
else:
    matching_path_list=glob.glob(pathname=os.path.join(Path(ckpts_path),"best_epoch=*.ckpt"))
    print("Matching paths found:",matching_path_list)
    confirm=input("Type 'ok' if the path to the best model is the expected one: ")
    if confirm=="ok":
        optimum_model_path=matching_path_list[0]
    else:
        print("Not found the proper best model path.")
print("Path selected:",optimum_model_path)
building_name="Agriculture"
optimum_tft = TemporalFusionTransformer.load_from_checkpoint(optimum_model_path)
if do_predictions:
    predictions=optimum_tft.predict(test_dataloader, return_y=True, trainer_kwargs=dict(accelerator="gpu" if torch.cuda.is_available() else "cpu"))
    save_variable(predictions, os.path.join(ckpts_path,"predictions_{}.pkl".format(building_name)))
    show_results(predictions, "{} building".format(building_name),saving_path=ckpts_path)
else:
    try:
        predictions=load_variable(os.path.join(ckpts_path,"predictions_{}.pkl".format(building_name)))
        show_results(predictions, "{} building".format(building_name),saving_path=ckpts_path)
    except:
        pass
if "batch_times.pkl" in os.listdir(ckpts_path):
    batch_times=pd.Series(load_variable(os.path.join(ckpts_path,"batch_times.pkl")),name="Elapsed times per training batch [s]")
    display(batch_times.describe())
    batch_per_training_epoch=(len(batch_times)//20)
    epoch_times=[]
    for i in range(20):
        epoch_times.append(
        batch_times.iloc[
            i*batch_per_training_epoch:(i+1)*batch_per_training_epoch
            ].sum()
        )
    epoch_times=pd.Series(epoch_times[1:],name="Elapsed times per training epoch [s]")
    display(epoch_times.describe())
    print("Total training time: {} s".format(batch_times.sum()))

In [ ]:
from tensorboard.backend.event_processing import event_accumulator
for nitem,item in enumerate(os.listdir("./Results/TFT/Optimization/{}/Twists_on_best/Statistical_significance".format(study_name))):
  ea=event_accumulator.EventAccumulator("./lightning_logs/Best_in_HyperbandPruner_stats_{}".format(item))
  ea.Reload()
  loss=pd.DataFrame(ea.Scalars("train_loss_epoch"))["value"]
  val_loss=pd.DataFrame(ea.Scalars("val_loss"))["value"]
  if not nitem:
      stat_loss=np.array(loss).reshape(1,-1)
      stat_val_loss=np.array(val_loss).reshape(1,-1)
  else:
      stat_loss=np.concat((stat_loss,np.array(loss).reshape(1,-1)),axis=0)
      stat_val_loss=np.concat((stat_val_loss,np.array(val_loss).reshape(1,-1)),axis=0)
display(stat_loss)
stat_final_loss={
  "min":stat_loss[:,-1].min().item(),
  "max":stat_loss[:,-1].max().item(),
  "mean":stat_loss[:,-1].mean().item(),
  "std":stat_loss[:,-1].std().item(),
}
display(stat_final_loss)
ave_loss=stat_loss.mean(axis=0)
ave_val_loss=stat_val_loss.mean(axis=0)
fig=plt.figure(
  figsize=(5,4), 
  dpi=300
  )
with plt.rc_context({"font.size":12}):
  plt.plot(np.arange(len(ave_loss)),ave_loss,linewidth=1.5,label="Training")
  plt.plot(np.arange(len(ave_val_loss)),ave_val_loss,linewidth=1.5,label="Validation")
  plt.fill_between(
    np.arange(len(ave_loss)),
    ave_loss-stat_loss.std(axis=0),
    ave_loss+stat_loss.std(axis=0),
    label="",
    fc="tab:blue",
    step=None,
    alpha=0.3
    )
  plt.fill_between(
    np.arange(len(ave_val_loss)),
    ave_val_loss-stat_val_loss.std(axis=0),
    ave_val_loss+stat_val_loss.std(axis=0),
    label="",
    fc="tab:orange",
    step=None,
    alpha=0.3
    )
  plt.ylabel("Average epoch quantile loss [-]")
  plt.xlabel('Epoch number [-]')
  plt.xlim(0,20)
  plt.legend(loc="upper right")
  plt.grid()
  plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True,nbins=10))
  plt.close("all")
  display(fig)

# TFT interpretation

## Functions

In [ ]:
def get_data(
        path_to_trial_folder:str|Path,
        what_to_do:Literal["save","load"]|None,
        building_name:str|None=None,
        custom_data:dict[Literal["train",
                                 "val",
                                 "test"],dict[str,pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader]]|None=None,
        )->dict[Literal["train","val","test"],
                dict[str,
                     pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]|pd.Series]]:
    path_to_data=os.path.join(path_to_trial_folder,"data_{}.pkl".format(building_name))    
    if what_to_do=="load":
        data=load_variable(path_to_data)
        return data
    data={
        "train":dict(
            df=train_df_tft,
            starting_date=pd.Timestamp(
                2019,7,1,0,0,0
            ),
            dataloader=train_dataloader
        ),
        "val":dict(
            df=val_df_tft,
            starting_date="to be defined",
            dataloader=val_dataloader
        ),
        "test":dict(
            df=df_test_x_tft,
            starting_date=pd.Timestamp(
                2022,7,1,0,0,0
            ),
            dataloader=test_dataloader
        ),
    }
    if data["val"]["starting_date"]=="to be defined":
        ending_date_train=data["train"]["starting_date"]+(data["train"]["df"].shape[0]-1)*pd.Timedelta("1h")
        data["val"]["starting_date"]=ending_date_train+pd.Timedelta("1h")
    if custom_data is not None:
        for which in data.keys():
            if which in custom_data.keys():
                data[which].update(custom_data[which])
    for which in data:
        data[which]["dataloader_list"]=list(data[which]["dataloader"])
    if what_to_do=="save":
        save_variable(
            data,
            path_to_data
        )
        return data
    if what_to_do is None:
        return data
    

In [ ]:
def item_in_batch_to_dates(
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]|pd.Series]],
        which:Literal["train","val","test"],
        item_idx:int,
        batch_idx:int,
        lookback_window_steps:int=n_past,
        lookforward_window_steps:int=n_future,
        consecutive_time_steps:bool=True,
        )->pd.DatetimeIndex:
    
    if consecutive_time_steps:
        time_dummy_to_date=pd.Series(
            pd.date_range(
                data[which]["starting_date"],freq="1h",
                periods=data[which]["df"].shape[0]
            ),
            index=data[which]["df"]["time_idx"]
        )
    else:
        msg="When consecutive_time_steps=False, you must provide custom_data in get_data(),\n\
            ensuring to have 'time_dummy_to_date' (key) conversion Series(value) provided\n\
            for which={} dataset.".format(which)
        assert "time_dummy_to_date" in data[which].keys(),msg
        time_dummy_to_date=copy.deepcopy(data[which]["time_dummy_to_date"])
    if batch_idx>=len(data[which]["dataloader"]):
        raise Exception("batch_idx={} exceeds the number of batches in the dataloader.\nIt must be within [0,...,{}].".format(
            batch_idx,len(data[which]["dataloader"])-1
            )
        )
    dict_x,(_,_)=data[which]["dataloader_list"][batch_idx]
    time_idxs_each_sequence_in_batch_decoder=(dict_x["decoder_time_idx"])
    starting_time_idxs_each_sequence_in_batch_decoder=time_idxs_each_sequence_in_batch_decoder[:,0]
    starting_time_idxs_each_sequence_in_batch=starting_time_idxs_each_sequence_in_batch_decoder-lookback_window_steps 
    batch_size=dict_x["decoder_time_idx"].shape[0]
    if item_idx>=batch_size:
        raise Exception("item_idx={} exceeds the number of sequences in batch_idx={}.\nIt must be within [0,...,{}].".format(
            item_idx,batch_idx,batch_size-1
            )
        )
    items_in_batch_to_dates={
        item:time_dummy_to_date.loc[
            list(
                range(
                    starting_time_idxs_each_sequence_in_batch[item].item(),
                    starting_time_idxs_each_sequence_in_batch[item].item()+lookback_window_steps+lookforward_window_steps
                )
            )
        ]
        for item in range(batch_size)
    }
    return items_in_batch_to_dates[item_idx]

In [ ]:
from typing import NamedTuple, Literal, Tuple
def get_model_info(
        path_to_model_ckpt:str|Path,
        print_info:bool=True,
        )->tuple[TemporalFusionTransformer,dict[str,list[str]]]:
    
    tft=TemporalFusionTransformer.load_from_checkpoint(path_to_model_ckpt)
    static_covariates_names=tft.hparams["static_categoricals"]+tft.hparams["static_reals"]
    encoder_variables_names=tft.hparams["time_varying_categoricals_encoder"]+tft.hparams["time_varying_reals_encoder"]
    decoder_variables_names=tft.hparams["time_varying_categoricals_decoder"]+tft.hparams["time_varying_reals_decoder"]
    feature_names_per_channel={
        "static_variables":static_covariates_names,
        "encoder_variables":encoder_variables_names,
        "decoder_variables":decoder_variables_names
    }
    if print_info:
        print("----Name of the features in each data channel----")
        for key,value in feature_names_per_channel.items():
            print(key+":",value)
        print("\n")
    return tft,feature_names_per_channel

In [ ]:
def handle_as_from_predict_raw(
        path_to_folder:str|Path,
        filename:str,
        what_to_do:Literal["save","load"],
        as_from_predict_raw:NamedTuple=None,
        )->None|dict:
    path_to_file=os.path.join(path_to_folder,filename)
    if what_to_do=="load":
        as_from_predict_raw_unpacked=load_variable(path_to_file)
        return as_from_predict_raw_unpacked
    field_names=as_from_predict_raw._fields
    as_from_predict_raw_unpacked={}
    if "output" not in field_names:
        for field_name in field_names:
            as_from_predict_raw_unpacked[field_name]=getattr(as_from_predict_raw,field_name)
    else:
        Output=getattr(as_from_predict_raw,"output")
        Output_unpacked={}
        for field_name in Output._fields:
            Output_unpacked[field_name]=getattr(Output,field_name)
        as_from_predict_raw_unpacked["output"]=Output_unpacked
        remaining_field_names=[field_name for field_name in field_names if field_name!="output"]
        for field_name in remaining_field_names:
            as_from_predict_raw_unpacked[field_name]=getattr(as_from_predict_raw,field_name)
    if what_to_do=="save":
        save_variable(as_from_predict_raw_unpacked,path_to_file)

def do_raw_predictions(
        tft:TemporalFusionTransformer,
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],        
        predict_kwargs:dict|None=None,
        print_info:bool=True,
        save_as_from_predict_raw:bool=False,
        path_to_trial_folder:str|Path|None=None,
        building_name:str|None=None,
        )->NamedTuple:
    dataloader=data[which]["dataloader"]
    if predict_kwargs is None:
        predict_kwargs=dict(
            return_x=True,
            return_index=True,
            return_decoder_lengths=True,
            return_y=True,
            trainer_kwargs=dict(
                accelerator="gpu" if torch.cuda.is_available() else "cpu",
                logger=False
            ),
        )
    try:
        as_from_predict_raw=tft.predict(dataloader,mode="raw",**predict_kwargs)
    except RuntimeError as e:
        if print_info:
            print("An error occurred.\nWhen using TFT.predict(DataLoader,...,return_y={}) (or similar return_something arguments set to\nTrue) on a DataLoader whose constructor's argument 'drop_last' is {} (in our case, by setting\ntrain=False in TimeSeriesDataSet.to_dataloader()'s arguments), if the total number of sequences\nin the TimeSeriesDataSet is not an exact multiple of DataLoader.batch_size={} then the last\nbatch in the DataLoader (which is not dropped) has a lower number of sequences in it compared\nto the batch_size, leading predict() to not being able to concatenate y tensors across their\nfirst dimension and to output the following error:".format(
                predict_kwargs["return_y"],
                dataloader.drop_last,
                dataloader.batch_size)
            )
            print(e)
            print("Because of this, in order to allow the prediction on the chosen DataLoader, all the\n'return_something' arguments in predict() will be automatically re-assigned to False.")
            print("Re-assignment in progress...")
        predict_kwargs=dict(
            return_x=False,
            return_index=False,
            return_decoder_lengths=False,
            return_y=False,
            trainer_kwargs=dict(
                accelerator="gpu" if torch.cuda.is_available() else "cpu",
                logger=False
            ),
        )
        if print_info:
            print("Re-assignment completed. Since all 'return_something' arguments are now set to False,\npredict(DataLoader,mode='raw',...) will return only the 'output' field name (Output namedtuple\ninstance) of the whole Prediction namedtuple normally returned when at least one of the\n'return_something' arguments is True.")
        as_from_predict_raw=tft.predict(dataloader,mode="raw",**predict_kwargs)
    if save_as_from_predict_raw:
        handle_as_from_predict_raw(
            path_to_trial_folder,
            "as_from_predict_raw_unpacked_{}.pkl".format(building_name),
            "save",as_from_predict_raw
        )
        return as_from_predict_raw
    else:
        return as_from_predict_raw      

### Variable selection weights

In [ ]:
from typing import Union
def get_VSWs(
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],        
        feature_names_per_channel:dict[str,list[str]],
        channel_name:Literal["static_variables","encoder_variables","decoder_variables"],
        batch_idx:int|None=None,
        item_idx:int|None=None,
        reduction:Literal["average","quantiles"]="average",
        quantile:float=None,
        print_info:bool=True,
        additional_title:str="",
        figsize:tuple[int,int]=(16,8),
        return_figure:bool=False,
        return_Series_VSWs:bool=False,
        fontsize:int=10,
        avoid_callout:bool=False,
        overlap_cumulative:bool|None=None,
        def_ofs_x:float=6,
        def_ofs_y:float=0,
        offsets:dict[str,dict[Literal["x","y"],float]]={},
        use_only_non_overlapping_sequences:bool=False,
        markerprops_update:dict[Literal["marker","markersize","markerfacecolor","markeredgecolor"],Any]|None=None,
        target_label:str="Target",
        no_title:bool=False,
        barh_settings_update:dict[
            Literal["bar_px","top_px","bottom_px","bar_height_data","width_in","dpi"],Any
        ]|None=None,
        pad_spaces:int|None=None,
        legend:bool=True,
        positional_index:int|None=None,
        square_brackets:bool=False,
        formatter:bool=True,
        force_xlabel:str|None=None
        )->None|plt.Figure|pd.Series|tuple[plt.Figure,pd.Series]:
    barh_settings=dict(
        bar_px=14,   
        top_px=14,  
        bottom_px=14,
        bar_height_data=0.8,
        width_in=6,
        dpi=100
        )
    if barh_settings_update is not None:
        barh_settings.update(barh_settings_update)
    def plot_barh_fixed_px_margins(
            names, values, barh_settings
            ):
        n = len(names)
        total_px = n * barh_settings["bar_px"] + barh_settings["top_px"] + barh_settings["bottom_px"]
        height_in = total_px / barh_settings["dpi"]

        fig, ax = plt.subplots(figsize=(barh_settings["width_in"], height_in), dpi=barh_settings["dpi"])

        y = np.arange(n)
        ax.barh(y, values, height=barh_settings["bar_height_data"], align='center')

        ax.set_yticks(y)
        ax.set_yticklabels(names)

        ax.set_ylim(-0.5, n - 0.5)

        bottom_frac = barh_settings["bottom_px"] / total_px
        top_frac = 1.0 - (barh_settings["top_px"] / total_px)
        fig.subplots_adjust(bottom=bottom_frac, top=top_frac)

        ax.margins(y=0)
        return fig,ax
    markerprops=dict(marker=".",markersize=12,markerfacecolor="red",markeredgecolor="k")
    if markerprops_update is not None:
        markerprops.update(markerprops_update)
    try:
        try:
            Output=getattr(as_from_predict_raw,"output") 
        except AttributeError:
            Output=as_from_predict_raw["output"]
    except AttributeError:
        Output=as_from_predict_raw
    try:
        dc_VSWs_tensor=getattr(Output,channel_name).cpu() 
    except AttributeError:
        dc_VSWs_tensor=Output[channel_name].cpu()
    dc_VSWs_array=np.array(dc_VSWs_tensor)
    if positional_index is not None:
        try:
            assert dc_VSWs_array.shape[1]==n_past
            positional_index_to_slice_idx=pd.Series(list(range(0,n_past)),index=range(-n_past+1,1))
        except AssertionError:
            assert dc_VSWs_array.shape[1]==n_future, "The channel is neither the past inputs nor the future inputs, positional index is not expected"
            positional_index_to_slice_idx=pd.Series(list(range(0,n_future)),index=range(1,n_future+1))
        tst_idx=positional_index_to_slice_idx.loc[positional_index]
        dc_VSWs_array=dc_VSWs_array[:,slice(tst_idx,tst_idx+1),:,:]

    dc_VSWs_per_feature={
        feature_name:dc_VSWs_array[:,:,:,feature_idx] for feature_idx,feature_name in enumerate(
            feature_names_per_channel[channel_name]
        )
    }
    dataloader=data[which]["dataloader"]
    if use_only_non_overlapping_sequences:
        nos_idxs=[]
        assert all([batch_idx is None, item_idx is None]),"You can't select the batch and item when using only the non-overlapping sequences."
        already_seen_decoder_time_idxs=torch.tensor([])
        ovs_idx=0
        for x_dict,_ in data[which]["dataloader_list"]:
            decoder_target=x_dict["decoder_target"]
            batch_size=decoder_target.shape[0]         
            for iti in range(batch_size):
                decoder_time_idxs=x_dict["decoder_time_idx"][iti,:]
                if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                    nos_idxs.append(ovs_idx)
                    already_seen_decoder_time_idxs=torch.cat(
                        (already_seen_decoder_time_idxs,decoder_time_idxs)
                    )
                ovs_idx+=1
        dc_VSWs_per_feature={
            feature_name:dc_VSWs_array[nos_idxs,:,:,feature_idx] for feature_idx,feature_name in enumerate(
                feature_names_per_channel[channel_name]
            )
        }
    if batch_idx is not None:
        batch_size=dataloader.batch_size
        number_of_sequences=dc_VSWs_array.shape[0]
        drop_last=dataloader.drop_last
        number_of_batches=math.floor(number_of_sequences/batch_size) if (drop_last and number_of_sequences%batch_size!=0) else math.ceil(number_of_sequences/batch_size)
        
        batch_to_seqs_slice={
            batch_num:slice(batch_num*batch_size,(batch_num+1)*batch_size) if batch_num!=(number_of_batches-1) else slice(batch_num*batch_size,None) for batch_num in range(number_of_batches)
        }
        dc_VSWs_per_feature={
            feature_name:dc_VSWs_array[batch_to_seqs_slice[batch_idx],:,:,feature_idx] for feature_idx,feature_name in enumerate(
                feature_names_per_channel[channel_name]
            )
        }
        if item_idx is not None:
            if item_idx>=batch_size:
                raise Exception("When you specify the index of the selected sequence (item={}) in the selected batch, it must be in [0,...,{}].".format(
                    item_idx,batch_size-1
                    )
                )
            item_slice=slice(item_idx,item_idx+1)
            dc_VSWs_per_feature={
                feature_name:feature_array[item_slice,:,:] for feature_name,feature_array in dc_VSWs_per_feature.items()
            }

    dc_VSWs_per_feature={
        feature_name:feature_array.reshape(-1) for feature_name,feature_array in dc_VSWs_per_feature.items()
    }
    dc_VSWs_per_feature={
        feature_name:feature_array.mean() if reduction=="average" else np.quantile(feature_array,quantile) for feature_name,feature_array in dc_VSWs_per_feature.items()
    }
    if print_info:
        print("----{}---- \nOriginal VSWs tensor shape:".format(channel_name),dc_VSWs_tensor.shape)
        print("Single feature VSWs ndarray shape:",dc_VSWs_array[:,:,:,0].shape)
        print("Single feature averaged VSW:" if reduction=="average" else "Single feature percentile VSW's P{:.0f}:".format(quantile*100))
        print(dc_VSWs_per_feature)
        print("\n") 
    with plt.rc_context({'font.size': fontsize}):
        VSWs_to_plot=np.array([weight for _,weight in dc_VSWs_per_feature.items()])*(100 if reduction=="average" else 1)
        VSWs_to_plot=pd.Series(VSWs_to_plot,index=feature_names_per_channel[channel_name]).sort_values()

        fig,ax=plot_barh_fixed_px_margins(
            names=VSWs_to_plot.index,
            values=VSWs_to_plot,
            barh_settings=barh_settings
        )
        if overlap_cumulative is None:
            overlap_cumulative=False if reduction!="average" else True
        if print_info:
            print("Features:",VSWs_to_plot.index,flush=True)
        if overlap_cumulative:
            assert reduction=="average","Percentiles don't necessarily add up to 100%, so it does not make sense to overlap\ntheir cumulative sum curve."
            ax.plot(
                np.cumsum(VSWs_to_plot),VSWs_to_plot.index,color="red",
                marker=markerprops["marker"],linestyle='-',
                markersize=markerprops["markersize"],
                markerfacecolor=markerprops["markerfacecolor"],
                markeredgecolor=markerprops["markeredgecolor"],
                label="Cumulative importance"
            )
            if legend:
                ax.legend(
                    loc="lower center",
                    bbox_to_anchor=(0.5,1)
                )
        if not avoid_callout:
            to_callout=copy.deepcopy(VSWs_to_plot) if not overlap_cumulative else np.cumsum(copy.deepcopy(VSWs_to_plot))
            all_offsets={ftn:{"x":def_ofs_x,"y":def_ofs_y} for ftn in to_callout.index}
            for ftn,ft_ofs in all_offsets.items():
                if ftn in offsets.keys():
                    ft_ofs.update(offsets[ftn])
            ax.plot(
                to_callout,to_callout.index,
                linestyle="none",
                marker=markerprops["marker"],
                markersize=markerprops["markersize"],
                markerfacecolor=markerprops["markerfacecolor"],
                markeredgecolor=markerprops["markeredgecolor"]
            )
            for feature_name in to_callout.index:
                offset_x=all_offsets[feature_name]["x"]
                offset_y=all_offsets[feature_name]["y"]
                ax.annotate(
                    "{:.2f}%".format(to_callout.loc[feature_name]) if reduction=="average" else "{:.2f}".format(to_callout.loc[feature_name]),
                    xy=(to_callout.loc[feature_name],feature_name),
                    xytext=(offset_x,offset_y),
                    color="red",
                    textcoords="offset points",
                    size="medium",
                    va="center",
                )
        if not no_title:
            if use_only_non_overlapping_sequences:
                title="{}' importance, {} over all non-overlapping sequences. ".format(
                    channel_name.replace("_"," ").capitalize(),
                    reduction if reduction=="average" else "percentile"
                )
            else:
                if batch_idx is None:
                    title="{}' importance, {} over all batches. ".format(
                        channel_name.replace("_"," ").capitalize(),
                        reduction if reduction=="average" else "percentile"
                    )
                else:
                    if item_idx is None:
                        title="{}' importance, {} at batch={}. ".format(
                            channel_name.replace("_"," ").capitalize(),
                            reduction if reduction=="average" else "percentile",
                            batch_idx
                        )
                    else:
                        title="{}' importance, {} at batch={} in its sequence={}. ".format(
                            channel_name.replace("_"," ").capitalize(),
                            reduction if reduction=="average" else "percentile",
                            batch_idx,
                            item_idx
                        )
            ax.set_title(title+"{}".format("n={}. ".format(positional_index) if positional_index is not None else "")+additional_title)
        if reduction=="average":
            if overlap_cumulative:
                ax.set_xlim(right=100)
            if formatter:
                ax.xaxis.set_major_formatter("{x}%")
        if reduction=="average":
            if formatter:
                xlbl="Importance"
            else:
                xlbl="Importance [%]"
        else:
            xlbl="P{:.0f}".format(quantile*100)
        if force_xlabel is None:
            ax.set_xlabel(xlbl)
        else:
            ax.set_xlabel(force_xlabel)
        yticks=ax.get_yticks()
        yticks_labels=ax.get_yticklabels()
        yticks_labels=[label.get_text() for label in yticks_labels]
        yticks_labels=[og_to_new_labels[label] if label!="target" else target_label for label in yticks_labels]
        if pad_spaces is not None:
            yticks_labels=["\u200A"*pad_spaces+label for label in yticks_labels]
        if square_brackets:
            yticks_labels=[lbl.replace("(","[").replace(")","]") for lbl in yticks_labels]
        ax.set_yticks(ticks=yticks,labels=yticks_labels)
        plt.close("all")
        if not return_figure:
            display(fig)
            if return_Series_VSWs:
                return VSWs_to_plot
        else:
            if not return_Series_VSWs:
                return fig
            else:
                return fig,VSWs_to_plot

### Attention

In [ ]:
def get_attention(
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],
        reduction:Literal["mean","sum"],
        normalize_attention_by_position_idxs_sum:bool,
        batch_idx:int|None=None,
        item_idx:int|None=None,
        set_attention_threshold:bool=False,
        custom_ticks:bool=True,
        plotting_time_interval:int=24,
        print_info:bool=True,
        head_idx:int|None=None,
        prediction_idx:int|None=None,
        figsize:tuple[int,int]=(16,8),
        linewidth:float=1.5,
        detail_representatives:bool=True,
        representative_quantiles:list[float]=[0.1,0.5,0.9],
        detail_prediction_timesteps:bool=False,
        prediction_idxs_to_plot:list[int]|None=None,
        add_pre_suptitle:str|None=None,
        return_attention:bool=False,
        return_figure:bool=False,
        return_dist:bool=False,
        use_dates:bool=False,
        reduce_across_homogeneous_seqs:bool=False,
        hour_of_preds_start_homogeneous_seqs:int|None=None,
        day_name:Literal["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]|None=None,
        zoom_from_to_idxs:tuple[bool,int,int]=(False,-n_past+1,n_future),
        separator_update:dict[Literal["enc","dec","x_offset","txt_size",
                                      "sep_color","sep_style","sep_linewidth",
                                      "box_y_fraction","box_fill_color","box_edge_color"],
                                      bool|float|str]={},
        ylims:tuple[float]|None=None,
        fontsize:float=11,
        forced_xlabel:str|None=None,
        forced_ylabel:str|None=None,
        only_legend:bool=False,
        scientific_on_y:bool=False,
        colormap:matplotlib.colors.Colormap|str|None="coolwarm",
        distinctipy_ops:dict[Literal["pastel_factor","colorblind_type","exclude_colors"],Any]|None=None,
        exclude_mean:bool=False,
        masking:Literal["immediate_past","up_to"]="immediate_past",
        dpi:int|None=None,
        for_graphical_abstract:bool=False,
        forced_rotation:float|None=None,
        pad:dict[Literal["x","y"],int]|None=None
        )->None|dict[int,dict[str,torch.Tensor]]|plt.Figure|torch.Tensor|tuple[plt.Figure,dict[int,dict[str,torch.Tensor]],torch.Tensor]:
    scientific_formatter=ScalarFormatter(useMathText=True)
    scientific_formatter.set_scientific(True)
    scientific_formatter.set_powerlimits((0,0))
    separator={
        "enc":False,"dec":False,"x_offset":8,"txt_size":11,"sep_color":"k",
        "sep_style":"-.","sep_linewidth":1,"box_y_fraction":0.96,
        "box_fill_color":"white","box_edge_color":"k"
    }
    separator.update(separator_update)
    def plot_attention(prediction_index,positional_indices,
                       representative_attentions,ax,color,
                       linewidth,detail_representatives:bool,
                       detail_prediction_timesteps:bool,
                       quantile:float|None=None)->None|str:
        if detail_prediction_timesteps:
            label=r"$\tau$"+"={}".format(prediction_index)
        if detail_representatives:
            label="P{:.0f}".format(quantile*100) if quantile is not None else "Mean"
        ax.plot(positional_indices,representative_attentions,color=color,
                linewidth=linewidth,label=label)      
    try:
        try:
            Output=getattr(as_from_predict_raw,"output")
        except:
            Output=as_from_predict_raw["output"]
    except AttributeError:
        Output=as_from_predict_raw
    who_pays_attention="Decoder"
    who_to_pay_attention_to=["encoder","decoder"]
    original_attention_to={}
    for who in who_to_pay_attention_to:
        try:
            tensor=getattr(Output,who+"_attention").cpu()
        except:
            tensor=Output[who+"_attention"].cpu()
        number_of_time_steps=tensor.shape[3]
        if who=="encoder":
            time_idxs=np.arange(-(number_of_time_steps-1),0+1)
        else:
            time_idxs=np.arange(1,number_of_time_steps+1)
        original_attention_to[who]=dict(
            tensor=tensor,
            time_idxs=pd.Series(time_idxs),    
            )
    if print_info:
        print("Original attention shapes:")
        print("To encoder: ",original_attention_to["encoder"]["tensor"].shape)
        print("To decoder: ",original_attention_to["decoder"]["tensor"].shape)
    number_of_attention_heads=original_attention_to["encoder"]["tensor"].shape[2]
    if head_idx is not None and head_idx>=number_of_attention_heads:
        raise Exception("The selected head's index must be an integer within set([0,...,{}])".format(number_of_attention_heads-1))
    if prediction_idx is not None and prediction_idx>(original_attention_to["decoder"]["time_idxs"]).max():
        raise Exception("The selected prediction time step's index must be an integer within set([1,...,{}])".format(max(original_attention_to["decoder"]["time_idxs"])))
    mathematical_prediction_idx=(original_attention_to["decoder"]["time_idxs"]==prediction_idx).idxmax()
    prediction_slice=slice(mathematical_prediction_idx,mathematical_prediction_idx+1) if prediction_idx is not None else slice(None)
    head_slice=slice(head_idx,head_idx+1) if head_idx is not None else slice(None)
    sliced_attention_to={}

    if not reduce_across_homogeneous_seqs:
        if batch_idx is not None:
            dataloader=data[which]["dataloader"]
            batch_size=dataloader.batch_size
            number_of_batches=len(dataloader)
            batch_to_seqs_slice={
                batch_num:slice(batch_num*batch_size,(batch_num+1)*batch_size) if batch_num!=(number_of_batches-1) else slice(batch_num*batch_size,None) for batch_num in range(number_of_batches)
            }
            original_attention_to["encoder"]["tensor"]=original_attention_to["encoder"]["tensor"][batch_to_seqs_slice[batch_idx],:,:,:]
            original_attention_to["decoder"]["tensor"]=original_attention_to["decoder"]["tensor"][batch_to_seqs_slice[batch_idx],:,:,:]

            if item_idx is not None:
                if item_idx>=batch_size:
                    raise Exception("When you specify the index of the selected sequence (item={}) in the selected batch, it must be in [0,...,{}].".format(
                        item_idx,batch_size-1
                        )
                    )
                item_slice=slice(item_idx,item_idx+1)
            else:
                item_slice=slice(0,item_idx)
            original_attention_to["encoder"]["tensor"]=original_attention_to["encoder"]["tensor"][item_slice,:,:,:]
            original_attention_to["decoder"]["tensor"]=original_attention_to["decoder"]["tensor"][item_slice,:,:,:]
    else:
        try:        
            assert "time_dummy_to_date" in data[which].keys()
            time_dummy_to_date=copy.deepcopy(data[which]["time_dummy_to_date"])
        except AssertionError:
            time_dummy_to_date=pd.Series(
                pd.date_range(
                    data[which]["starting_date"],freq="1h",
                    periods=data[which]["df"].shape[0]
                ),
                index=data[which]["df"]["time_idx"]
            )        
        overall_seqs_idx=0
        homogeneous_seqs_idxs={hour_of_preds_start:[] for hour_of_preds_start in range(0,n_future)}
        if day_name is not None:
            assert reduce_across_homogeneous_seqs,"Day selection makes sense only when you reduce across homogeneous sequences."
        for (x_dict,(_,_)) in data[which]["dataloader_list"]:
            local_batch_size=x_dict["decoder_time_idx"].shape[0]
            for local_item_idx in range(local_batch_size):
                decoder_time_dummies=x_dict["decoder_time_idx"][local_item_idx,:]
                first_decoder_time_dummy=decoder_time_dummies[0].item()
                first_prediction_hour=time_dummy_to_date.loc[first_decoder_time_dummy].hour
                if day_name is None:
                    homogeneous_seqs_idxs[first_prediction_hour].append(overall_seqs_idx)
                else:
                    if time_dummy_to_date.loc[first_decoder_time_dummy].day_name()==day_name:
                        homogeneous_seqs_idxs[first_prediction_hour].append(overall_seqs_idx)
                overall_seqs_idx+=1
        if hour_of_preds_start_homogeneous_seqs is None:
            raise Exception("When you pass reduce_across_homogeneous_seqs=True,\nyou must also pass a valid value for hour_of_preds_start_homogeneous_seqs,\nwhich must be in[0,...,{}]".format(n_future-1))
        selected_homogeneous_seqs_idxs=homogeneous_seqs_idxs[hour_of_preds_start_homogeneous_seqs]
        original_attention_to["encoder"]["tensor"]=original_attention_to["encoder"]["tensor"][selected_homogeneous_seqs_idxs,:,:,:]
        original_attention_to["decoder"]["tensor"]=original_attention_to["decoder"]["tensor"][selected_homogeneous_seqs_idxs,:,:,:]        

    for who in original_attention_to:
        sliced_attention_to[who]=dict(
            tensor=original_attention_to[who]["tensor"][:,prediction_slice,head_slice,:],
            time_idxs=original_attention_to[who]["time_idxs"]
        )
    concatenated_attention_to={
        "tensor":torch.cat(
            (sliced_attention_to["encoder"]["tensor"],sliced_attention_to["decoder"]["tensor"]),
            dim=-1
        ),
        "time_idxs":pd.Series(torch.cat(
            (torch.from_numpy(np.array(sliced_attention_to["encoder"]["time_idxs"])),
             torch.from_numpy(np.array(sliced_attention_to["decoder"]["time_idxs"]))),
            dim=-1
            ),
            index=np.arange(
                0,
                len(sliced_attention_to["encoder"]["time_idxs"])+len(sliced_attention_to["decoder"]["time_idxs"])
            )
        )
    }
    if set_attention_threshold:
        attention_threshold=1e-5
        concatenated_attention_to["tensor"][concatenated_attention_to["tensor"]<attention_threshold]=float("nan")
    representative_attention_dict={}
    prediction_idxs_to_iterate_over=[prediction_idx] if prediction_idx is not None else sliced_attention_to["decoder"]["time_idxs"]
    dist=0
    if return_dist and any([batch_idx is not None,item_idx is not None,head_idx is not None,prediction_idx is not None]):
        print("Warning: at least one among batch_idx, item_idx, head_idx and prediction_idx\nis not None but you asked to return the dist(t) vector, which is originally implemented\nin the TFT paper considering all heads, batches, items and prediciton indices.")
    for prediction_index in prediction_idxs_to_iterate_over:
        if prediction_idx is None:
            prediction_math_index=(sliced_attention_to["decoder"]["time_idxs"]==prediction_index).idxmax()
        else:
            prediction_math_index=0
        if masking=="immediate_past":
            masking_slice=slice(0,(concatenated_attention_to["time_idxs"]==prediction_index).idxmax())
        else:
            masking_slice=slice(0,(concatenated_attention_to["time_idxs"]==prediction_index).idxmax()+1)
        prediction_slice=slice(prediction_math_index,prediction_math_index+1)
        prediction_attention_to=dict(
            tensor=concatenated_attention_to["tensor"][:,prediction_slice,:,masking_slice],
            time_idxs=concatenated_attention_to["time_idxs"].loc[concatenated_attention_to["time_idxs"]<prediction_index if masking=="immediate_past" else concatenated_attention_to["time_idxs"]<=prediction_index]
        )
        prediction_attention_to["mean_across_heads"]=torch.nanmean(
            prediction_attention_to["tensor"],dim=2
        )
        attention_to_single_timestep={}
        attention_quantiles_single_timestep={}      
        for idx in range(prediction_attention_to["mean_across_heads"].shape[2]):
            slice_previous_timestep=slice(idx,idx+1)
            logical_idx=prediction_attention_to["time_idxs"].loc[idx]
            attention_to_single_timestep[logical_idx]=prediction_attention_to["mean_across_heads"][:,:,slice_previous_timestep]
            quantiles=[]
            for quantile in representative_quantiles:
                quantiles.append(torch.nanquantile(
                    attention_to_single_timestep[logical_idx],
                    q=quantile,
                    )
                )
            attention_quantiles_single_timestep[logical_idx]=torch.tensor(quantiles).reshape((1,len(quantiles)))
        prediction_attention_to["quantiles"]=torch.cat(
            [attention_quantiles_single_timestep[logical_idx] 
             for logical_idx in attention_quantiles_single_timestep],
             dim=0
        ) 
        prediction_attention_to["mean_across_batches"]=torch.nanmean(
            prediction_attention_to["mean_across_heads"],
            dim=0
        ).squeeze()
        prediction_attention_to["sum_across_batches"]=torch.nansum(
            prediction_attention_to["mean_across_heads"],
            dim=0
        ).squeeze()
        if return_dist:
            pjqj=prediction_attention_to["mean_across_batches"].cpu()*prediction_attention_to["mean_across_heads"].cpu().squeeze()
            square_root_pjqj=torch.sqrt(pjqj)
            Bhattacharya_coeff=torch.nansum(square_root_pjqj,dim=1)
            Comaniciu_dist=torch.sqrt(1-Bhattacharya_coeff)
            dist+=Comaniciu_dist

        representative_attention_dict[prediction_index]=dict(
            representative=prediction_attention_to["mean_across_batches"] if reduction=="mean" else prediction_attention_to["sum_across_batches"],
            quantiles=prediction_attention_to["quantiles"],
            time_idxs=prediction_attention_to["time_idxs"]
        )
        if normalize_attention_by_position_idxs_sum:
            representative_attention_dict[prediction_index]["quantiles"]=representative_attention_dict[prediction_index]["quantiles"]/(torch.nansum(representative_attention_dict[prediction_index]["representative"]))
            representative_attention_dict[prediction_index]["representative"]=representative_attention_dict[prediction_index]["representative"]/(torch.nansum(representative_attention_dict[prediction_index]["representative"]))
        if set_attention_threshold:
            representative_attention_dict[prediction_index]["representative"]=torch.nan_to_num(representative_attention_dict[prediction_index]["representative"],nan=0)
            representative_attention_dict[prediction_index]["quantiles"]=torch.nan_to_num(representative_attention_dict[prediction_index]["quantiles"],nan=0)
    if return_dist:
        dist=dist/len(prediction_idxs_to_iterate_over)
    with plt.rc_context({"font.size":fontsize}):
        if detail_prediction_timesteps:
            y_label="Representative "+r"$\alpha(t,n,\tau)$"
            if prediction_idxs_to_plot is None: 
                if prediction_idx is not None:
                    prediction_idxs_to_plot=[prediction_idx]
                else:
                    prediction_idxs_to_plot=[5,10,15,20]
            fig=plt.figure(figsize=figsize,dpi=dpi)
            ax=fig.gca()
            if any([separator["enc"],separator["dec"]]):
                ofs_x_txt=separator["x_offset"]
                size_txt=separator["txt_size"]
                ax.axvline(
                    x=original_attention_to["encoder"]["time_idxs"].iloc[-1],
                    color=separator["sep_color"],
                    linestyle=separator["sep_style"],
                    linewidth=separator["sep_linewidth"]
                )
                if separator["enc"]:
                    ax.annotate(
                        text="Encoder",
                        xycoords=("data","axes fraction"),
                        xy=(
                            original_attention_to["encoder"]["time_idxs"].iloc[-1],
                            separator["box_y_fraction"]
                        ),
                        textcoords="offset points",
                        xytext=(-ofs_x_txt,0),
                        size=size_txt,
                        bbox=dict(
                            boxstyle="round,pad=0.3", 
                            fc=separator["box_fill_color"], 
                            ec=separator["box_edge_color"]
                        ),
                        ha="right"
                    )
                if separator["dec"]:
                    ax.annotate(
                        text="Decoder",
                        xycoords=("data","axes fraction"),
                        xy=(
                            original_attention_to["encoder"]["time_idxs"].iloc[-1],
                            separator["box_y_fraction"]
                        ),
                        textcoords="offset points",
                        xytext=(+ofs_x_txt,0),
                        size=size_txt,
                        bbox=dict(
                            boxstyle="round,pad=0.3", 
                            fc=separator["box_fill_color"], 
                            ec=separator["box_edge_color"]
                        ),
                        ha="left"
                    )
            if distinctipy_ops is None:
                cmap=plt.get_cmap(colormap)
                colors=[cmap(i) for i in np.linspace(0,1,len(prediction_idxs_to_plot))]
            else:
                colors=distinctipy.get_colors(len(prediction_idxs_to_plot),**distinctipy_ops)
                cmap=distinctipy.get_colormap(colors)
            max=0
            min=np.inf
            for iterator,prediction_idx_to_plot in enumerate(prediction_idxs_to_plot):
                if not zoom_from_to_idxs[0]:
                    if representative_attention_dict[prediction_idx_to_plot]["representative"].max()>max:
                        max=representative_attention_dict[prediction_idx_to_plot]["representative"].max()
                    if representative_attention_dict[prediction_idx_to_plot]["representative"].min()<min:
                        min=representative_attention_dict[prediction_idx_to_plot]["representative"].min()            
                else:
                    if not any(representative_attention_dict[prediction_idx_to_plot]["time_idxs"]==zoom_from_to_idxs[1]):
                        raise Exception("Lower positional index given in zoom_from_to_idxs falls out of limits of prediction index={}.\nIt must be within [{},...,{}], please provide a valid integer.".format(
                            prediction_idx_to_plot,
                            representative_attention_dict[prediction_idx_to_plot]["time_idxs"].min(),
                            representative_attention_dict[prediction_idx_to_plot]["time_idxs"].max()
                            )
                        )
                    if zoom_from_to_idxs[2]<=zoom_from_to_idxs[1]:
                        raise Exception("Upper positional index in zoom_from_to_idxs can't be lower or equal to the lower position index.")
                    local_mask=(representative_attention_dict[prediction_idx_to_plot]["time_idxs"].loc[(representative_attention_dict[prediction_idx_to_plot]["time_idxs"]>=zoom_from_to_idxs[1])*(representative_attention_dict[prediction_idx_to_plot]["time_idxs"]<=zoom_from_to_idxs[2])]).index           
                    local_max=representative_attention_dict[prediction_idx_to_plot]["representative"][local_mask].max()
                    local_min=representative_attention_dict[prediction_idx_to_plot]["representative"][local_mask].min()
                    if local_max>max:
                        max=local_max
                    if local_min<min:
                        min=local_min
                plot_attention(prediction_idx_to_plot,
                            representative_attention_dict[prediction_idx_to_plot]["time_idxs"],
                            representative_attention_dict[prediction_idx_to_plot]["representative"],
                            ax,colors[iterator],linewidth,
                            detail_representatives=False,
                            detail_prediction_timesteps=True)
            hom_seqs_one_tau_only=reduce_across_homogeneous_seqs and len(prediction_idxs_to_plot)==1
            if custom_ticks:
                if for_graphical_abstract:
                    print("plotting_time_interval automatically set to 24 because of for_graphical_abstract=True.")
                    plotting_time_interval=24
                ticks=np.arange(
                    concatenated_attention_to["time_idxs"].min(),
                    concatenated_attention_to["time_idxs"].max(),
                    plotting_time_interval
                )
                if not use_dates:
                    labels=[str(tick) for tick in ticks]
                    if hom_seqs_one_tau_only:
                        dummy_dates=pd.date_range(
                            end=pd.Timestamp(
                                year=2025,month=1,day=15,
                                hour=hour_of_preds_start_homogeneous_seqs,
                                minute=0
                            ),
                            freq="h",
                            periods=n_past+1
                        )
                        dummy_dates=dummy_dates.append(
                            pd.date_range(
                                start=dummy_dates[-1]+pd.Timedelta("1h"),
                                freq="h",
                                periods=n_future-1
                            )
                        )
                        if plotting_time_interval<24 and 24%plotting_time_interval==0:
                            labels=[]
                            for date_idx,date in enumerate(dummy_dates):
                                if date_idx%plotting_time_interval==0:
                                    labels.append(date.strftime("%H:%M"))
                        else:
                            labels=[
                                date.strftime("%H:%M") for date in dummy_dates[slice(0,None,plotting_time_interval)]
                            ]                             
                else:
                    try:
                        assert "time_dummy_to_date" in data[which].keys()
                        consecutive_time_steps=False
                    except AssertionError:
                        consecutive_time_steps=True
                    if day_name is None:
                        dates=item_in_batch_to_dates(
                            data=data,which=which,item_idx=item_idx,batch_idx=batch_idx,
                            consecutive_time_steps=consecutive_time_steps
                        )
                    else:
                        dm_st=pd.Timestamp(year=2025,month=4,day=28,hour=hour_of_preds_start_homogeneous_seqs,minute=0,second=0)
                        dm_sh={"Tuesday":1,"Wednesday":2,"Thursday":3,"Friday":4,"Saturday":5,"Sunday":6}
                        if day_name!="Monday":
                            dm_st+=pd.Timedelta("1d")*dm_sh[day_name]
                        dates=pd.date_range(dm_st,periods=n_past+n_future,freq="h")
                    if plotting_time_interval<24 and 24%plotting_time_interval==0:
                        labels=[]
                        for date_idx,date in enumerate(dates):
                            if date_idx%plotting_time_interval==0:
                                if date_idx%24==0:
                                    if day_name is None:
                                        labels.append(date.strftime("%a %Y/%m/%d %H:%M"))
                                    else:
                                        labels.append(date.strftime("%a %H:%M"))
                                else:
                                    labels.append(date.strftime("%H:%M"))
                    else:
                        if day_name is None:
                            labels=[
                                date.strftime("%a %Y/%m/%d %H:%M") for date in dates[slice(0,None,plotting_time_interval)]
                            ]
                        else:
                            if not for_graphical_abstract:
                                labels=[
                                    date.strftime("%a %H:%M") for date in dates[slice(0,None,plotting_time_interval)]
                                ]  
                            else:
                                labels=[
                                    date.strftime("%A") for date in dates[slice(0,None,plotting_time_interval)]
                                ] 
                if not use_dates and not hom_seqs_one_tau_only:
                    if plotting_time_interval>4:
                        rotation=0
                    else:
                        rotation=90
                else:
                    rotation=90
                ax.set_xticks(
                    ticks=ticks,
                    labels=labels,
                    rotation=rotation
                )
                if pad is not None:
                    ax.tick_params(axis='x', pad=pad["x"])   
                    ax.tick_params(axis='y', pad=pad["y"])  
                if forced_rotation is not None:
                    ax.set_xticks(
                        ticks=ax.get_xticks(),
                        labels=ax.get_xticklabels(),
                        rotation=forced_rotation
                    )
                if not for_graphical_abstract:
                    locator=MaxNLocator()
                else:
                    locator=MaxNLocator(nbins=5)
                ax.yaxis.set_major_locator(locator)
                if not zoom_from_to_idxs[0]:
                    ax.set_xlim((concatenated_attention_to["time_idxs"].min(),
                                concatenated_attention_to["time_idxs"].max())) 
                else:
                    ax.set_xlim((zoom_from_to_idxs[1],zoom_from_to_idxs[2]))
                if ylims is None:
                    ax.set_ylim((min*(1-0.05),max*(1+0.05)))
                else:
                    ax.set_ylim(ylims)
            if forced_xlabel is None:
                ax.set_xlabel("Position index (n)" if not use_dates and not hom_seqs_one_tau_only else "")
            else:
                ax.set_xlabel(forced_xlabel)
            if forced_ylabel is None:
                ax.set_ylabel(y_label)
            else:
                ax.set_ylabel(forced_ylabel)
            if scientific_on_y:
                ax.yaxis.set_major_formatter(scientific_formatter)
            ax.grid(visible=True)
            ax.legend(loc="lower center",ncols=len(prediction_idxs_to_plot),bbox_to_anchor=(0.5,1))
            if not reduce_across_homogeneous_seqs:
                title_for_reduction="summed across all sequences." if reduction=="sum" else "across all sequences."
            else:
                title_for_reduction="summed across homogeneous sequences." if reduction=="sum" else "across homogeneous sequences."
            title_for_threshold=" All weights strictly lower than {} have been ignored.".format(attention_threshold) if set_attention_threshold else ""
            if batch_idx is None:
                title_for_batch_and_item=" All batches considered."
            else:
                title_for_batch_and_item=" Sequence index={} in batch index={}.".format(item_idx,batch_idx) if item_idx is not None else " All sequences in batch index={}.".format(batch_idx)
            title_for_normalization= r" For each prediction index, each of its representative attention weights ($\alpha$) is normalized according"+"\n to the sum of the representative attention weights across all position indices involved in that prediction index's attention.\n" if normalize_attention_by_position_idxs_sum else ""
            title_for_head=" all heads" if head_idx is None else " head={}".format(head_idx)
            explain_homogeneous="\nHomogeneous sequences'"+r" 1$^{st}$ prediction index"+r" ($\tau$=1)"+"{}".format(" on {}".format(day_name if day_name is not None else "any day"))+" at hour {}:00.".format(
                "0{}".format(hour_of_preds_start_homogeneous_seqs) if len(str(hour_of_preds_start_homogeneous_seqs))==1
                else "{}".format(hour_of_preds_start_homogeneous_seqs) 
            ) if reduce_across_homogeneous_seqs else ""
            overall_title="\nAttention weights: averaged across"+title_for_head+" and "+title_for_reduction+title_for_threshold+title_for_batch_and_item+explain_homogeneous
            pre_suptitle=add_pre_suptitle if add_pre_suptitle is not None else ""
            if not only_legend:
                fig.suptitle(pre_suptitle+title_for_normalization+overall_title)
            fig.tight_layout()
            plt.close("all")
        else:
            if prediction_idx is not None:
                y_label=r"$\alpha(t,n,\tau=$"+"{})".format(prediction_index)
                fig=plt.figure(figsize=figsize,dpi=dpi)
                ax=fig.gca()
                if any([separator["enc"],separator["dec"]]):
                    ofs_x_txt=separator["x_offset"]
                    size_txt=separator["txt_size"]
                    ax.axvline(
                        x=original_attention_to["encoder"]["time_idxs"].iloc[-1],
                        color=separator["sep_color"],
                        linestyle=separator["sep_style"],
                        linewidth=separator["sep_linewidth"]
                    )
                    if separator["enc"]:
                        ax.annotate(
                            text="Encoder",
                            xycoords=("data","axes fraction"),
                            xy=(
                                original_attention_to["encoder"]["time_idxs"].iloc[-1],
                                separator["box_y_fraction"]
                            ),
                            textcoords="offset points",
                            xytext=(-ofs_x_txt,0),
                            size=size_txt,
                            bbox=dict(
                                boxstyle="round,pad=0.3", 
                                fc=separator["box_fill_color"], 
                                ec=separator["box_edge_color"]
                            ),
                            ha="right"
                        )
                    if separator["dec"]:
                        ax.annotate(
                            text="Decoder",
                            xycoords=("data","axes fraction"),
                            xy=(
                                original_attention_to["encoder"]["time_idxs"].iloc[-1],
                                separator["box_y_fraction"]
                            ),
                            textcoords="offset points",
                            xytext=(+ofs_x_txt,0),
                            size=size_txt,
                            bbox=dict(
                                boxstyle="round,pad=0.3", 
                                fc=separator["box_fill_color"], 
                                ec=separator["box_edge_color"]
                            ),
                            ha="left"
                        )
                if distinctipy_ops is None:
                    cmap=plt.get_cmap(colormap)
                    colors=[cmap(i) for i in np.linspace(0,1,len(representative_quantiles)+1)]
                else:
                    colors=distinctipy.get_colors(len(representative_quantiles)+1,**distinctipy_ops)
                    cmap=distinctipy.get_colormap(colors)
                max=0
                min=np.inf
                for iterator,quantile in enumerate(representative_quantiles):
                    if not zoom_from_to_idxs[0]:
                        if representative_attention_dict[prediction_index]["quantiles"][:,iterator].squeeze().max()>max:
                            max=representative_attention_dict[prediction_index]["quantiles"][:,iterator].squeeze().max()
                        if representative_attention_dict[prediction_index]["quantiles"][:,iterator].squeeze().min()<min:
                            min=representative_attention_dict[prediction_index]["quantiles"][:,iterator].squeeze().min()                    
                    else:
                        if not any(representative_attention_dict[prediction_index]["time_idxs"]==zoom_from_to_idxs[1]):
                            raise Exception("Lower positional index given in zoom_from_to_idxs falls out of limits of prediction index={}.\nIt must be within [{},...,{}], please provide a valid integer.".format(
                                prediction_index,
                                representative_attention_dict[prediction_index]["time_idxs"].min(),
                                representative_attention_dict[prediction_index]["time_idxs"].max()
                                )
                            )  
                        if zoom_from_to_idxs[2]<=zoom_from_to_idxs[1]:
                            raise Exception("Upper positional index in zoom_from_to_idxs can't be lower or equal to the lower position index.")
                        local_mask=(representative_attention_dict[prediction_index]["time_idxs"].loc[(representative_attention_dict[prediction_index]["time_idxs"]>=zoom_from_to_idxs[1])*(representative_attention_dict[prediction_index]["time_idxs"]<=zoom_from_to_idxs[2])]).index             
                        local_max=representative_attention_dict[prediction_index]["quantiles"][:,iterator].squeeze()[local_mask].max()
                        local_min=representative_attention_dict[prediction_index]["quantiles"][:,iterator].squeeze()[local_mask].min()
                        if local_max>max:
                            max=local_max
                        if local_min<min:
                            min=local_min                    
                    plot_attention(prediction_index,
                                representative_attention_dict[prediction_index]["time_idxs"],
                                representative_attention_dict[prediction_index]["quantiles"][:,iterator].squeeze(),
                                ax,colors[iterator],linewidth,
                                detail_representatives=True,
                                detail_prediction_timesteps=False,
                                quantile=quantile)
                if not exclude_mean:
                    if not zoom_from_to_idxs[0]:
                        if representative_attention_dict[prediction_index]["representative"].max()>max:
                            max=representative_attention_dict[prediction_index]["representative"].max()
                        if representative_attention_dict[prediction_index]["representative"].min()<min:
                            min=representative_attention_dict[prediction_index]["representative"].min()
                    else:
                        if representative_attention_dict[prediction_index]["representative"][local_mask].max()>max:
                            max=representative_attention_dict[prediction_index]["representative"][local_mask].max()
                        if representative_attention_dict[prediction_index]["representative"][local_mask].min()<min:
                            min=representative_attention_dict[prediction_index]["representative"][local_mask].min()           
                    plot_attention(prediction_index,
                                representative_attention_dict[prediction_index]["time_idxs"],
                                representative_attention_dict[prediction_index]["representative"],
                                ax,colors[-1],linewidth,
                                detail_representatives=True,
                                detail_prediction_timesteps=False)
                if custom_ticks:
                    ticks=np.arange(
                        concatenated_attention_to["time_idxs"].min(),
                        concatenated_attention_to["time_idxs"].max(),
                        plotting_time_interval
                    )
                    if not use_dates:
                        labels=[str(tick) for tick in ticks]
                        if reduce_across_homogeneous_seqs:
                            dummy_dates=pd.date_range(
                                end=pd.Timestamp(
                                    year=2025,month=1,day=15,
                                    hour=hour_of_preds_start_homogeneous_seqs,
                                    minute=0
                                ),
                                freq="h",
                                periods=n_past+1
                            )
                            dummy_dates=dummy_dates.append(
                                pd.date_range(
                                    start=dummy_dates[-1]+pd.Timedelta("1h"),
                                    freq="h",
                                    periods=n_future-1
                                )
                            )
                            if plotting_time_interval<24 and 24%plotting_time_interval==0:
                                labels=[]
                                for date_idx,date in enumerate(dummy_dates):
                                    if date_idx%plotting_time_interval==0:
                                        labels.append(date.strftime("%H:%M"))
                            else:
                                labels=[
                                    date.strftime("%H:%M") for date in dummy_dates[slice(0,None,plotting_time_interval)]
                                ]                                                 
                    else:
                        try:
                            assert "time_dummy_to_date" in data[which].keys()
                            consecutive_time_steps=False
                        except AssertionError:
                            consecutive_time_steps=True                
                        if day_name is None:
                            dates=item_in_batch_to_dates(
                                data=data,which=which,item_idx=item_idx,batch_idx=batch_idx,
                                consecutive_time_steps=consecutive_time_steps
                            )
                        else:
                            dm_st=pd.Timestamp(year=2025,month=4,day=28,hour=hour_of_preds_start_homogeneous_seqs,minute=0,second=0)
                            dm_sh={"Tuesday":1,"Wednesday":2,"Thursday":3,"Friday":4,"Saturday":5,"Sunday":6}
                            if day_name!="Monday":
                                dm_st+=pd.Timedelta("1d")*dm_sh[day_name]
                            dates=pd.date_range(dm_st,periods=n_past+n_future,freq="h")
                        if plotting_time_interval<24 and 24%plotting_time_interval==0:
                            labels=[]
                            for date_idx,date in enumerate(dates):
                                if date_idx%plotting_time_interval==0:
                                    if date_idx%24==0:
                                        if day_name is None:
                                            labels.append(date.strftime("%a %Y/%m/%d %H:%M"))
                                        else:
                                            labels.append(date.strftime("%a %H:%M"))
                                    else:
                                        labels.append(date.strftime("%H:%M"))
                        else:
                            if day_name is None:
                                labels=[
                                    date.strftime("%a %Y/%m/%d %H:%M") for date in dates[slice(0,None,plotting_time_interval)]
                                ]  
                            else:
                                labels=[
                                    date.strftime("%a %H:%M") for date in dates[slice(0,None,plotting_time_interval)]
                                ]
                    if not use_dates and not reduce_across_homogeneous_seqs:
                        if plotting_time_interval>4:
                            rotation=0
                        else:
                            rotation=90
                    else:
                        rotation=90        
                    ax.set_xticks(
                        ticks=ticks,
                        labels=labels,
                        rotation=rotation
                    )
                    if pad is not None:
                        ax.tick_params(axis='x', pad=pad["x"])   
                        ax.tick_params(axis='y', pad=pad["y"])  
                    if forced_rotation is not None:
                        ax.set_xticks(
                            ticks=ax.get_xticks(),
                            labels=ax.get_xticklabels(),
                            rotation=forced_rotation
                        )
                    ax.yaxis.set_major_locator(MaxNLocator())
                    if not zoom_from_to_idxs[0]:
                        ax.set_xlim((concatenated_attention_to["time_idxs"].min(),
                                    concatenated_attention_to["time_idxs"].max()))
                    else:
                        ax.set_xlim((zoom_from_to_idxs[1],zoom_from_to_idxs[2])) 
                    if ylims is None:
                        ax.set_ylim((min*(1-0.05),max*(1+0.05)))
                    else:
                        ax.set_ylim(ylims)
                if forced_xlabel is None:
                    ax.set_xlabel("Position index (n)" if not use_dates and not reduce_across_homogeneous_seqs else "")
                else:
                    ax.set_xlabel(forced_xlabel)
                if forced_ylabel is None:
                    ax.set_ylabel(y_label)
                else:
                    ax.set_ylabel(forced_ylabel)
                if scientific_on_y:
                    ax.yaxis.set_major_formatter(scientific_formatter)
                ax.grid(visible=True)
                ax.legend(loc="lower center",ncols=len(representative_quantiles)+(1-exclude_mean),bbox_to_anchor=(0.5,1))
                if not reduce_across_homogeneous_seqs:
                    title_for_reduction="summed across all sequences." if reduction=="sum" else "across all sequences."
                else:
                    title_for_reduction="summed across homogeneous sequences." if reduction=="sum" else "across homogeneous sequences."            
                title_for_threshold=" All weights strictly lower than {} have been ignored.".format(attention_threshold) if set_attention_threshold else ""
                if batch_idx is None:
                    title_for_batch_and_item=" All batches considered."
                else:
                    title_for_batch_and_item=" Sequence index={} in batch index={}.".format(item_idx,batch_idx) if item_idx is not None else " All sequences in batch index={}.".format(batch_idx)            
                title_for_quantiles=" Quantiles are retrieved across sequences for each position index.\n"
                title_for_normalization= r" For each prediction index, each of its representative attention weights ($\alpha$) is normalized according to"+"\nthe sum of the representative attention weights across all position indices involved in that prediction index's attention.\n Quantiles for each prediction index are normalized as the attention weigths for that prediction index to ensure comparability." if normalize_attention_by_position_idxs_sum else ""
                title_for_head=" all heads" if head_idx is None else " head={}".format(head_idx)
                explain_homogeneous="\nHomogeneous sequences'"+r" 1$^{st}$ prediction index"+r" ($\tau$=1)"+"{}".format(" on {}".format(day_name if day_name is not None else "any day"))+" at hour {}:00.".format(
                    "0{}".format(hour_of_preds_start_homogeneous_seqs) if len(str(hour_of_preds_start_homogeneous_seqs))==1
                    else "{}".format(hour_of_preds_start_homogeneous_seqs) 
                ) if reduce_across_homogeneous_seqs else ""
                overall_title="\nAttention weights: averaged across"+title_for_head+" and "+title_for_reduction+title_for_quantiles+title_for_threshold+title_for_batch_and_item+explain_homogeneous            
                pre_suptitle=add_pre_suptitle if add_pre_suptitle is not None else ""
                if not only_legend:
                    fig.suptitle(pre_suptitle+title_for_normalization+overall_title)
                fig.tight_layout()
                plt.close("all")
            else:
                raise Exception("When detail_representatives={}, you must set one value for prediction_idx.".format(detail_representatives))
    if print_info:
        for who in who_to_pay_attention_to:
            print("----{}'s time steps paying attention to {}'s time steps----".format(
                who_pays_attention,who
                )
            )
            print("Original attention tensor shape:",original_attention_to[who]["tensor"].shape)
            if head_idx is not None or prediction_idx is not None:
                print("Head's index={}\nPrediction's index={}\nSelected attention tensor shape: {}".format(
                    head_idx,prediction_idx,sliced_attention_to[who]["tensor"].shape
                    )
                )
            print("\n")
        print("----{}'s time steps paying attention to both encoder and decoder's time steps----".format(
            who_pays_attention
            )
        )
        print("Concatenated attention tensor shape: {}".format(concatenated_attention_to["tensor"].shape))
    if not return_figure:
        plt.close("all")
        display(fig)
        if return_attention:
            if not return_dist:
                return representative_attention_dict 
            else:
                return representative_attention_dict,dist
        else:
            if return_dist:
                return dist  
    else:
        plt.close("all")
        if not return_attention:
            if not return_dist:
                return fig
            else: 
                return fig,dist 
        else: 
            if not return_dist:
                return fig,representative_attention_dict
            else:
                return fig,representative_attention_dict,dist       

### Comaniciu

In [ ]:
def get_Comaniciu_distance_plot(
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
        data:dict[Literal["train","val","test"],
                    dict[str,
                        pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        show_rescaled:bool=True,
        scaler_y=None,
        dist_threshold:float|Literal["mean"]|tuple[Literal["quantile"],float]="mean",
        figsize:tuple[float,float]=(16,9),
        plot_kwargs:dict[str,]={"linewidth":1},
        return_most_and_least_deviating:bool=True,
        N_least_deviating:int|None=5,
        return_also_all_deviations:bool=False,
        use_non_overlapping_sequences_from_00:bool=False,
        fontsize:float=11,
        xlabel:str|None=None,
        ylabel:str|None=None,
        ylabel_udm:str="kWh",
        dist_label:str|None=None,
        dpi:int|None=None,
        formatter:bool=True,
        stairplot:bool=False,
        brackets:str="curve"
        )->None|list[pd.DataFrame,pd.DataFrame]|list[pd.DataFrame,pd.DataFrame,pd.DataFrame]:
    attention_kwargs=dict(
        reduction="mean",
        normalize_attention_by_position_idxs_sum=False,
        set_attention_threshold=False,
        detail_prediction_timesteps=True,
        detail_representatives=False,
        head_idx=None,
        batch_idx=None,
        item_idx=None,
        prediction_idx=None,
        print_info=False,
        return_figure=True,
        return_dist=True
    )
    if use_non_overlapping_sequences_from_00:
        attention_kwargs["reduce_across_homogeneous_seqs"]=True
        attention_kwargs["hour_of_preds_start_homogeneous_seqs"]=0
    which="test"
    _,dist=get_attention(
        as_from_predict_raw,data,which,**attention_kwargs
    )
    dataloader_list=data[which]["dataloader_list"]
    
    if isinstance(dist_threshold,str):
        dist_threshold=dist.cpu().nanmean()
        additional_label=" (mean)"
    elif isinstance(dist_threshold,tuple):
        additional_label=" (P{:.0f})".format(dist_threshold[1]*100)
        dist_threshold=dist.cpu().nanquantile(dist_threshold[1])
    else:
        additional_label=""

    start=True
    already_seen_decoder_time_idxs=torch.tensor([])
    batch_idxs_significant=[]
    item_idxs_significant=[]
    dist_significant=[]
    least_deviating=pd.DataFrame(
        {
        "batch_idx":np.zeros((N_least_deviating),dtype=int),
        "item_idx":np.zeros((N_least_deviating),dtype=int),
        "dist":np.inf*np.ones((N_least_deviating))
        },
        index=np.arange(1,N_least_deviating+1)
    )
    if return_also_all_deviations:
        all_batch_idxs=[]
        all_item_idxs=[]
        all_dist=[]
    overall_seqs_idx=0
    nos_idx=0
    nos_last_encoder_time_dummies=[]
    for batch_index,(x_dict,(_,_)) in enumerate(dataloader_list):
        y=x_dict["decoder_target"]
        batch_size=y.shape[0]          
        for item_index in range(batch_size):
            decoder_time_idxs=x_dict["decoder_time_idx"][item_index,:]
            if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                nos_last_encoder_time_dummies.append(int(decoder_time_idxs[0]-1))
                if dist[
                        overall_seqs_idx if not use_non_overlapping_sequences_from_00 else nos_idx
                    ].item()>dist_threshold:  
                    batch_idxs_significant.append(batch_index)                
                    item_idxs_significant.append(item_index)
                    dist_significant.append(dist[
                        overall_seqs_idx if not use_non_overlapping_sequences_from_00 else nos_idx
                    ].item())
                if dist[
                        overall_seqs_idx if not use_non_overlapping_sequences_from_00 else nos_idx
                    ].item()<least_deviating["dist"].max():
                    least_deviating.sort_values(
                        by="dist",inplace=True,
                        ascending=True
                    )
                    least_deviating.iloc[-1,0]=batch_index
                    least_deviating.iloc[-1,1]=item_index
                    least_deviating.iloc[-1,2]=dist[
                        overall_seqs_idx if not use_non_overlapping_sequences_from_00 else nos_idx
                    ].item()               
                already_seen_decoder_time_idxs=torch.cat(
                    (already_seen_decoder_time_idxs,decoder_time_idxs)
                )
                if start:
                    target=y
                    start=False
                else:    
                    target=torch.cat(
                        (target,y),
                        dim=0
                    )
                nos_idx+=1
            if return_also_all_deviations:
                all_batch_idxs.append(batch_index)
                all_item_idxs.append(item_index)
                all_dist.append(dist[overall_seqs_idx if not use_non_overlapping_sequences_from_00 else nos_idx-1].item())
            overall_seqs_idx+=1     
    tau_max=n_future
    last_encoder_time_dummies=(already_seen_decoder_time_idxs-1).int().tolist()[:-(tau_max-1)]
    time_dummy_to_date=pd.Series(
        pd.date_range(
            data[which]["starting_date"],freq="1h",
            periods=data[which]["df"].shape[0]
        ),
        index=data[which]["df"]["time_idx"]
    )
    decoder_dates=time_dummy_to_date.loc[already_seen_decoder_time_idxs]
    if show_rescaled:
        target=torch.tensor(
            scaler_y.inverse_transform(target.cpu().reshape((-1,1))).reshape(-1)
        )
    else:
        target=target.cpu().reshape(-1)
    with plt.rc_context({"font.size":fontsize}):
        fig=plt.figure(figsize=figsize,dpi=dpi)
        prop_cycle=iter(plt.rcParams["axes.prop_cycle"])
        target_color=next(prop_cycle)["color"]
        dist_color=next(prop_cycle)["color"]
        ax_target=fig.gca()
        if not stairplot:
            ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
        else:
            ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
        if formatter:
            ax_target.yaxis.set_major_locator(MaxNLocator())
            ax_target.yaxis.set_major_formatter("{x} "+ylabel_udm if show_rescaled else "{x}")
        ax_target.set_ylim(bottom=0)
        if ylabel is None:
            ylabel="Target" if show_rescaled else "Target rescaled"
        ax_target.set_ylabel(
            ylabel if formatter else ylabel+" {}{}{}".format("(" if brackets=="curve" else "[",ylabel_udm,")" if brackets=="curve" else "]"),
            color=target_color
        )
        ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
        name=""
        for idx,character in enumerate(which):
            if idx==0:
                name+=character.upper()
            else:
                name+=character
        if xlabel is None:
            xlabel=name+" set dates."
        ax_target.set_xlabel(xlabel)

        more_than_threshold_idxs=pd.Series(
            dist.cpu()>dist_threshold,
            index=last_encoder_time_dummies if not use_non_overlapping_sequences_from_00 else nos_last_encoder_time_dummies
        )
        x_switch={}
        unknown="to_be_defined"
        list_counter=0
        x_switch[list_counter]=[]
        before=unknown
        for idx in more_than_threshold_idxs.index:
            current=more_than_threshold_idxs.loc[idx]
            if before==unknown and current:
                x_switch[list_counter].append(idx)
            else:   
                if current and before:
                    x_switch[list_counter].append(idx)
                elif not current and before:
                    list_counter+=1
                    x_switch[list_counter]=[]
                elif current and not before:
                    x_switch[list_counter].append(idx)
                else:
                    continue
            before=current
        _,ymax=ax_target.get_ylim()
        for list_idx,above_indices in x_switch.items():
            ax_target.fill_between(
                x=time_dummy_to_date.loc[above_indices],
                y1=0,
                y2=ymax,
                fc=dist_color,
                alpha=0.15,
                label="Significant Events [dist(t)>{:.4f}".format(dist_threshold)+additional_label+"]" if list_idx==0 else ""
            )
        ax_target.legend(loc="lower center",bbox_to_anchor=(0.5,1))
        ax_target.set_xlim(
            (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
            max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
        )
        ax_dist=ax_target.twinx()
        ax_dist.plot(
            time_dummy_to_date.loc[last_encoder_time_dummies if not use_non_overlapping_sequences_from_00 else (already_seen_decoder_time_idxs-1).int().tolist()],
            dist.cpu() if not use_non_overlapping_sequences_from_00 else dist.cpu().reshape(-1,1).repeat(1,tau_max).reshape(-1),
            color=dist_color
        )
        ax_dist.set_ylim(bottom=dist.cpu().min())
        if dist_label is None:
            dist_label="dist(t)"
        ax_dist.set_ylabel(dist_label,color=dist_color)
        ax_dist.tick_params(axis="y",color=dist_color,labelcolor=dist_color)
        fig.tight_layout()
        plt.close("all")
        display(fig)
        if return_most_and_least_deviating:
            if N_least_deviating is None:
                raise Exception("return_most_and_least_deviating=True but you did not pass the\n\
                                number of least deviating sequences to be returned.")
            batch_and_item_idxs_significant_df=pd.DataFrame(
                {"batch_idx":batch_idxs_significant,
                "item_idx":item_idxs_significant,
                "dist":dist_significant}
                ).sort_values(by="dist",ascending=False)
            batch_and_item_idxs_significant_df.set_index(
                np.arange(1,len(batch_idxs_significant)+1,dtype=int),inplace=True
            )
            least_deviating.sort_values(by="dist",ascending=True,inplace=True)
            least_deviating.set_index(np.arange(1,N_least_deviating+1,dtype=int),inplace=True)
            if not return_also_all_deviations:
                return [batch_and_item_idxs_significant_df,least_deviating]
            else:
                batch_and_item_deviations=pd.DataFrame(
                    {"batch_idx":all_batch_idxs,"item_idx":all_item_idxs,"Comaniciu":all_dist}
                )
                return [batch_and_item_idxs_significant_df,least_deviating,batch_and_item_deviations]

### Predictions

In [ ]:
def get_prediction_plot(
        tft:TemporalFusionTransformer,
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],
        batch_idx:int,
        item_idx:int=0,
        attention_kwargs:dict|None=None,        
        figsize:tuple[float,float]=(16,8),
        linewidth:float=1.5,
        show_rescaled:bool=False,
        scaler_y=None,
        custom_ticks:bool=True,
        plotting_time_interval:int=24,
        add_loss_to_title:bool=True,
        loss_as_from_built_in_implementation:bool=True,
        sklearn_metric_name:Literal["MAE","MSE","RMSE","MAPE","R2"]|None=None,
        visualize_attention:bool=True,
        show_future_observed:bool=True,
        suptitle:str|None=None,
        explicit_fill_zones:bool=False,
        use_dates:bool=False,
        return_loss_info:bool=False,
        return_figure:bool=False,
        tau:int=1,
        pred_label:str|None=None,
        attn_label:str|None=None,
        only_legend:bool=False,
        fontsize:float=11,
        scientific_on_attn:bool=False,
        pred_udm:str="kWh",
        normalize_attention_by_position_idxs_sum:bool=True,
        dpi:int|None=None,
        formatter:bool=True,
        stairplot:bool=False,
        brackets:str="curve",
        pred_lims:tuple[float]|None=None,
        attn_lims:tuple[float]|None=None,
        force_base_strftime:str|None=None
        )->None|dict[str,dict[str,float|str]]|tuple[plt.Figure,dict[str,dict[str,float|str]]]:
    scientific_formatter=ScalarFormatter(useMathText=True)
    scientific_formatter.set_scientific(True)
    scientific_formatter.set_powerlimits((0,0))
    try:
        try:
            Output=getattr(as_from_predict_raw,"output") 
        except AttributeError:
            Output=as_from_predict_raw["output"]
    except AttributeError:
        Output=as_from_predict_raw
    
    try:
        raw_predictions=getattr(Output,"prediction")
    except AttributeError:
        raw_predictions=Output["prediction"]

    quantiles_computed=pd.Series(tft.loss.quantiles)
    quantile_for_predictions=0.5
    if not (quantile_for_predictions in tft.loss.quantiles):
        raise Exception("By default, predictions are represented by the median (q=0.5, a.k.a. P50), but the loss used at TFT's initialization does not compute the median!\nAdd q=0.5 among the quantiles in QuantileLoss() at TFT's construction time.")
    quantile_for_predictions_idx=(quantiles_computed==quantile_for_predictions).idxmax()

    predictions=raw_predictions[:,:,quantile_for_predictions_idx]

    dataloader=data[which]["dataloader"]
    batch_size=dataloader.batch_size
    number_of_batches=len(dataloader)
    if not (batch_idx in np.arange(0,number_of_batches)):
        raise Exception("batch_idx selects the index of the selected batch. batch_idx={} but it must be in [0,...,#batches in DataLoader-1={}].".format(
            batch_idx,number_of_batches-1
            )
        )
    batch_to_seqs_slice={
        batch_num:slice(batch_num*batch_size,(batch_num+1)*batch_size) if batch_num!=(number_of_batches-1) else slice(batch_num*batch_size,None) for batch_num in range(number_of_batches)
    }
    predictions_in_batch=predictions[batch_to_seqs_slice[batch_idx],:]

    if not (item_idx in np.arange(0,batch_size)):
        raise Exception("item selects the index of the sequence within batch={} of which you want to\nvisualize the predictions. item={} but it must be in [0,...,#sequences in batch_{}-1={}].".format(
            batch_idx,item_idx,batch_idx,batch_size-1
            )
        )

    predictions_sequence_in_batch=predictions_in_batch[item_idx,:]
    
    assert tau in list(range(1,n_future+1)),"tau={} given but tau should be in\n{}.".format(tau,list(range(1,n_future)))
    attention_kwargs_selected=dict(
        as_from_predict_raw=as_from_predict_raw,
        data=data,
        which=which,
        reduction="mean",
        normalize_attention_by_position_idxs_sum=normalize_attention_by_position_idxs_sum,
        set_attention_threshold=True,
        prediction_idx=tau,
        custom_ticks=False,
        print_info=False,
        return_attention=True,
        return_figure=True,
        detail_representatives=False,
        detail_prediction_timesteps=True
    ) 
    if attention_kwargs is not None:
        assert all([not k in attention_kwargs_selected.keys() for k in attention_kwargs.keys()]),"Some of the keys in attention_kwargs are set by default."
        attention_kwargs_selected.update(attention_kwargs)
    x_dict,_=data[which]["dataloader_list"][batch_idx]
    encoder_target=x_dict["encoder_target"]
    decoder_target=x_dict["decoder_target"]
    encoder_target=encoder_target[item_idx,:]
    decoder_target=decoder_target[item_idx,:]
    _,attention_dict=get_attention(
        batch_idx=batch_idx,item_idx=item_idx,**attention_kwargs_selected
    )

    if show_rescaled:
        if scaler_y is None:
            raise Exception("When show_rescaled=True, you must pass a scaler to recover the original scale from the predictions. It should expect a (#time steps,1) vector.")
        predictions_sequence_in_batch=torch.tensor(
            scaler_y.inverse_transform(predictions_sequence_in_batch.cpu().reshape((-1,1))).reshape(-1)
        )
        encoder_target=torch.tensor(
            scaler_y.inverse_transform(encoder_target.reshape((-1,1))).reshape(-1)
        )
        decoder_target=torch.tensor(
            scaler_y.inverse_transform(decoder_target.reshape((-1,1))).reshape(-1)
        )

    with plt.rc_context({"font.size":fontsize}): 
        fig=plt.figure(figsize=figsize,dpi=dpi)
        ax_predictions=fig.gca()
        encoder_positions=np.arange(-n_past+1,1)
        decoder_positions=np.arange(0,24)
        prop_cycle = iter(plt.rcParams["axes.prop_cycle"])
        target_color = next(prop_cycle)["color"]
        predictions_color = next(prop_cycle)["color"]
        if not stairplot:
            ax_predictions.plot(encoder_positions,encoder_target,color=target_color,linewidth=linewidth,label="Observed")
        else:
            ax_predictions.step(encoder_positions,encoder_target,color=target_color,linewidth=linewidth,label="Observed",where="post")
        if show_future_observed:
            if not stairplot:
                ax_predictions.plot(decoder_positions,decoder_target,color=target_color,linewidth=linewidth)
            else:
                ax_predictions.step(decoder_positions,decoder_target,color=target_color,linewidth=linewidth,where="post")
        if not stairplot:
            ax_predictions.plot(
                decoder_positions,
                predictions_sequence_in_batch.cpu() ,
                color=predictions_color,linewidth=linewidth,label="Predicted (P50)")
        else:
            ax_predictions.step(
                decoder_positions,
                predictions_sequence_in_batch.cpu() ,
                color=predictions_color,linewidth=linewidth,label="Predicted (P50)",where="post")

        predictions_in_batch_all_quantiles=raw_predictions[batch_to_seqs_slice[batch_idx],:,:]
        predictions_for_loss=predictions_in_batch_all_quantiles[slice(item_idx,item_idx+1),:,:]
        if show_rescaled:
            for quantile_idx in quantiles_computed.index:
                predictions_for_loss=predictions_for_loss.cpu()
                predictions_for_loss[:,:,quantile_idx]=torch.tensor(scaler_y.inverse_transform(predictions_for_loss[:,:,quantile_idx].squeeze().reshape((-1,1))).reshape((1,-1)))
        if loss_as_from_built_in_implementation:
            title_for_loss=" Loss (across all quantiles predicted for this sequence)={}".format(
                tft.loss(
                    predictions_for_loss.cpu(),
                    decoder_target.reshape((1,-1))
                )
            )
            title_for_loss=title_for_loss+" kWh" if show_rescaled else title_for_loss+"."
        else:
            if sklearn_metric_name is None:
                raise Exception("When loss_as_from_built_in_implementation=False you must specify which metric you want to compute by passing a valid sklearn_metric_name.")
            else:
                predictions_for_sklearn_metrics=predictions_for_loss[:,:,quantile_for_predictions_idx].cpu().squeeze()
                from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score,mean_squared_error
                metrics_dict={
                    "MAE":dict(
                        value=mean_absolute_error(decoder_target,predictions_for_sklearn_metrics),
                        udm="(-)" if not show_rescaled else "kWh"
                    ),
                    "MSE":dict(
                        value=mean_squared_error(decoder_target,predictions_for_sklearn_metrics),
                        udm=r"(-)$^2$" if not show_rescaled else r"kWh$^2$"
                    ),
                    "RMSE":dict(
                        value=root_mean_squared_error(decoder_target,predictions_for_sklearn_metrics),
                        udm="(-)" if not show_rescaled else "kWh"
                    ),
                    "MAPE":dict(
                        value=torch.mean(
                            torch.abs(
                                decoder_target[decoder_target!=0] - predictions_for_sklearn_metrics[decoder_target!=0]
                            ) / decoder_target[decoder_target!=0]
                        ).item()*100,
                        udm="%"
                    ),                
                    "R2":dict(
                        value=r2_score(decoder_target,predictions_for_sklearn_metrics),
                        udm=""
                    )
                }
                title_for_loss=" Loss computed on predictions (P50): {}={:.2f} {}.".format(
                    sklearn_metric_name,
                    metrics_dict[sklearn_metric_name]["value"],
                    metrics_dict[sklearn_metric_name]["udm"]
                )

        predicted_quantiles_for_sequence=predictions_for_loss.cpu().squeeze()
        if not explicit_fill_zones:
            for quantile_idx in range(len(quantiles_computed)//2):
                ax_predictions.fill_between(
                    decoder_positions,
                    predicted_quantiles_for_sequence[:,quantile_idx],
                    predicted_quantiles_for_sequence[:,-quantile_idx-1], 
                    alpha=0.15,
                    fc=predictions_color,
                )
        else:
            alphas=np.linspace(
                0.15,
                0.45,
                max(
                    len(quantiles_computed.loc[quantiles_computed<quantile_for_predictions]),
                    len(quantiles_computed.loc[quantiles_computed>quantile_for_predictions])
                )
            )
            n_additional_cols=0
            for order in ["smaller","larger"]:
                if order=="smaller":
                    mask=quantiles_computed<quantile_for_predictions
                else:
                    mask=quantiles_computed>quantile_for_predictions
                count=0 if order=="smaller" else len(alphas)-1
                masked_computed_quantiles=quantiles_computed.loc[mask] if order=="smaller" else quantiles_computed.loc[mask]
                for quantile_idx in masked_computed_quantiles.index:
                    ax_predictions.fill_between(
                        decoder_positions,
                        predicted_quantiles_for_sequence[:,quantile_idx if order=="smaller" else quantile_idx-1],
                        predicted_quantiles_for_sequence[:,quantile_idx+1 if order=="smaller" else quantile_idx],
                        alpha=alphas[count],
                        fc=predictions_color,
                        label="[P{:.0f}, P{:.0f}]".format(
                            quantiles_computed.loc[quantile_idx if order=="smaller" else quantile_idx-1]*100,
                            quantiles_computed.loc[quantile_idx+1 if order=="smaller" else quantile_idx]*100,
                        ),
                        step=None if not stairplot else "post"
                    )
                    count=count+1 if order=="smaller" else count-1
                    n_additional_cols+=1
        if pred_label is None:
            pred_label="Target" if show_rescaled else "Target rescaled"
        ax_predictions.yaxis.set_major_locator(MaxNLocator())
        if formatter:
            ax_predictions.yaxis.set_major_formatter("{x} "+pred_udm if show_rescaled else "{x}")
            ax_predictions.set_ylabel(pred_label)
        else:
            ax_predictions.set_ylabel(pred_label+" {}{}{}".format("(" if brackets=="curve" else "[",pred_udm,")" if brackets=="curve" else "]"))
        ax_predictions.set_xlabel("Position index (n)" if not use_dates else "") 
        title="Predictions for sequence index={} in batch index={}.".format(
            item_idx,batch_idx
        )
        title=title+title_for_loss if add_loss_to_title else title
        ax_predictions.legend(
            loc="lower center",
            ncols=2 if not explicit_fill_zones else 2+n_additional_cols,
            bbox_to_anchor=(0.5,1)
        )
        if custom_ticks:
            ticks=np.arange(
                encoder_positions.min(),
                decoder_positions.max(),
                plotting_time_interval
            )
            if not use_dates:
                labels=[str(tick) for tick in ticks]
            else:
                if force_base_strftime is None:
                    base_strftime="%a %Y/%m/%d %H:%M"
                else:
                    base_strftime=force_base_strftime
                try:
                    assert "time_dummy_to_date" in data[which].keys()
                    consecutive_time_steps=False
                except AssertionError:
                    consecutive_time_steps=True
                dates=item_in_batch_to_dates(
                    data=data,which=which,item_idx=item_idx,batch_idx=batch_idx,
                    consecutive_time_steps=consecutive_time_steps
                )
                if plotting_time_interval<24 and 24%plotting_time_interval==0:
                    labels=[]
                    for date_idx,date in enumerate(dates):
                        if date_idx%plotting_time_interval==0:
                            if date_idx%24==0:
                                labels.append(date.strftime(base_strftime))
                            else:
                                labels.append(date.strftime("%H:%M"))
                else:
                    labels=[
                        date.strftime(base_strftime) for date in dates[slice(0,None,plotting_time_interval)]
                    ] 
            ax_predictions.set_xticks(
                ticks=ticks,
                labels=labels,
                rotation=0 if not use_dates else 90
            )
            ax_predictions.set_xlim((encoder_positions.min(),decoder_positions.max()))
            if pred_lims is None:
                ax_predictions.set_ylim(bottom=0)
            else:
                ax_predictions.set_ylim(pred_lims)
        if visualize_attention:
            ax_attention=ax_predictions.twinx()
            ax_attention.plot(attention_dict[tau]["time_idxs"],attention_dict[tau]["representative"],'-k',linewidth=linewidth,alpha=0.2)
            if attn_label is None:
                attn_label="Representative "+r"$\alpha(t,n,\tau$"+"={})".format(tau)
            ax_attention.set_ylabel(attn_label,color=(0,0,0,0.2))
            ax_attention.tick_params('y',color=(0,0,0,0.2),labelcolor=(0,0,0,0.2))
            if scientific_on_attn:
                ax_attention.yaxis.set_major_formatter(scientific_formatter)
            if custom_ticks:
                if attn_lims is None:
                    ax_attention.set_ylim(bottom=0)
                else:
                    ax_attention.set_ylim(attn_lims)
        for_suptitle=suptitle if suptitle is not None else ""
        str_percentiles=["P{:.0f}".format(quantile*100) for quantile in tft.loss.quantiles]
        str_concatenated_percentiles="["+", ".join(str_percentiles)+"]."
        if not only_legend:
            fig.suptitle(for_suptitle+" Percentiles computed: "+str_concatenated_percentiles+"\n"+title)
        fig.tight_layout()
        if not return_figure:
            plt.close("all")
            display(fig)
            if return_loss_info:
                return metrics_dict
        else:
            plt.close("all")
            if not return_loss_info:
                return fig
            else:
                return fig,metrics_dict

### Best/Worst

In [ ]:
def get_model_insights(
        tft:TemporalFusionTransformer,
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],        
        top_N:int,
        worst_M:int,
        scaler_y,
        sklearn_metric_name:Literal["MAE","MSE","RMSE","MAPE","R2"],
        search_without_rescaling:bool=False,
        return_all_seqs_metrics_df:bool=False,
        return_not_overlapped_seqs_batch_and_item:bool=False
        )->tuple[pd.DataFrame,pd.DataFrame]|tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame]|tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame]:

    from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
    better_if_lower=["MAE","MSE","RMSE","MAPE"]
    metrics_DF_top_N=pd.DataFrame(
        {
        "batch_idx":np.zeros((top_N),dtype=int),
        "item_idx":np.zeros((top_N),dtype=int),
        sklearn_metric_name:(np.inf if sklearn_metric_name in better_if_lower else -np.inf)*np.ones((top_N))
        },
        index=np.arange(1,top_N+1)
    )
    metrics_DF_worst_M=pd.DataFrame(
        {
        "batch_idx":np.zeros((worst_M),dtype=int),
        "item_idx":np.zeros((worst_M),dtype=int),
        sklearn_metric_name:(-np.inf if sklearn_metric_name in better_if_lower else np.inf)*np.ones((worst_M))
        },
        index=np.arange(1,worst_M+1)
    )

    try:
        try:
            Output=getattr(as_from_predict_raw,"output")
        except AttributeError:
            Output=as_from_predict_raw["output"]
    except AttributeError:
        Output=as_from_predict_raw
    
    try:
        raw_predictions=getattr(Output,"prediction")
    except:
        raw_predictions=Output["prediction"]
    quantiles_computed=pd.Series(tft.loss.quantiles)
    quantile_for_predictions=0.5
    if not (quantile_for_predictions in tft.loss.quantiles):
        raise Exception("By default, predictions are represented by the median (q=0.5, a.k.a. P50), but the loss used at TFT's initialization does not compute the median!\nAdd q=0.5 among the quantiles in QuantileLoss() at TFT's construction time.")
    quantile_for_predictions_idx=(quantiles_computed==quantile_for_predictions).idxmax()

    predictions=raw_predictions[:,:,quantile_for_predictions_idx]

    dataloader=data[which]["dataloader"]
    batch_size=dataloader.batch_size
    number_of_batches=len(dataloader)
    batch_to_seqs_slice={
        batch_num:slice(batch_num*batch_size,(batch_num+1)*batch_size) if batch_num!=(number_of_batches-1) else slice(batch_num*batch_size,None) for batch_num in range(number_of_batches)
    }    
    already_seen_decoder_time_idxs=torch.tensor([])
    not_overlapped_seqs_dict={
        "batch_idx":[],
        "item_idx":[],
    }
    all_seqs_metrics_dict={
        "batch_idx":[],
        "item_idx":[],
    }
    for name in ["MAE","MSE","RMSE","MAPE","R2"]:
        all_seqs_metrics_dict[name]=[]
    for batch_index,(x_dict,_) in enumerate(data[which]["dataloader_list"]):
        decoder_target=x_dict["decoder_target"]
        batch_size=decoder_target.shape[0]
        for item_index in range(batch_size):
            decoder_time_idxs=x_dict["decoder_time_idx"][item_index,:]
            if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                not_overlapped_seqs_dict["batch_idx"].append(batch_index)
                not_overlapped_seqs_dict["item_idx"].append(item_index)
                already_seen_decoder_time_idxs=torch.cat(
                    (already_seen_decoder_time_idxs,decoder_time_idxs)
                )
                decoder_target=decoder_target[item_index,:]
                predictions_in_batch=predictions[batch_to_seqs_slice[batch_index],:]

                predictions_sequence_in_batch=predictions_in_batch[item_index,:]

                if not search_without_rescaling:
                    predictions_sequence_in_batch=torch.tensor(
                        scaler_y.inverse_transform(predictions_sequence_in_batch.cpu().reshape((-1,1))).reshape(-1)
                    )
                    decoder_target=torch.tensor(
                        scaler_y.inverse_transform(decoder_target.cpu().reshape((-1,1))).reshape(-1)
                    )
                else:
                    predictions_sequence_in_batch=predictions_sequence_in_batch.cpu()
                    decoder_target=decoder_target.cpu()

                metrics_dict={
                    "MAE":mean_absolute_error(decoder_target,predictions_sequence_in_batch),
                    "MSE":mean_squared_error(decoder_target,predictions_sequence_in_batch),
                    "RMSE":root_mean_squared_error(decoder_target,predictions_sequence_in_batch),
                    "MAPE":torch.mean(
                        torch.abs(
                            decoder_target[decoder_target!=0] - predictions_sequence_in_batch[decoder_target!=0]
                        ) / decoder_target[decoder_target!=0]
                    ).item()*100,          
                    "R2":r2_score(decoder_target,predictions_sequence_in_batch),
                }
                metric=metrics_dict[sklearn_metric_name]

                if return_all_seqs_metrics_df:
                    all_seqs_metrics_dict["batch_idx"].append(batch_index)
                    all_seqs_metrics_dict["item_idx"].append(item_index)
                    for name in metrics_dict:
                        all_seqs_metrics_dict[name].append(metrics_dict[name]) 
                        
                if sklearn_metric_name in better_if_lower:
                    is_better=metric<metrics_DF_top_N[sklearn_metric_name].max()
                    is_worse=metric>metrics_DF_worst_M[sklearn_metric_name].min()
                else:
                    is_better=metric>metrics_DF_top_N[sklearn_metric_name].min()
                    is_worse=metric<metrics_DF_worst_M[sklearn_metric_name].max()
                if is_better:
                    metrics_DF_top_N.sort_values(
                        by=sklearn_metric_name,inplace=True,
                        ascending=True if sklearn_metric_name in better_if_lower else False
                    )
                    metrics_DF_top_N.iloc[-1,0]=batch_index
                    metrics_DF_top_N.iloc[-1,1]=item_index
                    metrics_DF_top_N.iloc[-1,2]=metric
                if is_worse:
                    metrics_DF_worst_M.sort_values(
                        by=sklearn_metric_name,inplace=True,
                        ascending=False if sklearn_metric_name in better_if_lower else True
                    )
                    metrics_DF_worst_M.iloc[-1,0]=batch_index
                    metrics_DF_worst_M.iloc[-1,1]=item_index
                    metrics_DF_worst_M.iloc[-1,2]=metric
            else:
                if return_all_seqs_metrics_df:
                    decoder_target=decoder_target[item_index,:]
                    predictions_in_batch=predictions[batch_to_seqs_slice[batch_index],:]
                    predictions_sequence_in_batch=predictions_in_batch[item_index,:]
                    if not search_without_rescaling:
                        predictions_sequence_in_batch=torch.tensor(
                            scaler_y.inverse_transform(predictions_sequence_in_batch.cpu().reshape((-1,1))).reshape(-1)
                        )
                        decoder_target=torch.tensor(
                            scaler_y.inverse_transform(decoder_target.cpu().reshape((-1,1))).reshape(-1)
                        )
                    else:
                        predictions_sequence_in_batch=predictions_sequence_in_batch.cpu()
                        decoder_target=decoder_target.cpu()

                    metrics_dict={
                        "MAE":mean_absolute_error(decoder_target,predictions_sequence_in_batch),
                        "MSE":mean_squared_error(decoder_target,predictions_sequence_in_batch),
                        "RMSE":root_mean_squared_error(decoder_target,predictions_sequence_in_batch),
                        "MAPE":torch.mean(
                            torch.abs(
                                decoder_target[decoder_target!=0] - predictions_sequence_in_batch[decoder_target!=0]
                            ) / decoder_target[decoder_target!=0]
                        ).item()*100,           
                        "R2":r2_score(decoder_target,predictions_sequence_in_batch),
                    }
                    all_seqs_metrics_dict["batch_idx"].append(batch_index)
                    all_seqs_metrics_dict["item_idx"].append(item_index)
                    for name in metrics_dict:
                        all_seqs_metrics_dict[name].append(metrics_dict[name])    

    metrics_DF_top_N.sort_values(
        by=sklearn_metric_name,inplace=True,
        ascending=True if sklearn_metric_name in better_if_lower else False
    )
    metrics_DF_worst_M.sort_values(
        by=sklearn_metric_name,inplace=True,
        ascending=False if sklearn_metric_name in better_if_lower else True
    )
    metrics_DF_top_N.set_index(np.arange(1,top_N+1,dtype=int),inplace=True)
    metrics_DF_worst_M.set_index(np.arange(1,worst_M+1,dtype=int),inplace=True)
    if not return_all_seqs_metrics_df:
        return metrics_DF_top_N,metrics_DF_worst_M
    else:
        all_seqs_metrics_df=pd.DataFrame(
            all_seqs_metrics_dict,
            index=np.arange(0,len(all_seqs_metrics_dict["batch_idx"]))
        )
        if not return_not_overlapped_seqs_batch_and_item:
            return metrics_DF_top_N,metrics_DF_worst_M,all_seqs_metrics_df
        else:
            not_overlapped_seqs_df=pd.DataFrame(not_overlapped_seqs_dict)
            return metrics_DF_top_N,metrics_DF_worst_M,all_seqs_metrics_df,not_overlapped_seqs_df

### Above/Below

In [ ]:
def get_above_and_below_metric(
        tft:TemporalFusionTransformer,
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
        sklearn_metric_name:Literal["MAE","MSE","RMSE","MAPE","R2"],
        all_seqs_metrics_df:pd.DataFrame,
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        show_rescaled:bool=True,
        scaler_y=None,
        upper_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.95),
        lower_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.05),
        figsize:tuple[float,float]=(16,9),
        building_name:str|None=None,
        plot_kwargs:dict[str,]={"linewidth":1},
        print_overall_metrics:bool=True,
        return_above_and_below_dfs:bool=False,
        )->None|list[pd.DataFrame,pd.DataFrame]:
    
    try:
        try:
            Output=getattr(as_from_predict_raw,"output")
        except AttributeError:
            Output=as_from_predict_raw["output"]
    except AttributeError:
        Output=as_from_predict_raw
    
    try:
        raw_predictions=getattr(Output,"prediction")
    except AttributeError:
        raw_predictions=Output["prediction"]
    quantiles_computed=pd.Series(tft.loss.quantiles)
    quantile_for_predictions=0.5
    if not (quantile_for_predictions in tft.loss.quantiles):
        raise Exception("By default, predictions are represented by the median (q=0.5, a.k.a. P50), but the loss used at TFT's initialization does not compute the median!\nAdd q=0.5 among the quantiles in QuantileLoss() at TFT's construction time.")
    quantile_for_predictions_idx=(quantiles_computed==quantile_for_predictions).idxmax()

    predictions=raw_predictions[:,:,quantile_for_predictions_idx]

    which="test"
    dataloader=data[which]["dataloader"]
    batch_size=dataloader.batch_size
    number_of_batches=len(dataloader)
    batch_to_seqs_slice={
        batch_num:slice(batch_num*batch_size,(batch_num+1)*batch_size) if batch_num!=(number_of_batches-1) else slice(batch_num*batch_size,None) for batch_num in range(number_of_batches)
    }
    dataloader_list=data[which]["dataloader_list"]
    metrics=np.array(all_seqs_metrics_df[sklearn_metric_name])
    if isinstance(upper_threshold,tuple):
        upper_label=" (P{:.1f})".format(upper_threshold[1]*100)
        upper_threshold=np.quantile(metrics,upper_threshold[1])
    else:
        upper_label=""
    if isinstance(lower_threshold,tuple):
        lower_label=" (P{:.1f})".format(lower_threshold[1]*100)
        lower_threshold=np.quantile(metrics,lower_threshold[1])
    else:
        lower_label=""

    start=True
    already_seen_decoder_time_idxs=torch.tensor([])
    batch_idxs_significant={"above":[],"below":[]}
    item_idxs_significant={"above":[],"below":[]}
    metrics_significant={"above":[],"below":[]}
    decoder_time_idxs_significant={"above":[],"below":[]}
    overall_seqs_idx=0
    for batch_index,(x_dict,(_,_)) in enumerate(dataloader_list):
        y=x_dict["decoder_target"]
        batch_size=y.shape[0]          
        for item_index in range(batch_size):
            decoder_time_idxs=x_dict["decoder_time_idx"][item_index,:]
            if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                if metrics[overall_seqs_idx]>upper_threshold:  
                    decoder_time_idxs_significant["above"].append(decoder_time_idxs)
                    batch_idxs_significant["above"].append(batch_index)                
                    item_idxs_significant["above"].append(item_index)
                    metrics_significant["above"].append(metrics[overall_seqs_idx])
                if metrics[overall_seqs_idx]<lower_threshold:  
                    decoder_time_idxs_significant["below"].append(decoder_time_idxs)
                    batch_idxs_significant["below"].append(batch_index)                
                    item_idxs_significant["below"].append(item_index)
                    metrics_significant["below"].append(metrics[overall_seqs_idx])  
                already_seen_decoder_time_idxs=torch.cat(
                    (already_seen_decoder_time_idxs,decoder_time_idxs)
                )
                predictions_in_batch=predictions[batch_to_seqs_slice[batch_index],:]
                predictions_sequence_in_batch=predictions_in_batch[item_index,:]           
                if start:
                    preds=predictions_sequence_in_batch
                    target=y[item_index,:]
                    start=False
                else:    
                    target=torch.cat(
                        (target,y[item_index,:]),
                        dim=0
                    )
                    preds=torch.cat(
                        (preds,predictions_sequence_in_batch)
                    )
            overall_seqs_idx+=1                
    tau_max=n_future
    last_encoder_time_dummies=(already_seen_decoder_time_idxs-1).int().tolist()[:-(tau_max-1)]
    try:
        assert "time_dummy_to_date" in data[which].keys()
        time_dummy_to_date=copy.deepcopy(data[which]["time_dummy_to_date"])
    except AssertionError:
        time_dummy_to_date=pd.Series(
            pd.date_range(
                data[which]["starting_date"],freq="1h",
                periods=data[which]["df"].shape[0]
            ),
            index=data[which]["df"]["time_idx"]
        )
    decoder_dates=time_dummy_to_date.loc[already_seen_decoder_time_idxs]

    n_not_overlapping_seqs=target.shape[0]
    percentage_not_overlapping_above=len(metrics_significant["above"])/n_not_overlapping_seqs
    percentage_not_overlapping_below=len(metrics_significant["below"])/n_not_overlapping_seqs
    if show_rescaled:
        target=torch.tensor(
            scaler_y.inverse_transform(target.cpu().reshape((-1,1))).reshape(-1)
        )
        preds=torch.tensor(
            scaler_y.inverse_transform(preds.cpu().reshape((-1,1))).reshape(-1)
        )
    else:
        target=target.cpu()
        preds=preds.cpu()
        
    from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
    overall_metrics={
        "MAE":mean_absolute_error(target,preds),
        "MSE":mean_squared_error(target,preds),
        "RMSE":root_mean_squared_error(target,preds),
        "MAPE":torch.mean(
            torch.abs(
                target[target!=0] - preds[target!=0]
            ) / target[target!=0]
        ).item()*100,             
        "R2":r2_score(target,preds),
    }
    names_to_udm={
        "MAE":" (-)" if not show_rescaled else " kWh",
        "MSE":r" (-)$^2$" if not show_rescaled else r" kWh$^2$",
        "RMSE":" (-)" if not show_rescaled else " kWh",
        "MAPE":"%",
        "R2":"",
    }  

    absolute_min=min([target.min(),preds.min()])
    absolute_max=max([target.max(),preds.max()])

    fig=plt.figure(figsize=figsize)    
    target_color="r"
    preds_color="b"
    bad_color="#FFD700" 
    good_color = "g"
    ax_target=fig.gca()
    ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
    ax_target.set_ylim(
        bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
        top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
    )
    ax_target.set_ylabel(
        "Observed Energy Consumption (kWh)" if show_rescaled else "Observed Energy Consumption (-)",
        color=target_color
    )
    ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
    name=""
    for idx,character in enumerate(which):
        if idx==0:
            name+=character.upper()
        else:
            name+=character
    ax_target.set_xlabel(name+" set dates.")
    better_if_lower=["MAE","MSE","RMSE","MAPE"]
    ymin,ymax=ax_target.get_ylim()
    for list_idx,above_indices in enumerate(decoder_time_idxs_significant["above"]):
        ax_target.fill_between(
            x=time_dummy_to_date.loc[above_indices],
            y1=ymin,
            y2=ymax,
            fc=bad_color if sklearn_metric_name in better_if_lower else good_color,
            alpha=0.35,
            label="{}>{:.4f}".format(sklearn_metric_name,upper_threshold)+names_to_udm[sklearn_metric_name]+upper_label if list_idx==0 else ""
        )

    for list_idx,below_indices in enumerate(decoder_time_idxs_significant["below"]):
        ax_target.fill_between(
            x=time_dummy_to_date.loc[below_indices],
            y1=ymin,
            y2=ymax,
            fc=good_color if sklearn_metric_name in better_if_lower else bad_color,
            alpha=0.35,
            label="{}<{:.4f}".format(sklearn_metric_name,lower_threshold)+names_to_udm[sklearn_metric_name]+lower_label if list_idx==0 else ""
        )

    ax_target.legend(loc="lower center",bbox_to_anchor=(0.5, 1.0),ncols=2)
    ax_target.set_xlim(
        (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
         max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
    )
    ax_preds=ax_target.twinx()
    ax_preds.plot(decoder_dates,preds,color=preds_color)
    ax_preds.set_ylim(
        bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
        top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
    )
    ax_preds.set_ylabel("Predicted Energy Consumption (P50) (kWh)" if show_rescaled else "Predicted Energy Consumption (P50) (-)",
                        color=preds_color)
    ax_preds.tick_params(axis="y",color=preds_color,labelcolor=preds_color)
    suptitle_strs=["{} building".format(building_name),"\nMetrics:"]
    for idx,(name,value) in enumerate(overall_metrics.items()):
        if idx!=(len(overall_metrics.items())-1):
            suptitle_strs.append("{}={:.2f}".format(name,value)+names_to_udm[name]+",")
        else:
            suptitle_strs.append("{}={:.2f}".format(name,value)+names_to_udm[name])
    fig.suptitle(" ".join(suptitle_strs))
    fig.tight_layout()
    plt.close("all")
    display(fig)
    print("Percentage of sequences with not overlapped decoder times above the metric's threshold:{:.2f}%".format(percentage_not_overlapping_above*100))
    print("Percentage of sequences with not overlapped decoder times below the metric's threshold:{:.2f}%".format(percentage_not_overlapping_below*100))
    if print_overall_metrics:
        print("----Overall metrics----")
        for name,value in overall_metrics.items():
            if name!="MSE":
                print("{} = {:.2f}".format(name,value)+names_to_udm[name])
            else:
                print("{} = {:.2f} kWh\u00B2".format(name,value))
    if return_above_and_below_dfs:
        above_df=pd.DataFrame(
            {"batch_idx":batch_idxs_significant["above"],
            "item_idx":item_idxs_significant["above"],
            sklearn_metric_name:metrics_significant["above"]}
            ).sort_values(by=sklearn_metric_name,ascending=False)
        above_df.set_index(
            np.arange(1,len(batch_idxs_significant["above"])+1,dtype=int),inplace=True
        )
        below_df=pd.DataFrame(
            {"batch_idx":batch_idxs_significant["below"],
            "item_idx":item_idxs_significant["below"],
            sklearn_metric_name:metrics_significant["below"]}
            ).sort_values(by=sklearn_metric_name,ascending=True)
        below_df.set_index(
            np.arange(1,len(batch_idxs_significant["below"])+1,dtype=int),inplace=True
        )
        return [above_df,below_df]

### Date to non-overlapping sequences

In [ ]:
def find_seqs_predicting_in_date(
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],
        date:pd.Timestamp,
        mode:Literal["up_to_characteristic","with_characteristic"],
        characteristic:Literal["year","month","day","exact"]="exact",
)->pd.DataFrame:
    assert characteristic in ["year","month","day","exact"],"Input not valid for characteristic."
    try:
        assert "time_dummy_to_date" in data[which].keys()
        time_dummy_to_date=copy.deepcopy(data[which]["time_dummy_to_date"])
    except AssertionError:
        time_dummy_to_date=pd.Series(
            pd.date_range(
                data[which]["starting_date"],freq="1h",
                periods=data[which]["df"].shape[0]
            ),
            index=data[which]["df"]["time_idx"]
        )
    if time_dummy_to_date.iloc[0].tz is None:
        if date.tz is not None:
            print("Warning: date is tz-aware while time_dummy_to_date is not, to compare them"\
                  "\ntime_dummy_to_date has been localized in the same tz as date provided.")
            time_dummy_to_date=time_dummy_to_date.dt.tz_localize(date.tz)
    else:
        if date.tz is None:
            print("Warning: date is tz-unaware while time_dummy_to_date is tz-aware, to compare\
                   \nthem date has been localized in the same tz as time_dummy_to_date.")
            date=date.tz_localize(time_dummy_to_date.iloc[0].tz)
    if mode=="up_to_characteristic":
        date_specs=np.array([date.day,date.month,date.year],dtype=int)
        if characteristic=="month":
            date_specs=date_specs[:2]
        if characteristic=="day":
            date_specs=date_specs[0].reshape(1)
    else:
        if characteristic=="day":
            date=date.day
        if characteristic=="month":
            date=date.month
        if characteristic=="year":
            date=date.year
    seqs_in_date={"batch_idx":[],"item_idx":[]}
    already_seen_decoder_time_idxs=torch.tensor([])
    for batch_idx,(x_dict,(_,_)) in enumerate(data[which]["dataloader_list"]):
        for item_idx in range(x_dict["decoder_time_idx"].shape[0]):
            decoder_time_idxs=x_dict["decoder_time_idx"][item_idx,:]
            if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                already_seen_decoder_time_idxs=torch.cat(
                    (already_seen_decoder_time_idxs,decoder_time_idxs)
                )
                decoder_dates=time_dummy_to_date.loc[decoder_time_idxs]
                for decoder_date in decoder_dates:
                    if mode=="up_to_characteristic": 
                        decoder_date_specs=np.array(
                            [decoder_date.day,decoder_date.month,decoder_date.year],
                            dtype=int
                        )
                        if characteristic=="month":
                            decoder_date_specs=decoder_date_specs[:2]
                        if characteristic=="day":
                            decoder_date_specs=decoder_date_specs[0].reshape(1)
                        if characteristic=="exact":
                            if decoder_date==date:
                                seqs_in_date["batch_idx"].append(batch_idx)
                                seqs_in_date["item_idx"].append(item_idx)
                                break                               
                        else:                       
                            if all(decoder_date_specs==date_specs):
                                seqs_in_date["batch_idx"].append(batch_idx)
                                seqs_in_date["item_idx"].append(item_idx)
                                break
                    else:
                        if characteristic=="day":
                            decoder_date=decoder_date.day
                        if characteristic=="month":
                            decoder_date=decoder_date.month
                        if characteristic=="year":
                            decoder_date=decoder_date.year
                        if decoder_date==date:
                            seqs_in_date["batch_idx"].append(batch_idx)
                            seqs_in_date["item_idx"].append(item_idx)
                            break
    seqs_in_date_DF=pd.DataFrame(seqs_in_date,index=np.arange(1,len(seqs_in_date["batch_idx"])+1))
    return seqs_in_date_DF

In [ ]:
def find_seqs_predicting_in_range(
        tft:TemporalFusionTransformer,
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],
        date_range:pd.DatetimeIndex,
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]]|None=None,
        scaler_y=None,
)->pd.DataFrame|tuple[pd.DataFrame,dict[str,float]]:
    try:
        assert "time_dummy_to_date" in data[which].keys()
        time_dummy_to_date=copy.deepcopy(data[which]["time_dummy_to_date"])
    except AssertionError:
        time_dummy_to_date=pd.Series(
            pd.date_range(
                data[which]["starting_date"],freq="1h",
                periods=data[which]["df"].shape[0]
            ),
            index=data[which]["df"]["time_idx"]
        )
    if time_dummy_to_date.iloc[0].tz is None:
        if date_range[0].tz is not None:
            print("Warning: date_range is tz-aware while time_dummy_to_date is not, to compare them"\
                  "\ntime_dummy_to_date has been localized in the same tz as date provided.")
            time_dummy_to_date=time_dummy_to_date.dt.tz_localize(date_range[0].tz)
    else:
        if date_range[0].tz is None:
            print("Warning: date_range is tz-unaware while time_dummy_to_date is tz-aware, to compare\
                   \nthem date_range has been localized in the same tz as time_dummy_to_date.")
            date_range=date_range.dt.tz_localize(time_dummy_to_date.iloc[0].tz)
    seqs_in_range={"batch_idx":[],"item_idx":[]}

    if as_from_predict_raw is not None:
        try:
            try:
                Output=getattr(as_from_predict_raw,"output")
            except AttributeError:
                Output=as_from_predict_raw["output"]
        except AttributeError:
            Output=as_from_predict_raw
        
        try:
            raw_predictions=getattr(Output,"prediction")
        except AttributeError:
            raw_predictions=Output["prediction"]
        quantiles_computed=pd.Series(tft.loss.quantiles)
        quantile_for_predictions=0.5
        if not (quantile_for_predictions in tft.loss.quantiles):
            raise Exception("By default, predictions are represented by the median (q=0.5, a.k.a. P50), but the loss used at TFT's initialization does not compute the median!\nAdd q=0.5 among the quantiles in QuantileLoss() at TFT's construction time.")
        quantile_for_predictions_idx=(quantiles_computed==quantile_for_predictions).idxmax()

        predictions=raw_predictions[:,:,quantile_for_predictions_idx]
        batch_size=data[which]["dataloader"].batch_size
        number_of_batches=len(data[which]["dataloader"])
        batch_to_seqs_slice={
            batch_num:slice(batch_num*batch_size,(batch_num+1)*batch_size) if batch_num!=(number_of_batches-1) else slice(batch_num*batch_size,None) for batch_num in range(number_of_batches)
        }
        start=True
    already_seen_decoder_time_idxs=torch.tensor([])
    for batch_idx,(x_dict,(_,_)) in enumerate(data[which]["dataloader_list"]):
        for item_idx in range(x_dict["decoder_time_idx"].shape[0]):
            decoder_time_idxs=x_dict["decoder_time_idx"][item_idx,:]
            if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                already_seen_decoder_time_idxs=torch.cat(
                    (already_seen_decoder_time_idxs,decoder_time_idxs)
                )
                decoder_dates=time_dummy_to_date.loc[decoder_time_idxs]
                if any(decoder_dates.isin(date_range)):
                    if as_from_predict_raw is not None:
                        y=x_dict["decoder_target"][item_idx]
                        predictions_in_batch=predictions[batch_to_seqs_slice[batch_idx],:]
                        predictions_sequence_in_batch=predictions_in_batch[item_idx,:]                
                        if start:
                            preds=predictions_sequence_in_batch
                            target=y
                            start=False
                        else:    
                            target=torch.cat(
                                (target,y),
                                dim=0
                            )
                            preds=torch.cat(
                                (preds,predictions_sequence_in_batch)
                            )                    
                    seqs_in_range["batch_idx"].append(batch_idx)
                    seqs_in_range["item_idx"].append(item_idx)
    if as_from_predict_raw is not None:
        assert scaler_y is not None,"You must provide the scaler if you want the overall metric along the period."
        target=torch.tensor(
            scaler_y.inverse_transform(target.cpu().reshape((-1,1))).reshape(-1)
        )
        preds=torch.tensor(
            scaler_y.inverse_transform(preds.cpu().reshape((-1,1))).reshape(-1)
        ) 
        from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
        overall_metrics={
            "MAE":mean_absolute_error(target,preds),
            "MSE":mean_squared_error(target,preds),
            "RMSE":root_mean_squared_error(target,preds),
            "MAPE":torch.mean(
                torch.abs(
                    target[target!=0] - preds[target!=0]
                ) / target[target!=0]
            ).item()*100,             
            "R2":r2_score(target,preds),
        }   
    seqs_in_range_DF=pd.DataFrame(seqs_in_range,index=np.arange(1,len(seqs_in_range["batch_idx"])+1))
    if as_from_predict_raw is None:
        return seqs_in_range_DF
    else:
        return seqs_in_range_DF,overall_metrics

In [ ]:
def get_prediction_NOS_in_range_w_percentiles(
        tft:TemporalFusionTransformer,
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],
        date_range:pd.DatetimeIndex,      
        figsize:tuple[float,float]=(16,8),
        linewidth:float=1.5,
        scaler_y=None,
        plotting_time_interval:int=24,
        add_loss_to_title:bool=True,
        sklearn_metric_name:Literal["MAE","MSE","RMSE","MAPE","R2"]|None=None,
        suptitle:str|None=None,
        ylabel="Target",
        ylabel_udm="kWh",
        xlabel="",
        only_legend:bool=False,
        fontsize:float=13,
        dpi:int|None=None,
        formatter:bool=True,
        for_graphical_abstract:bool=False,
        forced_rotation:int|None=None,
        pad:dict[Literal["x","y"]]|None=None,
        stairplot:bool=False,
        )->None|dict[str,dict[str,float|str]]|tuple[plt.Figure,dict[str,dict[str,float|str]]]:
    try:
        try:
            Output=getattr(as_from_predict_raw,"output") 
        except AttributeError:
            Output=as_from_predict_raw["output"]
    except AttributeError:
        Output=as_from_predict_raw
    
    try:
        raw_predictions=getattr(Output,"prediction")
    except AttributeError:
        raw_predictions=Output["prediction"]

    quantiles_computed=pd.Series(tft.loss.quantiles)
    quantile_for_predictions=0.5
    if not (quantile_for_predictions in tft.loss.quantiles):
        raise Exception("By default, predictions are represented by the median (q=0.5, a.k.a. P50), but the loss used at TFT's initialization does not compute the median!\nAdd q=0.5 among the quantiles in QuantileLoss() at TFT's construction time.")
    quantile_for_predictions_idx=(quantiles_computed==quantile_for_predictions).idxmax()

    predictions=raw_predictions[:,:,quantile_for_predictions_idx]

    dataloader=data[which]["dataloader"]
    batch_size=dataloader.batch_size
    number_of_batches=len(dataloader)
    batch_to_first_seqs_in_batch_slice={
        batch_num:batch_num*batch_size for batch_num in range(number_of_batches)
    }
    batch_and_item_in_range,metrics_in_range=find_seqs_predicting_in_range(tft,data,which,date_range,as_from_predict_raw,scaler_y)
    seqs_in_range=[batch_to_first_seqs_in_batch_slice[batch_idx] for batch_idx in batch_and_item_in_range["batch_idx"]]
    predictions_in_batches=predictions[seqs_in_range,:]

    decoder_target=torch.tensor([])
    for batch_idx in batch_and_item_in_range["batch_idx"]: 
        x_dict,_=data[which]["dataloader_list"][batch_idx]
        decoder_target=torch.concat((decoder_target,x_dict["decoder_target"][0,:]))
    predictions_in_batches=torch.tensor(
        scaler_y.inverse_transform(predictions_in_batches.cpu().reshape((-1,1))).reshape(-1)
    )
    decoder_target=torch.tensor(
        scaler_y.inverse_transform(decoder_target.reshape((-1,1))).reshape(-1)
    )
    with plt.rc_context({"font.size":fontsize}):
        fig=plt.figure(figsize=figsize,dpi=dpi)
        ax_predictions=fig.gca()
        prop_cycle = iter(plt.rcParams["axes.prop_cycle"])
        target_color = next(prop_cycle)["color"]
        predictions_color = next(prop_cycle)["color"]
        if not stairplot:
            ax_predictions.plot(date_range,decoder_target,color=target_color,linewidth=linewidth,label="Observed")
            ax_predictions.plot(
                date_range,
                predictions_in_batches.cpu(),
                color=predictions_color,linewidth=linewidth,label="Predicted (P50)" if not for_graphical_abstract else "Predicted")
        else:
            ax_predictions.step(date_range,decoder_target,color=target_color,linewidth=linewidth,label="Observed",where="post")
            ax_predictions.step(
                date_range,
                predictions_in_batches.cpu(),
                color=predictions_color,linewidth=linewidth,label="Predicted (P50)" if not for_graphical_abstract else "Predicted",where="post")
        metrics_udm={
            "MAE":"kWh",
            "MSE":r"kWh$^2$",
            "RMSE":"kWh",
            "MAPE":"%",                
            "R2":""
        }
        metrics_dict={mtc:{"value":metrics_in_range[mtc],"udm":metrics_udm[mtc]} for mtc in metrics_in_range.keys()}
        title_for_loss=" Loss computed on predictions (P50): {}={:.2f} {}.".format(
            sklearn_metric_name,
            metrics_dict[sklearn_metric_name]["value"],
            metrics_dict[sklearn_metric_name]["udm"]
        )
        
        predictions_in_batches_all_quantiles=raw_predictions[seqs_in_range,:,:].cpu()
        for seq_idx in range(predictions_in_batches_all_quantiles.shape[0]):
            for quantile_idx in quantiles_computed.index:
                predictions_in_batches_all_quantiles[seq_idx,:,quantile_idx]=torch.tensor(scaler_y.inverse_transform(predictions_in_batches_all_quantiles[seq_idx,:,quantile_idx].squeeze().reshape((-1,1))).reshape((1,-1)))

        predictions_in_batches_all_quantiles_concat=torch.tensor([])
        for seq_idx in range(predictions_in_batches_all_quantiles.shape[0]):
            predictions_in_batches_all_quantiles_concat=torch.concat(
                (predictions_in_batches_all_quantiles_concat,predictions_in_batches_all_quantiles[seq_idx,:,:]),dim=0
            )
        alphas=np.linspace(
            0.15,
            0.45,
            max(
                len(quantiles_computed.loc[quantiles_computed<quantile_for_predictions]),
                len(quantiles_computed.loc[quantiles_computed>quantile_for_predictions])
            )
        )
        n_additional_cols=0
        for order in ["smaller","larger"]:
            if order=="smaller":
                mask=quantiles_computed<quantile_for_predictions
            else:
                mask=quantiles_computed>quantile_for_predictions
            count=0 if order=="smaller" else len(alphas)-1
            masked_computed_quantiles=quantiles_computed.loc[mask] if order=="smaller" else quantiles_computed.loc[mask]
            for quantile_idx in masked_computed_quantiles.index:
                ax_predictions.fill_between(
                    date_range,
                    predictions_in_batches_all_quantiles_concat[:,quantile_idx if order=="smaller" else quantile_idx-1],
                    predictions_in_batches_all_quantiles_concat[:,quantile_idx+1 if order=="smaller" else quantile_idx],
                    alpha=alphas[count],
                    fc=predictions_color,
                    label="[P{:.0f}, P{:.0f}]".format(
                        quantiles_computed.loc[quantile_idx if order=="smaller" else quantile_idx-1]*100,
                        quantiles_computed.loc[quantile_idx+1 if order=="smaller" else quantile_idx]*100,
                    ) if not for_graphical_abstract else None,
                    step=None if not stairplot else "post"
                )
                count=count+1 if order=="smaller" else count-1
                n_additional_cols+=1
        ax_predictions.set_ylabel(ylabel)
        ax_predictions.set_xlabel(xlabel) 
        title="Predictions for sequences from {} to {}.".format(
            date_range.min().strftime("%d %B %Y"),date_range.max().strftime("%d %B %Y")
        )
        title=title+title_for_loss if add_loss_to_title else title
        if not for_graphical_abstract:
            ax_predictions.legend(
                loc="lower center",
                ncols=2+n_additional_cols,
                bbox_to_anchor=(0.5,1)
            )
        else:
            ax_predictions.legend(
                loc="lower center",
                ncols=2,
                bbox_to_anchor=(0.5,1)
            )
        ticks=pd.date_range(
            start=date_range.min(),
            end=date_range.max(),
            freq="{}h".format(plotting_time_interval),
        )
        if for_graphical_abstract:
            print("plotting_time_interval automatically set to 24 because of for_graphical_abstract=True.")
            plotting_time_interval=24
        if plotting_time_interval<24 and 24%plotting_time_interval==0:
            labels=[]
            for date_idx,date in enumerate(date_range):
                if date_idx%plotting_time_interval==0:
                    if date_idx%24==0:
                        labels.append(date.strftime("%a %Y/%m/%d %H:%M"))
                    else:
                        labels.append(date.strftime("%H:%M"))
        else:
            if not for_graphical_abstract:
                labels=[
                    date.strftime("%a %Y/%m/%d %H:%M") for date in date_range[slice(0,None,plotting_time_interval)]
                ] 
            else:
                labels=[
                    date.strftime("%A") for date in date_range[slice(0,None,plotting_time_interval)]
                ]
        ax_predictions.set_xticks(
            ticks=ticks,
            labels=labels,
            rotation=90
        )
        if forced_rotation is not None:
            ax_predictions.set_xticks(
            ticks=ax_predictions.get_xticks(),
            labels=ax_predictions.get_xticklabels(),
            rotation=forced_rotation
        )
        if pad is not None:
            ax_predictions.tick_params(axis='x', pad=pad["x"])  
            ax_predictions.tick_params(axis='y', pad=pad["y"])  
        if not for_graphical_abstract:
            ax_predictions.yaxis.set_major_locator(MaxNLocator())
        else:
            ax_predictions.yaxis.set_major_locator(MaxNLocator(nbins=5))
        if formatter:
            ax_predictions.yaxis.set_major_formatter("{x} "+ylabel_udm)
        ax_predictions.set_xlim((date_range.min(),date_range.max()))
        ax_predictions.set_ylim(bottom=0)
        for_suptitle=suptitle if suptitle is not None else ""
        str_percentiles=["P{:.0f}".format(quantile*100) for quantile in tft.loss.quantiles]
        str_concatenated_percentiles="["+", ".join(str_percentiles)+"]."
        if not only_legend:
            fig.suptitle(for_suptitle+" Percentiles computed: "+str_concatenated_percentiles+"\n"+title)
        fig.tight_layout()
        plt.close("all")
        display(fig)

In [ ]:
def all_seqs_predicting_in_range(
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],
        date_range:pd.DatetimeIndex,
)->pd.DataFrame|tuple[pd.DataFrame,dict[str,float]]:
    try:
        assert "time_dummy_to_date" in data[which].keys()
        time_dummy_to_date=copy.deepcopy(data[which]["time_dummy_to_date"])
    except AssertionError:
        time_dummy_to_date=pd.Series(
            pd.date_range(
                data[which]["starting_date"],freq="1h",
                periods=data[which]["df"].shape[0]
            ),
            index=data[which]["df"]["time_idx"]
        )
    if time_dummy_to_date.iloc[0].tz is None:
        if date_range[0].tz is not None:
            print("Warning: date_range is tz-aware while time_dummy_to_date is not, to compare them"\
                  "\ntime_dummy_to_date has been localized in the same tz as date provided.")
            time_dummy_to_date=time_dummy_to_date.dt.tz_localize(date_range[0].tz)
    else:
        if date_range[0].tz is None:
            print("Warning: date_range is tz-unaware while time_dummy_to_date is tz-aware, to compare\
                   \nthem date_range has been localized in the same tz as time_dummy_to_date.")
            date_range=date_range.dt.tz_localize(time_dummy_to_date.iloc[0].tz)
    seqs_in_range={"batch_idx":[],"item_idx":[]}
    for batch_idx,(x_dict,(_,_)) in enumerate(data[which]["dataloader_list"]):
        for item_idx in range(x_dict["decoder_time_idx"].shape[0]):
            decoder_time_idxs=x_dict["decoder_time_idx"][item_idx,:]
            decoder_dates=time_dummy_to_date.loc[decoder_time_idxs]
            if any(decoder_dates.isin(date_range)):
                seqs_in_range["batch_idx"].append(batch_idx)
                seqs_in_range["item_idx"].append(item_idx)
    seqs_in_range_DF=pd.DataFrame(seqs_in_range,index=np.arange(1,len(seqs_in_range["batch_idx"])+1))
    return seqs_in_range_DF

### Display

In [ ]:
def make_figures_dict_from_dfs(
        tft:TemporalFusionTransformer,
        as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
        data:dict[Literal["train","val","test"],
                  dict[str,
                       pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
        which:Literal["train","val","test"],                
        dfs_list:list[pd.DataFrame],
        dfs_names:list[str],
        feature_names_per_channel:dict[str, list[str]],
        sklearn_metric_name:Literal["MAE","MSE","RMSE","MAPE","R2"],
        scaler_y,
        channel_names:list[Literal['static_variables', 'encoder_variables', 'decoder_variables']],
        VSWs_kwargs:dict[Literal["reduction","quantile","figsize","additional_title"],]={},
        attention_kwargs:dict[Literal["reduction","plotting_time_interval","head_idx",
                                      "prediction_idx","figsize","detail_representatives",
                                      "representative_quantiles","detail_prediction_timesteps",
                                      "prediction_idxs_to_plot","add_pre_suptitle",
                                      "use_dates","linewidth"],]={},
        prediction_plot_kwargs:dict[Literal["figsize","linewidth","show_rescaled",
                                            "plotting_time_interval","suptitle",
                                            "use_dates","explicit_fill_zones"],]={}        
        )->dict[str,dict[int,dict[str,list[plt.Figure]|plt.Figure]]]:
    modified_dfs_list=[]
    for i,df in enumerate(dfs_list):
        try:
            assert len(df.index.unique())==len(df.index)
            modified_dfs_list.append(copy.deepcopy(df))
        except:
            print("Warning: {} does not have unique values in the index. \nA copy of the input with default index has been made so to make the function work.\nConsider changing the index values in case they meant something and you want to keep them.".format(dfs_names[i]))
            modified_dfs_list.append(copy.deepcopy(df.reset_index()))
    figures_dict={df_name:{} for df_name in dfs_names}
    for df_idx,df in enumerate(modified_dfs_list):
        name=dfs_names[df_idx]
        for idx in df.index:
            batch_idx=df.loc[idx,"batch_idx"]
            item_idx=df.loc[idx,"item_idx"]
            figures_dict[name][idx]={}
            figures_dict[name][idx]={"VSWs":[]}
            for channel_name in channel_names:
                figures_dict[name][idx]["VSWs"].append(
                    get_VSWs(
                        as_from_predict_raw,data,which,feature_names_per_channel,
                        channel_name,batch_idx,item_idx,
                        print_info=False,return_figure=True,**VSWs_kwargs
                    )
                )
            figures_dict[name][idx]["Attentions"]=get_attention(
                as_from_predict_raw,data,which,
                normalize_attention_by_position_idxs_sum=True,
                batch_idx=batch_idx,item_idx=item_idx,
                set_attention_threshold=True,custom_ticks=True,
                print_info=False,return_figure=True,**attention_kwargs,
            )
            figures_dict[name][idx]["Predictions"]=get_prediction_plot(
                tft,as_from_predict_raw,data,which,batch_idx,item_idx,
                custom_ticks=True,loss_as_from_built_in_implementation=False,
                sklearn_metric_name=sklearn_metric_name,return_figure=True,
                scaler_y=scaler_y,**prediction_plot_kwargs
            )
    return figures_dict

In [ ]:
from typing import Literal
def display_dfs(
        dfs_list:list[pd.DataFrame],
        dfs_names:list[str],
        start_to_end:list[tuple[int|str,int|str]|Literal["all"]],
        width:str="850px",
        height:str="200px"
    ):
    import ipywidgets as widgets
    tab_contents=[]
    for i in range(len(dfs_list)):
        tab_content=widgets.Output()
        with tab_content:
            df_sliced=dfs_list[i].loc[start_to_end[i][0]:start_to_end[i][1],:] if start_to_end[i]!="all" else dfs_list[i]
            display(df_sliced)
        tab_contents.append(tab_content)
    scrollable_layout=widgets.Layout(overflow_x="auto",overflow_y="auto",width=width,height=height)
    tabs=widgets.Tab(
        children=[
            widgets.Box([tab_content],layout=scrollable_layout) for tab_content in tab_contents
        ]
    )
    for i in range(len(dfs_list)):
        tabs.set_title(i,dfs_names[i])
    display(tabs)

In [ ]:
def display_figures(
        figures_dict:dict[str,dict[int,dict[str,list[plt.Figure]|plt.Figure]]],
        from_who:str,
        which_characteristics:list[Literal["VSWs","Attentions","Predictions"]]|Literal["all"],
        which_positions:list[int|str]|np.ndarray|Literal["all"],    
        use_tabs:bool=True,
        )->None:
    import ipywidgets as widgets
    tab_contents=[]
    if not (isinstance(which_positions,list) or isinstance(which_positions,np.ndarray)):
        if which_positions=="all":
            which_positions=figures_dict[from_who].keys()
    if which_characteristics=="all":
        which_characteristics=["VSWs","Attentions","Predictions"]
    for position in which_positions:
        if use_tabs:
            tab_content=widgets.Output()
        else:
            print("----n°{}----".format(position))
        for characteristic in which_characteristics:
            if use_tabs:
                with tab_content:
                    if characteristic=="VSWs":
                        for idx in range(len(figures_dict[from_who][position][characteristic])):
                            display(figures_dict[from_who][position][characteristic][idx])
                    else:
                        display(figures_dict[from_who][position][characteristic])
            else:
                if characteristic=="VSWs":
                    for idx in range(len(figures_dict[from_who][position][characteristic])):
                        display(figures_dict[from_who][position][characteristic][idx])
                else:
                    display(figures_dict[from_who][position][characteristic])
        if use_tabs:
            tab_contents.append(tab_content)
    if use_tabs:
        tabs=widgets.Tab(children=tab_contents)
        for idx,position in enumerate(which_positions):
            tabs.set_title(idx,"n°{}".format(position))
        display(tabs)


In [ ]:
def pair_metrics_with_batch_and_item_df(
        all_seqs_metrics_df:pd.DataFrame,
        batch_and_item_df:pd.DataFrame,
        judgment:Literal["Percentile","In the percent best performing","MinMaxScaler"],
        all_dev:pd.DataFrame|None=None,
        not_overlapped_seqs_batch_and_item_df:pd.DataFrame|None=None
        )->pd.DataFrame:
    from scipy.stats import percentileofscore
    assert all(["batch_idx" in batch_and_item_df,"item_idx" in batch_and_item_df]),"batch_and_item_df does not have proper column names, see docstring."
    assert all(["batch_idx" in all_seqs_metrics_df,"item_idx" in all_seqs_metrics_df]),"all_seqs_metrics_df does not have proper column names, see docstring."
    if all_dev is not None:
        assert all(["batch_idx" in all_dev,"item_idx" in all_dev]),"all_dev does not have proper column names, see docstring."
        assert all(
            [
                all(all_dev.index==all_seqs_metrics_df.index),
                all(all_dev.batch_idx==all_seqs_metrics_df.batch_idx),
                all(all_dev.item_idx==all_seqs_metrics_df.item_idx)
            ]
        ),"all_dev and all_seqs_metrics_df are expected to share index, batch_idx and item_idx values."
    batch_and_item_to_comaniciu=copy.deepcopy(all_dev)
    seqs_metrics=copy.deepcopy(all_seqs_metrics_df)
    seqs_to_pair=copy.deepcopy(batch_and_item_df)
    new_cols={metric:[] for metric in seqs_metrics.columns if not metric in ['batch_idx','item_idx']}
    metrics=[metric for metric in new_cols.keys()]
    better_if_lower=["MAE","MSE","RMSE","MAPE"]
    if all_dev is not None:
        assert judgment!="In the percent best performing","It does not make sense to say that Comaniciu is in the top % performing,\
             \nchange judgment='In the percent best performing' to something else."
        seqs_metrics=pd.concat(
            [seqs_metrics,batch_and_item_to_comaniciu.loc[:,"Comaniciu"]],
            axis=1
        )
        metrics.append("Comaniciu")
        new_cols["Comaniciu"]=[]
    if judgment=="MinMaxScaler":
        only_metrics=copy.deepcopy(seqs_metrics.loc[:,metrics])
        only_batch_and_item=copy.deepcopy(seqs_metrics.loc[:,["batch_idx","item_idx"]])
        metrics_scaler=MinMaxScaler(feature_range=(0,1))
        if not_overlapped_seqs_batch_and_item_df is not None:
            metrics_scaler.fit(
                only_metrics.loc[
                    (only_batch_and_item.batch_idx.isin(not_overlapped_seqs_batch_and_item_df.batch_idx))&(only_batch_and_item.item_idx.isin(not_overlapped_seqs_batch_and_item_df.item_idx)),:
                ]
            )
        else:
            metrics_scaler.fit(only_metrics)
        scaled_seqs_metrics=pd.concat(
            [only_batch_and_item,pd.DataFrame(
                metrics_scaler.transform(only_metrics),
                index=seqs_metrics.index,
                columns=metrics
            )],axis=1
        )
    if not_overlapped_seqs_batch_and_item_df is None:
        over="(all sequences)"
    else:
        over="(non-overlapping sequences)"
    for col in metrics:
        new_cols["{} {} {}".format(judgment,over,col)]=[]
    for row_label in seqs_to_pair.index:
        batch_idx=seqs_to_pair.loc[row_label,"batch_idx"]
        item_idx=seqs_to_pair.loc[row_label,"item_idx"]
        for metric_name in metrics:
            score=seqs_metrics.loc[
                (seqs_metrics.batch_idx==batch_idx)&(seqs_metrics.item_idx==item_idx),
                metric_name
            ].squeeze()
            new_cols[metric_name].append(score)
            if judgment!="MinMaxScaler":
                if not_overlapped_seqs_batch_and_item_df is not None:
                    set_of_values_to_contextualize=seqs_metrics.loc[
                        (seqs_metrics.batch_idx.isin(not_overlapped_seqs_batch_and_item_df.batch_idx))&(seqs_metrics.item_idx.isin(not_overlapped_seqs_batch_and_item_df.item_idx)),metric_name
                    ]
                else:
                    set_of_values_to_contextualize=seqs_metrics.loc[:,metric_name]
                pos=percentileofscore(
                    a=set_of_values_to_contextualize,
                    score=score
                )
                if judgment=="In the percent best performing":
                    if metric_name not in better_if_lower:
                        pos=100-pos
                new_cols["{} {} {}".format(judgment,over,metric_name)].append(
                    pos
                )                    
            else:
                new_cols["{} {} {}".format(judgment,over,metric_name)].append(
                    scaled_seqs_metrics.loc[
                        (scaled_seqs_metrics.batch_idx==batch_idx)&(scaled_seqs_metrics.item_idx==item_idx),
                        metric_name
                    ].squeeze()
                )                
    new_cols=pd.DataFrame(new_cols,index=seqs_to_pair.index)
    return pd.concat([seqs_to_pair,new_cols],axis=1)

### TimeHandler

In [ ]:
class TimeHandler():
    def __init__(
            self,
            data:dict[Literal["train","val","test"],
                            dict[str,
                            pd.DataFrame|pd.Timestamp|torch.utils.data.DataLoader|list[tuple[dict,tuple]]]],
            which:Literal["train","val","test"],
            all_seqs_metrics_df:pd.DataFrame,
            all_dev:pd.DataFrame|None=None,
            not_overlapped_seqs_batch_and_item_df:pd.DataFrame|None=None,
            non_fixed_or_regional_holidays:dict[str,
                                                dict[Literal["date","mode","characteristic"],
                                                    Union[pd.Timestamp,str]]]={},      
    )->None:
        self._data=copy.deepcopy(data)
        self._which=which
        self._periods=None
        self._all_seqs_metrics_df=copy.deepcopy(all_seqs_metrics_df)
        self._all_dev=copy.deepcopy(all_dev)
        self._not_overlapped_seqs_batch_and_item_df=copy.deepcopy(not_overlapped_seqs_batch_and_item_df)
        self._non_fixed_or_regional_holidays=non_fixed_or_regional_holidays
        self._holidays={
            "Capodanno":dict(
                date=pd.Timestamp(
                    year=2020,month=1,day=1
                ),
            ),
            "Epifania":dict(
                date=pd.Timestamp(
                    year=2020,month=1,day=6
                ),          
            ),
            "Liberazione dal nazifascismo":dict(
                date=pd.Timestamp(
                    year=2020,month=4,day=25
                ),          
            ),
            "Festa del lavoro":dict(
                date=pd.Timestamp(
                    year=2020,month=5,day=1
                ),         
            ),
            "Festa della Repubblica":dict(
                date=pd.Timestamp(
                    year=2020,month=6,day=2
                ),         
            ),
            "Ferragosto":dict(
                date=pd.Timestamp(
                    year=2020,month=8,day=15
                ),         
            ),
            "Ognissanti":dict(
                date=pd.Timestamp(
                    year=2020,month=11,day=1
                ),         
            ),
            "Immacolata Concezione":dict(
                date=pd.Timestamp(
                    year=2020,month=12,day=8
                ),        
            ),
            "Natale":dict(
                date=pd.Timestamp(
                    year=2020,month=12,day=25
                ),         
            ),
            "Santo Stefano":dict(
                date=pd.Timestamp(
                    year=2020,month=12,day=26
                ),            
            ),
        }
        for hld_dict in self._holidays.values():
            hld_dict.update(
                dict(
                    mode="up_to_characteristic",
                    characteristic="month",
                )
            )
        self._national_fixed_holidays=copy.deepcopy(self._holidays)
        self._holidays.update(self._non_fixed_or_regional_holidays)
        for value in self._holidays.values():
            value.update({"which":which})
    def compute_metrics_in_year(self,tft,as_from_predict_raw,scaler_y):
        metrics_in_year={yr:{} for yr in ["2019","2020","2021","2022","2023"]}
        each_year_date_range={
            yr:pd.date_range(
                start=pd.Timestamp(year=int(yr),month=1,day=1,hour=0,minute=0),
                end=pd.Timestamp(year=int(yr),month=12,day=31,hour=23,minute=0),
                freq="h"
            ) for yr in metrics_in_year.keys()
        }
        year_to_Comaniciu_avg_dict={}
        for yr in metrics_in_year.keys():
            seqs_in_year,metrics_in_year[yr]=find_seqs_predicting_in_range(
                tft,self._data,self._which,
                each_year_date_range[yr],
                as_from_predict_raw,scaler_y
            )
            year_to_Comaniciu_avg_dict[yr]=pair_metrics_with_batch_and_item_df(
                self._all_seqs_metrics_df,seqs_in_year,"Percentile",self._all_dev,
            )["Comaniciu"].mean()
        year_to_Comaniciu_avg=pd.Series(year_to_Comaniciu_avg_dict.values(),index=year_to_Comaniciu_avg_dict.keys(),name="Comaniciu")
        self._metrics_in_year=metrics_in_year
        self._year_to_Comaniciu_avg=year_to_Comaniciu_avg
    def compute_holidays_performance(
            self,
            judgment:Literal["Percentile","In the percent best performing","MinMaxScaler"],
            to_be_grouped:dict[str,list[str]]|None,
            ):
        batch_and_item_df_in_each_holiday={
            holiday_name:find_seqs_predicting_in_date(
                self._data,**holiday_dict
            ) for holiday_name,holiday_dict in self._holidays.items()
        }
        performance_in_each_holiday={
            holiday_name:pair_metrics_with_batch_and_item_df(
                self._all_seqs_metrics_df,holiday_batch_and_item_df,judgment,self._all_dev,
                self._not_overlapped_seqs_batch_and_item_df
            ) for holiday_name,holiday_batch_and_item_df in batch_and_item_df_in_each_holiday.items()
        }
        if to_be_grouped is not None:
            for grouped_holiday_name,original_holiday_names in to_be_grouped.items():
                assert grouped_holiday_name not in performance_in_each_holiday.keys(),"{} grouped holiday should not match any of the provided holidays at initialization, but it does.".format(grouped_holiday_name)
                print("Grouped holiday creation: {}".format(grouped_holiday_name),flush=True)
                grouped_holiday_performance=pd.DataFrame()
                for original_holiday_name in original_holiday_names:
                    assert original_holiday_name in performance_in_each_holiday.keys(),"{} was not provided at initialization time."
                    grouped_holiday_performance=pd.concat(
                        [grouped_holiday_performance,performance_in_each_holiday[original_holiday_name]],
                        axis=0
                    )
                    _=performance_in_each_holiday.pop(original_holiday_name)
                    print("{} joined to the group and removed.".format(original_holiday_name),flush=True)
                performance_in_each_holiday[grouped_holiday_name]=grouped_holiday_performance
                print("{} grouped holiday successfully added.".format(grouped_holiday_name),flush=True)
                print("\n",flush=True)
        all_holidays_performance=pd.DataFrame()
        for holiday_name,performance_df in performance_in_each_holiday.items():
            performance_df=copy.deepcopy(performance_df)
            performance_df[holiday_name]=[holiday_name for _ in range(performance_df.shape[0])]
            performance_df.set_index(
                holiday_name,
                drop=True,
                append=True,
                inplace=True
            )
            all_holidays_performance=pd.concat(
                [all_holidays_performance,performance_df],
                axis=0,
            )
        self._all_holidays_performance=all_holidays_performance
        self._holiday_names=self._all_holidays_performance.index.get_level_values(1).unique()
        self._single_index_all_holidays_performance=copy.deepcopy(self._all_holidays_performance)
        self._single_index_all_holidays_performance.reset_index(level=1,inplace=True)
        self._single_index_all_holidays_performance.rename({"level_1":"Holiday name"},axis=1,inplace=True)        
    def save_holidays_performance(self,path_to_csv)->None:
        self._all_holidays_performance.to_csv(path_to_csv)
    def display_holidays_performance(
            self,
            holiday_name:str|None=None,
            display_dfs_kwargs:dict[Literal["width","height"],str]={},
        )->None:
        if holiday_name is not None:
            assert holiday_name in self._holiday_names,"The holiday you look for is not in the ones nationally fixed nor custom provided."
            to_be_displayed=copy.deepcopy(
                self._single_index_all_holidays_performance.loc[
                    self._single_index_all_holidays_performance.loc[:,"Holiday name"]==holiday_name,:
                ]
            )
        else:
            to_be_displayed=copy.deepcopy(self._all_holidays_performance)
        display_dfs(
            dfs_list=[to_be_displayed],
            dfs_names=["Holidays"],
            start_to_end=["all"],
            **display_dfs_kwargs
        )
    def show_holidays_plots(
            self,
            holiday_name:str,
            make_figures_dict_from_dfs_kwargs:dict[
                Literal["tft","as_from_predict_raw","feature_names_per_channel",
                        "sklearn_metric_name","scaler_y","channel_names","VSWs_kwargs",
                        "attention_kwargs","prediction_plot_kwargs"]
            ]={},
            display_figures_kwargs:dict[Literal["which_characteristics","which_positions","use_tabs"]]={}
        )->None:
        assert holiday_name in self._holiday_names,"The holiday you look for is not in the ones nationally fixed nor custom provided."
        to_be_displayed=copy.deepcopy(
            self._single_index_all_holidays_performance.loc[
                self._single_index_all_holidays_performance.loc[:,"Holiday name"]==holiday_name,:
            ]
        )
        holidays_figs=make_figures_dict_from_dfs(
            data=self._data,
            which=self._which,
            dfs_list=[to_be_displayed],
            dfs_names=[holiday_name],
            **make_figures_dict_from_dfs_kwargs
        )
        display_figures(
            figures_dict=holidays_figs,
            from_who=holiday_name,
            **display_figures_kwargs
        )
    def holidays_summary_plot(
            self,
            metric_name:Literal["MAE","MSE","RMSE","MAPE","R2","Comaniciu"],
            by:Literal["Year","Day"]="Year",
            exclude_holidays:list[str]|None=None,
            exclude_hue_values:list[str]|None=None,
            fontsize:float=9,
            rotation:float=90,
            size:float=8,
            figure_kwargs:dict={"figsize":(13,6)},
            lineprops:dict=dict(linestyle="-.",linewidth=0.85),
            custom_ylims:tuple[float|None,float|None]|None=None,
            display_days_statistics:bool=False,
            custom_order:dict[str,int]|None=None,
            jitter:bool=False,
            swarmplot:bool=False,
            og_to_new:dict[str,str]|None=None,
            legend:bool=False,
            ref_fig:Figure|None=None,
            return_ax_and_fig:bool=False,
            broken_axis:dict[Literal["non_outliers_axis_lim","outliers_axis_lim","hspace","left_offset_ylabel"]]|None=None,
            formatter_USA_no_perc:bool=True,
            forced_ylabel:str|None=None
        )->None:
        with plt.rc_context({'font.size': fontsize}):
            df=copy.deepcopy(self._single_index_all_holidays_performance)
            year=[]
            day=[]
            for batch_idx,item_idx in zip(df["batch_idx"],df["item_idx"]):
                first_decoder_date=item_in_batch_to_dates(
                    self._data,self._which,item_idx,batch_idx,
                    consecutive_time_steps=not "time_dummy_to_date" in self._data[self._which].keys()
                ).iloc[-n_future]
                first_decoder_date_year=first_decoder_date.year
                first_decoder_date_day=first_decoder_date.strftime("%A")
                day.append(first_decoder_date_day)
                year.append(str(first_decoder_date_year))
            df["Day"]=day
            df["Year"]=year
            metrics_in_year={yr:mtc_dict for yr,mtc_dict in self._metrics_in_year.items() if yr in df["Year"].unique()}
            for mtc in ["MAE","MSE","RMSE","MAPE","R2"]:
                df[mtc+" in year"]=year
                df[mtc+" in year"]=df[mtc+" in year"].map(lambda yr:metrics_in_year[yr][mtc])
                df[mtc+" relative difference"]=(df[mtc]-df[mtc+" in year"])/df[mtc+" in year"]*100
            year_to_Comaniciu_avg=copy.deepcopy(self._year_to_Comaniciu_avg.loc[np.sort(df["Year"].unique()).tolist()])
            if custom_order is not None:
                assert all(pd.Series(custom_order.keys()).isin(self._holiday_names)),"Some names in custom_order are not within the holiday names saved."
                df["Holiday order"]=copy.deepcopy(df["Holiday name"].apply(lambda hdn:custom_order[hdn]))
                df.sort_values(by="Holiday order",ascending=True,inplace=True)
            if exclude_holidays:
                df=df.loc[~df["Holiday name"].isin(exclude_holidays),:]
            if exclude_hue_values:
                df=df.loc[~df[by].isin(exclude_hue_values),:]
            if broken_axis is None:
                fig=plt.figure(**figure_kwargs)
                ax=fig.gca()
            else:
                fig,(ax_out, ax)=plt.subplots(2,1,sharex=True,**figure_kwargs)
                fig.subplots_adjust(hspace=broken_axis["hspace"])  
            if metric_name=="Comaniciu":
                y=metric_name
            else:
                y=metric_name+" relative difference"
            if custom_ylims is not None:
                ymin=custom_ylims[0]
                ymax=custom_ylims[1]
                if ymin is not None and ymax is not None:
                    df=df.loc[(df[y]>=ymin)&(df[y]<=ymax)]
                else:
                    if ymin is None:
                        df=df.loc[df[y]<=ymax]
                    else:
                        df=df.loc[df[y]>=ymin]
            if by=="Year":
                ordered_hue=np.sort(df[by].unique())
            else:
                day_names=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
                numeric_sorted_day=np.sort(np.array([day_names.index(day) for day in df[by].unique()]))
                ordered_hue=np.array([day_names[idx] for idx in numeric_sorted_day])
            if broken_axis is None:
                if not swarmplot:
                    sns.stripplot(
                        data=df,x="Holiday name",y=y,
                        hue=by,jitter=jitter,ax=ax,size=size,
                        hue_order=ordered_hue,
                    )
                else:
                    sns.swarmplot(
                        data=df,x="Holiday name",y=y,
                        hue=by,ax=ax,size=size,
                        hue_order=ordered_hue,
                        warn_thresh=0
                    )
            else:
                if not swarmplot:
                    sns.stripplot(
                        data=df,x="Holiday name",y=y,
                        hue=by,jitter=jitter,ax=ax_out,size=size,
                        hue_order=ordered_hue,
                    )
                    sns.stripplot(
                        data=df,x="Holiday name",y=y,
                        hue=by,jitter=jitter,ax=ax,size=size,
                        hue_order=ordered_hue,
                    )
                else:
                    sns.swarmplot(
                        data=df,x="Holiday name",y=y,
                        hue=by,ax=ax_out,size=size,
                        hue_order=ordered_hue,
                        warn_thresh=0
                    )
                    sns.swarmplot(
                        data=df,x="Holiday name",y=y,
                        hue=by,ax=ax,size=size,
                        hue_order=ordered_hue,
                        warn_thresh=0
                    )
                ax_out.set_ylim(broken_axis["outliers_axis_lim"]) 
                ax.set_ylim(broken_axis["non_outliers_axis_lim"])  
                
                ax_out.spines.bottom.set_visible(False)
                ax_out.tick_params(axis='x', which='both', bottom=False, top=False)
                ax.spines.top.set_visible(False)
                ax.xaxis.tick_bottom()

                d=.5 
                kwargs=dict(
                    marker=[(-1,-d),(1,d)], markersize=12,
                    linestyle="none",
                    color='k',
                    mec='k', 
                    mew=1, 
                    clip_on=False
                )
                ax_out.plot([0, 1], [0, 0], transform=ax_out.transAxes, **kwargs)
                ax.plot([0, 1], [1, 1], transform=ax.transAxes, **kwargs)
                ax_out.set_ylabel("")
                ax.set_ylabel("")
            if metric_name=="Comaniciu" and by=="Year":
                palette=sns.color_palette()  
                color_mapping={category:palette[i] for i,category in enumerate(ordered_hue)}
                xticks=np.array(ax.get_xticks())
                for yr in np.sort(df["Year"].unique()):
                    ax.plot(xticks,np.ones(xticks.shape)*year_to_Comaniciu_avg.loc[yr],
                            color=color_mapping[yr],label=yr+" average",**lineprops)
            ax.set_xlabel("")
            if metric_name!="Comaniciu":
                if broken_axis is None:
                    if forced_ylabel is None:
                        ax.set_ylabel("{} (relative difference to the yearly {})".format(metric_name,metric_name))
                    else:
                        ax.set_ylabel(forced_ylabel)
                    if not formatter_USA_no_perc:
                        ax.yaxis.set_major_formatter("{x}%")
                    else:
                        ax.yaxis.set_major_formatter(
                            matplotlib.ticker.StrMethodFormatter("{x:,.1f}")
                            )
                else:
                    fig.supylabel(
                        "{} (relative difference to the yearly {})".format(
                            metric_name,metric_name
                            ) if forced_ylabel is None else forced_ylabel,
                        x=broken_axis['left_offset_ylabel'],
                        fontsize=fontsize
                        )
                    if not formatter_USA_no_perc:
                        ax.yaxis.set_major_formatter("{x}%")
                        ax_out.yaxis.set_major_formatter("{x}%")
                    else:
                        ax.yaxis.set_major_formatter(
                            matplotlib.ticker.StrMethodFormatter("{x:,.1f}")
                            )
                        ax_out.yaxis.set_major_formatter(
                            matplotlib.ticker.StrMethodFormatter("{x:,.1f}")
                            )
            xticks=ax.get_xticks()
            xticks_labels=ax.get_xticklabels()
            if og_to_new is not None:
                xticks_labels=[og_to_new[lb.get_text()] for lb in xticks_labels]
            ax.set_xticks(ticks=xticks,labels=xticks_labels,rotation=rotation)
            handles,labels=ax.get_legend_handles_labels()
            if metric_name=="Comaniciu" and by=="Year":
                label_to_handle={label:handle for handle,label in zip(handles,labels)}
                labels=np.sort(labels)
                handles=[label_to_handle[label] for label in labels]
            ax.legend_.remove()
            if broken_axis is None:
                ax.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1),ncol=df[by].nunique())
                if not legend:
                    lg=ax.get_legend()
                    lg.set_visible(legend)
                ax.grid(visible=True,alpha=0.5)
            else:
                ax_out.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1),ncol=df[by].nunique())
                if not legend:
                    lg=ax_out.get_legend()
                    lg.set_visible(legend)
                ax_out.grid(visible=True,alpha=0.5)
                ax.grid(visible=True,alpha=0.5)
            if ref_fig is not None:
                match_data_area(ref_fig,fig)
            plt.close("all")
            display(fig)
            if display_days_statistics:
                if custom_ylims is None:
                    print("Warning: statistics concerning the day<->performance make sense if you set custom_ylims.")
                value_counts=df["Day"].value_counts()
                non_weekend_count=value_counts.loc[~value_counts.index.isin(["Saturday","Sunday"])].sum()
                display(value_counts)
                print("Non-weekend days over total days in ylims: {}/{}".format(non_weekend_count,value_counts.sum()))
            if return_ax_and_fig:
                if broken_axis is None:
                    return ax,fig
                else:
                    return ax,ax_out,fig
    def add_period(
            self,
            tft:TemporalFusionTransformer,
            period_name:str,
            period:pd.DatetimeIndex,
            judgment:Literal["Percentile","In the percent best performing","MinMaxScaler"],
            as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],
            scaler_y,
        )->None:
        period_id="From {} to {}".format(
            period[0].strftime("%Y/%m/%d_h%H"),
            period[-1].strftime("%Y/%m/%d_h%H")
        )
        batch_and_item_in_range_df,aggregate_metrics_dict=find_seqs_predicting_in_range(
            tft,self._data,self._which,
            period,as_from_predict_raw,scaler_y
        )
        period_performance=pair_metrics_with_batch_and_item_df(
            self._all_seqs_metrics_df,
            batch_and_item_in_range_df,judgment,
            self._all_dev,self._not_overlapped_seqs_batch_and_item_df
        )
        period_dict={
            period_id:[period_performance,aggregate_metrics_dict]
        }
        if self._periods is None:
            self._periods={period_name:period_dict}
        else:
            if period_name in self._periods.keys():
                self._periods[period_name].update(period_dict)
            else:
                self._periods[period_name]=period_dict
    def compute_periods_performance(self)->None:
        _,d=next(iter(self._periods.items()))
        i=next(iter(d)) 
        aggregate_metrics={mtc:[] for mtc in d[i][1].keys()}
        aggregate_metrics["Period name"]=[]
        aggregate_metrics["ID"]=[]
        aggregate_metrics["Comaniciu (average in ID)"]=[]
        all_periods_performance=pd.DataFrame()
        for period_name,period_dict in self._periods.items():
            performance_df=pd.DataFrame()
            for period_id in period_dict.keys():
                id_performance_df=copy.deepcopy(period_dict[period_id][0])
                performance_df=pd.concat(
                    [performance_df,id_performance_df],
                    axis=0
                )
                aggregate_metrics["Period name"].append(period_name)
                aggregate_metrics["ID"].append(period_id)
                for metric_name,metric_value in period_dict[period_id][1].items():
                    aggregate_metrics[metric_name].append(metric_value)
                aggregate_metrics["Comaniciu (average in ID)"].append(id_performance_df.Comaniciu.mean())
            performance_df[period_name]=[period_name for _ in range(performance_df.shape[0])]
            performance_df.set_index(
                period_name,
                drop=True,
                append=True,
                inplace=True
            )
            all_periods_performance=pd.concat(
                [all_periods_performance,performance_df],
                axis=0, 
            )
        self._all_periods_performance=all_periods_performance
        self._periods_names=self._all_periods_performance.index.get_level_values(1).unique()
        self._single_index_all_periods_performance=copy.deepcopy(self._all_periods_performance)
        self._single_index_all_periods_performance.reset_index(level=1,inplace=True)
        self._single_index_all_periods_performance.rename({"level_1":"Period name"},axis=1,inplace=True)
        all_periods_aggregate_metrics=pd.DataFrame(aggregate_metrics)
        all_periods_aggregate_metrics.set_index(
            keys=["Period name","ID"],
            drop=True,
            append=False, 
            inplace=True
        )
        self._all_periods_aggregate_metrics=all_periods_aggregate_metrics
    def save_periods_performance(self,path_to_performance_csv,path_to_aggregate_metrics_csv)->None:
        self._all_periods_performance.to_csv(path_to_performance_csv)
        self._all_periods_aggregate_metrics.to_csv(path_to_aggregate_metrics_csv)
    def display_periods_performance(
            self,
            period_name:str|None=None,
            display_dfs_kwargs:dict[Literal["width","height"],str]={},
        )->None:
        if period_name is not None:
            assert period_name in self._periods_names,"The period you look for is not in the ones nationally fixed nor custom provided."
            to_be_displayed=copy.deepcopy(
                self._single_index_all_periods_performance.loc[
                    self._single_index_all_periods_performance.loc[:,"Period name"]==period_name,:
                ]
            )
            metrics_to_display=copy.deepcopy(
                self._all_periods_aggregate_metrics.xs(
                    key=period_name,level=0
                )
            )
        else:
            to_be_displayed=copy.deepcopy(self._all_periods_performance)
            metrics_to_display=copy.deepcopy(self._all_periods_aggregate_metrics)
        display_dfs(
            dfs_list=[to_be_displayed,metrics_to_display],
            dfs_names=["Sequence Metrics","Aggregate Metrics"],
            start_to_end=["all","all"],
            **display_dfs_kwargs
        )
    def show_periods_plots(
            self,
            period_name:str,
            only_the_worst:tuple[int,Literal["MAE","MSE","RMSE","MAPE","R2","Comaniciu"]]|None=None,
            make_figures_dict_from_dfs_kwargs:dict[
                Literal["tft","as_from_predict_raw","feature_names_per_channel",
                        "sklearn_metric_name","scaler_y","channel_names","VSWs_kwargs",
                        "attention_kwargs","prediction_plot_kwargs"]
            ]={},
            display_figures_kwargs:dict[Literal["which_characteristics","which_positions","use_tabs"]]={}
        )->None:
        assert period_name in self._periods_names,"The period you look for is not in the ones stored."
        to_be_displayed=copy.deepcopy(
            self._single_index_all_periods_performance.loc[
                self._single_index_all_periods_performance.loc[:,"Period name"]==period_name,:
            ]
        )
        if only_the_worst is not None:
            lower_is_better=["MAE","MSE","RMSE","MAPE"]
            if only_the_worst[1] in lower_is_better:
                to_be_displayed=to_be_displayed.sort_values(
                    by=only_the_worst[1],
                    ascending=False,
                ).head(only_the_worst[0])
            elif only_the_worst[1]=="Comaniciu":
                years=[]
                for batch_idx,item_idx in zip(to_be_displayed["batch_idx"],to_be_displayed["item_idx"]):
                    years.append(
                        str(
                            item_in_batch_to_dates(
                                self._data,self._which,item_idx,batch_idx,
                                consecutive_time_steps="time_dummy_to_date" not in self._data[self._which].keys()
                            ).iloc[-24].year
                        )
                    )
                years=pd.Series(years,index=to_be_displayed.index)
                to_be_displayed["Comaniciu in year"]=years.apply(
                    lambda yr:self._year_to_Comaniciu_avg.loc[yr]
                )
                to_be_displayed["abs_rel_diff_Comaniciu"]=(
                    (to_be_displayed["Comaniciu"]-to_be_displayed["Comaniciu in year"])/(to_be_displayed["Comaniciu in year"])*100
                ).abs()
                to_be_displayed=to_be_displayed.sort_values(
                    by="abs_rel_diff_Comaniciu",
                    ascending=False,
                ).head(only_the_worst[0])
            else:
                to_be_displayed=to_be_displayed.sort_values(
                    by=only_the_worst[1],
                    ascending=True,
                ).head(only_the_worst[0])
        periods_figs=make_figures_dict_from_dfs(
            data=self._data,
            which=self._which,
            dfs_list=[to_be_displayed],
            dfs_names=[period_name],
            **make_figures_dict_from_dfs_kwargs
        )
        display_figures(
            figures_dict=periods_figs,
            from_who=period_name,
            **display_figures_kwargs
        )
    def save_non_init_instance_attributes(self,path_to_trial,building_name,judgment)->None:
        attributes=copy.deepcopy(self.__dict__)
        init_attributes=[
            "_data",
            "_which",
            "_all_seqs_metrics_df",
            "_all_dev",
            "_not_overlapped_seqs_batch_and_item_df",
            "_non_fixed_or_regional_holidays",
            "_holidays",
            "_national_fixed_holidays"
        ]
        for attr in init_attributes:
            attributes.pop(attr)
        save_variable(
            attributes,
            os.path.join(path_to_trial,"TimeHandler_{}_non_init_attributes_{}.pkl".format(building_name,judgment))  
        )
    def load_instance_attributes(self,path_to_trial,building_name,judgment)->None:
        self.__dict__.update(load_variable(os.path.join(path_to_trial,"TimeHandler_{}_non_init_attributes_{}.pkl".format(building_name,judgment))))
    def period_summary_plot(
            self,
            period_name:str,
            metric_name:Literal["MAE","MSE","RMSE","MAPE","R2","Comaniciu"],
            exclude_years:list[str]|None=None,
            fontsize:float=9,
            rotation:float=90,
            size:float=60,
            figure_kwargs={"figsize":(13,6)},
            custom_ylims:tuple[float|None,float|None]|None=None
        )->None:
        with plt.rc_context({'font.size': fontsize}):
            df=copy.deepcopy(self._single_index_all_periods_performance)
            df=df.loc[df["Period name"]==period_name]
            df.reset_index(inplace=True)
            day_and_month=[]
            year_hue=[]
            for row_label in df.index:
                batch_idx=df.loc[row_label,"batch_idx"]
                item_idx=df.loc[row_label,"item_idx"]
                first_decoder_date=item_in_batch_to_dates(
                    self._data,self._which,item_idx,batch_idx,
                    consecutive_time_steps="time_dummy_to_date" not in self._data[self._which].keys()
                ).iloc[-n_future]
                year_hue.append(first_decoder_date.strftime("%Y"))
                day_and_month.append(first_decoder_date.replace(year=2000))
            if period_name=="Winter break":
                for idx,date in enumerate(day_and_month):
                    if date.month==1:
                        day_and_month[idx]=date.replace(year=2001)
            df["Day and month"]=day_and_month
            min_day_and_month=df["Day and month"].min()
            max_day_and_month=df["Day and month"].max()
            date_range=pd.date_range(
                start=min_day_and_month,
                end=max_day_and_month,
                freq="D"
            )
            df["Year"]=year_hue
            if exclude_years:
                df=df.loc[~df["Year"].isin(exclude_years),:]
            fig=plt.figure(**figure_kwargs)
            ax=fig.gca()
            if custom_ylims is not None:
                ymin=custom_ylims[0]
                ymax=custom_ylims[1]
                if ymin is not None and ymax is not None:
                    df=df.loc[(df[metric_name]>=ymin)&(df[metric_name]<=ymax)]
                else:
                    if ymin is None:
                        df=df.loc[df[metric_name]<=ymax]
                    else:
                        df=df.loc[df[metric_name]>=ymin]
            sns.scatterplot(
                data=df,x="Day and month",y=metric_name,
                hue="Year",ax=ax,s=size,hue_order=np.sort(df["Year"].unique())
            )
            names_to_udm={
                "MAE":" kWh",
                "MSE":r" kWh$^2$",
                "RMSE":" kWh",
                "MAPE":"%",
                "R2":"",
                "Comaniciu":""
            }
            ax.yaxis.set_major_formatter("{x}%")
            xticks=date_range
            new_xticks_labels=[date.strftime("%m/%d") for date in xticks]
            ax.set_xticks(ticks=xticks,labels=new_xticks_labels,rotation=rotation)
            ax.set_xlim((xticks.min(),xticks.max()))
            handles, labels = ax.get_legend_handles_labels()
            ax.legend_.remove()
            ax.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1), ncol=df["Year"].nunique())
            plt.close("all")
            display(fig)
    def periods_summary_plot(
            self,
            metric_name:Literal["MAE","MSE","RMSE","MAPE","R2","Comaniciu"],
            exclude_periods:list[str]|None=None,
            exclude_years:list[str]|None=None,
            fontsize:float=9,
            rotation:float=90,
            size:float=8,
            lineprops:dict=dict(linestyle="-.",linewidth=0.85),
            figure_kwargs={"figsize":(13,6)},
            custom_ylims:tuple[float|None,float|None]|None=None,
            custom_order:dict[str,int]|None=None,
            jitter:bool=False,
            swarmplot:bool=False,
            og_to_new:dict[str,str]|None=None,
            legend:bool=False,
            ref_fig:Figure|None=None,
            return_ax_and_fig:bool=False,
        )->None:
        with plt.rc_context({'font.size': fontsize}):
            df=copy.deepcopy(self._all_periods_aggregate_metrics)
            df.reset_index(inplace=True)
            new_ID=[]
            for string in df["ID"]:
                id_pieces=string.split(sep=" ")
                start_pieces=id_pieces[1].split(sep="/")
                year=start_pieces[0]
                new_ID.append(year)
            df["Year"]=new_ID
            df.drop(columns="ID",inplace=True)
            metrics_in_year={yr:mtc_dict for yr,mtc_dict in self._metrics_in_year.items() if yr in df["Year"].unique()}
            for mtc in ["MAE","MSE","RMSE","MAPE","R2"]:
                df[mtc+" in year"]=new_ID
                df[mtc+" in year"]=df[mtc+" in year"].map(lambda yr:metrics_in_year[yr][mtc])
                df[mtc+" relative difference"]=(df[mtc]-df[mtc+" in year"])/df[mtc+" in year"]*100
            year_to_Comaniciu_avg=copy.deepcopy(self._year_to_Comaniciu_avg.loc[np.sort(df["Year"].unique()).tolist()])
            if custom_order is not None:
                assert all(pd.Series(custom_order.keys()).isin(self._periods_names)),"Some names in custom_order are not within the period names saved."
                df["Period order"]=copy.deepcopy(df["Period name"].apply(lambda pdn:custom_order[pdn]))
                df.sort_values(by="Period order",ascending=True,inplace=True)
            if exclude_periods:
                df=df.loc[~df["Period name"].isin(exclude_periods),:]
            if exclude_years:
                df=df.loc[~df["Year"].isin(exclude_years),:]
            fig=plt.figure(**figure_kwargs)
            ax=fig.gca()
            if metric_name=="Comaniciu":
                y=metric_name+" (average in ID)"
            else:
                y=metric_name+" relative difference"
            if custom_ylims is not None:
                ymin=custom_ylims[0]
                ymax=custom_ylims[1]
                if ymin is not None and ymax is not None:
                    df=df.loc[(df[y]>=ymin)&(df[y]<=ymax)]
                    year_to_Comaniciu_avg=year_to_Comaniciu_avg.loc[
                        (year_to_Comaniciu_avg>=ymin)&(year_to_Comaniciu_avg<=ymax)
                    ]
                else:
                    if ymin is None:
                        df=df.loc[df[y]<=ymax]
                        year_to_Comaniciu_avg=year_to_Comaniciu_avg.loc[
                            year_to_Comaniciu_avg<=ymax
                        ]
                    else:
                        df=df.loc[df[y]>=ymin]
                        year_to_Comaniciu_avg=year_to_Comaniciu_avg.loc[
                            year_to_Comaniciu_avg>=ymin
                        ]
            if not swarmplot:
                sns.stripplot(
                    data=df,x="Period name",y=y,
                    hue="Year",jitter=jitter,ax=ax,size=size,hue_order=np.sort(df["Year"].unique())
                )
            else:
                sns.swarmplot(
                    data=df,x="Period name",y=y,
                    hue="Year",ax=ax,size=size,hue_order=np.sort(df["Year"].unique()),
                    warn_thresh=0
                )
            if metric_name=="Comaniciu":
                ax.set_ylabel("Comaniciu (average in the period)")
                palette=sns.color_palette() 
                color_mapping={category:palette[i] for i,category in enumerate(np.sort(df["Year"].unique()))}
                xticks=np.array(ax.get_xticks())
                for yr in np.sort(df["Year"].unique()):
                    ax.plot(xticks,np.ones(xticks.shape)*year_to_Comaniciu_avg.loc[yr],
                            color=color_mapping[yr],label=yr+" average",**lineprops)
            ax.set_xlabel("")
            if metric_name!="Comaniciu":
                ax.set_ylabel("{} (relative difference to the yearly {})".format(metric_name,metric_name))
                yticks=ax.get_yticks()
                yticks_labels=["{}%".format(label) for label in yticks]
                ax.set_yticks(ticks=yticks,labels=yticks_labels)
            xticks=ax.get_xticks()
            xticks_labels=ax.get_xticklabels()
            if og_to_new is not None:
                xticks_labels=[og_to_new[lb.get_text()] for lb in xticks_labels]
            ax.set_xticks(ticks=xticks,labels=xticks_labels,rotation=rotation)
            handles, labels = ax.get_legend_handles_labels()
            if metric_name=="Comaniciu":
                label_to_handle={label:handle for handle,label in zip(handles,labels)}
                labels=np.sort(labels)
                handles=[label_to_handle[label] for label in labels]
            ax.legend_.remove()
            ax.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1), ncol=df["Year"].nunique())
            if not legend:
                lg=ax.get_legend()
                lg.set_visible(legend)
            ax.grid(visible=True,alpha=0.5)
            if ref_fig is not None:
                match_data_area(ref_fig,fig)
            plt.close("all")
            display(fig)
            if return_ax_and_fig:
                return ax,fig            
    def get_period_above_and_below_metric(
            self,
            tft:TemporalFusionTransformer,
            as_from_predict_raw:Union[NamedTuple,dict[str,dict[str,torch.Tensor]|torch.Tensor]],     
            metric_name:Literal["MAE","MSE","RMSE","MAPE","R2","Comaniciu"],
            period_name:str,
            judgment:Literal["Percentile","In the percent best performing","MinMaxScaler"],
            show_rescaled:bool=True,
            scaler_y=None,
            upper_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.95),
            lower_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.05),
            figsize:tuple[float,float]=(16,9),
            building_name:str|None=None,
            plot_kwargs:dict[str,]={"linewidth":1},
            thresholds_wrt_non_overlapping_in_ID:bool=False,
            labels_rotation:float|None=None,
            fontsize:float=14,
            xlabel:str|None=None,
            legend_only:bool=False,
            return_only_figure:bool=False,
            stairplot:bool=False,
            dpi:int|None=None,
            target_name:str="electricity consumption (kWh)"
            )->None:
        try:
            try:
                Output=getattr(as_from_predict_raw,"output")
            except AttributeError:
                Output=as_from_predict_raw["output"]
        except AttributeError:
            Output=as_from_predict_raw
        
        try:
            raw_predictions=getattr(Output,"prediction")
        except AttributeError:
            raw_predictions=Output["prediction"]
        quantiles_computed=pd.Series(tft.loss.quantiles)
        quantile_for_predictions=0.5
        if not (quantile_for_predictions in tft.loss.quantiles):
            raise Exception("By default, predictions are represented by the median (q=0.5, a.k.a. P50), but the loss used at TFT's initialization does not compute the median!\nAdd q=0.5 among the quantiles in QuantileLoss() at TFT's construction time.")
        quantile_for_predictions_idx=(quantiles_computed==quantile_for_predictions).idxmax()

        predictions=raw_predictions[:,:,quantile_for_predictions_idx]

        dataloader=self._data[self._which]["dataloader"]
        batch_size=dataloader.batch_size
        number_of_batches=len(dataloader)
        batch_to_seqs_slice={
            batch_num:slice(batch_num*batch_size,(batch_num+1)*batch_size) if batch_num!=(number_of_batches-1) else slice(batch_num*batch_size,None) for batch_num in range(number_of_batches)
        }
        dataloader_list=self._data[self._which]["dataloader_list"]
        ids=list(self._periods[period_name].keys())
        print(
            "----ids in {}----\n{}".format(period_name,ids),
            flush=True
        )
        idx_in_ids=int(input("Type the index (from 0) of the selected id within the selected period's ids list:"))
        id_selected=ids[idx_in_ids]
        print("Selected id: {}".format(id_selected),flush=True)
        [_,start_piece,_,end_piece]=id_selected.split(sep=" ")
        start_date=pd.Timestamp(
            year=int(start_piece.split(sep="/")[0]),
            month=int(start_piece.split(sep="/")[1]),
            day=int(start_piece.split(sep="/")[2].split("_")[0]),
            hour=int(start_piece.split(sep="/")[2].split("_")[1].removeprefix("h")),
            minute=0
        )
        end_date=pd.Timestamp(
            year=int(end_piece.split(sep="/")[0]),
            month=int(end_piece.split(sep="/")[1]),
            day=int(end_piece.split(sep="/")[2].split("_")[0]),
            hour=int(end_piece.split(sep="/")[2].split("_")[1].removeprefix("h")),
            minute=0
        )
        id_performance=copy.deepcopy(self._periods[period_name][id_selected][0])
        if not thresholds_wrt_non_overlapping_in_ID:
            batch_and_item_all_seqs_in_id=all_seqs_predicting_in_range(
                self._data,self._which,
                pd.date_range(
                    start=start_date,
                    end=end_date,
                    freq="h"
                )
            )
        else:
            batch_and_item_all_seqs_in_id=find_seqs_predicting_in_range(
                tft,self._data,self._which,
                pd.date_range(
                    start=start_date,
                    end=end_date,
                    freq="h"
                )
            )            
        all_seqs_in_id_performance=pair_metrics_with_batch_and_item_df(
            self._all_seqs_metrics_df,
            batch_and_item_all_seqs_in_id,
            judgment,
            self._all_dev,
            self._not_overlapped_seqs_batch_and_item_df
        )
        metrics=all_seqs_in_id_performance[metric_name]
        if isinstance(upper_threshold,tuple):
            upper_label=" (P{:.1f})".format(upper_threshold[1]*100)
            upper_threshold=np.quantile(metrics,upper_threshold[1])
        else:
            upper_label=""
        if isinstance(lower_threshold,tuple):
            lower_label=" (P{:.1f})".format(lower_threshold[1]*100)
            lower_threshold=np.quantile(metrics,lower_threshold[1])
        else:
            lower_label=""

        start=True
        already_seen_decoder_time_idxs=torch.tensor([])
        batch_idxs_significant={"above":[],"below":[]}
        item_idxs_significant={"above":[],"below":[]}
        decoder_time_idxs_significant={"above":[],"below":[]}
        
        batch_and_item=list(zip(id_performance["batch_idx"],id_performance["item_idx"]))
        for batch_index,item_index in batch_and_item:
            x_dict,(y,_)=dataloader_list[batch_index]
            decoder_time_idxs=x_dict["decoder_time_idx"][item_index,:]
            if id_performance.loc[
                (id_performance["batch_idx"]==batch_index)&(id_performance["item_idx"]==item_index),
                metric_name
                ].squeeze()>upper_threshold:  
                decoder_time_idxs_significant["above"].append(decoder_time_idxs)
                batch_idxs_significant["above"].append(batch_index)                
                item_idxs_significant["above"].append(item_index)
            if id_performance.loc[
                (id_performance["batch_idx"]==batch_index)&(id_performance["item_idx"]==item_index),
                metric_name
                ].squeeze()<lower_threshold:  
                decoder_time_idxs_significant["below"].append(decoder_time_idxs)
                batch_idxs_significant["below"].append(batch_index)                
                item_idxs_significant["below"].append(item_index)
            already_seen_decoder_time_idxs=torch.cat(
                (already_seen_decoder_time_idxs,decoder_time_idxs)
            )
            predictions_in_batch=predictions[batch_to_seqs_slice[batch_index],:]
            predictions_sequence_in_batch=predictions_in_batch[item_index,:]             
            if start:
                preds=predictions_sequence_in_batch
                target=y[item_index,:]
                start=False
            else:    
                target=torch.cat(
                    (target,y[item_index,:]),
                    dim=0
                )
                preds=torch.cat(
                    (preds,predictions_sequence_in_batch)
                )

        try:
            assert "time_dummy_to_date" in self._data[self._which].keys()
            time_dummy_to_date=copy.deepcopy(self._data[self._which]["time_dummy_to_date"])
        except AssertionError:
            time_dummy_to_date=pd.Series(
                pd.date_range(
                    self._data[self._which]["starting_date"],freq="1h",
                    periods=self._data[self._which]["df"].shape[0]
                ),
                index=self._data[self._which]["df"]["time_idx"]
            )
        decoder_dates=time_dummy_to_date.loc[already_seen_decoder_time_idxs]
        if show_rescaled:
            target=torch.tensor(
                scaler_y.inverse_transform(target.cpu().reshape((-1,1))).reshape(-1)
            )
            preds=torch.tensor(
                scaler_y.inverse_transform(preds.cpu().reshape((-1,1))).reshape(-1)
            )
        else:
            target=target.cpu()
            preds=preds.cpu()
            
        from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
        overall_metrics={
            "MAE":mean_absolute_error(target,preds),
            "MSE":mean_squared_error(target,preds),
            "RMSE":root_mean_squared_error(target,preds),
            "MAPE":torch.mean(
                torch.abs(
                    target[target!=0] - preds[target!=0]
                ) / target[target!=0]
            ).item()*100,             
            "R2":r2_score(target,preds),
            "Comaniciu":id_performance.Comaniciu.mean()
        }
        names_to_udm={
            "MAE":" (-)" if not show_rescaled else " kWh",
            "MSE":r" (-)$^2$" if not show_rescaled else r" kWh$^2$",
            "RMSE":" (-)" if not show_rescaled else " kWh",
            "MAPE":"%",
            "R2":"",
            "Comaniciu":""
        }  

        absolute_min=min([target.min(),preds.min()])
        absolute_max=max([target.max(),preds.max()])

        fig=plt.figure(figsize=figsize,dpi=dpi)    
        target_color="r"
        preds_color="b"
        bad_color="#FFD700"
        good_color = "g"
        with plt.rc_context({"font.size":fontsize}):
            ax_target=fig.gca()
            if not stairplot:
                ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
            else:
                ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
            ax_target.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_target.set_ylabel(
                "Observed {}".format(target_name),
                color=target_color
            )
            ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
            if xlabel is None:
                name=""
                for idx,character in enumerate(self._which):
                    if idx==0:
                        name+=character.upper()
                    else:
                        name+=character
                ax_target.set_xlabel(name+" set dates.")
            else:
                ax_target.set_xlabel(xlabel)
            better_if_lower=["MAE","MSE","RMSE","MAPE","Comaniciu"]
            ymin,ymax=ax_target.get_ylim()
            for list_idx,above_indices in enumerate(decoder_time_idxs_significant["above"]):
                ax_target.fill_between(
                    x=time_dummy_to_date.loc[above_indices],
                    y1=ymin,
                    y2=ymax,
                    fc=bad_color if metric_name in better_if_lower else good_color,
                    alpha=0.35,
                    label="{}>{:.4f}".format(metric_name,upper_threshold)+names_to_udm[metric_name]+upper_label if list_idx==0 else ""
                )

            for list_idx,below_indices in enumerate(decoder_time_idxs_significant["below"]):
                ax_target.fill_between(
                    x=time_dummy_to_date.loc[below_indices],
                    y1=ymin,
                    y2=ymax,
                    fc=good_color if metric_name in better_if_lower else bad_color,
                    alpha=0.35,
                    label="{}<{:.4f}".format(metric_name,lower_threshold)+names_to_udm[metric_name]+lower_label if list_idx==0 else ""
                )

            ax_target.legend(loc="lower center",bbox_to_anchor=(0.5, 1.0),ncols=2)
            ax_target.set_xlim(
                (min(time_dummy_to_date.loc[already_seen_decoder_time_idxs]),
                max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
            )
            if labels_rotation is not None:
                for label in ax_target.get_xticklabels():
                    label.set_rotation(labels_rotation)
            ax_preds=ax_target.twinx()
            if not stairplot:
                ax_preds.plot(decoder_dates,preds,color=preds_color)
            else:
                ax_preds.step(decoder_dates,preds,color=preds_color,where="post")
            ax_preds.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_preds.set_ylabel("Predicted {}".format(target_name),color=preds_color)
            ax_preds.tick_params(axis="y",color=preds_color,labelcolor=preds_color)
            if not legend_only:
                suptitle_strs=["{} building".format(building_name)+": "+id_selected.lower(),"\nMetrics:"]
                for idx,(name,value) in enumerate(overall_metrics.items()):
                    if idx!=(len(overall_metrics.items())-1):
                        if name!="R2":
                            suptitle_strs.append("{}={:.2f}".format(name,value)+names_to_udm[name]+",")
                        else:
                            suptitle_strs.append("{}={:.3f}".format(name,value)+names_to_udm[name]+",")
                    else:
                        suptitle_strs.append("{}={:.4f}".format(name,value)+names_to_udm[name])
                fig.suptitle(" ".join(suptitle_strs))
            fig.tight_layout()
            plt.close("all")
            display(fig)
        if return_only_figure:
            return fig

# TFT evaluation

## Extend test to entire period

In [ ]:
# NO ABLATION AND/OR ADDITION
df_test_x_tft_extended=pd.concat(
    [df_train_x_tft,df_test_x_tft],ignore_index=True
)
display_dfs(
    [df,df_train_x_tft,df_test_x_tft,df_test_x_tft_extended],
    ["df","df_train_x_tft","df_test_x_tft","df_test_x_tft_extended"],
    start_to_end=["all","all","all","all"],
)
test_data_extended=TimeSeriesDataSet.from_dataset(train_data,df_test_x_tft_extended,stop_randomization=True,predict=False)
test_dataloader_extended=test_data_extended.to_dataloader(train=False,batch_size=1,num_workers=1,drop_last=True)

In [ ]:
# WITH MANIPULATION
df_test_x_tft_extended_manipulated=pd.concat(
    [tft_dfs_manipulated["train_df_tft"],
     tft_dfs_manipulated["val_df_tft"],
     tft_dfs_manipulated["df_test_x_tft"]],
    ignore_index=True
)
display_dfs(
    [df,tft_dfs_manipulated["train_df_tft"],tft_dfs_manipulated["val_df_tft"],
     tft_dfs_manipulated["df_test_x_tft"],df_test_x_tft_extended_manipulated],
    ["df","train_df_tft_manipulated","val_df_tft_manipulated",
     "df_test_x_tft_manipulated","df_test_x_tft_extended_manipulated"],
    start_to_end=["all","all","all","all","all"],
)
test_data_extended_manipulated=TimeSeriesDataSet.from_dataset(
    tft_tsds_manipulated["train_df_tft"],
    df_test_x_tft_extended_manipulated,
    stop_randomization=True,predict=False
)
test_dataloader_extended_manipulated=test_data_extended_manipulated.to_dataloader(train=False,batch_size=1,num_workers=1,drop_last=True)

## Task selection

In [ ]:
from_optuna_study=True
if from_optuna_study:
    _,_,trial_best_epoch=find_trial_in_study(completed_study,pick_best=True)
    path_to_trial_folder,tft=TFT_from_trial(paths["to_ckpts_folder"],study=completed_study,pick_best=True)
    path_to_model_ckpt=os.path.join(path_to_trial_folder,"epoch={}.ckpt".format(trial_best_epoch))
else:
    # # ----ABLATION:'R_MaT_OO'----
    # trial_best_epoch=16
    # id_manipulation='R_MaT_OO'

    # # ----ABLATION:'R_MaTAp(_OO'----
    # trial_best_epoch=7
    # id_manipulation='R_MaTAp(_OO'

    # # ----ABLATION:'R_MaTAp(P(_OO'----
    # trial_best_epoch=10
    # id_manipulation='R_MaTAp(P(_OO'

    # # ----ABLATION:'R_MaTAp(P(Mah_OO'----
    # trial_best_epoch=10
    # id_manipulation='R_MaTAp(P(Mah_OO'

    # # ----ABLATION:'R_MaTAp(P(MahMws2_OO'----
    # trial_best_epoch=8
    # id_manipulation='R_MaTAp(P(MahMws2_OO'

    # # ----ABLATION:'R_MaTAp(P(MahMws2SI_OO'----
    # trial_best_epoch=16
    # id_manipulation='R_MaTAp(P(MahMws2SI_OO'

    # # ----ABLATION:'R_MaTAp(P(MahMws2SITarget_OO'----
    # trial_best_epoch=6
    # id_manipulation='R_MaTAp(P(MahMws2SITarget_OO'

    # ----ADDITION:'A_Hi'----
    trial_best_epoch=19
    id_manipulation='A_Hi'
    
    # ----MANIPULATION: ANY OF THE CASES----
    path_to_trial_folder=Path("./Results/TFT/Optimization/HyperbandPruner/Twists_on_best/Change_features/{}".format(id_manipulation))
    
    path_to_model_ckpt=os.path.join(path_to_trial_folder,"best_epoch={}.ckpt".format(trial_best_epoch))
tft,feature_names_per_channel=get_model_info(path_to_model_ckpt,print_info=True)

In [ ]:
# building_name="Agriculture extended"
building_name="Agriculture"
# building_name="Architecture"
# building_name="Economics"
list_of_file_and_directory_names_in_trial_folder=os.listdir(path_to_trial_folder)
if os.path.join("data_{}.pkl".format(building_name)) in list_of_file_and_directory_names_in_trial_folder:
    what_to_do="load"
else:
    what_to_do="save"
if os.path.join("as_from_predict_raw_unpacked_{}.pkl".format(building_name)) in list_of_file_and_directory_names_in_trial_folder:
    do_raw_preds=False
else:
    do_raw_preds=True
if os.path.join("predictions_{}.pkl".format(building_name)) in list_of_file_and_directory_names_in_trial_folder:
    do_predictions=False
    have_you_checked_dataloader=True
else:
    do_predictions=True
    have_you_checked_dataloader=False
print("Scanning '{}'\nto look for existing data for the selected\nbuilding_name='{}',\nthese settings are defined according to the data found:\nwhat_to_do='{}'\ndo_raw_preds={}\ndo_predictions={}\nhave_you_checked_dataloader={}".format(
    path_to_trial_folder,building_name,what_to_do,do_raw_preds,do_predictions,have_you_checked_dataloader,
    )
)
print("custom_data in get_data() is irrelevant." if what_to_do=="load"  else "")

In [ ]:
data=get_data(
    path_to_trial_folder,
    what_to_do=what_to_do,
    building_name=building_name,  
    custom_data={
        # "train":dict(
        #     df=tft_dfs_manipulated["train_df_tft"],
        #     # Don't need to update the starting_date with respect to the original
        #     # model, since we don't change that, only the amount of features changed.
        #     dataloader=tft_dataloaders_manipulated["train_df_tft"],
        # ),
        # "val":dict(
        #     df=tft_dfs_manipulated["val_df_tft"],
        #     dataloader=tft_dataloaders_manipulated["val_df_tft"],
        # ),
        # "test":dict(
        #     df=tft_dfs_manipulated["df_test_x_tft"],
        #     starting_date=pd.Timestamp(
        #         year=2022,month=7,day=1,hour=0,minute=0,second=0
        #     ),
        #     dataloader=tft_dataloaders_manipulated["df_test_x_tft"]
        # ),
        
        # "test":dict(
        #     df=tft_dfs_manipulated["df_Architecture_building_test_x_tft"],
        #     starting_date=pd.Timestamp(
        #         year=2019,month=7,day=1,hour=0,minute=0,second=0
        #     ),
        #     dataloader=tft_dataloaders_manipulated["df_Architecture_building_test_x_tft"]
        # ),
        # "test":dict(
        #     df=tft_dfs_manipulated["df_Economics_building_test_x_tft"],
        #     starting_date=pd.Timestamp(
        #         year=2019,month=7,day=1,hour=0,minute=0,second=0
        #     ),
        #     dataloader=tft_dataloaders_manipulated["df_Economics_building_test_x_tft"]
        # ),
        # "test":dict(
        #     df=df_test_x_tft_extended_manipulated,
        #     starting_date=pd.Timestamp(
        #         year=2019,month=7,day=1,hour=0,minute=0,second=0
        #     ),
        #     dataloader=test_dataloader_extended_manipulated
        # )
    }
)

In [ ]:
from torchinfo import summary
summary(
    model=tft,
    verbose=1,
)

In [ ]:
if do_raw_preds:
    as_from_predict_raw=do_raw_predictions(
        tft,data,"test",
        save_as_from_predict_raw=True,
        path_to_trial_folder=path_to_trial_folder,
        building_name=building_name
    )
else:
    as_from_predict_raw=handle_as_from_predict_raw(
        path_to_trial_folder,
        "as_from_predict_raw_unpacked_{}.pkl".format(building_name),
        what_to_do="load"
    )

In [ ]:
best_N_seqs_df,worst_M_seqs_df,all_seqs_metrics_df,not_overlapped_seqs_batch_and_item_df=get_model_insights(
    tft,as_from_predict_raw,data,"test",
    top_N=5,
    worst_M=5,
    scaler_y=scaler_y,
    sklearn_metric_name="R2",
    return_all_seqs_metrics_df=True,
    return_not_overlapped_seqs_batch_and_item=True
)

In [ ]:
try:
    assert have_you_checked_dataloader
    if do_predictions:
        predictions=tft.predict(test_dataloader, return_y=True, trainer_kwargs=dict(accelerator="gpu" if torch.cuda.is_available() else "cpu"))
        save_variable(predictions, os.path.join(path_to_trial_folder,"predictions_{}.pkl".format(building_name)))
    else:
        predictions=load_variable(os.path.join(path_to_trial_folder,"predictions_{}.pkl".format(building_name)))
    show_results(
        predictions,"{} building".format(building_name),saving_path=path_to_trial_folder,
        start_timestep_preds=data["test"]["starting_date"]+n_past*pd.Timedelta("1h")
    )
except AssertionError:
    print("Check the dataloader in tft.predict() before running this cell, ensure it's correct.\nThen, change have_you_checked_dataloader to True and re-run this cell.")

In [ ]:
tft.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tft.to(device)

dummy_batch_x, _ = next(iter(data["test"]["dataloader"]))
dummy_batch_x = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in dummy_batch_x.items()}

print("Warming up the model...")
with torch.no_grad():
    for _ in range(5):
        _ = tft(dummy_batch_x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
print("Warm-up complete. Starting benchmark.")


inference_times = []

take=0
with torch.no_grad():
    for btc_idx, (x, y) in enumerate(data["test"]["dataloader"]):
        if btc_idx==take:
            take+=24 
            x = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in x.items()}

            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = perf_counter()

            _ = tft(x)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            end_time = perf_counter()
            
            elapsed_time = (end_time - start_time) * 1000 
            inference_times.append(elapsed_time)

avg_time = np.mean(inference_times)
std_time = np.std(inference_times)

print(f"\nMetrics across all {len(inference_times)} sequences:")
print(f"Average Inference Time: {avg_time:.2f} ms")
print(f"Standard Deviation: {std_time:.2f} ms")

## Variable selection weights

In [ ]:
channel_names=["encoder_variables","decoder_variables"]
for nmc,channel_name in enumerate(channel_names):
        get_VSWs(
            as_from_predict_raw,data,"test",
            feature_names_per_channel,channel_name,
            reduction="average",
            additional_title="Results on test dataset.",
            print_info=False,
            fontsize=16, 
            overlap_cumulative=True,
            def_ofs_y=-10,
            target_label="Agriculture building load (kWh)",
            no_title=True,
            barh_settings_update={
                "width_in":16,
                "bar_px":150, 
                "dpi":300 
            },
            pad_spaces=None if not nmc else 90,
            legend=not nmc,
            square_brackets=True,
            formatter=False
            )

## Attention weights

In [ ]:
get_attention(
    as_from_predict_raw,
    data,"test",
    reduction="mean",
    normalize_attention_by_position_idxs_sum=False,
    set_attention_threshold=True,
    custom_ticks=True,
    plotting_time_interval=6,
    detail_representatives=False,
    representative_quantiles=[0.1,0.5,0.9],
    detail_prediction_timesteps=True,
    prediction_idxs_to_plot=[6,12,18,24],
    linewidth=1.10,
    add_pre_suptitle="Results on {} test dataset.".format(building_name),
    print_info=False,
    separator_update={
        "enc":True,"dec":True,
        "box_y_fraction":0.94, 
        "txt_size":13,
    },
    figsize=(16,6),
    fontsize=15,
    forced_xlabel="Position index [-]",
    forced_ylabel="Average "+r"$\alpha(t,n,\tau)$"+" [-]",
    only_legend=True,
    scientific_on_y=True,
    colormap="coolwarm",
    masking="up_to",
    reduce_across_homogeneous_seqs=True,
    hour_of_preds_start_homogeneous_seqs=0,
    # day_name="Monday",
    # use_dates=True,
    dpi=300,
    forced_rotation=90,
)

In [ ]:
get_prediction_plot(
    tft,as_from_predict_raw,data,"test",
    batch_idx=4176,
    item_idx=0,
    show_rescaled=True,
    scaler_y=scaler_y,
    custom_ticks=True,
    add_loss_to_title=False,
    loss_as_from_built_in_implementation=False,
    sklearn_metric_name="R2",
    visualize_attention=True,
    show_future_observed=True,
    suptitle="Results on {} test dataset.".format(building_name),
    explicit_fill_zones=True,
    use_dates=True,
    plotting_time_interval=6,
    tau=24,
    dpi=300,
    formatter=False,
    attn_label=r"$\alpha(t,n,\tau=24)$"+" [-]",
    only_legend=True,
    fontsize=14.5,
    figsize=(18,8),
    scientific_on_attn=False,
    normalize_attention_by_position_idxs_sum=False,
    pred_label="Agriculture building load",
    pred_udm="kWh",
    brackets="square",
    stairplot=False,
)

In [ ]:
get_prediction_NOS_in_range_w_percentiles(
    tft,as_from_predict_raw,data,"test",
    date_range=pd.date_range(
        start=pd.Timestamp(year=2023,month=1,day=2,hour=0,minute=0),
        end=pd.Timestamp(year=2023,month=1,day=8,hour=23,minute=0),
        freq="h"
    ), # W4
    scaler_y=scaler_y,
    add_loss_to_title=True,
    sklearn_metric_name="MAPE",
    suptitle="Results on {} test dataset.".format(building_name),
    plotting_time_interval=4,
    ylabel="Electricity demand [kWh]",
    only_legend=True,
    figsize=(18,8),
    fontsize=14.5,
    dpi=300,
    formatter=False,
)

## Comaniciu

In [ ]:
most_dev,least_dev,all_dev=get_Comaniciu_distance_plot(
    as_from_predict_raw,data,
    show_rescaled=True,
    scaler_y=scaler_y,
    dist_threshold=("quantile",0.85),
    return_most_and_least_deviating=True,
    N_least_deviating=5,
    return_also_all_deviations=True,
    use_non_overlapping_sequences_from_00=True,
    fontsize=14,
    figsize=(16,6),
    xlabel="Test set dates", 
    ylabel="Agriculture building load", 
    ylabel_udm="kWh",
    brackets="square", 
    dist_label=r"$\mathit{dist(t)}$"+" [-]",
    dpi=300,
    formatter=False
)

## Festivities and performance

In [ ]:
time_handler=TimeHandler(
    data,"test",
    all_seqs_metrics_df,
    all_dev=all_dev,
    not_overlapped_seqs_batch_and_item_df=not_overlapped_seqs_batch_and_item_df,
    non_fixed_or_regional_holidays={
        "Pasqua 2020":dict(
            date=pd.Timestamp(
                year=2020,month=4,day=12
            ),
            mode="up_to_characteristic",
            characteristic="year"
        ),
        "Pasqua 2021":dict(
            date=pd.Timestamp(
                year=2021,month=4,day=4
            ),
            mode="up_to_characteristic",
            characteristic="year"
        ),
        "Pasqua 2022":dict(
            date=pd.Timestamp(
                year=2022,month=4,day=17
            ),
            mode="up_to_characteristic",
            characteristic="year"
        ),
        "Pasqua 2023":dict(
            date=pd.Timestamp(
                year=2023,month=4,day=9
            ),
            mode="up_to_characteristic",
            characteristic="year"
        ),
        "Lunedì di Pasqua 2020":dict(
            date=pd.Timestamp(
                year=2020,month=4,day=13
            ),
            mode="up_to_characteristic",
            characteristic="year"
        ),        
        "Lunedì di Pasqua 2021":dict(
            date=pd.Timestamp(
                year=2021,month=4,day=5
            ),
            mode="up_to_characteristic",
            characteristic="year"
        ),        
        "Lunedì di Pasqua 2022":dict(
            date=pd.Timestamp(
                year=2022,month=4,day=18
            ),
            mode="up_to_characteristic",
            characteristic="year"
        ),
        "Lunedì di Pasqua 2023":dict(
            date=pd.Timestamp(
                year=2023,month=4,day=10
            ),
            mode="up_to_characteristic",
            characteristic="year"
        ),
        "Santa Rosalia":dict(
            date=pd.Timestamp(
                year=2019,month=7,day=15
            ),
            mode="up_to_characteristic",
            characteristic="month"
        ),
    },
)
judgment="MinMaxScaler"
list_of_file_and_directory_names_in_trial_folder=os.listdir(path_to_trial_folder)
if os.path.join("TimeHandler_{}_non_init_attributes_{}.pkl".format(building_name,judgment)) in list_of_file_and_directory_names_in_trial_folder:
    compute_TimeHandler=False
else:
    compute_TimeHandler=True
if not compute_TimeHandler:
    time_handler.load_instance_attributes(path_to_trial_folder,building_name,judgment)
    print("compute_TimeHandler={}. time_handler now has all its attributes pre-loaded.".format(compute_TimeHandler))

In [ ]:
if compute_TimeHandler:
    time_handler.compute_metrics_in_year(tft,as_from_predict_raw,scaler_y)
display(time_handler._metrics_in_year)
display(time_handler._year_to_Comaniciu_avg)

In [ ]:
if compute_TimeHandler:
    time_handler.compute_holidays_performance(
        judgment,
        to_be_grouped={
            "Pasqua":["Pasqua 2020","Pasqua 2021","Pasqua 2022","Pasqua 2023"],
            "Lunedì di Pasqua":[
                "Lunedì di Pasqua 2020","Lunedì di Pasqua 2021","Lunedì di Pasqua 2022","Lunedì di Pasqua 2023"
            ]
        }
    )
time_handler.display_holidays_performance(
    holiday_name="Santa Rosalia"
)

In [ ]:
_,ref_holidays_Eco=time_handler.holidays_summary_plot(
    metric_name="MAPE",
    by="Year",
    display_days_statistics=True,
    lineprops=dict(linestyle="-.",linewidth=0.85),
    custom_order={
        "Capodanno":1,
        "Epifania":2,
        "Pasqua":3,
        "Lunedì di Pasqua":4,
        "Liberazione dal nazifascismo":5,
        "Festa del lavoro":6,
        "Festa della Repubblica":7,
        "Santa Rosalia":8,
        "Ferragosto":9,
        "Ognissanti":10,
        "Immacolata Concezione":11,
        "Natale":12,
        "Santo Stefano":13
    },
    figure_kwargs={
        "figsize":(16,4.5),
        "dpi":300
        },
    fontsize=13,
    size=10,
    swarmplot=True,
    og_to_new={
        "Capodanno":"New Year",
        "Epifania":"Epiphany",
        "Pasqua":"Easter",
        "Lunedì di Pasqua":"Easter Monday",
        "Liberazione dal nazifascismo":"25 April",
        "Festa del lavoro":"Labour Day",
        "Festa della Repubblica":"Republic Day",
        "Santa Rosalia":"Saint Rosalia",
        "Ferragosto":"Ferragosto",
        "Ognissanti":"All Saints' Day",
        "Immacolata Concezione":"Immaculate\nConception",
        "Natale":"Christmas",
        "Santo Stefano":"Boxing Day"
    },
    return_ax_and_fig=True,
    legend=True,
    forced_ylabel="MAPE (rel. diff. to the yearly MAPE) [%]",
    formatter_USA_no_perc=True
)

In [ ]:
periods_AA_2018_2019={   
        "2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2019,month=7,day=1,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2019,month=7,day=19,hour=23,minute=0,second=0),
                freq="h"
        ),
        "Summer break":pd.date_range(
                start=pd.Timestamp(year=2019,month=7,day=22,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2019,month=8,day=30,hour=23,minute=0,second=0),
                freq="h"
        ),
        "September exam session":pd.date_range(
                start=pd.Timestamp(year=2019,month=9,day=2,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2019,month=9,day=20,hour=23,minute=0,second=0),
                freq="h"
        ),
}
periods_AA_2019_2020={
        "In-itinere 1st semester exam session":pd.date_range(
                start=pd.Timestamp(year=2019,month=11,day=11,hour=0,minute=0,second=0), 
                end=pd.Timestamp(year=2019,month=11,day=15,hour=23,minute=0,second=0),
                freq="h"
        ),  
        "Winter break":pd.date_range(
                start=pd.Timestamp(year=2019,month=12,day=23,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=1,day=6,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "1st semester exam session":pd.date_range(
                start=pd.Timestamp(year=2020,month=1,day=14,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=2,day=28,hour=23,minute=0,second=0),
                freq="h"
        ),
        "Covid (national lockdown and up to businesses restart)":pd.date_range(
                start=pd.Timestamp(year=2020,month=3,day=9,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=5,day=17,hour=23,minute=0,second=0),
                freq="h"
        ),
        "In-itinere 2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2020,month=4,day=14,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=4,day=24,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2020,month=6,day=8,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=7,day=17,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "Summer break":pd.date_range(
                start=pd.Timestamp(year=2020,month=7,day=20,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=8,day=31,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "September exam session":pd.date_range(
                start=pd.Timestamp(year=2020,month=9,day=1,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=9,day=18,hour=23,minute=0,second=0),
                freq="h"
        ),
}
periods_AA_2020_2021={
        "Pre 1st semester break":pd.date_range(
                start=pd.Timestamp(year=2020,month=9,day=21,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=9,day=25,hour=23,minute=0,second=0),
                freq="h"
        ),
        "In-itinere 1st semester exam session":pd.date_range(
                start=pd.Timestamp(year=2020,month=11,day=9,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2020,month=11,day=13,hour=23,minute=0,second=0),
                freq="h"
        ),
        "Winter break":pd.date_range(
                start=pd.Timestamp(year=2020,month=12,day=23,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2021,month=1,day=6,hour=23,minute=0,second=0),
                freq="h"
        ),
        "1st semester exam session":pd.date_range(
                start=pd.Timestamp(year=2021,month=1,day=18,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2021,month=2,day=26,hour=23,minute=0,second=0),
                freq="h"
        ),
        "In-itinere 2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2021,month=4,day=6,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2021,month=4,day=16,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2021,month=6,day=7,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2021,month=7,day=16,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "Summer break":pd.date_range(
                start=pd.Timestamp(year=2021,month=7,day=19,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2021,month=8,day=31,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "September exam session":pd.date_range(
                start=pd.Timestamp(year=2021,month=9,day=1,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2021,month=9,day=17,hour=23,minute=0,second=0),
                freq="h"
        ),
}
periods_AA_2021_2022={   
        "Pre 1st semester break":pd.date_range(
                start=pd.Timestamp(year=2021,month=9,day=20,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2021,month=9,day=24,hour=23,minute=0,second=0),
                freq="h"
        ),
        "In-itinere 1st semester exam session":pd.date_range(
                start=pd.Timestamp(year=2021,month=11,day=8,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2021,month=11,day=12,hour=23,minute=0,second=0),
                freq="h"
        ),
        "Winter break":pd.date_range(
                start=pd.Timestamp(year=2021,month=12,day=24,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2022,month=1,day=9,hour=23,minute=0,second=0),
                freq="h"
        ),
        "1st semester exam session":pd.date_range(
                start=pd.Timestamp(year=2022,month=1,day=17,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2022,month=2,day=25,hour=23,minute=0,second=0),
                freq="h"
        ),
        "In-itinere 2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2022,month=4,day=11,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2022,month=4,day=22,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2022,month=6,day=6,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2022,month=7,day=15,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "Summer break":pd.date_range(
                start=pd.Timestamp(year=2022,month=7,day=18,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2022,month=8,day=31,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "September exam session":pd.date_range(
                start=pd.Timestamp(year=2022,month=9,day=1,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2022,month=9,day=16,hour=23,minute=0,second=0),
                freq="h"
        ),
}
periods_AA_2022_2023={   
        "Pre 1st semester break":pd.date_range(
                start=pd.Timestamp(year=2022,month=9,day=19,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2022,month=9,day=23,hour=23,minute=0,second=0),
                freq="h"
        ),
        "In-itinere 1st semester exam session":pd.date_range(
                start=pd.Timestamp(year=2022,month=11,day=7,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2022,month=11,day=11,hour=23,minute=0,second=0),
                freq="h"
        ),
        "Winter break":pd.date_range(
                start=pd.Timestamp(year=2022,month=12,day=24,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2023,month=1,day=8,hour=23,minute=0,second=0),
                freq="h"
        ),
        "1st semester exam session":pd.date_range(
                start=pd.Timestamp(year=2023,month=1,day=16,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2023,month=2,day=25,hour=23,minute=0,second=0),
                freq="h"
        ),
        "In-itinere 2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2023,month=4,day=11,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2023,month=4,day=19,hour=23,minute=0,second=0),
                freq="h"
        ), 
        "2nd semester exam session":pd.date_range(
                start=pd.Timestamp(year=2023,month=6,day=7,hour=0,minute=0,second=0),
                end=pd.Timestamp(year=2023,month=6,day=30,hour=23,minute=0,second=0),
                freq="h"
        ), 
}
periods=[periods_AA_2018_2019,periods_AA_2019_2020,
         periods_AA_2020_2021,periods_AA_2021_2022,periods_AA_2022_2023]

In [ ]:
if compute_TimeHandler:
    for period_AA in periods:
        for period_name,period in period_AA.items():
            time_handler.add_period(
                tft,
                period_name=period_name,
                period=period,
                judgment=judgment,
                as_from_predict_raw=as_from_predict_raw,
                scaler_y=scaler_y
            )
    time_handler.compute_periods_performance()

In [ ]:
time_handler.display_periods_performance(
)

In [ ]:
if compute_TimeHandler:
    time_handler.save_non_init_instance_attributes(path_to_trial_folder,building_name,judgment)

In [ ]:
time_handler.add_period(
    tft,"Test",
    pd.date_range(
        start=pd.Timestamp(year=2022,month=7,day=8,hour=0,minute=0),
        # start=pd.Timestamp(year=2019,month=7,day=8,hour=0,minute=0),
        end=pd.Timestamp(year=2023,month=6,day=30,hour=23,minute=0),
        freq="h"
    ),
    judgment=judgment,
    as_from_predict_raw=as_from_predict_raw,
    scaler_y=scaler_y
)

In [ ]:
for each_yr in [2019,2020,2021,2022,2023]:
    time_handler.add_period(
        tft,"Year-specific performance",
        pd.date_range(
            start=pd.Timestamp(year=each_yr,month=1,day=1,hour=0,minute=0),
            end=pd.Timestamp(year=each_yr,month=12,day=31,hour=23,minute=0),
            freq="h"
        ),
        judgment=judgment,
        as_from_predict_raw=as_from_predict_raw,
        scaler_y=scaler_y
    )

In [ ]:
tft_in_test_fig=time_handler.get_period_above_and_below_metric(
    tft=tft,
    as_from_predict_raw=as_from_predict_raw,     
    metric_name="MAPE",
    # period_name="Year-specific performance",
    period_name="Test",
    judgment=judgment,
    scaler_y=scaler_y,
    upper_threshold=("quantile",0.975),
    lower_threshold=("quantile",0.025),
    building_name="Agriculture" if building_name=="Agriculture extended" else building_name,
    thresholds_wrt_non_overlapping_in_ID=True,
    labels_rotation=0,
    xlabel="",
    fontsize=13,
    figsize=(16,4.5),
    legend_only=True,
    return_only_figure=True,
    stairplot=True,
    dpi=600,
    target_name="electric load (kW)"
)

# ETSformer

## Auxiliary functions

In [ ]:
from torch.utils.data import DataLoader
from ETSformer.models import ETSformer
from torch.optim import Adam
from torch.nn import MSELoss,L1Loss
clone=False
update=False

import subprocess
subprocess.run(["git","--version"])
if clone:
    subprocess.run(["git","clone","https://github.com/salesforce/ETSformer"])
if update:
    subprocess.run(["git","pull"])

In [ ]:
ETSf_hps=namedtuple(
    "ETSf_hps",
    [
        "e_layers","d_layers","enc_in","d_model","n_heads","c_out","d_ff","label_len",
        "seq_len","pred_len","K","output_attention","dropout","activation","std"
    ],
    defaults=[0.2,"sigmoid",0.0]
)

In [ ]:
from torch.utils.data import Dataset
class etsf_dataset(Dataset):
    def __init__(
            self,df_raw:pd.DataFrame, 
            target:str,
            dummies:list[str],
            lookback_window_size:int=n_past,
            horizon_window_size:int=n_future,
            dummies_as_features:bool=False):
        self._df_raw=copy.deepcopy(df_raw)
        self.seq_len=lookback_window_size
        self.pred_len=horizon_window_size
        self._dummies=dummies
        self.target=target
        self._dummies_as_features=dummies_as_features
        self.__read_data__()

    def __read_data__(self):
        self._time_idx=copy.deepcopy(self._df_raw.loc[:,"time_idx"]).to_numpy()
        self._df_raw.drop(columns="time_idx",inplace=True)
        cols=list(self._df_raw.columns)
        cols.remove(self.target)
        if not self._dummies_as_features:
            for col in self._dummies:
                cols.remove(col)
        df_stamp=copy.deepcopy(self._df_raw.loc[:,self._dummies])
        df_data=copy.deepcopy(self._df_raw.loc[:,cols+[self.target]])
        self._enc_in=len(df_data.columns)

        data=df_data.to_numpy()
        data_stamp=df_stamp.to_numpy()

        self.data_x=copy.deepcopy(data)
        self.data_y=copy.deepcopy(data)
        self.data_stamp=data_stamp
    def __len__(self):
        return len(self.data_x)-self.seq_len-self.pred_len+1
    def __getitem__(self, index):
        max_index=len(self)-1
        if index<0 or index>max_index:
            raise IndexError("Index must be an integer between 0 and {}".format(max_index))
        s_begin=index
        s_end=s_begin+self.seq_len
        r_begin=s_end 
        r_end=r_begin+self.pred_len

        encoder_time_idxs=self._time_idx[s_begin:s_end]
        decoder_time_idxs=self._time_idx[r_begin:r_end]
        seq_x=self.data_x[s_begin:s_end]
        seq_y=self.data_y[r_begin:r_end]
        seq_x_mark=self.data_stamp[s_begin:s_end]
        seq_y_mark=self.data_stamp[r_begin:r_end]
        
        return seq_x, seq_y, seq_x_mark, seq_y_mark, encoder_time_idxs, decoder_time_idxs

In [ ]:
class HandleOptimizer():
    def __init__(
            self,
            model:torch.nn.Module,
            learning_rate:float,
            min_lr:float,
            lradj:Literal['exponential','schedule','cos',
                          'cos_with_warmup','exponential_with_warmup',
                          'constant','reduce_on_plateau','reduce_on_plateau_all'],
            train_epochs:int,
            warmup_epochs:int,
            smoothing_learning_rate:float=0,
            damping_learning_rate:float=0,
            ):
        self._model=model
        self._learning_rate=learning_rate
        self._smoothing_learning_rate=smoothing_learning_rate
        self._damping_learning_rate=damping_learning_rate
        self._min_lr=min_lr
        self._lradj=lradj
        self._train_epochs=train_epochs
        self._warmup_epochs=warmup_epochs
        self._select_optimizer()
        self._last_lr={}
        self._reduce_on_plateau_counter={}
        self._best_monitor={}
    def _select_optimizer(self):
        if 'warmup' in self._lradj:
            lr = self._min_lr
        else:
            lr = self._learning_rate
        if self._smoothing_learning_rate > 0:
            smoothing_lr = self._smoothing_learning_rate
        else:
            smoothing_lr = 100 * self._learning_rate

        if self._damping_learning_rate > 0:
            damping_lr = self._damping_learning_rate
        else:
            damping_lr = 100 * self._learning_rate

        nn_params = []
        smoothing_params = []
        damping_params = []
        for k, v in self._model.named_parameters():
            if k[-len('_smoothing_weight'):] == '_smoothing_weight':
                smoothing_params.append(v)
            elif k[-len('_damping_factor'):] == '_damping_factor':
                damping_params.append(v)
            else:
                nn_params.append(v)

        from torch.optim import Adam
        model_optim = Adam([
            {'params': nn_params, 'lr': lr, 'name': 'nn'},
            {'params': smoothing_params, 'lr': smoothing_lr, 'name': 'smoothing'},
            {'params': damping_params, 'lr': damping_lr, 'name': 'damping'},
        ])

        self._optimizer=model_optim
    def adjust_learning_rate(self,epoch,monitor:float|None=None,mode:Literal["min","max"]|optuna.study.StudyDirection|None=None,patience:int|None=None):
        for param_group in self._optimizer.param_groups:
            if param_group['name'] == 'smoothing':
                if self._lradj!='reduce_on_plateau_all': 
                    continue
                else:
                    learning_rate=self._smoothing_learning_rate if self._smoothing_learning_rate>0 else self._learning_rate*100
            elif param_group['name'] == 'damping':
                if self._lradj!='reduce_on_plateau_all':
                    continue
                else:
                    learning_rate=self._damping_learning_rate if self._damping_learning_rate>0 else self._learning_rate*100
            else:
                learning_rate = self._learning_rate

            if self._lradj == 'exponential':
                lr_adjust = {epoch: learning_rate * (0.5 ** ((epoch - 1) // 1))}
            elif self._lradj == 'schedule':
                lr_adjust = {
                    2: 5e-5, 4: 1e-5, 6: 5e-6, 8: 1e-6,
                    10: 5e-7, 15: 1e-7, 20: 5e-8
                }
            elif self._lradj == 'cos':
                lr_adjust = {epoch: learning_rate * 0.5 * (1. + math.cos(math.pi * epoch / self._train_epochs))}
            elif self._lradj == 'cos_with_warmup':
                if epoch <= self._warmup_epochs:
                    lr = self._min_lr + (learning_rate - self._min_lr) * (epoch / (self._warmup_epochs))
                else:
                    curr_epoch = epoch - self._warmup_epochs
                    total_epochs = self._train_epochs - self._warmup_epochs
                    lr = learning_rate * 0.5 * (1. + math.cos(math.pi * curr_epoch / total_epochs))
                lr_adjust = {epoch: lr}
            elif self._lradj == 'exponential_with_warmup':
                if epoch <= self._warmup_epochs:
                    lr = self._min_lr + (learning_rate - self._min_lr) * (epoch / (self._warmup_epochs))
                else:
                    curr_epoch = epoch - self._warmup_epochs
                    lr = learning_rate * (0.5 ** ((curr_epoch) // 1))
                lr_adjust = {epoch: lr}
            elif self._lradj=='constant':
                lr=learning_rate
                lr_adjust={epoch:lr}
            elif 'reduce_on_plateau' in self._lradj:
                assert all(
                    [monitor is not None,mode is not None,patience is not None]
                ),"monitor, mode, and patience must be provided when 'reduce_on_plateau' is in self._lradj."
                if epoch==1:
                    self._last_lr.update({param_group["name"]:learning_rate})
                    self._best_monitor.update({param_group["name"]:monitor})
                    self._reduce_on_plateau_counter.update({param_group["name"]:0})
                elif self._reduce_on_plateau_counter[param_group["name"]]<patience:
                    if mode=="max" or mode==optuna.study.StudyDirection.MAXIMIZE:
                        if monitor>self._best_monitor[param_group["name"]]:
                            self._best_monitor[param_group["name"]]=monitor
                            self._reduce_on_plateau_counter[param_group["name"]]=0
                        else:
                            self._reduce_on_plateau_counter[param_group["name"]]+=1
                    elif mode=="min" or mode==optuna.study.StudyDirection.MINIMIZE:
                        if monitor<self._best_monitor[param_group["name"]]:
                            self._best_monitor[param_group["name"]]=monitor
                            self._reduce_on_plateau_counter[param_group["name"]]=0
                        else:
                            self._reduce_on_plateau_counter[param_group["name"]]+=1
                if self._reduce_on_plateau_counter[param_group["name"]]<patience:
                    lr=self._last_lr[param_group["name"]]
                else:
                    lr=self._last_lr[param_group["name"]]/10
                    self._last_lr[param_group["name"]]=lr
                    self._reduce_on_plateau_counter[param_group["name"]]=0
                lr_adjust={epoch:lr}
            else:
                raise NotImplementedError

            if epoch in lr_adjust.keys():
                lr = lr_adjust[epoch]
                param_group['lr'] = lr

In [ ]:
from typing import Literal,Union
class CustomEarlyStopping():
    def __init__(
            self,
            patience:int,
            min_delta:float=0.0,
            mode:Literal["min","max"]="min",
            check_finite:bool=True
    )->None:
        assert min_delta>=0,"min_delta should be >=0."
        self._min_delta=min_delta
        self._patience=patience
        self._mode=mode
        self._check_finite=check_finite
        self._best_validation_metric_value=np.inf if self._mode=="min" else -np.inf
        self._epochs_run_without_improvement=0
        self._progressive_epoch_number=0
        self._should_save=False
    def is_best(
            self,
            validation_metric_value
    )->bool:
        if self._mode=="max":
            if validation_metric_value>=(self._best_validation_metric_value+self._min_delta):
                return True
            else:
                return False
        else:
            if validation_metric_value<=(self._best_validation_metric_value-self._min_delta):
                return True
            else:
                return False
    def should_stop(
            self,
            validation_metric_value:float,
    )->bool:
        self._progressive_epoch_number+=1
        if (np.isnan(validation_metric_value) or np.isinf(validation_metric_value)) and self._check_finite:
            print("check_finite triggered: validation_metric_value is NaN or inf. Training stopped.")
            self.__init__(
                self._patience,
                self._min_delta,
                self._mode,
                self._check_finite
            )
            return True
        if self._progressive_epoch_number<self._patience:
            if self.is_best(validation_metric_value):
                self._should_save=True
                self._best_validation_metric_value=validation_metric_value
                self._epochs_run_without_improvement=0
            else:
                self._should_save=False
                self._epochs_run_without_improvement+=1
            return False
        else:
            if self.is_best(validation_metric_value):
                self._should_save=True
                self._best_validation_metric_value=validation_metric_value
                self._epochs_run_without_improvement=0
                return False
            else:
                self._should_save=False
                self._epochs_run_without_improvement+=1
                if self._epochs_run_without_improvement>=self._patience:
                    self.__init__(
                        self._patience,
                        self._min_delta,
                        self._mode,
                        self._check_finite
                    )
                    return True
                else:
                    return False

### etsfInterpreter

In [ ]:
ETSf_interpretability=namedtuple(
    "ETSf_interpretability",
    ["target","levels","growths","seasons","predictions","metrics","test_data"]
)
class etsfInterpreter():
    from typing import NamedTuple
    def __init__(
            self,scaler_y,
            level_color:str,growth_color:str,season_color:str,trend_color:str,
            path_to_etsf_results:str|Path=Path("./Results/ETSf")
        ):
        self._scaler_y=scaler_y
        self._path_to_etsf_results=path_to_etsf_results
        self._level_color=level_color
        self._growth_color=growth_color
        self._season_color=season_color
        self._trend_color=trend_color
    def train(
            self,
            model:torch.nn.Module,
            handle_optimizer:HandleOptimizer,
            criterion:torch.nn.modules.loss._Loss,
            train_dataloader:torch.utils.data.DataLoader,
            val_dataloader:torch.utils.data.DataLoader,
            accelerator:Literal["cuda","cpu","auto"]="auto",
            gradient_clip_val:float|None=None,
            clear_cache_on_train_epoch_end:bool=True,
            early_stopping:CustomEarlyStopping|None=None,
            mode:Literal["max","min"]="min",
            reduce_on_plateau_patience:int|None=None,
            save_last:bool=False,
            create_tensorboard_logs:bool=True,
            experiment_name:str|None=None,
            step_log_interval:int=50,
            timeout=8*3600
            ):
        import warnings
        from tqdm.notebook import tqdm
        from time import time
        from datetime import datetime
        from torch.nn.utils import clip_grad_norm_
        from torch.utils.tensorboard.writer import SummaryWriter
        if "reduce_on_plateau" in handle_optimizer._lradj:
            assert reduce_on_plateau_patience is not None,"When 'reduce_on_plateau' is in handle_optimizer._lradj, you must provide reduce_on_plateau_patience."
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            def save_pytorch_model(model,path_to_pth_folder,pth_filename:str)->None:
                torch.save(model.state_dict(),os.path.join(path_to_pth_folder,pth_filename+".pth"))
            now=datetime.now()
            str_now=now.strftime("%Y_%m_%d_h%H_m%M_s%S")
            _experiment_name=str_now if experiment_name is None else experiment_name
            path_to_pth_folder=os.path.join(self._path_to_etsf_results,_experiment_name)
            os.makedirs(path_to_pth_folder,exist_ok=True)
            summary_writer=SummaryWriter(log_dir="etsf_logs/{}".format(_experiment_name))
            if accelerator=="auto":
                device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
            else:
                device=torch.device(accelerator)
            model.to(device)
            global_step=0
            best_val_loss=-np.inf if mode=="max" else np.inf
            model_to_be_saved=model
            epoch_to_be_saved=0
            tic=time()
            batch_times=[]
            on_gpu=device.type=="cuda"
            for epoch_num in range(handle_optimizer._train_epochs):
                model.train(mode=True)
                train_epoch_loss=0
                if create_tensorboard_logs:
                    list_param_groups=handle_optimizer._optimizer.param_groups
                    for param_group in list_param_groups:
                        summary_writer.add_scalar(
                            tag="Optimizer: {} LR".format(param_group["name"]),
                            scalar_value=param_group["lr"],
                            global_step=global_step
                        )
                with tqdm(
                    iterable=train_dataloader,
                    desc="Train epoch {}/{}".format(epoch_num,(handle_optimizer._train_epochs-1)),
                    leave=True,
                    unit="step",
                    ) as decorated_train_dataloader:
                    for X_batch,y_batch,X_batch_mark,y_batch_mark,_,_ in decorated_train_dataloader:
                        X_batch=X_batch.float().to(device)
                        y_batch=y_batch.float().to(device)
                        X_batch_mark=X_batch_mark.float().to(device)
                        y_batch_mark=y_batch_mark.float().to(device)
                        handle_optimizer._optimizer.zero_grad()
                        dec_inp = torch.zeros_like(y_batch[:, -model.configs.pred_len:, :]).float()
                        dec_inp = torch.cat([y_batch[:, :model.configs.label_len, :], dec_inp], dim=1).float().to(device)

                        if on_gpu:
                            torch.cuda.synchronize(device)
                        batch_tic=perf_counter()
                        outputs = model(X_batch, X_batch_mark, dec_inp, y_batch_mark)

                        f_dim = -1
                        y_pred=outputs[:, -model.configs.pred_len:, f_dim:].to(device)
                        y_batch = y_batch[:, -model.configs.pred_len:, f_dim:].to(device)
                        train_batch_loss=criterion(y_pred,y_batch)
                        train_batch_loss.backward()
                        if gradient_clip_val is not None:
                            clip_grad_norm_(model.parameters(),max_norm=gradient_clip_val)
                        handle_optimizer._optimizer.step()
                        if on_gpu:
                            torch.cuda.synchronize(device)
                        batch_elapsed=perf_counter()-batch_tic
                        batch_times.append(batch_elapsed)
                        
                        decorated_train_dataloader.set_postfix({"Batch loss":"{:.3f}".format(train_batch_loss.item())})
                        if create_tensorboard_logs and (global_step+1)%step_log_interval==0:
                            summary_writer.add_scalar(
                                tag="Batch (step) loss: training",
                                scalar_value=train_batch_loss.item(),
                                global_step=global_step
                            )
                        train_epoch_loss+=train_batch_loss.item()
                        global_step+=1
                    train_epoch_loss/=len(train_dataloader)
                    if create_tensorboard_logs:
                        summary_writer.add_scalar(
                            tag="Epoch (average across batches) loss: training",
                            scalar_value=train_epoch_loss,
                            global_step=global_step
                        )
                model.train(mode=False)
                val_epoch_loss=0
                with torch.no_grad():
                    with tqdm(
                        iterable=val_dataloader,
                        desc="Validation check",
                        leave=True,
                        unit="batch"
                    ) as decorated_val_dataloader:
                        for X_batch,y_batch,X_batch_mark,y_batch_mark,_,_ in decorated_val_dataloader:
                            X_batch=X_batch.float().to(device)
                            y_batch=y_batch.float().to(device)
                            X_batch_mark=X_batch_mark.float().to(device)
                            y_batch_mark=y_batch_mark.float().to(device)
                            dec_inp = torch.zeros_like(y_batch[:, -model.configs.pred_len:, :]).float()
                            dec_inp = torch.cat([y_batch[:, :model.configs.label_len, :], dec_inp], dim=1).float().to(device)
                            outputs = model(X_batch, X_batch_mark, dec_inp, y_batch_mark)
                            f_dim = -1
                            y_pred=outputs[:, -model.configs.pred_len:, f_dim:].to(device)
                            y_batch = y_batch[:, -model.configs.pred_len:, f_dim:].to(device)
                            val_batch_loss=criterion(y_pred,y_batch).item()
                            decorated_val_dataloader.set_postfix({"Batch loss":"{:.3f}".format(val_batch_loss)})
                            val_epoch_loss+=val_batch_loss
                val_epoch_loss/=len(val_dataloader)
                if create_tensorboard_logs:
                    summary_writer.add_scalar(
                        tag="Epoch (average across batches) loss: validation",
                        scalar_value=val_epoch_loss,
                        global_step=global_step
                    )
                if early_stopping is not None:
                    if early_stopping.should_stop(val_epoch_loss):
                        print("Early stopping triggered at epoch {} end. Training stopped.".format(epoch_num),flush=True)
                        if save_last:
                            save_pytorch_model(model,path_to_pth_folder,"last_epoch={}".format(epoch_num))
                        break
                    else:
                        if early_stopping._should_save:
                            best_val_loss=val_epoch_loss
                            model_to_be_saved=model
                            epoch_to_be_saved=epoch_num
                else:
                    if mode=="max" and val_epoch_loss>best_val_loss:
                        best_val_loss=val_epoch_loss
                        model_to_be_saved=model
                        epoch_to_be_saved=epoch_num
                    if mode=="min" and val_epoch_loss<best_val_loss:
                        best_val_loss=val_epoch_loss
                        model_to_be_saved=model
                        epoch_to_be_saved=epoch_num
                handle_optimizer.adjust_learning_rate(
                    epoch=epoch_num+1,
                    monitor=val_epoch_loss if "reduce_on_plateau" in handle_optimizer._lradj else None,
                    mode=mode if "reduce_on_plateau" in handle_optimizer._lradj else None,
                    patience=reduce_on_plateau_patience if "reduce_on_plateau" in handle_optimizer._lradj else None
                )
                toc=time()
                if toc-tic>=timeout:
                    print("Timeout={} s triggered. Training stopped at epoch {}.".format(toc-tic,epoch_num))
                    if save_last:
                        save_pytorch_model(model,path_to_pth_folder,"last_epoch={}".format(epoch_num))
                    break
                if clear_cache_on_train_epoch_end:
                    import gc
                    torch.cuda.empty_cache()
                    gc.collect()
            if save_last:
                save_pytorch_model(model,path_to_pth_folder,"last_epoch={}".format(epoch_num))
            save_pytorch_model(model_to_be_saved,path_to_pth_folder,"best_epoch={}".format(epoch_to_be_saved))
            save_variable(model_to_be_saved.configs,os.path.join(path_to_pth_folder,"configs.pkl"))
            save_variable(
                batch_times,
                os.path.join(
                    path_to_pth_folder,
                    "batch_times.pkl"
                )
            )
            if create_tensorboard_logs:
                summary_writer.add_hparams(
                    hparam_dict=model_to_be_saved.configs._asdict(),
                    metric_dict={"best_val_loss":best_val_loss},
                    run_name="hparams"
                )
                summary_writer.flush()
                summary_writer.close()
    def load(self,building_name:str,source:Literal["experiment","optuna study","path"],pick_best:bool=True,trial_number:int=0,pth_filepath:str|Path|None=None):
        if source=="experiment":
            print("Experiments available in {}:".format(self._path_to_etsf_results),flush=True)
            experiments_available=[
                item for item in os.listdir(self._path_to_etsf_results) if os.path.isdir(
                    os.path.join(self._path_to_etsf_results,item)
                    ) and item!="Optimization"
                ]
            print(experiments_available,flush=True)
            index_experiment=int(input("Type the index corresponding to the experiment you want to load:"))
            experiment_name=experiments_available[index_experiment]
            path_to_pth_folder=os.path.join(self._path_to_etsf_results,experiment_name)
            configs=load_variable(os.path.join(path_to_pth_folder,"configs.pkl"))
            model=ETSformer(configs)
            import glob
            matching_path_list=glob.glob(pathname=os.path.join(Path(path_to_pth_folder),"*_epoch=*.pth"))
            print("\nModels saved in {}:".format(path_to_pth_folder),flush=True)
            print(matching_path_list,flush=True)
            index=int(input("Type the index corresponding to the model you want to load:"))
            model.load_state_dict(torch.load(matching_path_list[index],weights_only=True))
            self._loaded_model=model
            print("{} correctly loaded in self.\n".format(matching_path_list[index]),flush=True)
            self._path_to_pth_folder=path_to_pth_folder
            try:
                self._Interpretation=load_variable(os.path.join(self._path_to_pth_folder,"Interpretation_{}.pkl".format(building_name)))
                print("{} correctly loaded in self.".format("Interpretation_{}.pkl".format(building_name)),flush=True)
                self._building_name=building_name
            except FileNotFoundError as e:
                print(e,flush=True)
                print("You should predict() with the loaded model on {} building\nso to load its Interpretation namedtuple in self.".format(building_name))
        elif source=="optuna study":
            print("Studies available in {}:".format(
                os.path.join(self._path_to_etsf_results,"Optimization")
            ),flush=True)
            studies_available=[
                item for item in os.listdir(os.path.join(self._path_to_etsf_results,"Optimization")) if os.path.isdir(
                    os.path.join(self._path_to_etsf_results,"Optimization",item)
                    )
                ]
            print(studies_available,flush=True)
            index_study=int(input("Type the index corresponding to the study you want to load:"))
            study_name=studies_available[index_study]
            path_to_checkpoints=os.path.join(
                self._path_to_etsf_results,"Optimization/{}/Checkpoints".format(study_name)
            )
            study=load_variable(
                os.path.join(self._path_to_etsf_results,"Optimization/{}/{}.pkl".format(study_name,study_name))
            )
            selected_trial,selected_trial_number,selected_trial_best_epoch_number=find_trial_in_study(study,pick_best=pick_best,trial_number=trial_number)
            path_to_trial_ckpt=os.path.join(
                path_to_checkpoints,
                "trial_{}".format(selected_trial_number),
                "epoch={}.pth".format(selected_trial_best_epoch_number)
            )
            print('Trial selected (enumerated from 0): {}/{}\n'.format(selected_trial_number,len(study.trials)-1))
            print('Distributions to sample tentative hyperparameters from:')
            for key,value in selected_trial.distributions.items():
                print('->{}:{}'.format(key,value))
            print('\nHyperparameters:')
            for key,value in selected_trial.params.items():
                if key=="model_dimension":
                    print('->{}:{}'.format(key,round(value/selected_trial.params["number_of_heads"])*selected_trial.params["number_of_heads"]))
                else:
                    print('->{}:{}'.format(key,value))
            print('\nBest epoch (enumerated from 0): {}/{}\n'.format(selected_trial_best_epoch_number,selected_trial.last_step))
            path_to_trial=os.path.join(
                path_to_checkpoints,
                "trial_{}".format(selected_trial_number)
            )
            configs=load_variable(os.path.join(path_to_trial,"configs.pkl"))
            model=ETSformer(configs)
            model.load_state_dict(torch.load(path_to_trial_ckpt,weights_only=True))
            self._loaded_model=model
            print("{} correctly loaded in self.\n".format(path_to_trial_ckpt),flush=True)
            self._loaded_study=study
            print("{} correctly loaded in self.\n".format(study_name),flush=True)
            self._path_to_pth_folder=path_to_trial
            try:
                self._Interpretation=load_variable(os.path.join(self._path_to_pth_folder,"Interpretation_{}.pkl".format(building_name)))
                print("{} correctly loaded in self.".format("Interpretation_{}.pkl".format(building_name)),flush=True)
                self._building_name=building_name
            except FileNotFoundError as e:
                print(e,flush=True)
                print("You should predict() with the loaded model on {} building\nso to load its Interpretation namedtuple in self.".format(building_name))       
        elif source=="path":
            try:
                configs=load_variable(os.path.join(os.path.dirname(pth_filepath),"configs.pkl"))
            except FileNotFoundError:
                print("No 'configs.pkl' file was found in the folder where pth_filename={} is into. Save it or move it there.".format(pth_filepath),flush=True)
            model=ETSformer(configs)
            model.load_state_dict(torch.load(pth_filepath,weights_only=True))
            self._loaded_model=model
            print("{} correctly loaded in self.\n".format(pth_filepath),flush=True)
            self._path_to_pth_folder=os.path.dirname(pth_filepath)
            try:
                self._Interpretation=load_variable(os.path.join(self._path_to_pth_folder,"Interpretation_{}.pkl".format(building_name)))
                print("{} correctly loaded in self.".format("Interpretation_{}.pkl".format(building_name)),flush=True)
                self._building_name=building_name
            except FileNotFoundError as e:
                print(e,flush=True)
                print("You should predict() with the loaded model on {} building\nso to load its Interpretation namedtuple in self.".format(building_name))
        else:
            raise NotImplementedError()
    def predict(
            self,
            building_name:str,
            starting_date:pd.Timestamp,
            test_df:pd.DataFrame,
            test_dataloader:torch.utils.data.DataLoader,
            accelerator:Literal["cuda","cpu","auto"]="auto",
            )->None:
        from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
        from tqdm.notebook import tqdm
        assert "_loaded_model" in self.__dict__.keys(),"Use load_experiment() prior to predict() call, no loaded model found."
        if accelerator=="auto":
            device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            device=torch.device(accelerator)
        self._loaded_model.to(device)
        self._loaded_model.train(mode=False)
        predictions=torch.tensor([],dtype=float).to(device)
        levels=torch.tensor([],dtype=float).to(device)
        growths=torch.tensor([],dtype=float).to(device)
        seasons=torch.tensor([],dtype=float).to(device)        
        target=torch.tensor([],dtype=float).to(device)
        all_seqs_metrics_dict={
            "batch_idx":[],
            "item_idx":[],
        }
        for name in ["MAE","MSE","RMSE","MAPE","R2"]:
            all_seqs_metrics_dict[name]=[]
        with torch.no_grad():
            with tqdm(
                iterable=test_dataloader,
                desc="Predictions",
                leave=True,
                unit="batch"
            ) as decorated_test_dataloader:
                for batch_index,(X_batch,y_batch,X_batch_mark,y_batch_mark,_,_) in enumerate(decorated_test_dataloader):
                    X_batch=X_batch.float().to(device)
                    y_batch=y_batch.float().to(device)
                    X_batch_mark=X_batch_mark.float().to(device)
                    y_batch_mark=y_batch_mark.float().to(device)
                    dec_inp = torch.zeros_like(y_batch[:, -self._loaded_model.configs.pred_len:, :]).float()
                    dec_inp = torch.cat([y_batch[:, :self._loaded_model.configs.label_len, :], dec_inp], dim=1).float().to(device)
                    level,growth,season=self._loaded_model(
                        X_batch, X_batch_mark, dec_inp, y_batch_mark,decomposed=True
                    )
                    outputs=self._loaded_model(X_batch, X_batch_mark, dec_inp, y_batch_mark)
                    f_dim = -1
                    y_pred=outputs[:, -self._loaded_model.configs.pred_len:, f_dim:].to(device)
                    y_batch = y_batch[:, -self._loaded_model.configs.pred_len:, f_dim:].to(device)
                    level=level[:,:,f_dim:].to(device)
                    growth=growth[:,:,f_dim:].to(device)
                    season=season[:,:,f_dim:].to(device)
                    predictions=torch.cat(
                        (predictions,y_pred)
                    )
                    target=torch.cat(
                        (target,y_batch)
                    )
                    levels=torch.cat(
                        (levels,level)
                    )
                    growths=torch.cat(
                        (growths,growth)
                    )
                    seasons=torch.cat(
                        (seasons,season)
                    )
                    for item_index in range(y_batch.shape[0]):
                        y_item=torch.squeeze(y_batch[item_index,:,:])
                        pred_item=torch.squeeze(y_pred[item_index,:,:])
                        pred_item=torch.tensor(
                            scaler_y.inverse_transform(pred_item.cpu().reshape((-1,1))).reshape(-1)
                        )
                        y_item=torch.tensor(
                            scaler_y.inverse_transform(y_item.cpu().reshape((-1,1))).reshape(-1)
                        )
                        metrics_dict={
                            "MAE":mean_absolute_error(y_item,pred_item),
                            "MSE":mean_squared_error(y_item,pred_item),
                            "RMSE":root_mean_squared_error(y_item,pred_item),
                            "MAPE":torch.mean(
                                torch.abs(
                                    y_item[y_item!=0] - pred_item[y_item!=0]
                                ) / y_item[y_item!=0]
                            ).item()*100,           
                            "R2":r2_score(y_item,pred_item),
                        }
                        all_seqs_metrics_dict["batch_idx"].append(batch_index)
                        all_seqs_metrics_dict["item_idx"].append(item_index)
                        for name in metrics_dict:
                            all_seqs_metrics_dict[name].append(metrics_dict[name])
            predictions=torch.squeeze(predictions)
            target=torch.squeeze(target)
            levels=torch.squeeze(levels)
            levels=torch.repeat_interleave(levels.reshape((-1,1)),n_future,dim=1)      
            growths=torch.squeeze(growths)
            seasons=torch.squeeze(seasons)
            all_seqs_metrics_df=pd.DataFrame(
                all_seqs_metrics_dict,
                index=np.arange(0,len(all_seqs_metrics_dict["batch_idx"]))
            )
            Interpretation=ETSf_interpretability(
                target=target,
                levels=levels,
                growths=growths,
                seasons=seasons,
                predictions=predictions,
                metrics=all_seqs_metrics_df,
                test_data={
                    "starting_date":starting_date,
                    "df":copy.deepcopy(test_df),
                    "dataloader":copy.deepcopy(test_dataloader),
                    "dataloader_list":list(copy.deepcopy(test_dataloader))
                }
            )
        save_variable(Interpretation,os.path.join(self._path_to_pth_folder,"Interpretation_{}.pkl".format(building_name)))
        self._Interpretation=Interpretation
        print("{}\nloaded in self._Interpretation. It'll be the one used for visualizations.".format(os.path.join(self._path_to_pth_folder,"Interpretation_{}.pkl".format(building_name))),flush=True)
            
    def visualize_performance(
            self,
            sklearn_metric_name:Literal["MAE","MSE","RMSE","MAPE","R2"],
            upper_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.95),
            lower_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.05),
            show_rescaled:bool=True,
            figsize:tuple[float,float]=(16,9),
            plot_kwargs:dict[str,]={"linewidth":1},
            return_above_and_below_dfs:bool=False,
            test_period:pd.DatetimeIndex|None=None,
            thresholds_wrt_non_overlapping_in_test:bool=True,
            return_scores:bool=False,
            fontsize:float=14,
            legend_only:bool=False,
            xlabel:str|None="Test set dates.",
            return_only_figure:bool=False,
            significance_level:float=0.05,
            return_remainder:bool=False,
            rotation:int=0,
            target_name:str="Energy Consumption",
            target_udm:str="kWh",
            stairplot:bool=False,
            dpi:int|None=None,
            use_axis_formatters:bool=False,
            bins_y:int=10,
            byhour:list[int]=[0,6,12,18],
            )->None|list[pd.DataFrame,pd.DataFrame]|dict[str,float]|tuple[list[pd.DataFrame,pd.DataFrame],dict[str,float]]:
        assert "_Interpretation" in self.__dict__.keys(),"You need to load_experiment() for the wanted model and building before interpreting it."
        predictions=self._Interpretation.predictions
        y=self._Interpretation.target
        dataloader_list=self._Interpretation.test_data["dataloader_list"]
        time_dummy_to_date=pd.Series(
            pd.date_range(
                self._Interpretation.test_data["starting_date"],freq="1h",
                periods=self._Interpretation.test_data["df"].shape[0]
            ),
            index=self._Interpretation.test_data["df"]["time_idx"]
        )
        if test_period is None:
            tp_was_none=True
            test_period=copy.deepcopy(time_dummy_to_date)
        else:
            tp_was_none=False
            test_period=pd.Series(
                test_period,
                index=time_dummy_to_date.loc[time_dummy_to_date.isin(test_period)].index
            )
        if thresholds_wrt_non_overlapping_in_test:
            not_overlapping_batch_and_item_in_test=[]
            overall_seqs_idx=0
            already_seen_decoder_time_idxs=torch.tensor([])
            for batch_index,(_,_,_,_,_,decoder_batch_idxs) in enumerate(dataloader_list):       
                for item_index in range(decoder_batch_idxs.shape[0]):
                    decoder_time_idxs=decoder_batch_idxs[item_index,:]
                    if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                        if all(pd.Series(decoder_time_idxs).isin(test_period.index)):
                            already_seen_decoder_time_idxs=torch.cat(
                                (already_seen_decoder_time_idxs,decoder_time_idxs)
                            )
                            not_overlapping_batch_and_item_in_test.append(overall_seqs_idx)
                    overall_seqs_idx+=1
            not_overlapped_metrics=copy.deepcopy(
                self._Interpretation.metrics.loc[not_overlapping_batch_and_item_in_test,sklearn_metric_name]
            )
        metrics=np.array(self._Interpretation.metrics[sklearn_metric_name])
        if isinstance(upper_threshold,tuple):
            upper_label=" (P{:.1f})".format(upper_threshold[1]*100)
            upper_threshold=np.quantile(
                not_overlapped_metrics if thresholds_wrt_non_overlapping_in_test else metrics,
                upper_threshold[1]
            )
        else:
            upper_label=""
        if isinstance(lower_threshold,tuple):
            lower_label=" (P{:.1f})".format(lower_threshold[1]*100)
            lower_threshold=np.quantile(
                not_overlapped_metrics if thresholds_wrt_non_overlapping_in_test else metrics,
                lower_threshold[1]
            )
        else:
            lower_label=""
        start=True
        already_seen_decoder_time_idxs=torch.tensor([])
        batch_idxs_significant={"above":[],"below":[]}
        item_idxs_significant={"above":[],"below":[]}
        metrics_significant={"above":[],"below":[]}
        decoder_time_idxs_significant={"above":[],"below":[]}
        overall_seqs_idx=0
        for batch_index,(_,_,_,_,_,decoder_batch_idxs) in enumerate(dataloader_list):       
            for item_index in range(decoder_batch_idxs.shape[0]):
                decoder_time_idxs=decoder_batch_idxs[item_index,:]
                if not torch.isin(already_seen_decoder_time_idxs,decoder_time_idxs).any():
                    if all(pd.Series(decoder_time_idxs).isin(test_period.index)):
                        if metrics[overall_seqs_idx]>upper_threshold:  
                            decoder_time_idxs_significant["above"].append(decoder_time_idxs)
                            batch_idxs_significant["above"].append(batch_index)                
                            item_idxs_significant["above"].append(item_index)
                            metrics_significant["above"].append(metrics[overall_seqs_idx])
                        if metrics[overall_seqs_idx]<lower_threshold:  
                            decoder_time_idxs_significant["below"].append(decoder_time_idxs)
                            batch_idxs_significant["below"].append(batch_index)                
                            item_idxs_significant["below"].append(item_index)
                            metrics_significant["below"].append(metrics[overall_seqs_idx])                                
                        already_seen_decoder_time_idxs=torch.cat(
                            (already_seen_decoder_time_idxs,decoder_time_idxs)
                        )
                        predictions_sequence_in_batch=predictions[overall_seqs_idx,:]
                        if start:
                            preds=predictions_sequence_in_batch
                            target=y[overall_seqs_idx,:]
                            lvs=self._Interpretation.levels[overall_seqs_idx,:]
                            gws=self._Interpretation.growths[overall_seqs_idx,:]
                            szs=self._Interpretation.seasons[overall_seqs_idx,:]
                            start=False
                        else:    
                            target=torch.cat((target,y[overall_seqs_idx,:]))
                            preds=torch.cat((preds,predictions_sequence_in_batch))
                            lvs=torch.cat((lvs,self._Interpretation.levels[overall_seqs_idx,:]))
                            gws=torch.cat((gws,self._Interpretation.growths[overall_seqs_idx,:]))
                            szs=torch.cat((szs,self._Interpretation.seasons[overall_seqs_idx,:]))
                overall_seqs_idx+=1                
        tau_max=n_future
        last_encoder_time_dummies=(already_seen_decoder_time_idxs-1).int().tolist()[:-(tau_max-1)]
        decoder_dates=time_dummy_to_date.loc[already_seen_decoder_time_idxs]

        if show_rescaled:
            target=torch.tensor(
                self._scaler_y.inverse_transform(target.cpu().reshape((-1,1))).reshape(-1)
            )
            preds=torch.tensor(
                self._scaler_y.inverse_transform(preds.cpu().reshape((-1,1))).reshape(-1)
            )
            lvs=torch.tensor(
                self._scaler_y.inverse_transform(lvs.cpu().reshape((-1,1))).reshape(-1)
            )
            gws=torch.tensor(
                self._scaler_y.inverse_transform(gws.cpu().reshape((-1,1))).reshape(-1)
            )
            szs=torch.tensor(
                self._scaler_y.inverse_transform(szs.cpu().reshape((-1,1))).reshape(-1)
            )
        else:
            target=target.cpu()
            preds=preds.cpu()
            lvs=lvs.cpu()
            gws=gws.cpu()
            szs=szs.cpu()
            
        from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
        overall_metrics={
            "MAE":mean_absolute_error(target,preds),
            "MSE":mean_squared_error(target,preds),
            "RMSE":root_mean_squared_error(target,preds),
            "MAPE":torch.mean(
                torch.abs(
                    target[target!=0] - preds[target!=0]
                ) / target[target!=0]
            ).item()*100,             
            "R2":r2_score(target,preds),
        }
        names_to_udm={
            "MAE":" (-)" if not show_rescaled else " kWh",
            "MSE":r" (-)$^2$" if not show_rescaled else r" kWh$^2$",
            "RMSE":" (-)" if not show_rescaled else " kWh",
            "MAPE":"%",
            "R2":"",
        }  

        absolute_min=min([target.min(),preds.min()])
        absolute_max=max([target.max(),preds.max()])

        fig_preds=plt.figure(figsize=figsize,dpi=dpi)    
        target_color="r"
        preds_color="b"
        bad_color="#FFD700" 
        good_color = "g"
        with plt.rc_context({"font.size":fontsize}):
            ax_target=fig_preds.gca()
            if not stairplot:
                ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
            else:
                ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
            ax_target.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_target.set_ylabel(
                "Observed "+target_name+" ({})".format(target_udm) if show_rescaled else "Observed "+target_name+" (-)",
                color=target_color
            )
            ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
            if xlabel is not None:
                ax_target.set_xlabel(xlabel)
            better_if_lower=["MAE","MSE","RMSE","MAPE"]
            ymin,ymax=ax_target.get_ylim()
            for list_idx,above_indices in enumerate(decoder_time_idxs_significant["above"]):
                ax_target.fill_between(
                    x=time_dummy_to_date.loc[above_indices],
                    y1=ymin,
                    y2=ymax,
                    fc=bad_color if sklearn_metric_name in better_if_lower else good_color,
                    alpha=0.35,
                    label="{}>{:.4f}".format(sklearn_metric_name,upper_threshold)+names_to_udm[sklearn_metric_name]+upper_label if list_idx==0 else ""
                )

            for list_idx,below_indices in enumerate(decoder_time_idxs_significant["below"]):
                ax_target.fill_between(
                    x=time_dummy_to_date.loc[below_indices],
                    y1=ymin,
                    y2=ymax,
                    fc=good_color if sklearn_metric_name in better_if_lower else bad_color,
                    alpha=0.35,
                    label="{}<{:.4f}".format(sklearn_metric_name,lower_threshold)+names_to_udm[sklearn_metric_name]+lower_label if list_idx==0 else ""
                )

            ax_target.legend(loc="lower center",bbox_to_anchor=(0.5, 1.0),ncols=2)
            ax_target.set_xlim(
                (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
                max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
            )
            for label in ax_target.get_xticklabels():
                label.set_rotation(rotation)
            ax_preds=ax_target.twinx()
            if not stairplot:
                ax_preds.plot(decoder_dates,preds,color=preds_color)
            else:
                ax_preds.step(decoder_dates,preds,color=preds_color,where="post")
            ax_preds.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_preds.set_ylabel(
                "Predicted "+target_name+" ({})".format(target_udm) if show_rescaled else "Predicted "+target_name+" (-)",
                color=preds_color)
            ax_preds.tick_params(axis="y",color=preds_color,labelcolor=preds_color)
            if not legend_only:
                suptitle_strs=["{} building".format(building_name),"\nMetrics:"]
                for idx,(name,value) in enumerate(overall_metrics.items()):
                    if idx!=(len(overall_metrics.items())-1):
                        suptitle_strs.append("{}={:.2f}".format(name,value)+names_to_udm[name]+",")
                    else:
                        suptitle_strs.append("{}={:.3f}".format(name,value)+names_to_udm[name])
                fig_preds.suptitle(" ".join(suptitle_strs))
            fig_preds.tight_layout()
            
            fig_levels=plt.figure(figsize=figsize,dpi=dpi) 
            absolute_min=min([target.min(),lvs.min()])
            absolute_max=max([target.max(),lvs.max()])   
            ax_target=fig_levels.gca()
            if not stairplot:
                ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
            else:
                ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
            ax_target.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_target.set_ylabel(
                "Observed "+target_name+" ({})".format(target_udm) if show_rescaled else "Observed "+target_name+" (-)",
                color=target_color
            )
            ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
            if xlabel is not None:
                ax_target.set_xlabel(xlabel)
            ax_target.set_xlim(
                (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
                max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
            )
            if use_axis_formatters:
                ax_target.xaxis.set_major_locator(HourLocator(byhour))
                xticks=ax_target.get_xticks()
                xlabels=[]
                for i,tick in enumerate(xticks):
                    date=num2date(tick)
                    if i==0:
                        if date.hour!=0:
                            xlabels.append(date.strftime("%a %d %b %Y, %H:%M"))
                        else:
                            xlabels.append(date.strftime("%a %d %b %Y"))
                    else:
                        to_add=""
                        if date.day!=prev_stats["day"]:
                            to_add+=" %a %d"
                        if date.month!=prev_stats["month"]:
                            to_add+=" %b"
                        if date.year!=prev_stats["year"]:
                            to_add+=" %Y"
                        if to_add!="":
                            to_add+=", "
                        fmt=(to_add+"%H:%M").lstrip()
                        if fmt!="%a %d %b %Y, %H:%M":
                            xlabels.append(date.strftime(fmt))
                        else:
                            xlabels.append(date.strftime(to_add.rstrip(" ,")))
                    prev_stats={"year":date.year,"month":date.month,"day":date.day}
                ax_target.set_xticks(xticks,xlabels,rotation=rotation)
                ax_target.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            else:
                for label in ax_target.get_xticklabels():
                    label.set_rotation(rotation)
            ax_lvs=ax_target.twinx()
            if use_axis_formatters:
                ax_lvs.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            if not stairplot:
                ax_lvs.plot(decoder_dates,lvs,color=self._level_color)
            else:
                ax_lvs.step(decoder_dates,lvs,color=self._level_color,where="post")
            ax_lvs.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_lvs.set_ylabel(
                "Predicted {}: Level ({})".format(target_name,target_udm) if show_rescaled else "Predicted {}: Level (-)".format(target_name),
                color=self._level_color)
            ax_lvs.tick_params(axis="y",color=self._level_color,labelcolor=self._level_color)
            if not legend_only:
                fig_levels.suptitle("Observed target VS Predicted level")
            fig_levels.tight_layout()

            fig_growth_alone=plt.figure(figsize=figsize,dpi=dpi)
            absolute_min=gws.min()
            absolute_max=gws.max()  
            ax_growth_alone=fig_growth_alone.gca()
            if not stairplot:
                ax_growth_alone.plot(decoder_dates,gws,color=self._growth_color)
            else:
                ax_growth_alone.step(decoder_dates,gws,color=self._growth_color,where="post")
            ax_growth_alone.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_growth_alone.set_ylabel(
                "Predicted {}: Growth ({})".format(target_name,target_udm) if show_rescaled else "Predicted {}: Growth (-)".format(target_name),
                color=self._growth_color)
            ax_growth_alone.tick_params(axis="y",color=self._growth_color,labelcolor=self._growth_color)
            if use_axis_formatters:
                ax_growth_alone.xaxis.set_major_locator(HourLocator(byhour))
                xticks=ax_growth_alone.get_xticks()
                xlabels=[]
                for i,tick in enumerate(xticks):
                    date=num2date(tick)
                    if i==0:
                        if date.hour!=0:
                            xlabels.append(date.strftime("%a %d %b %Y, %H:%M"))
                        else:
                            xlabels.append(date.strftime("%a %d %b %Y"))
                    else:
                        to_add=""
                        if date.day!=prev_stats["day"]:
                            to_add+=" %a %d"
                        if date.month!=prev_stats["month"]:
                            to_add+=" %b"
                        if date.year!=prev_stats["year"]:
                            to_add+=" %Y"
                        if to_add!="":
                            to_add+=", "
                        fmt=(to_add+"%H:%M").lstrip()
                        if fmt!="%a %d %b %Y, %H:%M":
                            xlabels.append(date.strftime(fmt))
                        else:
                            xlabels.append(date.strftime(to_add.rstrip(" ,")))
                    prev_stats={"year":date.year,"month":date.month,"day":date.day}
                ax_growth_alone.set_xticks(xticks,xlabels,rotation=rotation)
                ax_growth_alone.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            else:
                for label in ax_growth_alone.get_xticklabels():
                    label.set_rotation(rotation)
            if xlabel is not None:
                ax_growth_alone.set_xlabel(xlabel)
            ax_growth_alone.set_xlim(
                (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
                max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
            )
            if not legend_only:
                fig_growth_alone.suptitle("Predicted growth")
            fig_growth_alone.tight_layout()

            fig_growths=plt.figure(figsize=figsize,dpi=dpi) 
            absolute_min=min([target.min(),gws.min()])
            absolute_max=max([target.max(),gws.max()])   
            ax_target=fig_growths.gca()
            if not stairplot:
                ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
            else:
                ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
            ax_target.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_target.set_ylabel(
                "Observed {} ({})".format(target_name,target_udm) if show_rescaled else "Observed {} (-)".format(target_name),
                color=target_color
            )
            ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
            if xlabel is not None:
                ax_target.set_xlabel(xlabel)
            ax_target.set_xlim(
                (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
                max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
            )
            if use_axis_formatters:
                ax_target.xaxis.set_major_locator(HourLocator(byhour))
                xticks=ax_target.get_xticks()
                xlabels=[]
                for i,tick in enumerate(xticks):
                    date=num2date(tick)
                    if i==0:
                        if date.hour!=0:
                            xlabels.append(date.strftime("%a %d %b %Y, %H:%M"))
                        else:
                            xlabels.append(date.strftime("%a %d %b %Y"))
                    else:
                        to_add=""
                        if date.day!=prev_stats["day"]:
                            to_add+=" %a %d"
                        if date.month!=prev_stats["month"]:
                            to_add+=" %b"
                        if date.year!=prev_stats["year"]:
                            to_add+=" %Y"
                        if to_add!="":
                            to_add+=", "
                        fmt=(to_add+"%H:%M").lstrip()
                        if fmt!="%a %d %b %Y, %H:%M":
                            xlabels.append(date.strftime(fmt))
                        else:
                            xlabels.append(date.strftime(to_add.rstrip(" ,")))
                    prev_stats={"year":date.year,"month":date.month,"day":date.day}
                ax_target.set_xticks(xticks,xlabels,rotation=rotation)
                ax_target.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            else:
                for label in ax_target.get_xticklabels():
                    label.set_rotation(rotation)
            ax_gws=ax_target.twinx()
            if use_axis_formatters:
                ax_gws.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            if not stairplot:
                ax_gws.plot(decoder_dates,gws,color=self._growth_color)
            else:
                ax_gws.step(decoder_dates,gws,color=self._growth_color,where="post")
            ax_gws.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_gws.set_ylabel("Predicted {}: Growth ({})".format(target_name,target_udm) if show_rescaled else "Predicted {}: Growth (-)".format(target_name),
                                color=self._growth_color)
            ax_gws.tick_params(axis="y",color=self._growth_color,labelcolor=self._growth_color)
            if not legend_only:
                fig_growths.suptitle("Observed target VS Predicted growth")
            fig_growths.tight_layout()

            fig_trend=plt.figure(figsize=figsize,dpi=dpi) 
            absolute_min=min([target.min(),(lvs+gws).min()])
            absolute_max=max([target.max(),(lvs+gws).max()])   
            ax_target=fig_trend.gca()
            if not stairplot:
                ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
            else:
                ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
            ax_target.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_target.set_ylabel(
                "Observed {} ({})".format(target_name,target_udm) if show_rescaled else "Observed {} (-)".format(target_name),
                color=target_color
            )
            ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
            if xlabel is not None:
                ax_target.set_xlabel(xlabel)
            ax_target.set_xlim(
                (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
                max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
            )
            if use_axis_formatters:
                ax_target.xaxis.set_major_locator(HourLocator(byhour))
                xticks=ax_target.get_xticks()
                xlabels=[]
                for i,tick in enumerate(xticks):
                    date=num2date(tick)
                    if i==0:
                        if date.hour!=0:
                            xlabels.append(date.strftime("%a %d %b %Y, %H:%M"))
                        else:
                            xlabels.append(date.strftime("%a %d %b %Y"))
                    else:
                        to_add=""
                        if date.day!=prev_stats["day"]:
                            to_add+=" %a %d"
                        if date.month!=prev_stats["month"]:
                            to_add+=" %b"
                        if date.year!=prev_stats["year"]:
                            to_add+=" %Y"
                        if to_add!="":
                            to_add+=", "
                        fmt=(to_add+"%H:%M").lstrip()
                        if fmt!="%a %d %b %Y, %H:%M":
                            xlabels.append(date.strftime(fmt))
                        else:
                            xlabels.append(date.strftime(to_add.rstrip(" ,")))
                    prev_stats={"year":date.year,"month":date.month,"day":date.day}
                ax_target.set_xticks(xticks,xlabels,rotation=rotation)
                ax_target.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            else:
                for label in ax_target.get_xticklabels():
                    label.set_rotation(rotation)
            ax_trend=ax_target.twinx()
            if use_axis_formatters:
                ax_trend.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            if not stairplot:
                ax_trend.plot(decoder_dates,(lvs+gws),color=self._trend_color)
            else:
                ax_trend.step(decoder_dates,(lvs+gws),color=self._trend_color,where="post")
            ax_trend.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_trend.set_ylabel("Predicted {}: Trend ({})".format(target_name,target_udm) if show_rescaled else "Predicted {}: Trend (-)".format(target_name),
                                color=self._trend_color)
            ax_trend.tick_params(axis="y",color=self._trend_color,labelcolor=self._trend_color)
            if not legend_only:
                fig_trend.suptitle("Observed target VS Predicted trend (level+growth)")
            fig_trend.tight_layout()

            fig_seasons=plt.figure(figsize=figsize,dpi=dpi)    
            absolute_min=min([target.min(),szs.min()])
            absolute_max=max([target.max(),szs.max()])
            print("Mean of season: {}".format(szs.mean())+" kWh" if show_rescaled else "")
            ax_target=fig_seasons.gca()
            if not stairplot:
                ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
            else:
                ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
            ax_target.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_target.set_ylabel(
                "Observed {} ({})".format(target_name,target_udm) if show_rescaled else "Observed {} (-)".format(target_name),
                color=target_color
            )
            ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
            if xlabel is not None:
                ax_target.set_xlabel(xlabel)
            ax_target.set_xlim(
                (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
                max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
            )
            if use_axis_formatters:
                ax_target.xaxis.set_major_locator(HourLocator(byhour))
                xticks=ax_target.get_xticks()
                xlabels=[]
                for i,tick in enumerate(xticks):
                    date=num2date(tick)
                    if i==0:
                        if date.hour!=0:
                            xlabels.append(date.strftime("%a %d %b %Y, %H:%M"))
                        else:
                            xlabels.append(date.strftime("%a %d %b %Y"))
                    else:
                        to_add=""
                        if date.day!=prev_stats["day"]:
                            to_add+=" %a %d"
                        if date.month!=prev_stats["month"]:
                            to_add+=" %b"
                        if date.year!=prev_stats["year"]:
                            to_add+=" %Y"
                        if to_add!="":
                            to_add+=", "
                        fmt=(to_add+"%H:%M").lstrip()
                        if fmt!="%a %d %b %Y, %H:%M":
                            xlabels.append(date.strftime(fmt))
                        else:
                            xlabels.append(date.strftime(to_add.rstrip(" ,")))
                    prev_stats={"year":date.year,"month":date.month,"day":date.day}
                ax_target.set_xticks(xticks,xlabels,rotation=rotation)
                ax_target.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            else:
                for label in ax_target.get_xticklabels():
                    label.set_rotation(rotation)
            ax_szs=ax_target.twinx()
            if use_axis_formatters:
                ax_szs.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            if not stairplot:
                ax_szs.plot(decoder_dates,szs,color=self._season_color)
            else:
                ax_szs.step(decoder_dates,szs,color=self._season_color,where="post")
            ax_szs.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_szs.set_ylabel("Predicted {}: Season ({})".format(target_name,target_udm) if show_rescaled else "Predicted {}: Season (-)".format(target_name),
                                color=self._season_color)
            ax_szs.tick_params(axis="y",color=self._season_color,labelcolor=self._season_color)
            if not legend_only:    
                fig_seasons.suptitle("Observed target VS Predicted season")
            fig_seasons.tight_layout()

            fig_remainder=plt.figure(figsize=figsize,dpi=dpi)
            remainder=target-szs-gws-lvs
            print("Mean of remainder: {}".format(remainder.mean())+" kWh" if show_rescaled else "")
            from statsmodels.stats.diagnostic import acorr_ljungbox
            from statsmodels.stats.stattools import jarque_bera
        
            if tp_was_none:
                ljungbox_df=acorr_ljungbox(remainder,lags=168,boxpierce=False)
                print("Ljung-Box acorrelation test on remainder:")
                print("Null hypothesis: data is not correlated.")
                print("Alternative hypothesis: data exhibits serial correlation.")
                display(ljungbox_df)
                if len(ljungbox_df.loc[ljungbox_df["lb_pvalue"]>significance_level])>0:
                    display(ljungbox_df.loc[ljungbox_df["lb_pvalue"]>significance_level])
                    print("These are the lags up to which the p-value is larger than the significance level.")
                    print("Null hypothesis can't be rejected. Data seems not correlated.")
                else:
                    print("There are no p-values larger than the significance level across all lags")
                    print("Null hypothesis can be rejected. Data seem to exhibit serial correlation.")

                _,jb_pvalue,_,_=jarque_bera(remainder)
                print("Jarque-Bera normality test on residuals:")
                print("Null hypothesis: data is normally distributed.")
                print("Alternative hypothesis: data is not normally distributed.")
                print("p-value: {}".format(jb_pvalue))
                if jb_pvalue<=significance_level:
                    print("Null hypothesis can be rejected. Data seems not normally distributed.")
                else:
                    print("Null hypothesis can't be rejected. Data seem to follow the normal distribution.")
            absolute_min=remainder.min()
            absolute_max=remainder.max()  
            ax_remainder=fig_remainder.gca()
            if not stairplot:
                ax_remainder.plot(decoder_dates,remainder,color="k")
            else:
                ax_remainder.step(decoder_dates,remainder,color="k",where="post")
            ax_remainder.set_ylim(
                bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
            )
            ax_remainder.set_ylabel(
                "Remainder {} ({})".format(target_name,target_udm) if show_rescaled else "Remainder {} (-)".format(target_name),
                color="k"
            )
            ax_remainder.tick_params(axis="y",color="k",labelcolor="k")
            if xlabel is not None:
                ax_remainder.set_xlabel(xlabel)
            ax_remainder.set_xlim(
                (min(time_dummy_to_date.loc[last_encoder_time_dummies]),
                max(time_dummy_to_date.loc[already_seen_decoder_time_idxs]))
            )
            if use_axis_formatters:
                ax_remainder.xaxis.set_major_locator(HourLocator(byhour))
                xticks=ax_remainder.get_xticks()
                xlabels=[]
                for i,tick in enumerate(xticks):
                    date=num2date(tick)
                    if i==0:
                        if date.hour!=0:
                            xlabels.append(date.strftime("%a %d %b %Y, %H:%M"))
                        else:
                            xlabels.append(date.strftime("%a %d %b %Y"))
                    else:
                        to_add=""
                        if date.day!=prev_stats["day"]:
                            to_add+=" %a %d"
                        if date.month!=prev_stats["month"]:
                            to_add+=" %b"
                        if date.year!=prev_stats["year"]:
                            to_add+=" %Y"
                        if to_add!="":
                            to_add+=", "
                        fmt=(to_add+"%H:%M").lstrip()
                        if fmt!="%a %d %b %Y, %H:%M":
                            xlabels.append(date.strftime(fmt))
                        else:
                            xlabels.append(date.strftime(to_add.rstrip(" ,")))
                    prev_stats={"year":date.year,"month":date.month,"day":date.day}
                ax_remainder.set_xticks(xticks,xlabels,rotation=rotation)
                ax_remainder.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
            else:
                for label in ax_remainder.get_xticklabels():
                    label.set_rotation(rotation)
            if not legend_only:
                fig_remainder.suptitle("Remainder")
            fig_remainder.tight_layout()

            plt.close("all")
            display(fig_preds)
            display(fig_levels)
            display(fig_growth_alone)
            display(fig_growths)
            display(fig_trend)
            display(fig_seasons)
            display(fig_remainder)
        if return_only_figure:
            return fig_preds
        else:
            if return_above_and_below_dfs:
                above_df=pd.DataFrame(
                    {"batch_idx":batch_idxs_significant["above"],
                    "item_idx":item_idxs_significant["above"],
                    sklearn_metric_name:metrics_significant["above"]}
                    ).sort_values(by=sklearn_metric_name,ascending=False)
                above_df.set_index(
                    np.arange(1,len(batch_idxs_significant["above"])+1,dtype=int),inplace=True
                )
                below_df=pd.DataFrame(
                    {"batch_idx":batch_idxs_significant["below"],
                    "item_idx":item_idxs_significant["below"],
                    sklearn_metric_name:metrics_significant["below"]}
                    ).sort_values(by=sklearn_metric_name,ascending=True)
                below_df.set_index(
                    np.arange(1,len(batch_idxs_significant["below"])+1,dtype=int),inplace=True
                )
                if not return_scores:
                    if not return_remainder:
                        return [above_df,below_df]
                    else:
                        return [above_df,below_df],remainder
                else:
                    if not return_remainder:
                        return [above_df,below_df],overall_metrics
                    else:
                        return [above_df,below_df],overall_metrics,remainder
            else:
                if return_scores:
                    if not return_remainder:
                        return overall_metrics
                    else:
                        return overall_metrics,remainder
    def visualize_item(
            self,
            batch_idx:int,
            item_idx:int=0,
            figsize:tuple[float,float]=(16,8),
            linewidth:float=1.5,
            show_rescaled:bool=False,
            custom_ticks:bool=True,
            plotting_time_interval:int=24,
            add_loss_to_title:bool=True,
            show_future_observed:bool=True,
            show_level:bool=True,
            show_growth:bool=True,
            show_season:bool=True,
            use_dates:bool=False,
            )->None:      
        predictions=self._Interpretation.predictions
        dataloader=self._Interpretation.test_data["dataloader"]
        batch_size=dataloader.batch_size
        number_of_batches=len(dataloader)
        if not (batch_idx in np.arange(0,number_of_batches)):
            raise Exception("batch_idx selects the index of the selected batch. batch_idx={} but it must be in [0,...,#batches in DataLoader-1={}].".format(
                batch_idx,number_of_batches-1
                )
            )
        batch_to_seqs_slice={
            batch_num:slice(batch_num*batch_size,(batch_num+1)*batch_size) if batch_num!=(number_of_batches-1) else slice(batch_num*batch_size,None) for batch_num in range(number_of_batches)
        }
        predictions_in_batch=predictions[batch_to_seqs_slice[batch_idx],:]
        
        if not (item_idx in np.arange(0,batch_size)):
            raise Exception("item selects the index of the sequence within batch={} of which you want to\nvisualize the predictions. item={} but it must be in [0,...,#sequences in batch_{}-1={}].".format(
                batch_idx,item_idx,batch_idx,batch_size-1
                )
            )

        predictions_sequence_in_batch=predictions_in_batch[item_idx,:]
        level_in_item_at_decoder=self._Interpretation.levels[batch_to_seqs_slice[batch_idx],:][item_idx,:].cpu()
        growth_in_item_at_decoder=self._Interpretation.growths[batch_to_seqs_slice[batch_idx],:][item_idx,:].cpu()
        season_in_item_at_decoder=self._Interpretation.seasons[batch_to_seqs_slice[batch_idx],:][item_idx,:].cpu()
        X_batch,y_batch,_,_,encoder_time_idxs_batch,decoder_time_idxs_batch=self._Interpretation.test_data["dataloader_list"][batch_idx]
        encoder_target=X_batch[:,:,-1]
        decoder_target=y_batch[:,:,-1]
        encoder_target=encoder_target[item_idx,:]
        decoder_target=decoder_target[item_idx,:]
        if show_rescaled:
            predictions_sequence_in_batch=torch.tensor(
                self._scaler_y.inverse_transform(predictions_sequence_in_batch.cpu().reshape((-1,1))).reshape(-1)
            )
            encoder_target=torch.tensor(
                self._scaler_y.inverse_transform(encoder_target.reshape((-1,1))).reshape(-1)
            )
            decoder_target=torch.tensor(
                self._scaler_y.inverse_transform(decoder_target.reshape((-1,1))).reshape(-1)
            )
            level_in_item_at_decoder=torch.tensor(
                self._scaler_y.inverse_transform(level_in_item_at_decoder.reshape((-1,1))).reshape(-1)
            )
            growth_in_item_at_decoder=torch.tensor(
                self._scaler_y.inverse_transform(growth_in_item_at_decoder.reshape((-1,1))).reshape(-1)
            )
            season_in_item_at_decoder=torch.tensor(
                self._scaler_y.inverse_transform(season_in_item_at_decoder.reshape((-1,1))).reshape(-1)
            )
            
        fig=plt.figure(figsize=figsize)
        ax_predictions=fig.gca()
        encoder_positions=np.arange(
            start=-(self._loaded_model.configs.seq_len-1),
            stop=1
        )
        decoder_positions=np.arange(
            start=1,
            stop=self._loaded_model.configs.pred_len+1
        )
        prop_cycle = iter(plt.rcParams["axes.prop_cycle"])
        target_color = next(prop_cycle)["color"]
        predictions_color = next(prop_cycle)["color"]
        level_color=self._level_color
        growth_color=self._growth_color
        season_color=self._season_color
        ax_predictions.plot(encoder_positions,encoder_target,color=target_color,linewidth=linewidth,label="Observed")
        if show_future_observed:
            ax_predictions.plot(decoder_positions,decoder_target,color=target_color,linewidth=linewidth)
        ax_predictions.plot(
            decoder_positions,
            predictions_sequence_in_batch.cpu(),
            color=predictions_color,linewidth=linewidth,label="Predicted")
        from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score,mean_squared_error
        metrics_dict={
            "MAE":dict(
                value=mean_absolute_error(decoder_target,predictions_sequence_in_batch.cpu()),
                udm=" (-)" if not show_rescaled else " kWh"
            ),
            "MSE":dict(
                value=mean_squared_error(decoder_target,predictions_sequence_in_batch.cpu()),
                udm=r" (-)$^2$" if not show_rescaled else r" kWh$^2$"
            ),
            "RMSE":dict(
                value=root_mean_squared_error(decoder_target,predictions_sequence_in_batch.cpu()),
                udm=" (-)" if not show_rescaled else " kWh"
            ),
            "MAPE":dict(
                value=torch.mean(
                    torch.abs(
                        decoder_target[decoder_target!=0] - predictions_sequence_in_batch.cpu()[decoder_target!=0]
                    ) / decoder_target[decoder_target!=0]
                ).item()*100,
                udm="%"
            ),                
            "R2":dict(
                value=r2_score(decoder_target,predictions_sequence_in_batch.cpu()),
                udm=""
            )
        }
        title_for_loss=" Loss computed on predictions: "
        metrics_str=", ".join(["{}={:.2f}{}".format(
            mtc_name,
            metrics_dict[mtc_name]["value"],
            metrics_dict[mtc_name]["udm"]
        ) for mtc_name in list(metrics_dict.keys())])
        title_for_loss+=metrics_str+"."
        if show_level:
            ax_predictions.plot(
                decoder_positions,
                level_in_item_at_decoder,
                color=level_color,linewidth=linewidth,label="Level")
        if show_growth:
            ax_predictions.plot(
                decoder_positions,
                growth_in_item_at_decoder,
                color=growth_color,linewidth=linewidth,label="Growth")
        if show_season:
            ax_predictions.plot(
                decoder_positions,
                season_in_item_at_decoder,
                color=season_color,linewidth=linewidth,label="Season")
        ax_predictions.set_ylabel("Target (kWh)" if show_rescaled else "Target rescaled (-)")
        ax_predictions.set_xlabel("Position index (n)" if not use_dates else "") 
        title="Predictions for sequence index={} in batch index={}.".format(
            item_idx,batch_idx
        )
        title=title+title_for_loss if add_loss_to_title else title
        ax_predictions.legend(
            loc="lower center",
            ncols=2+show_level+show_growth+show_season,
            bbox_to_anchor=(0.5,1)
        )
        if custom_ticks:
            ticks=np.arange(
                encoder_positions.min(),
                decoder_positions.max(),
                plotting_time_interval
            )
            if not use_dates:
                labels=[str(tick) for tick in ticks]
            else:
                time_dummy_to_date=pd.Series(
                    pd.date_range(
                        self._Interpretation.test_data["starting_date"],freq="1h",
                        periods=self._Interpretation.test_data["df"].shape[0]
                    ),
                    index=self._Interpretation.test_data["df"]["time_idx"]
                )
                dates=time_dummy_to_date.loc[
                    torch.cat(
                        (encoder_time_idxs_batch[item_idx,:],decoder_time_idxs_batch[item_idx,:])
                    )
                ]
                if plotting_time_interval<24 and 24%plotting_time_interval==0:
                    labels=[]
                    for date_idx,date in enumerate(dates):
                        if date_idx%plotting_time_interval==0:
                            if date_idx%24==0:
                                labels.append(date.strftime("%a %Y/%m/%d %H:%M"))
                            else:
                                labels.append(date.strftime("%H:%M"))
                else:
                    labels=[
                        date.strftime("%a %Y/%m/%d %H:%M") for date in dates[slice(0,None,plotting_time_interval)]
                    ] 
            ax_predictions.set_xticks(
                ticks=ticks,
                labels=labels,
                rotation=0 if not use_dates else 90
            )
            ax_predictions.set_xlim((encoder_positions.min(),decoder_positions.max()))
        fig.suptitle(title)
        fig.tight_layout()
        plt.close("all")
        display(fig)
    def optimize_hyperparameters(
            self,
            study_name:str,
            pruner:optuna.pruners.BasePruner,
            n_trials:int,
            max_epochs:int,
            criterion:torch.nn.modules.loss._Loss,
            train_dataloader:torch.utils.data.DataLoader,
            val_dataloader:torch.utils.data.DataLoader,
            layers_distribution:dict[Literal["low","high","step","log"],int|bool]|list[int]={
                "low":1,"high":4,"step":1,"log":False
            },
            latent_space_distribution:dict[Literal["low","high","step","log"],int|bool]|list[int]=[
                16,32,64,128,256
            ],
            heads_distribution:dict[Literal["low","high","step","log"],int|bool]|list[int]={
                "low":1,"high":10,"step":1,"log":False
            },
            K_distribution:dict[Literal["low","high","step","log"],int|bool]|list[int]={
                "low":1,"high":3,"step":1,"log":False
            },
            ff_distribution:dict[Literal["low","high","step","log"],int|bool]|list[int]|None=None,
            dropout_distribution:dict[Literal["low","high","step","log"],float|bool]|list[float]={
                "low":0.2,"high":0.2,"step":None,"log":False
            },
            gradient_clipping_distribution:dict[Literal["low","high","step","log"],float|bool]|list[float]={
                "low":1.0,"high":1.0,"step":None,"log":False
            },
            nn_LR_distribution:dict[Literal["low","high","step","log"],float|bool]|list[float]={
                "low":1e-4,"high":1e-1,"step":None,"log":True
            },
            smoothing_LR_distribution:dict[Literal["low","high","step","log"],float|bool]|list[float]|None=None,
            damping_LR_distribution:dict[Literal["low","high","step","log"],float|bool]|list[float]|None=None,
            scheduler_distribution:list[str]|None=["cos_with_warmup"],
            timeout:int|float=8*3600,
            verbose:int|None=1,
            direction:optuna.study.StudyDirection=optuna.study.StudyDirection.MINIMIZE,
            accelerator:Literal["cuda","cpu","auto"]="auto",
            reduce_on_plateau_patience:int|None=None,
            clear_cache_on_train_epoch_end:bool=True,
            early_stopping:CustomEarlyStopping|None=None,
            create_tensorboard_logs:bool=True,
            step_log_interval:int=50,
    ):
        import optuna.logging
        logging_level={
            None:optuna.logging.get_verbosity(),
            0:optuna.logging.WARNING,
            1:optuna.logging.INFO,
            2:optuna.logging.DEBUG,
        }
        optuna.logging.set_verbosity(logging_level[verbose])
        if ff_distribution is None:
            if isinstance(latent_space_distribution,dict):
                ff_distribution={
                    "low":4*latent_space_distribution["low"],
                    "high":4*latent_space_distribution["high"],
                    "step":4*latent_space_distribution["step"],
                    "log":latent_space_distribution["log"]
                }
            elif isinstance(latent_space_distribution,list):
                ff_distribution=(4*np.array(latent_space_distribution,dtype=int)).tolist()
            else:
                raise Exception("ff_distribution must be either a dict or a list or None.")
        from typing import get_args,get_type_hints
        if scheduler_distribution is None:          
            scheduler_distribution=list(get_args(get_type_hints(HandleOptimizer.__init__)["lradj"]))
        def objective(trial:optuna.Trial):
            early_stopping.__init__(
                patience=early_stopping._patience,
                min_delta=early_stopping._min_delta,
                mode=early_stopping._mode,
                check_finite=early_stopping._check_finite
            )
            import warnings
            from torch.nn.utils import clip_grad_norm_
            from torch.utils.tensorboard.writer import SummaryWriter
            gradient_clip_val=trial.suggest_float(
                "gradient_clip_val",**gradient_clipping_distribution
            ) if isinstance(gradient_clipping_distribution,dict) else trial.suggest_categorical(
                "gradient_clip_val",gradient_clipping_distribution
            )
            n_lys=trial.suggest_int(
                "encoder_and_decoder_layers",**layers_distribution
            ) if isinstance(layers_distribution,dict) else trial.suggest_categorical(
                "encoder_and_decoder_layers",layers_distribution
            )
            d_model_suggested=trial.suggest_int(
                "model_dimension",**latent_space_distribution
            ) if isinstance(latent_space_distribution,dict) else trial.suggest_categorical(
                "model_dimension",latent_space_distribution
            )
            n_heads_suggested=trial.suggest_int(
                "number_of_heads",**heads_distribution
            ) if isinstance(heads_distribution,dict) else trial.suggest_categorical(
                "number_of_heads",heads_distribution
            )
            d_model_suggested=round(d_model_suggested/n_heads_suggested)*n_heads_suggested
            configs=ETSf_hps(
                e_layers=n_lys,
                d_layers=n_lys,
                enc_in=train_dataloader.dataset._enc_in, 
                d_model=d_model_suggested,
                dropout=trial.suggest_float(
                    "dropout",**dropout_distribution
                ) if isinstance(dropout_distribution,dict) else trial.suggest_categorical(
                    "dropout",dropout_distribution
                ), 
                n_heads=n_heads_suggested,
                c_out=train_dataloader.dataset._enc_in, 
                label_len=0,
                seq_len=n_past,
                pred_len=n_future,
                K=trial.suggest_int(
                    "K",**K_distribution
                ) if isinstance(K_distribution,dict) else trial.suggest_categorical(
                    "K",K_distribution
                ), 
                d_ff=trial.suggest_int(
                    "feedforward_dimension",**ff_distribution
                ) if isinstance(ff_distribution,dict) else trial.suggest_categorical(
                    "feedforward_dimension",ff_distribution
                ),
                output_attention=False, 
            )
            model=ETSformer(
                configs
            )
            learning_rate_suggested=trial.suggest_float(
                "learning_rate",**nn_LR_distribution
            ) if isinstance(nn_LR_distribution,dict) else trial.suggest_categorical(
                "learning_rate",nn_LR_distribution
            )
            if smoothing_LR_distribution is None:
                smoothing_LR_suggested=100*learning_rate_suggested
            else:
                smoothing_LR_suggested=trial.suggest_float(
                    "smoothing_learning_rate",**smoothing_LR_distribution
                ) if isinstance(smoothing_LR_distribution,dict) else trial.suggest_categorical(
                    "smoothing_learning_rate",smoothing_LR_distribution
                )
            if damping_LR_distribution is None:
                damping_LR_suggested=100*learning_rate_suggested
            else:
                damping_LR_suggested=trial.suggest_float(
                    "damping_learning_rate",**damping_LR_distribution
                ) if isinstance(damping_LR_distribution,dict) else trial.suggest_categorical(
                    "damping_learning_rate",damping_LR_distribution
                )
            handle_optimizer=HandleOptimizer(
                model,
                learning_rate=learning_rate_suggested,
                min_lr=1e-30,
                lradj=trial.suggest_categorical("learning_rate_scheduler",scheduler_distribution),
                train_epochs=max_epochs,
                warmup_epochs=3,
                smoothing_learning_rate=smoothing_LR_suggested,
                damping_learning_rate=damping_LR_suggested,
            )
            if "reduce_on_plateau" in handle_optimizer._lradj:
                assert reduce_on_plateau_patience is not None,"When 'reduce_on_plateau' is in handle_optimizer._lradj, you must provide reduce_on_plateau_patience."            
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                def save_pytorch_model(model,path_to_pth_folder,pth_filename:str)->None:
                    torch.save(model.state_dict(),os.path.join(path_to_pth_folder,pth_filename+".pth"))
                _experiment_name=study_name
                path_to_pth_folder=os.path.join(
                    self._path_to_etsf_results,
                    "Optimization/{}/Checkpoints/trial_{}".format(_experiment_name,trial.number),
                )
                os.makedirs(path_to_pth_folder,exist_ok=True)
                summary_writer=SummaryWriter(log_dir="etsf_logs/optuna_{}/trial_{}".format(_experiment_name,trial.number))
                if accelerator=="auto":
                    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
                else:
                    device=torch.device(accelerator)
                model.to(device)
                global_step=0
                best_val_loss=-np.inf if trial.study.direction==optuna.study.StudyDirection.MAXIMIZE else np.inf
                best_epoch_num=0 
                for epoch_num in range(handle_optimizer._train_epochs):
                    model.train(mode=True)
                    train_epoch_loss=0
                    if create_tensorboard_logs:
                        list_param_groups=handle_optimizer._optimizer.param_groups
                        for param_group in list_param_groups:
                            summary_writer.add_scalar(
                                tag="Optimizer: {} LR".format(param_group["name"]),
                                scalar_value=param_group["lr"],
                                global_step=global_step
                            )
                    for X_batch,y_batch,X_batch_mark,y_batch_mark,_,_ in train_dataloader:
                        X_batch=X_batch.float().to(device)
                        y_batch=y_batch.float().to(device)
                        X_batch_mark=X_batch_mark.float().to(device)
                        y_batch_mark=y_batch_mark.float().to(device)
                        handle_optimizer._optimizer.zero_grad()
                        dec_inp = torch.zeros_like(y_batch[:, -model.configs.pred_len:, :]).float()
                        dec_inp = torch.cat([y_batch[:, :model.configs.label_len, :], dec_inp], dim=1).float().to(device)

                        outputs = model(X_batch, X_batch_mark, dec_inp, y_batch_mark)

                        f_dim = -1
                        y_pred=outputs[:, -model.configs.pred_len:, f_dim:].to(device)
                        y_batch = y_batch[:, -model.configs.pred_len:, f_dim:].to(device)
                        train_batch_loss=criterion(y_pred,y_batch)
                        if create_tensorboard_logs and (global_step+1)%step_log_interval==0:
                            summary_writer.add_scalar(
                                tag="Batch (step) loss: training",
                                scalar_value=train_batch_loss.item(),
                                global_step=global_step
                            )
                        train_epoch_loss+=train_batch_loss.item()
                        train_batch_loss.backward()
                        if gradient_clip_val is not None:
                            clip_grad_norm_(model.parameters(),max_norm=gradient_clip_val)
                        handle_optimizer._optimizer.step()
                        global_step+=1
                    train_epoch_loss/=len(train_dataloader)
                    if create_tensorboard_logs:
                        summary_writer.add_scalar(
                            tag="Epoch (average across batches) loss: training",
                            scalar_value=train_epoch_loss,
                            global_step=global_step
                        )
                    model.train(mode=False)
                    val_epoch_loss=0
                    with torch.no_grad():
                        for X_batch,y_batch,X_batch_mark,y_batch_mark,_,_ in val_dataloader:
                            X_batch=X_batch.float().to(device)
                            y_batch=y_batch.float().to(device)
                            X_batch_mark=X_batch_mark.float().to(device)
                            y_batch_mark=y_batch_mark.float().to(device)
                            dec_inp = torch.zeros_like(y_batch[:, -model.configs.pred_len:, :]).float()
                            dec_inp = torch.cat([y_batch[:, :model.configs.label_len, :], dec_inp], dim=1).float().to(device)
                            outputs = model(X_batch, X_batch_mark, dec_inp, y_batch_mark)
                            f_dim = -1
                            y_pred=outputs[:, -model.configs.pred_len:, f_dim:].to(device)
                            y_batch = y_batch[:, -model.configs.pred_len:, f_dim:].to(device)
                            val_batch_loss=criterion(y_pred,y_batch).item()
                            val_epoch_loss+=val_batch_loss
                    val_epoch_loss/=len(val_dataloader)
                    if create_tensorboard_logs:
                        summary_writer.add_scalar(
                            tag="Epoch (average across batches) loss: validation",
                            scalar_value=val_epoch_loss,
                            global_step=global_step
                        )
                    if early_stopping is not None:
                        if early_stopping.should_stop(val_epoch_loss):
                            trial.report(val_epoch_loss,epoch_num)
                            if (np.isnan(val_epoch_loss) or np.isinf(val_epoch_loss)):
                                print("Trial {} is going to be pruned at epoch {} end because of check_finite triggering...".format(trial.number,epoch_num),flush=True)                    
                                trial.set_user_attr("check_finite_triggered",True)
                                try:
                                    os.rename(
                                        src=os.path.join(path_to_pth_folder,"epoch.pth"),
                                        dst=os.path.join(path_to_pth_folder,"epoch={}.pth".format(best_epoch_num))
                                    )
                                except FileNotFoundError:
                                    save_pytorch_model(model,path_to_pth_folder,"epoch")
                                    save_variable(model.configs,os.path.join(path_to_pth_folder,"configs.pkl"))
                                    os.rename(
                                        src=os.path.join(path_to_pth_folder,"epoch.pth"),
                                        dst=os.path.join(path_to_pth_folder,"epoch={}.pth".format(best_epoch_num))
                                    )
                                if create_tensorboard_logs:
                                    hparams_to_log=model.configs._asdict()
                                    hparams_to_log.update(trial.params)
                                    summary_writer.add_hparams(
                                        hparam_dict=hparams_to_log,
                                        metric_dict={"best_val_loss":best_val_loss},
                                        run_name="hparams"
                                    )
                                    summary_writer.flush()
                                    summary_writer.close()
                                raise optuna.TrialPruned()
                            if trial.should_prune():
                                os.rename(
                                    src=os.path.join(path_to_pth_folder,"epoch.pth"),
                                    dst=os.path.join(path_to_pth_folder,"epoch={}.pth".format(best_epoch_num))
                                )
                                if create_tensorboard_logs:
                                    hparams_to_log=model.configs._asdict()
                                    hparams_to_log.update(trial.params)
                                    summary_writer.add_hparams(
                                        hparam_dict=hparams_to_log,
                                        metric_dict={"best_val_loss":best_val_loss},
                                        run_name="hparams"
                                    )
                                    summary_writer.flush()
                                    summary_writer.close()
                                print("Trial {} is going to be pruned at epoch {} end...".format(trial.number,epoch_num),flush=True)                    
                                raise optuna.TrialPruned()
                            else:
                                print("Early stopping triggered at epoch {} end. Trial judged as completed before reaching max_epochs={}.".format(epoch_num,max_epochs),flush=True)
                                trial.set_user_attr("EarlyStopped",True)
                                break
                        else:
                            if early_stopping._should_save:
                                best_val_loss=val_epoch_loss
                                best_epoch_num=epoch_num
                                save_pytorch_model(model,path_to_pth_folder,"epoch")
                                save_variable(model.configs,os.path.join(path_to_pth_folder,"configs.pkl"))
                    else:
                        if trial.study.direction==optuna.study.StudyDirection.MAXIMIZE and val_epoch_loss>best_val_loss:
                            best_val_loss=val_epoch_loss
                            best_epoch_num=epoch_num
                            save_pytorch_model(model,path_to_pth_folder,"epoch")
                            save_variable(model.configs,os.path.join(path_to_pth_folder,"configs.pkl"))
                        if trial.study.direction==optuna.study.StudyDirection.MINIMIZE and val_epoch_loss<best_val_loss:
                            best_val_loss=val_epoch_loss
                            best_epoch_num=epoch_num
                            save_pytorch_model(model,path_to_pth_folder,"epoch")
                            save_variable(model.configs,os.path.join(path_to_pth_folder,"configs.pkl"))
                    handle_optimizer.adjust_learning_rate(
                        epoch=epoch_num+1,
                        monitor=val_epoch_loss if "reduce_on_plateau" in handle_optimizer._lradj else None,
                        mode=trial.study.direction if "reduce_on_plateau" in handle_optimizer._lradj else None,
                        patience=reduce_on_plateau_patience if "reduce_on_plateau" in handle_optimizer._lradj else None
                    )
                    trial.report(val_epoch_loss,epoch_num)
                    if trial.should_prune():
                        os.rename(
                            src=os.path.join(path_to_pth_folder,"epoch.pth"),
                            dst=os.path.join(path_to_pth_folder,"epoch={}.pth".format(best_epoch_num))
                        )
                        if create_tensorboard_logs:
                            hparams_to_log=model.configs._asdict()
                            hparams_to_log.update(trial.params)
                            summary_writer.add_hparams(
                                hparam_dict=hparams_to_log,
                                metric_dict={"best_val_loss":best_val_loss},
                                run_name="hparams"
                            )
                            summary_writer.flush()
                            summary_writer.close()
                        print("Trial {} is going to be pruned at epoch {} end...".format(trial.number,epoch_num),flush=True)                    
                        raise optuna.TrialPruned()
                    if clear_cache_on_train_epoch_end:
                        import gc
                        torch.cuda.empty_cache()
                        gc.collect()
                os.rename(
                    src=os.path.join(path_to_pth_folder,"epoch.pth"),
                    dst=os.path.join(path_to_pth_folder,"epoch={}.pth".format(best_epoch_num))
                )
                if create_tensorboard_logs:
                    hparams_to_log=model.configs._asdict()
                    hparams_to_log.update(trial.params)
                    summary_writer.add_hparams(
                        hparam_dict=hparams_to_log,
                        metric_dict={"best_val_loss":best_val_loss},
                        run_name="hparams"
                    )
                    summary_writer.flush()
                    summary_writer.close()
                return best_val_loss
        try:
            if any(pd.Series(
                    os.listdir(
                        os.path.join(
                            self._path_to_etsf_results,
                            "Optimization/{}".format(study_name),
                        )
                    )
                ).isin(["{}.pkl".format(study_name)])
            ):
              study=load_variable(os.path.join(self._path_to_etsf_results,"Optimization/{}/{}.pkl".format(study_name,study_name)))
              print("{} correctly loaded and resumed from its end.\nThe first new trial is trial n°{} (enumerated from 0)".format(
                  os.path.join(self._path_to_etsf_results,"Optimization/{}/{}.pkl".format(study_name,study_name)),
                  len(study.trials)),flush=True
                )
            else:
                print("'{}' folder exists in '{}' but there are no '{}.pkl' to resume, a new study will be created...".format(
                    study_name,os.path.join(self._path_to_etsf_results,"Optimization"),study_name
                    ),flush=True
                )
                study=optuna.study.create_study(pruner=pruner,direction=direction)
        except FileNotFoundError as e:
            print(e,flush=True)
            print("Since no '{}' folder exists in '{}', a new study will be created...".format(
                study_name,os.path.join(self._path_to_etsf_results,"Optimization")
                ),flush=True
            )
            study=optuna.study.create_study(pruner=pruner,direction=direction)
        if n_trials>0:
            study.optimize(
                func=objective,
                n_trials=n_trials,
                timeout=timeout,
                gc_after_trial=True,
            )
            save_variable(
                study,
                os.path.join(
                    self._path_to_etsf_results,
                    "Optimization/{}/{}.pkl".format(study_name,study_name),
                )
            )
            print("Study named '{}' correctly saved at '{}'.".format(
                study_name,os.path.join(
                    self._path_to_etsf_results,
                    "Optimization/{}/{}.pkl".format(study_name,study_name),
                    )
                ),flush=True
            )
        else:
            print("n_trials<=0, thus no new trials are run. The study is returned as is.")
        return study
    def predict_inference_time_only(
        self,
        test_dataset: etsf_dataset,
        accelerator: Literal["cuda", "cpu", "auto"] = "auto",
    ) -> dict:

        assert "_loaded_model" in self.__dict__.keys(), "No loaded model found."
        
        test_dataloader = DataLoader(
            test_dataset,batch_size=1,shuffle=False,drop_last=False,num_workers=0
        ) 
        if accelerator == "auto":
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            device = torch.device(accelerator)

        self._loaded_model.to(device)
        self._loaded_model.train(mode=False)

        try:
            warmup_batch = next(iter(test_dataloader))
            assert warmup_batch[0].shape[0]==1,"The dataloader does not have a unitary batch size"
            w_X, w_y, w_X_mark, w_y_mark = warmup_batch[0], warmup_batch[1], warmup_batch[2], warmup_batch[3]
            
            w_X = w_X.float().to(device)
            w_y = w_y.float().to(device)
            w_X_mark = w_X_mark.float().to(device)
            w_y_mark = w_y_mark.float().to(device)
            
            w_dec_inp = torch.zeros_like(w_y[:, -self._loaded_model.configs.pred_len:, :]).float().to(device)
            w_dec_inp = torch.cat([w_y[:, :self._loaded_model.configs.label_len, :], w_dec_inp], dim=1).float().to(device)
            
            with torch.no_grad():
                for _ in range(5):
                    _ = self._loaded_model(w_X, w_X_mark, w_dec_inp, w_y_mark)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
        except Exception as e:
            print(f"Warm-up failed ({e}). Proceeding straight to benchmarking.")

        per_sequence_inference_times = []

        take=0
        with torch.no_grad():
            for btc_idx, (X_batch, y_batch, X_batch_mark, y_batch_mark, _, _) in enumerate(test_dataloader):
                if btc_idx==take: 
                    take+=24
                    X_batch = X_batch.float().to(device)
                    y_batch = y_batch.float().to(device)
                    X_batch_mark = X_batch_mark.float().to(device)
                    y_batch_mark = y_batch_mark.float().to(device)
                    
                    dec_inp = torch.zeros_like(y_batch[:, -self._loaded_model.configs.pred_len:, :]).float()
                    dec_inp = torch.cat([y_batch[:, :self._loaded_model.configs.label_len, :], dec_inp], dim=1).float().to(device)
                    
                    if device.type == 'cuda':
                        torch.cuda.synchronize()
                    start_time = perf_counter()
                    
                    _ = self._loaded_model(X_batch, X_batch_mark, dec_inp, y_batch_mark)
                    
                    if device.type == 'cuda':
                        torch.cuda.synchronize()
                    end_time = perf_counter()
                    batch_elapsed_time = (end_time - start_time) * 1000
                    
                    per_sequence_inference_times.append(batch_elapsed_time)

        avg_seq_time = np.mean(per_sequence_inference_times)
        std_seq_time = np.std(per_sequence_inference_times)
        
        print("\n" + "="*50)
        print(f"PER-SEQUENCE INFERENCE STATISTICS ({len(per_sequence_inference_times)} sequences):")
        print(f"Average Time per Sequence: {avg_seq_time:.2f} ms")
        print(f"Standard Deviation:        {std_seq_time:.2f} ms")
        print("="*50 + "\n")

## Evaluation

### DFs, DSs, DLs

In [ ]:
tbr=["group_ids"]
train_df_etsf=manipulate_features(train_df_tft,to_be_removed=tbr,return_id_manipulation=False)
val_df_etsf=manipulate_features(val_df_tft,to_be_removed=tbr,return_id_manipulation=False,print_info=False)
test_df_etsf=manipulate_features(df_test_x_tft,to_be_removed=tbr,return_id_manipulation=False,print_info=False)
test_Architecture_df_etsf=manipulate_features(df_Architecture_building_test_x_tft,to_be_removed=tbr,return_id_manipulation=False,print_info=False)
test_Economics_df_etsf=manipulate_features(df_Economics_building_test_x_tft,to_be_removed=tbr,return_id_manipulation=False,print_info=False)
test_Agriculture_extended_df_etsf=manipulate_features(df_test_x_tft_extended,to_be_removed=tbr,return_id_manipulation=False,print_info=False)
dummies_as_features=False
print("\nDUMMIES{} USED AS FEATURES,{} AS IN THE DEFAULT ETSF IMPLEMENTATION.".format(
    " NOT" if not dummies_as_features else ""," NOT" if dummies_as_features else ""
    )
)
dms=["Day_of_week","Month_of_year"]
train_ds_etsf=etsf_dataset(
    df_raw=train_df_etsf,
    target="target",
    dummies=dms,
    dummies_as_features=dummies_as_features,
)
val_ds_etsf=etsf_dataset(
    df_raw=val_df_etsf,
    target="target",
    dummies=dms,
    dummies_as_features=dummies_as_features,
)
test_ds_etsf=etsf_dataset(
    df_raw=test_df_etsf,
    target="target",
    dummies=dms,
    dummies_as_features=dummies_as_features,
)
test_Architecture_ds_etsf=etsf_dataset(
    df_raw=test_Architecture_df_etsf,
    target="target",
    dummies=dms,
    dummies_as_features=dummies_as_features,
)
test_Economics_ds_etsf=etsf_dataset(
    df_raw=test_Economics_df_etsf,
    target="target",
    dummies=dms,
    dummies_as_features=dummies_as_features,
)
test_Agriculture_extended_ds_etsf=etsf_dataset(
    df_raw=test_Agriculture_extended_df_etsf,
    target="target",
    dummies=dms,
    dummies_as_features=dummies_as_features,
)
train_loader = DataLoader(
    train_ds_etsf,batch_size=64,shuffle=True,drop_last=True,num_workers=0
)
val_loader = DataLoader(
    val_ds_etsf,batch_size=64,shuffle=False,drop_last=False,num_workers=0,
)
test_loader = DataLoader(
    test_ds_etsf,batch_size=64,shuffle=False,drop_last=False,num_workers=0
)
test_Architecture_loader = DataLoader(
    test_Architecture_ds_etsf,batch_size=64,shuffle=False,drop_last=False,num_workers=0
)
test_Economics_loader = DataLoader(
    test_Economics_ds_etsf,batch_size=64,shuffle=False,drop_last=False,num_workers=0
)
test_Agriculture_extended_loader = DataLoader(
    test_Agriculture_extended_ds_etsf,batch_size=64,shuffle=False,drop_last=False,num_workers=0
)

In [ ]:
def print_n_seqs_etsf_dls(des:str,df,dl):
    n=0
    for batched_seq_x, _, _, _, _, _ in dl:
        n+=batched_seq_x.shape[0]
    print(des)
    print("# samples in df: {}".format(df.shape[0]))
    print("# sequences in dl: {}".format(n))
    print("# non overlapping sequences in dl: {}".format(math.floor((n-1)/n_future)+1))
print_n_seqs_etsf_dls("----Training----",train_df_etsf,train_loader)
print_n_seqs_etsf_dls("----Validation----",val_df_etsf,val_loader)
print_n_seqs_etsf_dls("----Test----",test_df_etsf,test_loader)
print_n_seqs_etsf_dls("----Test extended----",test_Architecture_df_etsf,test_Architecture_loader)

In [ ]:
etsf_interpreter=etsfInterpreter(
    scaler_y,
    level_color="#3c4142",
    growth_color="#a0bf16",
    season_color="#c48efd",
    trend_color="#01889f"
)

### Tuning

In [ ]:
max_epochs_study=50
criterion_ets=MSELoss(reduction="mean")
study=etsf_interpreter.optimize_hyperparameters(
    study_name="HyperbandPruner_with_EarlyStopping",
    pruner=optuna.pruners.HyperbandPruner(
        max_resource=max_epochs_study,
        min_resource=2,
        reduction_factor=2,
        bootstrap_count=0
    ),
    n_trials=0, 
    max_epochs=max_epochs_study,
    early_stopping=CustomEarlyStopping(patience=8),
    criterion=criterion_ets,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    reduce_on_plateau_patience=4,
    timeout=11*3600
)

In [ ]:
building_name="Agriculture"
etsf_interpreter.load(
    building_name,
    source="optuna study",
    pick_best=True
)

In [ ]:
from torchinfo import summary
summary(
    model=etsf_interpreter._loaded_model,
    verbose=1,
)

In [ ]:
do_predictions=False
if do_predictions:
    etsf_interpreter.predict(
        building_name,
        starting_date=pd.Timestamp(year=2022,month=7,day=1,hour=0,minute=0),
        test_df=test_df_etsf,
        test_dataloader=test_loader,
        accelerator="cuda"
    )

In [ ]:
etsf_interpreter.predict_inference_time_only(
    test_dataset=test_ds_etsf,
)

In [ ]:
etsf_in_test_fig=etsf_interpreter.visualize_performance(
# abv_blw_etsf_dfs,scores,remainder=etsf_interpreter.visualize_performance(
    sklearn_metric_name="MAPE",
    upper_threshold=("quantile",0.975),
    lower_threshold=("quantile",0.025),
    thresholds_wrt_non_overlapping_in_test=True,
    # return_above_and_below_dfs=True,
    # return_scores=True,
    # return_remainder=True,
    return_only_figure=True,
    # test_period=pd.date_range(
    #     start=pd.Timestamp(year=2023,month=1,day=1,hour=0,minute=0),
    #     end=pd.Timestamp(year=2023,month=6,day=30,hour=23,minute=0),
    #     freq="1h"
    # ),
    xlabel="",
    fontsize=13,
    legend_only=True,
    figsize=(16,4.5),
    target_name="electric load",
    target_udm="kW",
    stairplot=True,
)

In [ ]:
val_losses_df=find_trial_in_study(etsf_interpreter._loaded_study,return_only_val_losses_df=True)
display(val_losses_df.sort_values(by="trial best validation loss"))

In [ ]:
get_hyperparameters_exploration(
    etsf_interpreter._loaded_study,
    hps_to_ignore=["gradient_clip_val","dropout","learning_rate_scheduler","smoothing_learning_rate","damping_learning_rate"],
    hps_in_log_scale=["learning_rate"],
    hps_in_discrete_scale=["encoder_and_decoder_layers","number_of_heads","K","model_dimension","feedforward_dimension"],
    hps_categorical=["model_dimension","feedforward_dimension"],
    to_shrink=["encoder_and_decoder_layers","number_of_heads","K"],
    custom_ticks=["encoder_and_decoder_layers","number_of_heads","K","model_dimension","feedforward_dimension"],
    n_custom_ticks=[4,10,3,5,5],
    figsize=(16,9),
    sns_histplot_kwargs=dict(multiple="stack"),
    markerfacecolor="lime",
    markeredgecolor="black",
    fontsize=24,
    metric_name="MAPE",
    markersize=15,
    marker="o",
    dpi=600,
    to_approx={"learning_rate":4}
)

### Training significance

In [ ]:
configs=ETSf_hps(
    e_layers=4, 
    d_layers=4,
    enc_in=train_ds_etsf._enc_in, 
    d_model=252, 
    dropout=0.2, 
    n_heads=9,
    c_out=train_ds_etsf._enc_in, 
    label_len=0,  
    seq_len=n_past, 
    pred_len=n_future, 
    K=3, 
    d_ff=64, 
    output_attention=False, 
)
ETSf=ETSformer(
    configs
)
num_linear_warmup_epochs=3
max_epochs_etsf=50 
criterion_ets=MSELoss(reduction="mean") 
reduce_on_plateau_patience_etsf=4 
handle_optimizer=HandleOptimizer(
    model=ETSf,
    learning_rate=0.0026379446077118797,
    min_lr=1e-30,
    lradj="cos_with_warmup",
    train_epochs=max_epochs_etsf,
    warmup_epochs=num_linear_warmup_epochs,
)
etsf_interpreter=etsfInterpreter(
    scaler_y,
    level_color="#3c4142",
    growth_color="#a0bf16",
    season_color="#c48efd",
    trend_color="#01889f"
)
do_train=True
if do_train:
    etsf_interpreter.train(
        ETSf,
        handle_optimizer,
        criterion_ets,
        train_loader,
        val_loader,
        accelerator="auto",
        gradient_clip_val=1.0,
        clear_cache_on_train_epoch_end=False, 
        early_stopping=None,
        mode="min",
        reduce_on_plateau_patience=reduce_on_plateau_patience_etsf,
        save_last=True,
        create_tensorboard_logs=False,
        experiment_name="Statistical_significance/{}".format(datetime.datetime.now().strftime("%d_%m_%Y_%H_%M_%S")),
        timeout=8*3600
    )

In [ ]:
building_name="Agriculture"
trainings_available=[
  item for item in os.listdir(os.path.join(Path("./Results/ETSf"),"Statistical_significance"))
]
print(trainings_available,flush=True)
index_training=int(input("Type the index corresponding to the training you want to load:"))
training_name=trainings_available[index_training]
for item in os.listdir(os.path.join(r"./Results/ETSf/Statistical_significance",training_name)):
    if "best_epoch" in item:
        fln=item
        break
etsf_interpreter.load(
    building_name,
    source="path",
    pth_filepath=os.path.join(r"./Results/ETSf/Statistical_significance",training_name,fln)
)

In [ ]:
if "batch_times.pkl" in os.listdir(os.path.join(r"./Results/ETSf/Statistical_significance",training_name)):
    batch_times=pd.Series(load_variable(os.path.join(r"./Results/ETSf/Statistical_significance",training_name,"batch_times.pkl")),name="Elapsed times per training batch")
    display(batch_times.describe())
    batch_per_training_epoch=(len(batch_times)//50)
    epoch_times=[]
    for i in range(50):
        epoch_times.append(
        batch_times.iloc[
            i*batch_per_training_epoch:(i+1)*batch_per_training_epoch
            ].sum()
        )
    epoch_times=pd.Series(epoch_times[1:],name="Elapsed times per training epoch [s]") 
    display(epoch_times.describe())
    print("Total training time: {} s".format(batch_times.sum()))

In [ ]:
from tensorboard.backend.event_processing import event_accumulator
for nitem,item in enumerate(os.listdir("./Results/ETSf/Statistical_significance")):
    ea=event_accumulator.EventAccumulator("./etsf_logs/Statistical_significance/{}".format(item))
    ea.Reload()
    loss=pd.DataFrame(ea.Scalars("Epoch (average across batches) loss: training"))["value"]
    val_loss=pd.DataFrame(ea.Scalars("Epoch (average across batches) loss: validation"))["value"]
    if not nitem:
        stat_loss=np.array(loss).reshape(1,-1)
        stat_val_loss=np.array(val_loss).reshape(1,-1)
    else:
        stat_loss=np.concat((stat_loss,np.array(loss).reshape(1,-1)),axis=0)
        stat_val_loss=np.concat((stat_val_loss,np.array(val_loss).reshape(1,-1)),axis=0)
display(stat_loss)
stat_final_loss={
    "min":stat_loss[:,-1].min().item(),
    "max":stat_loss[:,-1].max().item(),
    "mean":stat_loss[:,-1].mean().item(),
    "std":stat_loss[:,-1].std().item(),
}
display(stat_final_loss)
ave_loss=stat_loss.mean(axis=0)
ave_val_loss=stat_val_loss.mean(axis=0)
fig=plt.figure(
  figsize=(5,4),
  dpi=300
)
with plt.rc_context({"font.size":12}): 
  plt.plot(np.arange(len(ave_loss)),ave_loss,linewidth=1.5,label="Training")
  plt.plot(np.arange(len(ave_val_loss)),ave_val_loss,linewidth=1.5,label="Validation")
  plt.fill_between(
    np.arange(len(ave_loss)),
    ave_loss-stat_loss.std(axis=0),
    ave_loss+stat_loss.std(axis=0),
    label="",
    fc="tab:blue",
    step=None,
    alpha=0.3
    )
  plt.fill_between(
    np.arange(len(ave_val_loss)),
    ave_val_loss-stat_val_loss.std(axis=0),
    ave_val_loss+stat_val_loss.std(axis=0),
    label="",
    fc="tab:orange",
    step=None,
    alpha=0.3
    )
  plt.ylabel("Average epoch MSE loss [-]")
  plt.xlabel('Epoch number [-]')
  plt.xlim(0,50)
  plt.legend(loc="upper right")
  plt.grid()
  plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True,nbins=10))
  plt.close("all")
  display(fig)

# LSTM Pytorch (computational efficiency, GPU)

In [ ]:
class PyTorchLSTM(torch.nn.Module):
    def __init__(self, n_features=9, units=[64, 32, 128], dropout=0.12, n_future=24):
        super().__init__()
        
        self.lstm1 = torch.nn.LSTM(input_size=n_features, hidden_size=units[0], batch_first=True)
        self.dropout1 = torch.nn.Dropout(dropout)
        
        self.lstm2 = torch.nn.LSTM(input_size=units[0], hidden_size=units[1], batch_first=True)
        self.dropout2 = torch.nn.Dropout(dropout)
        
        self.dense1 = torch.nn.Linear(units[1], units[2])
        self.dropout3 = torch.nn.Dropout(dropout)
        
        self.dense2 = torch.nn.Linear(units[2], n_future)

    def forward(self, x):
        out1, _ = self.lstm1(x)
        out1 = self.dropout1(out1)
        
        out2, _ = self.lstm2(out1)
        out2_last_step = out2[:, -1, :] 
        out2_last_step = self.dropout2(out2_last_step)
        
        out3 = self.dense1(out2_last_step)
        out3 = self.dropout3(out3)
        
        predictions = self.dense2(out3)
        return predictions

In [ ]:
def train_pytorch_lstm_for_benchmarking(
    X_train: np.ndarray,
    y_train: np.ndarray,
    validation_split: float = 0.2, 
    epochs: int = 200,
    lr: float = 0.0021,
    batch_size:int = 16,
    gradient_clip_val: float = 1.0,
    device_str: str = "cuda" if torch.cuda.is_available() else "cpu",
    path_to_lstm_results: str | Path = Path("./Results/LSTM"),
):
    split_idx = int(len(X_train) * (1 - validation_split))
    
    X_tr, y_tr = X_train[:split_idx], y_train[:split_idx]
    X_va, y_va = X_train[split_idx:], y_train[split_idx:]

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    X_va_t = torch.tensor(X_va, dtype=torch.float32)
    y_va_t = torch.tensor(y_va, dtype=torch.float32)

    from torch.utils.data import TensorDataset, DataLoader
    train_dataset = TensorDataset(X_tr_t, y_tr_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    val_dataset = TensorDataset(X_va_t, y_va_t)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = PyTorchLSTM()
    device = torch.device(device_str)
    model.to(device)
    
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    exp_name = "Experiment {}".format(datetime.datetime.now().strftime("%d_%m_%Y_%H_%M_%S"))
    ckpts_path = os.path.join(path_to_lstm_results, "Pytorch LSTM", exp_name)
    os.makedirs(ckpts_path, exist_ok=True)

    batch_times = []
    on_gpu = device.type == "cuda"
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            
            X_batch = X_batch.float().to(device)
            y_batch = y_batch.float().to(device)
            
            optimizer.zero_grad()
            
            if on_gpu:
                torch.cuda.synchronize(device)
            batch_tic = perf_counter()
            
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            
            if gradient_clip_val is not None:
                torch.torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_val)
                
            optimizer.step()
            
            if on_gpu:
                torch.cuda.synchronize(device)
            batch_times.append(perf_counter() - batch_tic)
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        model.eval()
        val_loss_epoch = 0.0
        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                X_val_batch = X_val_batch.float().to(device)
                y_val_batch = y_val_batch.float().to(device)
                
                val_preds = model(X_val_batch)
                val_loss_batch = criterion(val_preds, y_val_batch)
                val_loss_epoch += val_loss_batch.item()
                
        val_loss_epoch /= len(val_loader)
        
        if val_loss_epoch < best_val_loss:
            best_val_loss = val_loss_epoch
            best_model_path = os.path.join(ckpts_path, "best_model.pth")
            torch.save(model.state_dict(), best_model_path)
            
    save_variable(
        batch_times,
        os.path.join(ckpts_path, "batch_times.pkl")
    )
    return model,ckpts_path

In [ ]:
def benchmark_pytorch_lstm_inference(
    model: torch.nn.Module, 
    X_test: np.ndarray, 
    y_test: np.ndarray,
    device_str: str = "auto"
):
    if device_str == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device_str)
        
    model.to(device)
    model.eval() 
    
    X_ts_t = torch.tensor(X_test, dtype=torch.float32)
    y_ts_t = torch.tensor(y_test, dtype=torch.float32)

    from torch.utils.data import TensorDataset, DataLoader

    test_dataset = TensorDataset(X_ts_t, y_ts_t)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    print("Warming up PyTorch LSTM...")
    try:
        warmup_batch = next(iter(test_loader))
        w_X = warmup_batch[0].float().to(device)
        
        assert w_X.dim() == 3, f"Errore: atteso tensore 3D, ricevuto {w_X.dim()}D"
        assert w_X.shape[0] == 1, f"Errore: atteso batch_size=1, ricevuto {w_X.shape[0]}"
            
        with torch.no_grad():
            for _ in range(5):
                _ = model(w_X)
            if device.type == 'cuda':
                torch.cuda.synchronize()
        print("Warm-up complete. Starting individual sequence benchmarking.")
    except Exception as e:
        print(f"Warm-up failed: {e}. Proceeding straight to benchmarking.")


    per_sequence_inference_times = []
    raw_predictions = []
    
    with torch.no_grad():
        for batch in test_loader:
            X_batch = batch[0].float().to(device)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = perf_counter()
            
            out = model(X_batch)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            end_time = perf_counter()
            
            elapsed_time = (end_time - start_time) * 1000 
            per_sequence_inference_times.append(elapsed_time)
            
            raw_predictions.append(out.cpu().numpy())

    preds_da_per_seq_inf = np.vstack(raw_predictions)


    avg_seq_time = np.mean(per_sequence_inference_times)
    std_seq_time = np.std(per_sequence_inference_times)

    print("\n" + "="*50)
    print(f"PYTORCH LSTM PER-SEQUENCE INFERENCE STATISTICS ({len(per_sequence_inference_times)} sequences processed):")
    print(f"Average Time per Sequence: {avg_seq_time:.2f} ms")
    print(f"Standard Deviation:        {std_seq_time:.2f} ms")
    print("="*50 + "\n")
    
    return per_sequence_inference_times, preds_da_per_seq_inf

In [ ]:
do_train=False
if do_train:
    lstm_pytorch,ckpts_path=train_pytorch_lstm_for_benchmarking(
        X_train=X_train_all_seqs,
        y_train=y_train_all_seqs,
    )
else:
    lstm_pytorch = PyTorchLSTM()
    ckpts_path=r"./Results/LSTM/Pytorch LSTM/Experiment 05_06_2026_14_18_44/best_model.pth" 
    lstm_pytorch.load_state_dict(torch.load(ckpts_path, map_location="cpu"))
batch_times=pd.Series(load_variable(os.path.join(ckpts_path,"batch_times.pkl")),name="Elapsed times per training batch")
display(batch_times.describe())
batch_per_training_epoch=(len(batch_times)//200)
epoch_times=[]
for i in range(200):
    epoch_times.append(
    batch_times.iloc[
        i*batch_per_training_epoch:(i+1)*batch_per_training_epoch
        ].sum()
    )
epoch_times=pd.Series(epoch_times[1:],name="Elapsed times per training epoch")
display(epoch_times.describe())
print("Total training time: {} s".format(batch_times.sum()))

In [ ]:
_,_=benchmark_pytorch_lstm_inference(
    model=lstm_pytorch,
    X_test=X_test,
    y_test=y_test,
)

# SARIMAX

## Process

In [ ]:
class Process():
    def __init__(
            self,ts:pd.Series,
            building:str|None="Agriculture",
            ts_name:str|None="electricity consumption",
            udm:str|None="kWh",
            freq:str="1h",
            reference_process:Union["Process",None]=None,
            dpi:int|None=None
            ):
        assert ts.index.is_monotonic_increasing and (ts.index.diff(1).dropna()==pd.Timedelta(freq)).all(),"Time series' index must be monotonically increasing and without gaps."
        self._building=building
        self._ts_name=ts_name
        self._udm=udm
        self._freq=freq
        self._ts=copy.deepcopy(ts)
        self._lmbda=None
        self._reference_process=copy.deepcopy(reference_process)
        self._transf_log=[]
        self._dpi=dpi
    def rolling(
            self,stat:Literal["mean","mean and deviation","variance"],days:int=365,detrend:bool=True,
            xlims:tuple[pd.Timestamp|None]|None=None, fontsize:float=12,
            custom_ylabel:dict[Literal["building","ts_name","udm"],str]|None=None,
            stairplot:bool=False,
            ):
        with plt.rc_context({"font.size":fontsize}):
            figlist=[]
            fig=plt.figure(figsize=(16,9),dpi=self._dpi)
            ax=fig.gca()
            if "mean" in stat:
                rolling=self._ts.rolling(
                    window=days*24, 
                    center=True,
                    closed="right",
                    min_periods=round(days*24/2)
                ).mean()
                if not stairplot:
                    ax.plot(self._ts.index,self._ts,label="Observed")
                    ax.plot(rolling.index,rolling,label="Rolling mean over {} days".format(days))
                else:
                    ax.step(self._ts.index,self._ts,label="Observed",where="post")
                    ax.step(rolling.index,rolling,label="Rolling mean over {} days".format(days),where="post")
                if custom_ylabel is None:
                    ax.set_ylabel("{} building: {} ({})".format(self._building,self._ts_name,self._udm))
                else:
                    ax.set_ylabel("{} building: {} ({})".format(custom_ylabel["building"],custom_ylabel["ts_name"],custom_ylabel["udm"]))
                ncols=2
                if "deviation" in stat:
                    ncols+=1
                    rolling_std=self._ts.rolling(
                        window=days*24, 
                        center=True,
                        closed="right",
                        min_periods=round(days*24/2)
                    ).std()
                    if not stairplot:
                        ax.plot(rolling_std.index,rolling_std,color="tab:red",label="Rolling deviation over {} days".format(days))
                    else:
                        ax.step(rolling_std.index,rolling_std,color="tab:red",label="Rolling deviation over {} days".format(days),where="post")
                if xlims is None:
                    ax.set_xlim((min(self._ts.index),max(self._ts.index)))
                else:
                    ax.set_xlim(left=xlims[0],right=xlims[1])
                ax.legend(loc="lower center",ncols=ncols,bbox_to_anchor=(0.5,1))
                figlist.append(fig)
                if detrend:
                    self._detrended_tg=self._ts-rolling
                    fig=plt.figure(figsize=(16,9),dpi=self._dpi)
                    ax=fig.gca()
                    if not stairplot:
                        ax.plot(self._detrended_tg.index,self._detrended_tg)
                    else:
                        ax.step(self._detrended_tg.index,self._detrended_tg,where="post")
                    if custom_ylabel is None:
                        ax.set_ylabel("{} building: detrended {} ({})".format(self._building,self._ts_name,self._udm))
                    else:
                        ax.set_ylabel("{} building: detrended {} ({})".format(custom_ylabel["building"],custom_ylabel["ts_name"],custom_ylabel["udm"]))
                    if xlims is None:
                        ax.set_xlim((min(self._detrended_tg.index),max(self._detrended_tg.index)))
                    else:
                        ax.set_xlim(left=xlims[0],right=xlims[1])
                    figlist.append(fig)
            else:
                rolling=self._ts.rolling(
                    window=days*24, 
                    center=True,
                    closed="right",
                    min_periods=round(days*24/2)
                ).std()
                if not stairplot:
                    ax.plot(self._ts.index,self._ts,color="tab:blue")
                else:
                    ax.step(self._ts.index,self._ts,color="tab:blue",where="post")
                if custom_ylabel is None:
                    ax.set_ylabel("{} building: {} ({})".format(self._building,self._ts_name,self._udm),color="tab:blue")
                else:
                    ax.set_ylabel("{} building: {} ({})".format(custom_ylabel["building"],custom_ylabel["ts_name"],custom_ylabel["udm"]))
                ax.tick_params('y',color="tab:blue",labelcolor="tab:blue")
                if xlims is None:
                    ax.set_xlim((min(self._ts.index),max(self._ts.index)))
                else:
                    ax.set_xlim(left=xlims[0],right=xlims[1])
                ax_var=ax.twinx()
                ax_var.plot(rolling.index,rolling,color="tab:orange")
                ax_var.set_ylabel("Rolling variance over {} days (kWh".format(days)+r"$^2$"+")",color="tab:orange")
                ax_var.tick_params('y',color="tab:orange",labelcolor="tab:orange")
                figlist.append(fig)
            plt.close("all")
            for fig in figlist:
                display(fig)
    def periodogram(self,detrend:Literal["constant","linear","_detrended_tg"],where:Literal["pre","post","mid"]="mid",fontsize:float=12):
        from scipy.signal import periodogram
        if detrend=="_detrended_tg":
            assert "_detrended_tg" in self.__dict__.keys(),"Run self.rolling(stat='mean',...) to save self._detrended_tg."
        fs=24*365
        fqs,spc=periodogram(
            self._detrended_tg if detrend=="_detrended_tg" else self._ts,
            fs=fs,
            detrend=False if detrend=="_detrended_tg" else detrend,
            window="boxcar",
            scaling="spectrum"
        )
        with plt.rc_context({"font.size":fontsize}):
            fig=plt.figure(figsize=(16,9),dpi=self._dpi)
            ax=fig.gca()
            ax.step(fqs,spc,where=where)
            ax.set_xscale("log")
            ax.set_xticks(
                ticks=[1,2,4,6,12,24,52,104,365],
                labels=[
                    "Yearly (f=1)",
                    "Semiannual (f=2)",
                    "Quarterly (f=4)",
                    "Bimonthly (f=6)",
                    "Monthly (f=12)",
                    "Biweekly (f=24)",
                    "Weekly (f=52)",
                    "Semiweekly (f=104)",
                    "Daily (f=365)"
                ],
                rotation=90
            )
            ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
            ax.set_ylabel("Squared magnitude spectrum (kWh"+r"$^2$"+")")
            ax.set_xlabel("Frequency (cycles/year)")
            ax.set_title("Periodogram")
            plt.close("all")
            display(fig)
    def normalize(self,shift:float=10,revert:bool=False):
        from scipy.stats import boxcox
        if not revert:
            assert "normalize" not in self._transf_log,"Processes are not meant to be normalized more than once."
            if self._reference_process is None:
                self._shift=shift
                self._ts+=self._shift 
                output,self._lmbda=boxcox(self._ts)
            else:
                assert "normalize" in self._reference_process._transf_log,"The reference process was not normalized."
                self._shift=self._reference_process._shift
                print("Shift specification is ignored as self._shift is recovered from the reference process.")
                self._ts+=self._shift 
                self._lmbda=self._reference_process._lmbda
                print("self._lmbda is recovered from the reference process.")
                output=boxcox(self._ts,lmbda=self._lmbda)
            output=pd.Series(output,index=self._ts.index)
            self._ts=output
            self._transf_log.append("normalize")
        else:
            assert "_lmbda" in self.__dict__.keys(),"Process is not normalized."
            if self._lmbda==0:
                output=np.exp(self._ts)-self._shift
            else:
                output=(self._lmbda*self._ts+1)**(1/self._lmbda)-self._shift
            self._ts=output
            del self._lmbda
            del self._shift
    def scale(self,feature_range:tuple[float]=(0,1),revert:bool=False):
        from sklearn.preprocessing import MinMaxScaler
        if not revert:
            assert "scale" not in self._transf_log,"Processes are not meant to be scaled more than once."
            if self._reference_process is None:
                self._scaler=MinMaxScaler(feature_range=feature_range)
                self._ts=pd.Series(
                    self._scaler.fit_transform(
                        np.array(copy.deepcopy(self._ts)).reshape(-1,1)
                    ).reshape(-1),
                    index=self._ts.index,
                )
            else:
                assert "scale" in self._reference_process._transf_log,"The reference process was not scaled."
                self._scaler=self._reference_process._scaler
                print("Feature range specification is ignored as self._scaler is recovered from the reference process.")
                self._ts=pd.Series(
                    self._scaler.transform(
                        np.array(copy.deepcopy(self._ts)).reshape(-1,1)
                    ).reshape(-1),
                    index=self._ts.index,
                )
            self._transf_log.append("scale")
        else:
            assert "_scaler" in self.__dict__.keys(),"Process is not scaled."
            self._ts=pd.Series(
                self._scaler.inverse_transform(
                    np.array(copy.deepcopy(self._ts)).reshape(-1,1)
                ).reshape(-1),
                index=self._ts.index
            )
            del self._scaler
    def differenciate(self,periods:int|list[int]|np.ndarray[int]=[1]):
        assert self._ts.index.is_monotonic_increasing,"Index of the process is not monotonically increasing, sort it ascendingly."
        assert not bool((self._ts.index.diff(1).dropna()>pd.Timedelta(self._freq)).sum()),"There are gaps in the Index of the process, fill them."
        if isinstance(periods,int):
            periods=[periods]
        if self._reference_process is not None:
            assert "differenciate" in self._reference_process._transf_log,"The reference process was not differenciated."
            periods=[]
            for diff_dict in self._reference_process._diff_log.values():
                periods.append(diff_dict["period"])
            print("Periods specification is ignored as the periods are recovered from the reference process.")
        for period in periods:
            if "_diff_log" not in self.__dict__.keys():
                self._diff_log={0:{"pre":self._ts,"period":period}}
            else:
                self._diff_log.update({len(self._diff_log.keys()):{"pre":self._ts,"period":period}})
            self._ts=self._ts.diff(periods=period).dropna()
        self._transf_log.append("differenciate")
    def integrate(self):
        assert "_diff_log" in self.__dict__.keys(),"The process is not differenciated."
        ids_diff=list(self._diff_log.keys())
        ids_diff.reverse()
        for id_diff in ids_diff:
            self._ts=pd.concat(
                [
                    self._diff_log[id_diff]["pre"].iloc[
                        :self._diff_log[id_diff]["period"]
                    ],
                    self._diff_log[id_diff]["pre"].shift(
                        self._diff_log[id_diff]["period"]
                    ).dropna()+self._ts
                ]
            )
            del self._diff_log[id_diff]
        del self._diff_log
    def back_to_original(self):
        transf_list=copy.deepcopy(self._transf_log)
        transf_count=pd.Series(transf_list).value_counts()
        if "normalize" in transf_count.index:
            assert transf_count.loc["normalize"]==1,"The process was normalized more than once."
        if "scale" in transf_count.index:
            assert transf_count.loc["scale"]==1,"The process was scaled more than once."
        transf_list.reverse()
        for transf_name in transf_list:
            if transf_name=="scale":
                self.scale(revert=True)
            elif transf_name=="normalize":
                self.normalize(revert=True)
            else:
                self.integrate()
        self._transf_log.clear()
    def transform_as_reference_process(self):
        assert self._reference_process is not None,"This method is meant to transform a process as its reference process,\nbut reference_process was not given at self.__init__() time."
        assert len(self._transf_log)==0,"The process must not have already undergone any transformation prior to this method."
        reference_transf_list=copy.deepcopy(self._reference_process._transf_log)
        for reference_transf in reference_transf_list:
            if reference_transf=="normalize":
                self.normalize()
            elif reference_transf=="scale":
                self.scale()
            elif reference_transf=="differenciate":
                self.differenciate()
        if len(self._transf_log)==0:
            print("The reference process was not transformed. Thus, the current process remained unaltered.")
    def evaluate_stationarity(
            self,kpss_regression:Literal["c","ct"]="c",
            adf_regression:Literal["c","ct","ctt","n"]="c",
            significance_level:float=0.05,
            nlags:int=7*24,
            ):
        from statsmodels.tsa.stattools import kpss,adfuller
        import warnings
        wrn_list=[]
        with warnings.catch_warnings(record=True) as wrn_iterable:
            print("----KPSS test----")
            print("Null hypothesis: the process is stationary around a {}.".format(
                "constant (i.e. is level-stationary)" if kpss_regression=="c" else "trend (i.e. is trend-stationary)"
                )
            )
            print("Alternative hypotesis: the process has a unit root (non-stationary).")
            kpss_res=kpss(self._ts,regression=kpss_regression,nlags=nlags)
            kpss_dict={
                "Test statistic":kpss_res[0],
                "p-value":kpss_res[1],
                "Number of lags considered":kpss_res[2],
            }
            kpss_dict.update({"Critical value ({})".format(sl):cv for sl,cv in kpss_res[3].items()})
            print(pd.Series(kpss_dict.values(),index=kpss_dict.keys()))
            print("\n")
            if kpss_dict["p-value"]<=significance_level:
                print("Null hypothesis can be rejected. The process is non-stationary.")
            else:
                print("Null hypothesis can't be rejected. The process is {}-stationary.".format(
                    "level" if kpss_regression=="c" else "trend"
                    )
                )
            wrn_list.extend(wrn_iterable)
            if len(wrn_list)>0:
                print("\nWarnings occurred during the test:")
            for wrn in wrn_list:
                print("{}:{}".format(wrn.category.__name__,wrn.message))
        wrn_list.clear() 
        with warnings.catch_warnings(record=True) as wrn_iterable:
            print("\n----ADF test----")
            print("Null hypothesis: the process has a unit root (non-stationary).")
            print("Alternative hypotesis: the process does not have a unit root.")
            adf_reg_const={
                "c":"constant only (default).",
                "ct":"constant and trend.",
                "ctt":"constant, and linear and quadratic trend.",
                "n":"no constant, no trend."
            }
            print("Regression with {}".format(adf_reg_const[adf_regression]))
            adf_res=adfuller(self._ts,regression=adf_regression,maxlag=nlags)
            adf_dict={
                "Test statistic":adf_res[0],
                "p-value":adf_res[1],
                "Number of lags considered":adf_res[2],
                "Number of observations considered":adf_res[3],
            }
            adf_dict.update({"Critical value ({})".format(sl):cv for sl,cv in adf_res[4].items()})
            print(pd.Series(adf_dict.values(),index=adf_dict.keys()))
            print("\n")
            if adf_dict["p-value"]<=significance_level:
                print("Null hypothesis can be rejected. The process does not have a unit root.")
            else:
                print("Null hypothesis can't be rejected. The process has a unit root.")
            wrn_list.extend(wrn_iterable)
            if len(wrn_list)>0:
                print("\nWarnings occurred during the test:")
            for wrn in wrn_list:
                print("{}:{}".format(wrn.category.__name__,wrn.message))
        print("\n----In conclusion----")
        if kpss_dict["p-value"]<=significance_level and adf_dict["p-value"]<=significance_level:
            print("Conflicting results. The process might lay in between stationarity and non-stationarity.")
        elif kpss_dict["p-value"]>significance_level and adf_dict["p-value"]<=significance_level:
            print("The process is likely to be {}-stationary.".format(
                "level" if kpss_regression=="c" else "trend"
                )
            )
        elif kpss_dict["p-value"]<=significance_level and adf_dict["p-value"]>significance_level:
            print("The process is likely to be difference-stationary.")
        elif kpss_dict["p-value"]>significance_level and adf_dict["p-value"]>significance_level:
            print("Conflicting results. The process might lay in between stationarity and non-stationarity.")
    def acf_and_pacf(
            self,nlags:int=7*24,include_zero_lag:bool=True,
            significance_level:float=0.05,
            ylims:dict[Literal["acf","pacf"],tuple[float]|None]={"acf":None,"pacf":None},
            one_every_nlags:int|None=None,
            fontsize:float=11
        ):
        from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
        with plt.rc_context({"font.size":fontsize}):
            fig,axs=plt.subplots(nrows=2,ncols=1,squeeze=True,figsize=(16,9),dpi=self._dpi)
            plot_acf(
                self._ts,ax=axs[0],lags=nlags,zero=include_zero_lag,
                alpha=significance_level,
                title="{}: Autocorrelation{}. Significance level={:.1f}%".format(
                    self._ts_name.capitalize()," (lag zero {}cluded)".format("ex" if not include_zero_lag else "in"),
                    significance_level*100
                )
            )
            if one_every_nlags is None:
                one_every_nlags=24 
            _,right_lim=axs[0].get_xlim()
            nticks=math.floor(right_lim/one_every_nlags)
            ticks=[int(not include_zero_lag)]
            ticks.extend([i*one_every_nlags for i in range(1,nticks+1)])
            axs[0].set_xticks(
                ticks=ticks,
                labels=[str(tick) for tick in ticks]
            )
            if ylims["acf"] is not None:
                axs[0].set_ylim(ylims["acf"])
            plot_pacf(
                self._ts,ax=axs[1],lags=nlags,zero=include_zero_lag,
                alpha=significance_level,
                title="{}: Partial Autocorrelation{}. Significance level={:.1f}%".format(
                    self._ts_name.capitalize()," (lag zero {}cluded)".format("ex" if not include_zero_lag else "in"),
                    significance_level*100
                )
            )
            _,right_lim=axs[1].get_xlim()
            nticks=math.floor(right_lim/one_every_nlags)
            ticks=[int(not include_zero_lag)]
            ticks.extend([i*one_every_nlags for i in range(1,nticks+1)])
            axs[1].set_xticks(
                ticks=ticks,
                labels=[str(tick) for tick in ticks]
            )
            if ylims["pacf"] is not None:
                axs[1].set_ylim(ylims["pacf"])
            for ax in axs:
                ax.set_xlabel("Lag")
            fig.tight_layout()
            plt.close("all")
            display(fig)
    def transfer_state_to(self,other:"Process"):
        assert all(
            [
                self._building==other._building,
                self._freq==other._freq,
            ],
        ),"Building and frequency of the time series must be the same."
        if "_lmbda" in self.__dict__.keys():
            other._lmbda=copy.deepcopy(self._lmbda)
            print("self._lmbda succesfully transfered.")
        if "_shift" in self.__dict__.keys():
            other._shift=copy.deepcopy(self._shift)
            print("self._shift succesfully transfered.")
        if "_scaler" in self.__dict__.keys():
            other._scaler=copy.deepcopy(self._scaler)
            print("self._scaler succesfully transfered.")
        if "_transf_log" in self.__dict__.keys():
            other._transf_log=copy.deepcopy(self._transf_log)
            print("self._transf_log succesfully transfered.")
        if "_diff_log" in self.__dict__.keys():
            other._diff_log=copy.deepcopy(self._diff_log)
            print("self._diff_log succesfully transfered.")

## Preprocessing

In [ ]:
edg_train=Process(
    ts=df_train.set_index(datetime_series[train_index])["Agriculture_building_load"],dpi=600,
    ts_name="electric load",
    udm="kW"
)

In [ ]:
# edg_train.rolling(
#     stat="mean",
#     detrend=False,
#     stairplot=True,
# )

In [ ]:
edg_train.normalize()

In [ ]:
# edg_train.periodogram(
#     detrend="_detrended_tg",
#     where="post"
# )

In [ ]:
# edg_train.evaluate_stationarity(kpss_regression="c",adf_regression="c")

In [ ]:
# edg_train.acf_and_pacf(
#     include_zero_lag=False,
#     nlags=24*7,
#     one_every_nlags=6,
# )

In [ ]:
# edg_train.differenciate(
#     periods=24,
# )

In [ ]:
edg_train.scale(feature_range=(1,2))

In [ ]:
exg_dow_train=Process(
    ts=df_train.set_index(datetime_series[train_index])["Day_of_week"],
    ts_name="day of the week indicator",
    udm="-"
)
exg_dow_train.scale(feature_range=(1,2))
exg_moy_train=Process(
    ts=df_train.set_index(datetime_series[train_index])["Month_of_year"],
    ts_name="month of the year indicator",
    udm="-"
)
exg_moy_train.scale(feature_range=(1,2))
exg_train=pd.concat([exg_dow_train._ts,exg_moy_train._ts],axis=1)
exg_train.columns=["Day of week","Month of year"]
exg_dow_test=Process(
    ts=df_test.set_index(datetime_series[test_index])["Day_of_week"],
    ts_name="day of the week indicator",
    udm="-",
    reference_process=exg_dow_train
)
exg_dow_test.transform_as_reference_process()
exg_moy_test=Process(
    ts=df_test.set_index(datetime_series[test_index])["Month_of_year"],
    ts_name="month of the year indicator",
    udm="-",
    reference_process=exg_moy_train
)
exg_moy_test.transform_as_reference_process()
exg_test=pd.concat([exg_dow_test._ts,exg_moy_test._ts],axis=1)
exg_test.columns=["Day of week","Month of year"]

In [ ]:
edg_test=Process(
    ts=df_test.set_index(datetime_series[test_index])["Agriculture_building_load"],
    reference_process=edg_train
)
edg_test.transform_as_reference_process()

In [ ]:
edg_train_Architecture=Process(
    ts=df_Architecture_building.set_index(datetime_series)["Agriculture_building_load"].loc[
        datetime_series[train_index]
    ],
    building="Architecture",
    reference_process=edg_train
)
edg_train_Architecture.transform_as_reference_process()
edg_test_Architecture=Process(
    ts=df_Architecture_building.set_index(datetime_series)["Agriculture_building_load"].loc[
        datetime_series[test_index]
    ],
    building="Architecture",
    reference_process=edg_train
)
edg_test_Architecture.transform_as_reference_process()
edg_train_Economics=Process(
    ts=df_Economics_building.set_index(datetime_series)["Agriculture_building_load"].loc[
        datetime_series[train_index]
    ],
    building="Economics",
    reference_process=edg_train
)
edg_train_Economics.transform_as_reference_process()
edg_test_Economics=Process(
    ts=df_Economics_building.set_index(datetime_series)["Agriculture_building_load"].loc[
        datetime_series[test_index]
    ],
    building="Economics",
    reference_process=edg_train
)
edg_test_Economics.transform_as_reference_process()

In [ ]:
exg_dow_all=Process(
    ts=df.set_index(datetime_series)["Day_of_week"],
    ts_name="day of the week indicator",
    udm="-",
    reference_process=exg_dow_train
)
exg_dow_all.transform_as_reference_process()
exg_moy_all=Process(
    ts=df.set_index(datetime_series)["Month_of_year"],
    ts_name="month of the year indicator",
    udm="-",
    reference_process=exg_moy_train
)
exg_moy_all.transform_as_reference_process()
exg_all=pd.concat([exg_dow_all._ts,exg_moy_all._ts],axis=1)
exg_all.columns=["Day of week","Month of year"]

In [ ]:
edg_all=Process(
    ts=df.set_index(datetime_series)["Agriculture_building_load"],
    reference_process=edg_train
)
edg_all.transform_as_reference_process()

In [ ]:
edg_Architecture_all=Process(
    ts=df_Architecture_building.set_index(datetime_series)["Agriculture_building_load"],
    building="Architecture",
    reference_process=edg_train
)
edg_Architecture_all.transform_as_reference_process()
edg_Economics_all=Process(
    ts=df_Economics_building.set_index(datetime_series)["Agriculture_building_load"],
    building="Economics",
    reference_process=edg_train
)
edg_Economics_all.transform_as_reference_process()

In [ ]:
edg_all.rolling("mean",detrend=False)
edg_Architecture_all.rolling("mean",detrend=False)
edg_Economics_all.rolling("mean",detrend=False)

## Functions

In [ ]:
from statsmodels.graphics.tsaplots import plot_predict
from statsmodels.tsa.arima.model import ARIMA,ARIMAResults
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
def get_smx_id(
        model:Union[ARIMA,SARIMAX,None]=None,order:int|list[int]|None=None,
        seasonal_order:int|list[int]|None=None
    )->str:
    if model is None:
        assert all([order is not None,seasonal_order is not None]),"order and seasonal order become mandatory arguments when model is None."
        ns_orders=order
        s_orders=seasonal_order
    else:
        ns_orders=model.order
        s_orders=model.seasonal_order
    def list_orders(orders)->str:
        if isinstance(orders,int) or isinstance(orders,np.integer):
            orders=[orders]
        str_orders="_".join(["{}".format(order) for order in orders])
        return str_orders
    id=""
    for i,type in enumerate(["p","d","q"]):
        id+=type
        id+="="
        id+=list_orders(ns_orders[i])
        id+="_"
    for i,type in enumerate(["P","D","Q","m"]):
        id+=type
        id+="="
        id+=list_orders(s_orders[i])
        if i!=3:
            id+="_"
    return id

In [ ]:
def fit_predict_apply_in_sliding_window(
        order:tuple[int|list[int],int,int|list[int]],
        seasonal_order:tuple[int|list[int],int,int|list[int],int],
        edg_test:Process,
        edg_test_generalization:list[Process],
        exg_test:pd.Series|pd.DataFrame,
        exg_test_generalization:dict["str",pd.Series|pd.DataFrame],
        lookback_start:pd.Timestamp,
        n_past:int=n_past,
        n_future:int=n_future,
        freq:str="1h",
        significance_level:float=0.05,
        plot:bool=False,
        avoid_apply:bool=False,
        with_ci:bool=True,
        recover_original_scale:bool=True,
        tol:float=1e-10,
        figsize:tuple[float]=(16,9),
        linewidth:float=1.5,
        show_future_observed:bool=True,
        use_dates:bool=True,
        custom_ticks:bool=True,
        add_loss_to_title:bool=True,
        plotting_time_interval:int=24,
        full_output:bool=False,
        include_target_in_full_output:bool=False,
        take_times:bool=False
)->dict[str,pd.DataFrame|pd.Series]|tuple[dict[str,pd.DataFrame|pd.Series],pd.Timestamp,dict[str,dict[str,dict[str,float]]],pd.DatetimeIndex,dict[str,torch.Tensor]]|None:
    p_max=max(order[0]) if isinstance(order[0],list) else order[0]
    q_max=max(order[2]) if isinstance(order[2],list) else order[2]
    P_max=max(seasonal_order[0]) if isinstance(seasonal_order[0],list) else seasonal_order[0]
    Q_max=max(seasonal_order[2]) if isinstance(seasonal_order[2],list) else seasonal_order[2]
    AR_max_lag=p_max+P_max*seasonal_order[3]+order[1]+seasonal_order[1]*seasonal_order[3]
    MA_max_lag=q_max+Q_max*seasonal_order[3]
    assert AR_max_lag<=n_past,"AR lags exceed the lookback window size! Max AR lag={}.".format(AR_max_lag)
    assert MA_max_lag<=n_past,"MA lags exceed the lookback window size! Max MA lag={}.".format(MA_max_lag)
    assert isinstance(edg_test,Process),"edg_test must be a Process."
    assert isinstance(edg_test._ts.index,pd.DatetimeIndex),"edg_test._ts must have a DatetimeIndex."
    assert isinstance(exg_test.index,pd.DatetimeIndex),"exg_test must have a DatetimeIndex."
    assert all(edg_test._ts.index==exg_test.index),"Endogenous and exogenous variables to fit on don't share the index."
    for pcs in edg_test_generalization:
        assert all(edg_test._ts.index==pcs._ts.index),"{} endogenous time series' index does not align with edg_test._ts.index".format(pcs._building)
    for bdg,bdg_exg in exg_test_generalization.items():
        assert all(bdg_exg.index==pcs._ts.index),"{} exogenous time series' index does not align with edg_test._ts.index".format(bdg)
    if lookback_start.tz is None:
        print("lookback_start is tz-unaware. It'll be localized in the same tz of the edg_test._ts.index.", flush=True)
        lookback_start=lookback_start.tz_localize(edg_test._ts.index.min().tz)
    edg_test_generalization_copy={pcs._building:copy.deepcopy(pcs) for pcs in edg_test_generalization}
    lookback_end=lookback_start+pd.Timedelta(freq)*(n_past-1)
    smx=ARIMA(
        endog=copy.deepcopy(edg_test)._ts.loc[lookback_start:lookback_end],
        exog=exg_test.loc[lookback_start:lookback_end],
        order=order,
        seasonal_order=seasonal_order,
        trend="n",
        missing="raise"
    )

    start_window_time=perf_counter()
    smx_results=smx.fit(low_memory=False)
    lookahead_start=lookback_end+pd.Timedelta(freq)
    lookahead_end=lookahead_start+pd.Timedelta(freq)*(n_future-1)
    
    forecasts=smx_results.get_forecast(
        steps=24,
        exog=exg_test.loc[lookahead_start:lookahead_end]
    )
    end_window_time=perf_counter() 
    window_time=end_window_time-start_window_time 
    
    all_forecasts={edg_test._building:forecasts}
    all_edg_test={edg_test._building:copy.deepcopy(edg_test)}
    if not avoid_apply:
        for bdg,pcs in edg_test_generalization_copy.items():
            assert bdg not in all_forecasts.keys(),"Duplicate building names were found at apply() time. {} is duplicate.".format(bdg)
            smx_generalization_results=smx_results.apply(
                endog=pcs._ts.loc[lookback_start:lookback_end],
                exog=exg_test_generalization[bdg].loc[lookback_start:lookback_end]
            )
            all_forecasts[bdg]=smx_generalization_results.get_forecast(
                steps=24,
                exog=exg_test_generalization[bdg].loc[lookahead_start:lookahead_end]
            )
            all_edg_test[bdg]=pcs
    
    all_forecasts_with_ci={}
    all_metrics_dict={}
    all_lookahead_observed={}
    for bdg,fcs in all_forecasts.items():
        if not recover_original_scale:
            to_concat=[fcs.predicted_mean,fcs.conf_int(alpha=significance_level)]
        else:
            mean_process=Process(
                ts=fcs.predicted_mean,
                building=bdg,
                ts_name="forecasted electricity consumption (predicted mean)",
            )
            lower_process=Process(
                ts=fcs.conf_int(alpha=significance_level)["lower y"],
                building=bdg,
                ts_name="forecasted electricity consumption (lower limit)",
            )
            upper_process=Process(
                ts=fcs.conf_int(alpha=significance_level)["upper y"],
                building=bdg,
                ts_name="forecasted electricity consumption (upper limit)",
            )
            all_edg_test[bdg].transfer_state_to(mean_process)
            all_edg_test[bdg].transfer_state_to(lower_process)
            all_edg_test[bdg].transfer_state_to(upper_process)
            mean_process.back_to_original()
            lower_process.back_to_original()
            upper_process.back_to_original()
            mean_process._ts.name="predicted_mean"
            lower_process._ts.name="lower y"
            upper_process._ts.name="upper y"
            to_concat=[mean_process._ts,lower_process._ts,upper_process._ts]
            all_edg_test[bdg].back_to_original()
        forecasts_with_ci=pd.concat(
            to_concat,
            axis=1
        )
        lookahead_observed=torch.tensor(copy.deepcopy(all_edg_test[bdg]._ts.loc[lookahead_start:lookahead_end]).to_numpy())
        from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score,mean_squared_error
        metrics_dict={
            "MAE":dict(
                value=mean_absolute_error(lookahead_observed,forecasts_with_ci["predicted_mean"]),
                udm=" (-)" if not recover_original_scale else " kWh"
            ),
            "MSE":dict(
                value=mean_squared_error(lookahead_observed,forecasts_with_ci["predicted_mean"]),
                udm=r" (-)$^2$" if not recover_original_scale else r" kWh$^2$"
            ),
            "RMSE":dict(
                value=root_mean_squared_error(lookahead_observed,forecasts_with_ci["predicted_mean"]),
                udm=" (-)" if not recover_original_scale else " kWh"
            ),
            "MAPE":dict(
                value=torch.mean(
                    torch.abs(
                        lookahead_observed[lookahead_observed>=tol] - torch.tensor(copy.deepcopy(forecasts_with_ci["predicted_mean"]).to_numpy())[lookahead_observed>=tol]
                    ) / lookahead_observed[lookahead_observed>=tol]
                ).item()*100,
                udm="%"
            ),                
            "R2":dict(
                value=r2_score(lookahead_observed,forecasts_with_ci["predicted_mean"]),
                udm=""
            )
        }
        all_forecasts_with_ci[bdg]=forecasts_with_ci
        all_metrics_dict[bdg]=metrics_dict
        all_lookahead_observed[bdg]=lookahead_observed
        if plot:
            fig=plt.figure(figsize=figsize)
            ax=fig.gca()
            encoder_positions=np.arange(
                start=-n_past+1,
                stop=1
            )
            decoder_positions=np.arange(
                start=1,
                stop=n_future+1
            )
            prop_cycle = iter(plt.rcParams["axes.prop_cycle"])
            target_color = next(prop_cycle)["color"]
            predictions_color = next(prop_cycle)["color"]
            ax.plot(encoder_positions,all_edg_test[bdg]._ts.loc[lookback_start:lookback_end],color=target_color,linewidth=linewidth,label="Observed")
            if show_future_observed:
                ax.plot(decoder_positions,lookahead_observed,color=target_color,linewidth=linewidth)
            ax.plot(
                decoder_positions,
                forecasts_with_ci["predicted_mean"],
                color=predictions_color,linewidth=linewidth,label="Predicted mean"
            )
            if with_ci:
                ax.fill_between(
                    x=decoder_positions,
                    y1=forecasts_with_ci["lower y"],
                    y2=forecasts_with_ci["upper y"],
                    fc=predictions_color,
                    alpha=0.15,
                    label="Confidence interval={:.1f}%".format((1-significance_level)*100)
                )
                bottom=min(
                    [
                        0,
                        all_edg_test[bdg]._ts.loc[lookback_start:lookback_end].min(),
                        lookahead_observed.min(),
                        forecasts_with_ci["lower y"].min(),
                    ]
                )
            bottom=min(
                [
                    0,
                    all_edg_test[bdg]._ts.loc[lookback_start:lookback_end].min(),
                    lookahead_observed.min(),
                    forecasts_with_ci["predicted_mean"].min(),
                ]
            )
            title_for_loss=" Loss computed on predicted mean: "
            metrics_str=", ".join(["{}={:.2f}{}".format(
                mtc_name,
                metrics_dict[mtc_name]["value"],
                metrics_dict[mtc_name]["udm"]
            ) for mtc_name in list(metrics_dict.keys())])
            title_for_loss+=metrics_str+"."
            ax.set_ylabel("Target (kWh)" if recover_original_scale else "Target rescaled (-)")
            ax.set_xlabel("Position index (n)" if not use_dates else "") 
            title="{} building: predictions.".format(bdg)
            title=title+title_for_loss if add_loss_to_title else title
            ax.legend(
                loc="lower center",
                ncols=2 if not with_ci else 3,
                bbox_to_anchor=(0.5,1)
            )
            if custom_ticks:
                ticks=np.arange(
                    encoder_positions.min(),
                    decoder_positions.max(),
                    plotting_time_interval
                )
                if not use_dates:
                    labels=[str(tick) for tick in ticks]
                else:
                    dates=pd.date_range(lookback_start,lookahead_end,freq=freq)
                    if plotting_time_interval<24 and 24%plotting_time_interval==0:
                        labels=[]
                        for date_idx,date in enumerate(dates):
                            if date_idx%plotting_time_interval==0:
                                if date_idx%24==0:
                                    labels.append(date.strftime("%a %Y/%m/%d %H:%M"))
                                else:
                                    labels.append(date.strftime("%H:%M"))
                    else:
                        labels=[
                            date.strftime("%a %Y/%m/%d %H:%M") for date in dates[slice(0,None,plotting_time_interval)]
                        ] 
                ax.set_xticks(
                    ticks=ticks,
                    labels=labels,
                    rotation=0 if not use_dates else 90
                )
                ax.set_xlim((encoder_positions.min(),decoder_positions.max()))
            ax.set_ylim(bottom=bottom)
            fig.suptitle(title)
            fig.tight_layout()
            plt.close("all")
            display(fig)
    if not plot:
        if with_ci:
            if not full_output:
                return all_forecasts_with_ci
            else:
                if not include_target_in_full_output:
                    return all_forecasts_with_ci,lookback_start+n_future*pd.Timedelta(freq),all_metrics_dict,pd.date_range(lookahead_start,lookahead_end,freq=freq)
                else:
                    return all_forecasts_with_ci,lookback_start+n_future*pd.Timedelta(freq),all_metrics_dict,pd.date_range(lookahead_start,lookahead_end,freq=freq),all_lookahead_observed
        else:
            if not full_output:
                return {bdg:fcs_w_ci["predicted_mean"] for bdg,fcs_w_ci in all_forecasts_with_ci.items()}
            else:
                if not include_target_in_full_output:
                    if not take_times:
                        return {bdg:fcs_w_ci["predicted_mean"] for bdg,fcs_w_ci in all_forecasts_with_ci.items()},lookback_start+n_future*pd.Timedelta(freq),all_metrics_dict,pd.date_range(lookahead_start,lookahead_end,freq=freq)
                    else:
                        return {bdg:fcs_w_ci["predicted_mean"] for bdg,fcs_w_ci in all_forecasts_with_ci.items()},lookback_start+n_future*pd.Timedelta(freq),all_metrics_dict,pd.date_range(lookahead_start,lookahead_end,freq=freq),window_time
                else:
                    return {bdg:fcs_w_ci["predicted_mean"] for bdg,fcs_w_ci in all_forecasts_with_ci.items()},lookback_start+n_future*pd.Timedelta(freq),all_metrics_dict,pd.date_range(lookahead_start,lookahead_end,freq=freq),all_lookahead_observed

In [ ]:
fpa_in_sw_in_period_Summary=namedtuple(
    "fpa_in_sw_in_period_Summary",
    field_names=[
        "first_pred","last_pred","all_predicted_mean","all_metrics",
        "lookback_starts","lookahead_windows","all_edg_test",
        "exg_test","exg_test_generalization"
    ]
)
def fit_predict_apply_in_sliding_windows_in_period(
        order:tuple[int|list[int],int,int|list[int]]|None=None,
        seasonal_order:tuple[int|list[int],int,int|list[int],int]|None=None,
        edg_test:Process|None=None,
        edg_test_generalization:list[Process]|None=None,
        exg_test:pd.Series|pd.DataFrame|None=None,
        exg_test_generalization:dict["str",pd.Series|pd.DataFrame]|None=None,
        first_pred:pd.Timestamp|None=None,
        last_pred:pd.Timestamp|None=None,
        n_past:int=n_past,
        n_future:int=n_future,
        freq:str="1h",
        metric_name:Literal["MAE","MSE","RMSE","MAPE","R2"]="MAPE",
        tol:float=1e-10,
        upper_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.95),
        lower_threshold:float|tuple[Literal["quantile"],float]=("quantile",0.05),
        plot:bool=True,
        avoid_apply:bool=False,
        figsize:tuple[float,float]=(16,9),
        plot_kwargs:dict[str,]={"linewidth":1},
        fontsize:float=14,
        xlabel:str|None="Test set dates.",
        labels_rotation:float|None=None,
        return_significant_and_metrics:bool=False,
        load_results:bool=False,
        path_to_smx_models_folder:Path=Path("./Results/SMX/statsmodels_tsa_arima_model_ARIMA/Window_fit"),
        legend_only:bool=False,
        return_only_figure:bool=False,
        stairplot:bool=False,
        dpi:int|None=None,
        target_name:str="electricity consumption (kWh)",
        take_times:bool=False
)->None|tuple[dict[str,dict[str,pd.DataFrame]],dict[str,dict[str,float]]]:
    if not load_results:
        assert all([
            order is not None,
            seasonal_order is not None,
            edg_test is not None,
            edg_test_generalization is not None,
            exg_test is not None,
            exg_test_generalization is not None,
        ]),"When no saved_results are provided, the following arguments become mandatory:\
            \norder, seasonal_order, edg_test, edg_test_generalization, exg_test, exg_test_generalization."
        import sys
        original_stdout=sys.stdout
        import io
        import warnings
        if first_pred is None:
            first_pred=edg_test._ts.index.min()+n_past*pd.Timedelta(freq)
        if last_pred is None:
            last_pred=edg_test._ts.index.max()
        assert all([first_pred.minute==0,last_pred.minute==0]),"first_pred date and last_pred date must have minute=0."
        try:
            assert first_pred.tz is not None,"first_pred date is tz-unaware. It'll be localized as edg_test._ts.index."
        except AssertionError as e:
            print(e,flush=True)
            first_pred=first_pred.tz_localize(edg_test._ts.index.min().tz)
        try:
            assert last_pred.tz is not None,"last_pred date is tz-unaware. It'll be localized as edg_test._ts.index."
        except AssertionError as e:
            print(e,flush=True)
            last_pred=last_pred.tz_localize(edg_test._ts.index.min().tz)
        try:
            assert first_pred.hour==0,"first_pred date must with hour=0. first_pred will be adjusted coherently."
        except AssertionError as e:
            print(e,flush=True)
            first_pred=pd.Timestamp(year=first_pred.year,month=first_pred.month,day=first_pred.day,hour=0,minute=0,tz=first_pred.tz)
        try:
            assert last_pred.hour==23,"last_pred date must be with hour=23. last_pred will be adjusted coherently."
        except AssertionError as e:
            print(e,flush=True)
            last_pred=pd.Timestamp(year=last_pred.year,month=last_pred.month,day=last_pred.day,hour=23,minute=0,tz=last_pred.tz)
        assert first_pred>=edg_test._ts.index.min()+n_past*pd.Timedelta(freq),"first_pred is out of edg_test._ts.index."
        assert last_pred<=edg_test._ts.index.max(),"last_pred is out of edg_test._ts.index."
        if take_times:
            window_times=[]
        next_lookback_start=first_pred-n_past*pd.Timedelta(freq)
        lookback_starts=[]
        lookahead_windows=[]
        start=True
        while next_lookback_start<=(last_pred-n_past*pd.Timedelta(freq)-n_future*pd.Timedelta(freq)+pd.Timedelta(freq)):
            lookback_starts.append(next_lookback_start)
            if not take_times:
                all_lka_mean,next_lookback_start,all_metrics_dict,lka=fit_predict_apply_in_sliding_window(
                    order,seasonal_order,edg_test,edg_test_generalization,exg_test,
                    exg_test_generalization,next_lookback_start,with_ci=False,
                    full_output=True,avoid_apply=avoid_apply
                )
            else:
                all_lka_mean,next_lookback_start,all_metrics_dict,lka,window_time=fit_predict_apply_in_sliding_window(
                    order,seasonal_order,edg_test,edg_test_generalization,exg_test,
                    exg_test_generalization,next_lookback_start,with_ci=False,
                    full_output=True,avoid_apply=avoid_apply,
                    take_times=take_times
                )
                window_times.append(window_time)

            if start:
                all_predicted_mean={bdg:pdm for bdg,pdm in all_lka_mean.items()}
                mtcs=all_metrics_dict[next(iter(all_metrics_dict))].keys()
                all_metrics={bdg:{mtc:[] for mtc in mtcs} for bdg in all_metrics_dict.keys()}
                start=False
            else:
                for bdg in all_predicted_mean.keys():
                    all_predicted_mean[bdg]=pd.concat(
                        [all_predicted_mean[bdg],all_lka_mean[bdg]]
                    )
            for bdg in all_metrics.keys():
                for mtc in mtcs:
                    all_metrics[bdg][mtc].append(all_metrics_dict[bdg][mtc]["value"])
            lookahead_windows.append(lka)
            if len(lookback_starts)==1:
                captured_output=io.StringIO()
                sys.stdout=captured_output
                warnings.simplefilter("ignore")
        sys.stdout=original_stdout
        if take_times:
            warmup_steps = 1
            if len(window_times) > warmup_steps:
                benchmarked_times = window_times[warmup_steps:]
            else:
                benchmarked_times = window_times

            avg_window_time = np.mean(benchmarked_times)
            std_window_time = np.std(benchmarked_times)
            
            print("\n" + "="*60)
            print(f"SLIDING WINDOW OPERATIONAL LATENCY ({len(benchmarked_times)} non-warmup windows tracked):")
            print(f"Metrics represent combined Fit + Inference on an Agriculture window.")
            print("-" * 60)
            print(f"  Average Time per window: {avg_window_time:.2f} s")
            print(f"  Standard Deviation:             {std_window_time:.2f} s")
            print("="*60 + "\n", flush=True)
        warnings.resetwarnings()
        all_metrics_significant={}
        all_edg_test={edg_test._building:copy.deepcopy(edg_test)}
        assert all([pcs._building not in all_edg_test.keys() for pcs in edg_test_generalization]),"Duplicate buildings found across edg_test and edg_test_generalization."
        all_edg_test.update({pcs._building:copy.deepcopy(pcs) for pcs in edg_test_generalization})
        results_to_be_saved=fpa_in_sw_in_period_Summary(
            first_pred=first_pred,
            last_pred=last_pred,
            all_predicted_mean=all_predicted_mean,
            all_metrics=all_metrics,
            lookback_starts=lookback_starts,
            lookahead_windows=lookahead_windows,
            all_edg_test=all_edg_test,
            exg_test=exg_test,
            exg_test_generalization=exg_test_generalization
        )
        smx_id=get_smx_id(order=order,seasonal_order=seasonal_order)
        path_to_smx_results_folder=os.path.join(path_to_smx_models_folder,smx_id)
        os.makedirs(path_to_smx_results_folder,exist_ok=True)
        period_id="{}_to_{}".format(
            first_pred.strftime("%Y_%m_%d")+"_h{}".format(
                "0{}".format(first_pred.hour) if len(str(first_pred.hour))==1 else first_pred.hour
            ),
            last_pred.strftime("%Y_%m_%d")+"_h{}".format(
                "0{}".format(last_pred.hour) if len(str(last_pred.hour))==1 else last_pred.hour
            )
        )
        if period_id+".pkl" in os.listdir(path_to_smx_results_folder):
            print("'{}' results already exists in '{}'.".format(period_id,path_to_smx_results_folder),flush=True)
            from datetime import datetime
            sfx="_"+datetime.now().strftime("%Y_%m_%d_h%H_m%M_s%S")
            period_id="{}{}".format(period_id,sfx)
            print("The new period_id is '{}'.".format(period_id),flush=True)
        save_variable(
            results_to_be_saved,
            os.path.join(path_to_smx_results_folder,period_id+".pkl")
        )
        if take_times:
            save_variable(
                window_times,
                os.path.join(path_to_smx_results_folder,period_id+"_window_times"+".pkl")
            )
    else:
        all_metrics_significant={}
        smx_id=get_smx_id(order=order,seasonal_order=seasonal_order)
        assert os.path.exists(path_to_smx_models_folder),"No sliding window fits have been done yet."
        assert smx_id in os.listdir(path_to_smx_models_folder),"There are no saved results for '{}'.".format(smx_id)
        path_to_smx_results_folder=os.path.join(path_to_smx_models_folder,smx_id)
        file_ids=[id for id in os.listdir(path_to_smx_results_folder)]
        print("Results available in '{}':\n{}".format(path_to_smx_results_folder,file_ids),flush=True)
        file_id=file_ids[int(input("Type the index (from 0) of the results to be loaded:"))]
        saved_results=load_variable(os.path.join(path_to_smx_results_folder,file_id))
        all_edg_test=copy.deepcopy(saved_results.all_edg_test)
        if first_pred is None:
            first_pred=saved_results.first_pred
        if last_pred is None:
            last_pred=saved_results.last_pred
        try:
            assert first_pred.tz is not None,"first_pred date is tz-unaware. It'll be localized as edg_test._ts.index."
        except AssertionError as e:
            print(e,flush=True)
            first_pred=first_pred.tz_localize(all_edg_test[next(iter(all_edg_test))]._ts.index.min().tz)
        try:
            assert last_pred.tz is not None,"last_pred date is tz-unaware. It'll be localized as edg_test._ts.index."
        except AssertionError as e:
            print(e,flush=True)
            last_pred=last_pred.tz_localize(all_edg_test[next(iter(all_edg_test))]._ts.index.min().tz)
        try:
            assert first_pred.hour==0,"first_pred date must with hour=0. first_pred will be adjusted coherently."
        except AssertionError as e:
            print(e,flush=True)
            first_pred=pd.Timestamp(year=first_pred.year,month=first_pred.month,day=first_pred.day,hour=0,minute=0,tz=first_pred.tz)
        try:
            assert last_pred.hour==23,"last_pred date must be with hour=23. last_pred will be adjusted coherently."
        except AssertionError as e:
            print(e,flush=True)
            last_pred=pd.Timestamp(year=last_pred.year,month=last_pred.month,day=last_pred.day,hour=23,minute=0,tz=last_pred.tz)
        assert all([first_pred.minute==0,last_pred.minute==0]),"first_pred date and last_pred date must have minute=0."
        assert first_pred>=saved_results.first_pred,"Saved results' first_pred is after the provided first_pred."
        assert last_pred<=saved_results.last_pred,"Saved results' last_pred is before the provided last_pred."
        all_predicted_mean={
            bdg:copy.deepcopy(saved_results.all_predicted_mean[bdg]).loc[
                first_pred:last_pred
            ] for bdg in saved_results.all_predicted_mean.keys()
        }
        idx_new_first=saved_results.lookback_starts.index(first_pred-n_past*pd.Timedelta(freq))
        idx_new_last=saved_results.lookback_starts.index(last_pred-(n_past+n_future-1)*pd.Timedelta(freq))
        lookback_starts=copy.deepcopy(
            saved_results.lookback_starts[
                idx_new_first:idx_new_last+1
            ]
        )
        lookahead_windows=copy.deepcopy(
            saved_results.lookahead_windows[
                idx_new_first:idx_new_last+1
            ]
        )
        mtcs=saved_results.all_metrics[next(iter(saved_results.all_metrics))].keys()
        all_metrics={
            bdg:{
                mtc:copy.deepcopy(
                    saved_results.all_metrics[bdg][mtc][idx_new_first:idx_new_last+1]
                ) for mtc in mtcs
            } for bdg in saved_results.all_metrics.keys()
        }
    figs=[]
    all_overall_metrics={}
    for bdg in all_predicted_mean.keys():
        if isinstance(upper_threshold,tuple):
            upper_label=" (P{:.1f})".format(upper_threshold[1]*100)
            bdg_upper_threshold=np.quantile(all_metrics[bdg][metric_name],upper_threshold[1])
        else:
            upper_label=""
        if isinstance(lower_threshold,tuple):
            lower_label=" (P{:.1f})".format(lower_threshold[1]*100)
            bdg_lower_threshold=np.quantile(all_metrics[bdg][metric_name],lower_threshold[1])
        else:
            lower_label=""
        windows_and_metrics=pd.DataFrame({
            "lkb_start":lookback_starts,
            "lka_window":lookahead_windows,
            metric_name:all_metrics[bdg][metric_name]
        })
        lka_significant={}
        lka_significant["above"]=windows_and_metrics.loc[
            windows_and_metrics[metric_name]>bdg_upper_threshold,"lka_window"
        ]
        lka_significant["below"]=windows_and_metrics.loc[
            windows_and_metrics[metric_name]<bdg_lower_threshold,"lka_window"
        ]
        metrics_significant={}
        metrics_significant["above"]=windows_and_metrics.loc[
            lka_significant["above"].index,["lkb_start",metric_name]
        ]
        metrics_significant["below"]=windows_and_metrics.loc[
            lka_significant["below"].index,["lkb_start",metric_name]
        ]
        from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
        edg_in_period=copy.deepcopy(all_edg_test[bdg])
        edg_in_period.back_to_original()
        target=torch.tensor(copy.deepcopy(edg_in_period._ts.loc[first_pred:last_pred]).to_numpy())
        preds=torch.tensor(copy.deepcopy(all_predicted_mean[bdg]).to_numpy())
        overall_metrics={
            "MAE":mean_absolute_error(target,preds),
            "MSE":mean_squared_error(target,preds),
            "RMSE":root_mean_squared_error(target,preds),
            "MAPE":torch.mean(
                torch.abs(
                    target[target>=tol] - preds[target>=tol]
                ) / target[target>=tol]
            ).item()*100,             
            "R2":r2_score(target,preds),
        }
        all_overall_metrics[bdg]=overall_metrics
        names_to_udm={
            "MAE":" kWh",
            "MSE":r" kWh$^2$",
            "RMSE":" kWh",
            "MAPE":"%",
            "R2":"",
        }

        if plot:
            absolute_min=min([target.min(),preds.min()])
            absolute_max=max([target.max(),preds.max()])
            with plt.rc_context({"font.size":fontsize}):
                fig=plt.figure(figsize=figsize,dpi=dpi)    
                target_color="r"
                preds_color="b"
                bad_color="#FFD700" 
                good_color = "g"
                ax_target=fig.gca()
                decoder_dates=pd.date_range(first_pred,last_pred,freq=freq)
                if not stairplot:
                    ax_target.plot(decoder_dates,target,color=target_color,**plot_kwargs)
                else:
                    ax_target.step(decoder_dates,target,color=target_color,**plot_kwargs,where="post")
                ax_target.set_ylim(
                    bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                    top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
                )
                ax_target.set_ylabel(
                    "Observed {}".format(target_name),
                    color=target_color
                )
                ax_target.tick_params(axis="y",color=target_color,labelcolor=target_color)
                if xlabel is not None:
                    ax_target.set_xlabel(xlabel)
                better_if_lower=["MAE","MSE","RMSE","MAPE"]
                ymin,ymax=ax_target.get_ylim()
                for list_idx,above_dates in enumerate(lka_significant["above"]):
                    ax_target.fill_between(
                        x=above_dates,
                        y1=ymin,
                        y2=ymax,
                        fc=bad_color if metric_name in better_if_lower else good_color,
                        alpha=0.35,
                        label="{}>{:.4f}".format(metric_name,bdg_upper_threshold)+names_to_udm[metric_name]+upper_label if list_idx==0 else ""
                    )

                for list_idx,below_dates in enumerate(lka_significant["below"]):
                    ax_target.fill_between(
                        x=below_dates,
                        y1=ymin,
                        y2=ymax,
                        fc=good_color if metric_name in better_if_lower else bad_color,
                        alpha=0.35,
                        label="{}<{:.4f}".format(metric_name,bdg_lower_threshold)+names_to_udm[metric_name]+lower_label if list_idx==0 else ""
                    )

                ax_target.legend(loc="lower center",bbox_to_anchor=(0.5, 1.0),ncols=2)
                ax_target.set_xlim((first_pred,last_pred))
                if labels_rotation is not None:
                    for label in ax_target.get_xticklabels():
                        label.set_rotation(labels_rotation)
                ax_preds=ax_target.twinx()
                if not stairplot:
                    ax_preds.plot(decoder_dates,preds,color=preds_color)
                else:
                    ax_preds.step(decoder_dates,preds,color=preds_color,where="post")
                ax_preds.set_ylim(
                    bottom=absolute_min*0.98 if absolute_min>0 else absolute_min*1.02,
                    top=absolute_max*1.02 if absolute_max>0 else absolute_max*0.98
                )
                ax_preds.set_ylabel("Predicted {}".format(target_name),color=preds_color)
                ax_preds.tick_params(axis="y",color=preds_color,labelcolor=preds_color)
                if not legend_only:
                    suptitle_strs=["{} building".format(edg_in_period._building)+": from {} to {}".format(
                        first_pred.strftime("%Y/%m/%d")+"_h{}".format(
                            "0{}".format(first_pred.hour) if len(str(first_pred.hour))==1 else first_pred.hour
                        ),
                        last_pred.strftime("%Y/%m/%d")+"_h{}".format(
                            "0{}".format(last_pred.hour) if len(str(last_pred.hour))==1 else last_pred.hour
                        )
                    ),"\nMetrics:"]
                    for name,value in overall_metrics.items():
                        if name!="R2":
                            suptitle_strs.append("{}={:.2f}".format(name,value)+names_to_udm[name]+",")
                        else:
                            suptitle_strs.append("{}={:.3f}".format(name,value)+names_to_udm[name])
                    fig.suptitle(" ".join(suptitle_strs))
                fig.tight_layout()
                figs.append(fig)
                all_metrics_significant[bdg]=metrics_significant
    if plot:
        plt.close("all")
        for fig in figs:
            display(fig)
    if return_only_figure:
        if avoid_apply:
            return figs[0]
        else:
            return figs
    if return_significant_and_metrics:
        return all_metrics_significant,all_overall_metrics

In [ ]:
import optuna
def tune_orders_with_lkb_constraint_and_pruning(
        study_name:str,
        pruner:optuna.pruners.BasePruner,
        desired_valid_non_duplicate_trials:int,
        possible_m:list[int],
        edg_train:Process,
        edg_train_generalization:list[Process],
        exg_train:pd.Series|pd.DataFrame,
        exg_train_generalization:dict["str",pd.Series|pd.DataFrame],
        n_trials:int|None=None,
        dlim:int=1,
        Dlim:int=1,
        Plim:int|None=None,
        Qlim:int|None=None,
        plim:int|None=None,
        qlim:int|None=None,
        lkb_size:int=n_past,
        first_pred:pd.Timestamp|None=None,
        last_pred:pd.Timestamp|None=None,
        freq:str="1h",
        criterion:Literal["MAE","MAPE","RMSE","MSE","R2"]="MSE",
        tol:float=1e-10,
        timeout:int|float=8*3600,
        verbose:int|None=1,
        direction:optuna.study.StudyDirection=optuna.study.StudyDirection.MINIMIZE,
        path_to_studies_folder=Path("./Results/SMX/statsmodels_tsa_arima_model_ARIMA/Optimization")
):
    import sys
    import io
    import warnings
    from numpy.linalg import LinAlgError
    from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score
    original_stdout=sys.stdout
    assert desired_valid_non_duplicate_trials>=0,"desired_valid_non_duplicate_trials has to be >=0."
    assert max(possible_m)<=lkb_size,"The longest season can't be longer than the lkb_size."
    if first_pred is None:
        first_pred=edg_train._ts.index.min()+n_past*pd.Timedelta(freq)
    if last_pred is None:
        last_pred=edg_train._ts.index.max()
    assert all([first_pred.minute==0,last_pred.minute==0]),"first_pred date and last_pred date must have minute=0."
    try:
        assert first_pred.tz is not None,"first_pred date is tz-unaware. It'll be localized as edg_train._ts.index."
    except AssertionError as e:
        print(e,flush=True)
        first_pred=first_pred.tz_localize(edg_train._ts.index.min().tz)
    try:
        assert last_pred.tz is not None,"last_pred date is tz-unaware. It'll be localized as edg_train._ts.index."
    except AssertionError as e:
        print(e,flush=True)
        last_pred=last_pred.tz_localize(edg_train._ts.index.min().tz)
    try:
        assert first_pred.hour==0,"first_pred date must with hour=0. first_pred will be adjusted coherently."
    except AssertionError as e:
        print(e,flush=True)
        first_pred=pd.Timestamp(year=first_pred.year,month=first_pred.month,day=first_pred.day,hour=0,minute=0,tz=first_pred.tz)
    try:
        assert last_pred.hour==23,"last_pred date must be with hour=23. last_pred will be adjusted coherently."
    except AssertionError as e:
        print(e,flush=True)
        last_pred=pd.Timestamp(year=last_pred.year,month=last_pred.month,day=last_pred.day,hour=23,minute=0,tz=last_pred.tz)
    assert first_pred>=edg_train._ts.index.min()+n_past*pd.Timedelta(freq),"first_pred is out of edg_train._ts.index."
    assert last_pred<=edg_train._ts.index.max(),"last_pred is out of edg_train._ts.index."
    import optuna.logging
    logging_level={
        None:optuna.logging.get_verbosity(),
        0:optuna.logging.WARNING,
        1:optuna.logging.INFO,
        2:optuna.logging.DEBUG,
    }
    optuna.logging.set_verbosity(logging_level[verbose])
    def objective(trial:optuna.Trial):
        Plim_local=copy.deepcopy(Plim)
        Qlim_local=copy.deepcopy(Qlim)
        plim_local=copy.deepcopy(plim)
        qlim_local=copy.deepcopy(qlim)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m=trial.suggest_categorical("m",possible_m)
            D=trial.suggest_int("D",low=0,high=Dlim,step=1,log=False)
            if D==0:
                dmax=lkb_size
                d=trial.suggest_int("d",low=1,high=min([dlim,dmax]),step=1,log=False)
            else:
                dmax=lkb_size-m*D
                d=trial.suggest_int("d",low=0,high=min([dlim,dmax]),step=1,log=False)
            Pmax=math.floor((lkb_size-d-m*D)/m)
            if Plim_local is None:
                Plim_local=Pmax
            P=trial.suggest_int("P",low=0,high=min([Plim_local,Pmax]),step=1,log=False)
            pmax=lkb_size-d-m*D-m*P
            if plim_local is None:
                plim_local=pmax
            if P==0:
                p=trial.suggest_int("p",low=0,high=min([plim_local,pmax]),step=1,log=False)
            else:
                p=trial.suggest_int("p",low=0,high=min([plim_local,pmax,m-1]),step=1,log=False)
            Qmax=math.floor(lkb_size/m)
            if Qlim_local is None:
                Qlim_local=Qmax
            Q=trial.suggest_int("Q",low=0,high=min([Qlim_local,Qmax]),step=1,log=False)
            qmax=lkb_size-m*Q
            if qlim_local is None:
                qlim_local=qmax
            if Q==0:
                q=trial.suggest_int("q",low=0,high=min([qlim_local,qmax]),step=1,log=False)
            else:
                q=trial.suggest_int("q",low=0,high=min([qlim_local,qmax,m-1]),step=1,log=False)
            smx_id=get_smx_id(
                order=(p,d,q),
                seasonal_order=(P,D,Q,m)
            )
            try:
                assert all([d+m*D+p+m*P<=n_past,q+m*Q<=n_past])
                if smx_id not in study.user_attrs.keys():
                    study.set_user_attr("ValidNonDuplicate_counter",study.user_attrs["ValidNonDuplicate_counter"]+1)
                    if study.user_attrs["ValidNonDuplicate_counter"]==desired_valid_non_duplicate_trials:
                        study.stop()
                    study.set_user_attr(
                        smx_id,
                        {
                            "trial_number":trial.number,
                            "order":(p,d,q),
                            "seasonal_order":(P,D,Q,m)
                        }
                    )
                    next_lookback_start=first_pred-n_past*pd.Timedelta(freq)
                    lookback_starts=[]
                    lookahead_windows=[]
                    prediction_day_idx=0
                    start=True
                    while next_lookback_start<=(last_pred-n_past*pd.Timedelta(freq)-n_future*pd.Timedelta(freq)+pd.Timedelta(freq)):
                        try:
                            try:
                                sys.stdout=io.StringIO()
                                lookback_starts.append(next_lookback_start)
                                lka_predicted,next_lookback_start,all_metrics_dict,lka,lka_observed=fit_predict_apply_in_sliding_window(
                                    (p,d,q),(P,D,Q,m),edg_train,edg_train_generalization,exg_train,
                                    exg_train_generalization,next_lookback_start,with_ci=False,
                                    full_output=True,avoid_apply=True,freq=freq,include_target_in_full_output=True
                                )
                                if start:
                                    pdm_up_to_prediction_day_idx={bdg:pdm for bdg,pdm in lka_predicted.items()}
                                    obs_up_to_prediction_day_idx={bdg:obs for bdg,obs in lka_observed.items()}
                                    mtcs=all_metrics_dict[next(iter(all_metrics_dict))].keys()
                                    all_metrics={bdg:{mtc:[] for mtc in mtcs} for bdg in all_metrics_dict.keys()}
                                    start=False
                                else:
                                    for bdg in pdm_up_to_prediction_day_idx.keys():
                                        pdm_up_to_prediction_day_idx[bdg]=pd.concat(
                                            [pdm_up_to_prediction_day_idx[bdg],lka_predicted[bdg]]
                                        )
                                    for bdg in obs_up_to_prediction_day_idx.keys():
                                        obs_up_to_prediction_day_idx[bdg]=torch.concat(
                                            [obs_up_to_prediction_day_idx[bdg],lka_observed[bdg]]
                                        )
                                for bdg in all_metrics.keys():
                                    for mtc in mtcs:
                                        all_metrics[bdg][mtc].append(all_metrics_dict[bdg][mtc]["value"])
                                lookahead_windows.append(lka)
                                preds=torch.tensor(copy.deepcopy(pdm_up_to_prediction_day_idx[edg_train._building]).to_numpy())
                                target=copy.deepcopy(obs_up_to_prediction_day_idx[edg_train._building])
                                metrics_up_to_prediction_day_idx={
                                    "MAE":mean_absolute_error(target,preds),
                                    "MSE":mean_squared_error(target,preds),
                                    "RMSE":root_mean_squared_error(target,preds),
                                    "MAPE":torch.mean(
                                        torch.abs(
                                            target[target>=tol] - preds[target>=tol]
                                        ) / target[target>=tol]
                                    ).item()*100,             
                                    "R2":r2_score(target,preds),
                                }
                                trial.report(
                                    value=metrics_up_to_prediction_day_idx[criterion],
                                    step=prediction_day_idx
                                )
                                if trial.should_prune():
                                    sys.stdout=original_stdout
                                    print("Trial {} is going to be pruned at prediction_day_idx={} end.".format(
                                        trial.number,prediction_day_idx
                                        ),flush=True
                                    )
                                    all_edg_train={edg_train._building:copy.deepcopy(edg_train)}
                                    assert all([pcs._building not in all_edg_train.keys() for pcs in edg_train_generalization]),"Duplicate buildings found across edg_train and edg_train_generalization."
                                    all_edg_train.update({pcs._building:copy.deepcopy(pcs) for pcs in edg_train_generalization})
                                    results_to_be_saved=fpa_in_sw_in_period_Summary(
                                        first_pred=first_pred,
                                        last_pred=lka.max(),
                                        all_predicted_mean=pdm_up_to_prediction_day_idx,
                                        all_metrics=all_metrics,
                                        lookback_starts=lookback_starts,
                                        lookahead_windows=lookahead_windows,
                                        all_edg_test=all_edg_train,
                                        exg_test=exg_train,
                                        exg_test_generalization=exg_train_generalization
                                    )
                                    path_to_smx_results_folder=os.path.join(
                                        path_to_studies_folder,
                                        study_name,
                                        "trial_{}".format(trial.number),
                                        smx_id
                                    )
                                    os.makedirs(path_to_smx_results_folder,exist_ok=True)
                                    period_id="{}_to_{}".format(
                                        first_pred.strftime("%Y_%m_%d")+"_h{}".format(
                                            "0{}".format(first_pred.hour) if len(str(first_pred.hour))==1 else first_pred.hour
                                        ),
                                        lka.max().strftime("%Y_%m_%d")+"_h{}".format(
                                            "0{}".format(lka.max().hour) if len(str(lka.max().hour))==1 else lka.max().hour
                                        )
                                    )
                                    if period_id+".pkl" in os.listdir(path_to_smx_results_folder):
                                        print("'{}' results already exists in '{}'.".format(period_id,path_to_smx_results_folder),flush=True)
                                        from datetime import datetime
                                        sfx="_"+datetime.now().strftime("%Y_%m_%d_h%H_m%M_s%S")
                                        period_id="{}{}".format(period_id,sfx)
                                        print("The new period_id is '{}'.".format(period_id),flush=True)
                                    save_variable(
                                        results_to_be_saved,
                                        os.path.join(path_to_smx_results_folder,period_id+".pkl")
                                    )
                                    raise optuna.exceptions.TrialPruned
                                prediction_day_idx+=1
                            except LinAlgError:
                                study.set_user_attr("LinAlgError_counter",study.user_attrs["LinAlgError_counter"]+1)
                                sys.stdout=original_stdout
                                print("Trial {} ('{}') is going to be pruned because it encountered a LinAlgError at prediction_day_idx={} end.".format(trial.number,smx_id,prediction_day_idx),flush=True)
                                trial.set_user_attr("LinAlgError",True)
                                score=np.inf if direction==optuna.study.StudyDirection.MINIMIZE else -np.inf
                                trial.report(value=score,step=prediction_day_idx)
                                raise optuna.exceptions.TrialPruned
                        except ValueError:
                            study.set_user_attr("ValueError_counter",study.user_attrs["ValueError_counter"]+1)
                            sys.stdout=original_stdout
                            print("Trial {} ('{}') is going to be pruned because it encountered a ValueError (NaN values at metrics computation time) at prediction_day_idx={} end.".format(trial.number,smx_id,prediction_day_idx),flush=True)
                            trial.set_user_attr("ValueError",True)
                            score=np.inf if direction==optuna.study.StudyDirection.MINIMIZE else -np.inf
                            trial.report(value=score,step=prediction_day_idx)
                            raise optuna.exceptions.TrialPruned
                    sys.stdout=original_stdout
                    all_edg_train={edg_train._building:copy.deepcopy(edg_train)}
                    assert all([pcs._building not in all_edg_train.keys() for pcs in edg_train_generalization]),"Duplicate buildings found across edg_train and edg_train_generalization."
                    all_edg_train.update({pcs._building:copy.deepcopy(pcs) for pcs in edg_train_generalization})
                    results_to_be_saved=fpa_in_sw_in_period_Summary(
                        first_pred=first_pred,
                        last_pred=last_pred,
                        all_predicted_mean=pdm_up_to_prediction_day_idx,
                        all_metrics=all_metrics,
                        lookback_starts=lookback_starts,
                        lookahead_windows=lookahead_windows,
                        all_edg_test=all_edg_train,
                        exg_test=exg_train,
                        exg_test_generalization=exg_train_generalization
                    )
                    path_to_smx_results_folder=os.path.join(
                        path_to_studies_folder,
                        study_name,
                        "trial_{}".format(trial.number),
                        smx_id
                    )
                    os.makedirs(path_to_smx_results_folder,exist_ok=True)
                    period_id="{}_to_{}".format(
                        first_pred.strftime("%Y_%m_%d")+"_h{}".format(
                            "0{}".format(first_pred.hour) if len(str(first_pred.hour))==1 else first_pred.hour
                        ),
                        last_pred.strftime("%Y_%m_%d")+"_h{}".format(
                            "0{}".format(last_pred.hour) if len(str(last_pred.hour))==1 else last_pred.hour
                        )
                    )
                    if period_id+".pkl" in os.listdir(path_to_smx_results_folder):
                        print("'{}' results already exists in '{}'.".format(period_id,path_to_smx_results_folder),flush=True)
                        from datetime import datetime
                        sfx="_"+datetime.now().strftime("%Y_%m_%d_h%H_m%M_s%S")
                        period_id="{}{}".format(period_id,sfx)
                        print("The new period_id is '{}'.".format(period_id),flush=True)
                    save_variable(
                        results_to_be_saved,
                        os.path.join(path_to_smx_results_folder,period_id+".pkl")
                    )
                else:
                    study.set_user_attr("ValidDuplicate_counter",study.user_attrs["ValidDuplicate_counter"]+1)
                    print("'{}' was already fit in trial n°{}. Trial {} is going to be pruned without being fit again.".format(smx_id,study.user_attrs[smx_id]["trial_number"],trial.number),flush=True)
                    trial.set_user_attr("duplicate",True)
                    score=np.inf if direction==optuna.study.StudyDirection.MINIMIZE else -np.inf
                    trial.report(value=score,step=0)
                    raise optuna.exceptions.TrialPruned
            except AssertionError:
                if smx_id not in study.user_attrs.keys():
                    study.set_user_attr(
                        smx_id,
                        {
                            "trial_number":trial.number,
                            "order":(p,d,q),
                            "seasonal_order":(P,D,Q,m)
                        }
                    )
                    study.set_user_attr("ExceedsLookbackNonDuplicate_counter",study.user_attrs["ExceedsLookbackNonDuplicate_counter"]+1)
                else:
                    study.set_user_attr("ExceedsLookbackDuplicate_counter",study.user_attrs["ExceedsLookbackDuplicate_counter"]+1)
                    trial.set_user_attr("duplicate",True)
                print("Trial {} ('{}') exceeds the maximum lag allowed by lkb_size. This trial is going to be pruned without being fit.".format(trial.number,smx_id),flush=True)
                trial.set_user_attr("max_lag_exceeding_lkb",True)
                score=np.inf if direction==optuna.study.StudyDirection.MINIMIZE else -np.inf
                trial.report(value=score,step=0)
                raise optuna.exceptions.TrialPruned
            return metrics_up_to_prediction_day_idx[criterion]
    try:
        if any(
            pd.Series(
                os.listdir(
                    os.path.join(path_to_studies_folder,"{}".format(study_name))
                )
            ).isin(["{}.pkl".format(study_name)])
        ):
            study=load_variable(os.path.join(path_to_studies_folder,"{}/{}.pkl".format(study_name,study_name)))
            print("{} correctly loaded and resumed from its end.\nThe first new trial is trial n°{} (enumerated from 0)".format(
                os.path.join(path_to_studies_folder,"{}/{}.pkl".format(study_name,study_name)),
                len(study.trials)),flush=True
            )
        else:
            print("'{}' folder exists in '{}' but there are no '{}.pkl' to resume, a new study will be created...".format(
                study_name,os.path.join(path_to_studies_folder),study_name
                ),flush=True
            )
            study=optuna.study.create_study(pruner=pruner,direction=direction)
            os.makedirs(
                os.path.join(
                path_to_studies_folder,
                study_name
                ),
                exist_ok=True
            )
            study.set_user_attr("study_name",study_name)
            study.set_user_attr("ValidNonDuplicate_counter",0)        
            study.set_user_attr("ValidDuplicate_counter",0)
            study.set_user_attr("LinAlgError_counter",0)
            study.set_user_attr("ValueError_counter",0)
            study.set_user_attr("ExceedsLookbackNonDuplicate_counter",0)
            study.set_user_attr("ExceedsLookbackDuplicate_counter",0)
    except FileNotFoundError as e:
        print(e,flush=True)
        print("Since no '{}' folder exists in '{}', a new study will be created...".format(
            study_name,os.path.join(path_to_studies_folder)
            ),flush=True
        )
        study=optuna.study.create_study(pruner=pruner,direction=direction)
        os.makedirs(
            os.path.join(
            path_to_studies_folder,
            study_name
            ),
            exist_ok=True
        )
        study.set_user_attr("study_name",study_name)
        study.set_user_attr("ValidNonDuplicate_counter",0)        
        study.set_user_attr("ValidDuplicate_counter",0)
        study.set_user_attr("LinAlgError_counter",0)
        study.set_user_attr("ValueError_counter",0)
        study.set_user_attr("ExceedsLookbackNonDuplicate_counter",0)
        study.set_user_attr("ExceedsLookbackDuplicate_counter",0)
    if desired_valid_non_duplicate_trials>0:
        study.optimize(
            func=objective,
            n_trials=n_trials,
            timeout=timeout,
            gc_after_trial=True,
        )
        save_variable(
            study,
            os.path.join(
                path_to_studies_folder,
                "{}/{}.pkl".format(study_name,study_name),
            )
        )
        print("Study named '{}' correctly saved at '{}'.".format(
            study_name,os.path.join(
                path_to_studies_folder,
                "{}/{}.pkl".format(study_name,study_name),
                )
            ),flush=True
        )
    else:
        print("desired_valid_non_duplicate_trials=0, thus no new trials are run. The study is returned as is.")
    return study

## Tuning of the orders

In [ ]:
study=tune_orders_with_lkb_constraint_and_pruning(
    study_name="dlim=1_Dlim=1_plim=24_qlim=24_Plim=None_Qlim=None_m=168_1_week",
    desired_valid_non_duplicate_trials=0,
    pruner=optuna.pruners.SuccessiveHalvingPruner(
        min_resource=2,
        reduction_factor=2,
        min_early_stopping_rate=0,
        bootstrap_count=0
    ),
    possible_m=[168],
    plim=24,
    qlim=24,
    edg_train=edg_train,
    edg_train_generalization=[edg_train_Architecture,edg_train_Economics],
    exg_train=exg_train,
    exg_train_generalization={"Architecture":exg_train,"Economics":exg_train},
    first_pred=pd.Timestamp(year=2021,month=10,day=18,hour=0,minute=0),
    last_pred=pd.Timestamp(year=2021,month=10,day=24,hour=23,minute=0),
    criterion="MSE",
    timeout=24*3600
)

In [ ]:
val_losses_df=find_trial_in_study(
    study,return_only_val_losses_df=True,
    pick_best=True,
)
display(val_losses_df.sort_values(
    by="trial best validation loss").rename(
    columns={"trial best epoch number":"Best up to prediction_day_idx","trial best validation loss":"Best MSE (kWh^2) up to prediction_day_idx"}
    )
)

In [ ]:
trial_number=None
was_found=False
for mdl_id in [md_id for md_id in study.user_attrs.keys() if md_id not in [
        'study_name','ValidNonDuplicate_counter','ValidDuplicate_counter',
        'LinAlgError_counter','ValueError_counter','ExceedsLookbackNonDuplicate_counter',
        'ExceedsLookbackDuplicate_counter',
        ]
    ]:
    if trial_number is not None:
        if study.user_attrs[mdl_id]["trial_number"]==trial_number:
            order_to_load=study.user_attrs[mdl_id]["order"]
            seasonal_order_to_load=study.user_attrs[mdl_id]["seasonal_order"]
            was_found=True
            num_pms=np.array(
                [order_to_load[0],order_to_load[2]],
                dtype=int
            ).sum()+np.array(
                [seasonal_order_to_load[0],seasonal_order_to_load[2]],
                dtype=int
            ).sum()+len(exg_train.columns)
            break
    else:
        if study.user_attrs[mdl_id]["trial_number"]==study.best_trial.number:
            order_to_load=study.user_attrs[mdl_id]["order"]
            seasonal_order_to_load=study.user_attrs[mdl_id]["seasonal_order"]
            was_found=True
            num_pms=np.array(
                [order_to_load[0],order_to_load[2]],
                dtype=int
            ).sum()+np.array(
                [seasonal_order_to_load[0],seasonal_order_to_load[2]],
                dtype=int
            ).sum()+len(exg_train.columns)
            break
assert was_found,"trial_number={} not found.".format(trial_number)
print("Number of model parameters: {}".format(num_pms),flush=True)
print("Model parameters:",study.trials[trial_number].params if trial_number is not None else study.best_params)
fit_predict_apply_in_sliding_windows_in_period(
    order=order_to_load,
    seasonal_order=seasonal_order_to_load,
    upper_threshold=("quantile",0.975),
    lower_threshold=("quantile",0.025),
    load_results=True,
    path_to_smx_models_folder=os.path.join(
        Path("./Results/SMX/statsmodels_tsa_arima_model_ARIMA/Optimization"),
        "{}".format(study.user_attrs["study_name"]),
        "trial_{}".format(
            study.best_trial.number if trial_number is None else trial_number
        )
    ),
    fontsize=15,
    labels_rotation=90,
    xlabel=None
)

In [ ]:
models_specification={"Trial number":[],"Specification":[]}
study_stats={}
for key in study.user_attrs.keys():
    if key=="study_name":
        print("Study name:",study.user_attrs[key])
    elif key in [
        'ValidNonDuplicate_counter','ValidDuplicate_counter',
        'LinAlgError_counter','ValueError_counter','ExceedsLookbackNonDuplicate_counter',
        'ExceedsLookbackDuplicate_counter',
        ]:
        study_stats[key]=study.user_attrs[key]
    else:
        models_specification["Specification"].append(key)
        models_specification["Trial number"].append(study.user_attrs[key]["trial_number"])
print(
    pd.Series(
        models_specification["Specification"],
        index=models_specification["Trial number"],
        name="Trial specification"
    )
)
print("\n")
print(
    pd.Series(
        study_stats.values(),
        index=study_stats.keys(),
        name="Study statistics"
    )
)

In [ ]:
get_hyperparameters_exploration(
    study,
    hps_in_discrete_scale=["p","d","q","P","D","Q"],
    xaxis_integer=["p","d","q","P","D","Q"],
    to_shrink=["p","d","q","P","D","Q"], 
    custom_ticks=["D","d"],
    n_custom_ticks=[2,2],
    xlabel_as_hp_name=["p","d","q"],
    figsize=(16,9),
    sns_histplot_kwargs=dict(multiple="stack"),
    markerfacecolor="lime",
    markeredgecolor="black",
    fontsize=24,
    metric_name="MAPE",
    force_hps_non_const_to_be=["p","d","q","P","D","Q"],
    best_as="lowest_objective_return",
    markersize=15,
    marker="o",
    dpi=600
)

In [ ]:
trls=study.get_trials(states=[optuna.trial.TrialState.COMPLETE])
cptrls_stats={key:[] for key in trls[0].params.keys()}
cptrls_stats["MSE (kWh^2)"]=[]
for trl in trls:
    for key,value in trl.params.items():
        cptrls_stats[key].append(value)
    cptrls_stats["MSE (kWh^2)"].append(trl.value)
cptrls_stats=pd.DataFrame(cptrls_stats)
cptrls_stats=cptrls_stats.sort_values(by="MSE (kWh^2)",ascending=True,ignore_index=True)

In [ ]:
display(cptrls_stats)

In [ ]:
display(cptrls_stats.loc[cptrls_stats["d"]==0])
display(cptrls_stats.loc[cptrls_stats["d"]==1])

In [ ]:
# Computational efficiency
do_fit=True
if do_fit:
    smx_sliding_fit_metrics_significant,_=fit_predict_apply_in_sliding_windows_in_period(
        order=(0,0,0),
        seasonal_order=(0,1,0,168),
        edg_test=edg_test,
        edg_test_generalization=[edg_test_Architecture,edg_test_Economics],
        exg_test=exg_test,
        exg_test_generalization={"Architecture":exg_test,"Economics":exg_test},
        upper_threshold=("quantile",0.975),
        lower_threshold=("quantile",0.025),
        return_significant_and_metrics=True,
        avoid_apply=True,
        take_times=True
    )

In [ ]:
window_times=load_variable(r"./Results/SMX/statsmodels_tsa_arima_model_ARIMA/Window_fit/p=0_d=0_q=0_P=0_D=1_Q=0_m=168/2022_07_08_h00_to_2023_06_30_h23_2026_06_06_h16_m42_s12_window_times.pkl")
warmup_steps = 1
if len(window_times) > warmup_steps:
    benchmarked_times = window_times[warmup_steps:]
else:
    benchmarked_times = window_times

avg_window_time = np.mean(benchmarked_times)
std_window_time = np.std(benchmarked_times)

print("\n" + "="*60)
print(f"SLIDING WINDOW OPERATIONAL LATENCY ({len(benchmarked_times)} non-warmup windows tracked):")
print(f"Metrics represent combined Fit + Inference on an Agriculture window.")
print("-" * 60)
print(f" Average Time per window: {avg_window_time:.5f} s")
print(f" Standard Deviation: {std_window_time:.5f} s")
print(f" Total fit and inference time: {np.sum(window_times):.5f} s")
print("="*60 + "\n", flush=True)

In [ ]:
# Evaluation
do_fit=False
if do_fit:
    smx_sliding_fit_metrics_significant,_=fit_predict_apply_in_sliding_windows_in_period(
        order=(0,0,0),
        seasonal_order=(0,1,0,168),
        edg_test=edg_all,
        edg_test_generalization=[edg_Architecture_all,edg_Economics_all],
        exg_test=exg_all,
        exg_test_generalization={"Architecture":exg_all,"Economics":exg_all},
        upper_threshold=("quantile",0.975),
        lower_threshold=("quantile",0.025),
        return_significant_and_metrics=True,
        avoid_apply=False
    )
else:
    smx_in_test_fig=fit_predict_apply_in_sliding_windows_in_period(
        order=(0,0,0),
        seasonal_order=(0,1,0,168),
        avoid_apply=True,
        upper_threshold=("quantile",0.975),
        lower_threshold=("quantile",0.025),
        load_results=True,
        xlabel="",
        fontsize=13,
        legend_only=True,
        figsize=(16,4.5),
        return_only_figure=True,
        dpi=600,
        stairplot=True,
        target_name="electric load (kW)"
    )

# All models in a period

In [ ]:
path_to_figs_all_models_in_test=Path("./Results")
do_save=False
if do_save:
    figs_all_models_in_test={
        "lstm":lstm_in_test_fig,
        "tft":tft_in_test_fig,
        "etsf":etsf_in_test_fig,
        "smx":smx_in_test_fig
    }
    for mdl,fig_in_test in figs_all_models_in_test.items():
        save_variable(fig_in_test,os.path.join(path_to_figs_all_models_in_test,"{}_in_test_Agri.pkl".format(mdl)))
else:
    figs_all_models_in_test={}
    for mdl in ["lstm","tft","etsf","smx"]:
        figs_all_models_in_test[mdl]=load_variable(os.path.join(path_to_figs_all_models_in_test,"{}_in_test_Agri.pkl".format(mdl)))

In [ ]:
x_and_y={"labels":labels_common_plot["Agri"]}
for mdl,fig_in_test in figs_all_models_in_test.items():
    for ax in fig_in_test.axes:
        for line in ax.get_lines():
            if line.get_color()=="b":
                x,y=line.get_data()
                x_and_y[mdl]=pd.Series(y,index=x)
                if x_and_y[mdl].index.min().tz is None:
                    s=x_and_y[mdl]
                    s.index=s.index.tz_localize(tz="UTC")

### Performance significance

In [ ]:
display(x_and_y.keys())
test_errors={}
lbls_in_test=x_and_y["labels"].loc[x_and_y["labels"].index.isin(pd.date_range(
    start=pd.Timestamp(year=2022,month=7,day=8,hour=0,tz="utc"),
    end=pd.Timestamp(year=2023,month=6,day=30,hour=23,tz="utc"),
    freq="1h"
))].squeeze()
for mdl in [k for k in x_and_y.keys() if k!="labels"]:
    test_errors[mdl]=abs(lbls_in_test-x_and_y[mdl])
print("Check on MAE: ", {"{} MAE".format(mdl):test_errors[mdl].mean().item() for mdl in test_errors.keys()})
from scipy.stats import friedmanchisquare,wilcoxon
from statsmodels.stats.multitest import multipletests

stat, p_friedman = friedmanchisquare(
    test_errors["tft"],test_errors["smx"],test_errors["lstm"],test_errors["etsf"]
)
print(f"Friedman Test p-value: {p_friedman}")
sl=0.05
if p_friedman < sl:
    print("p-value lower than the significance level ({:.2f}%). Significant difference found among models.\nProceeding to pairwise Wilcoxon tests...\n".format(sl*100))

    _, p_smx = wilcoxon(test_errors["tft"], test_errors["smx"], alternative='less')
    _, p_lstm = wilcoxon(test_errors["tft"], test_errors["lstm"], alternative='less')
    _, p_etsf = wilcoxon(test_errors["tft"], test_errors["etsf"], alternative='less')

    raw_p_values = [p_smx, p_lstm, p_etsf]

    reject, adj_p_values, _, _ = multipletests(raw_p_values, alpha=sl, method='holm')

    models = ["SARIMAX", "LSTM", "ETSformer"]
    for i, model in enumerate(models):
        status = "superior" if reject[i] else "not superior"
        print(f"TFT vs {model}:")
        print(f" Raw p: {raw_p_values[i]} | Adjusted p: {adj_p_values[i]} -> TFT is {status}")
else:
    print("p-value larger than or equal to the significance level ({:.2f}%). No significant difference found among the models overall.".format(sl*100))

In [ ]:
def show_all_models_in_period(
    x_and_y:dict[pd.Series],
    period:pd.DatetimeIndex,
    lkb_in_period:bool=True,
    labels={"labels":"Observed","lstm":"LSTM","tft":"TFT","etsf":"ETSformer","smx":"SARIMAX"},
    figsize:tuple[float]=(16,9),
    linewidths={"labels":0.75,"lstm":0.75,"tft":0.75,"etsf":0.75,"smx":0.75},
    styles={"labels":"-","lstm":"--","tft":"-.","etsf":(0,(1,2)),"smx":(0,(3,1,1,1,1,1))},
    ylims:tuple[float]|None=None,
    fontsize:float=13,
    rotation:float=0,
    bins_y:int=10,
    byhour:list[int]=[0,6,12,18],
    legend:bool=True,
    xlabel:str|None="Date",
    fig_ref:Figure|None=None,
    return_fig:bool=False,
    show_only_labels:bool=False,
    separator_update:dict[
        Literal[
            "active","week_count","x_offset","txt_size",
            "sep_color","sep_style","sep_linewidth",
            "box_y_fraction","box_fill_color","box_edge_color"
        ],bool|float|str
    ]={},
    dpi:int|None=None,
    square_brackets:bool=False,
    ylabel:str="Electricity demand (kWh)",
    stairplot:bool=False
    ):
    separator={
        "active":False,"week_count":1,"x_offset":6.5,"sep_linewidth":0.75,
        "box_y_fraction":0.90,"txt_size":11,"sep_color":"k","sep_style":"-.",
        "box_fill_color":"white","box_edge_color":"k"
    }
    separator.update(separator_update)
    plot_period=copy.deepcopy(period)
    with plt.rc_context({"font.size":fontsize}):
        fig=plt.figure(figsize=figsize,dpi=dpi)
        ax=fig.gca()
        colors=matplotlib.cm.tab10(range(5))
        for i,(key,series) in enumerate(x_and_y.items()):
            if key=="labels":
                if not stairplot:
                    ax.plot(
                        series.loc[series.index.isin(plot_period)],
                        color=colors[i],
                        linestyle=styles[key],
                        label=labels[key],
                        linewidth=linewidths[key])
                else:
                    ax.step(
                        series.loc[series.index.isin(plot_period)].index,
                        series.loc[series.index.isin(plot_period)],
                        color=colors[i],
                        linestyle=styles[key],
                        label=labels[key],
                        linewidth=linewidths[key],
                        where="post")
                ax.set_xlim([plot_period.min(),plot_period.max()])
                if lkb_in_period:
                    plot_period=plot_period[168:]
            else:
                if show_only_labels:
                    continue
                if not stairplot:
                    ax.plot(
                        series.loc[series.index.isin(plot_period)],
                        color=colors[i],
                        linestyle=styles[key],
                        label=labels[key],
                        linewidth=linewidths[key])
                else:
                    ax.step(
                        series.loc[series.index.isin(plot_period)].index,
                        series.loc[series.index.isin(plot_period)],
                        color=colors[i],
                        linestyle=styles[key],
                        label=labels[key],
                        linewidth=linewidths[key],
                        where="post")
        if legend:
            ax.legend(loc="lower center",bbox_to_anchor=(0.5, 1.0),ncols=5 if not show_only_labels else 1)
        if ylims is not None:
            ax.set_ylim(ylims)
        ax.xaxis.set_major_locator(HourLocator(byhour))
        xticks=ax.get_xticks()
        xlabels=[]
        for date in period:
            if separator["active"] and date.day_name()=="Sunday" and date.hour==23:
                ofs_x_txt=separator["x_offset"]
                size_txt=separator["txt_size"]
                ax.axvline(
                    x=date,
                    color=separator["sep_color"],
                    linestyle=separator["sep_style"],
                    linewidth=separator["sep_linewidth"]
                )
                ax.annotate(
                    text="W{}".format(separator["week_count"]),
                    xycoords=("data","axes fraction"),
                    xy=(
                        date,
                        separator["box_y_fraction"]
                    ),
                    textcoords="offset points",
                    xytext=(-ofs_x_txt,0),
                    size=size_txt,
                    bbox=dict(
                        boxstyle="round,pad=0.3", 
                        fc=separator["box_fill_color"], 
                        ec=separator["box_edge_color"]
                    ),
                    ha="right"
                )
                separator["week_count"]+=1
        for i,tick in enumerate(xticks):
            date=num2date(tick)
            if i==0:
                if date.hour!=0:
                    xlabels.append(date.strftime("%a %d %b %Y, %H:%M"))
                else:
                    xlabels.append(date.strftime("%a %d %b %Y"))
            else:
                to_add=""
                if date.day!=prev_stats["day"]:
                    to_add+=" %a %d"
                if date.month!=prev_stats["month"]:
                    to_add+=" %b"
                if date.year!=prev_stats["year"]:
                    to_add+=" %Y"
                if to_add!="":
                    to_add+=", "
                fmt=(to_add+"%H:%M").lstrip()
                if fmt!="%a %d %b %Y, %H:%M":
                    xlabels.append(date.strftime(fmt))
                else:
                    xlabels.append(date.strftime(to_add.rstrip(" ,")))
            prev_stats={"year":date.year,"month":date.month,"day":date.day}
        ax.set_xticks(xticks,xlabels,rotation=rotation)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=bins_y))
        if square_brackets:
            ylabel=ylabel.replace("(","[").replace(")","]")
        ax.set_ylabel(ylabel)
        if xlabel is not None:
            ax.set_xlabel(xlabel)
    if fig_ref is not None:
        match_data_area(fig_ref,fig)
    else:
        fig.tight_layout()
    plt.close("all")
    display(fig)
    if return_fig:
        return fig

In [ ]:
show_all_models_in_period(
    x_and_y,
    period=pd.date_range(
        start=pd.Timestamp(year=2022,month=12,day=19,hour=0,minute=0,tz="UTC"),
        end=pd.Timestamp(year=2023,month=1,day=1,hour=23,minute=0,tz="UTC"),
        freq="1h"
    ),
    lkb_in_period=True,
    styles={"labels":"-","lstm":"-","tft":"-","etsf":"-","smx":"-"},
    figsize=(16,5.5),
    ylims=(0,300),
    rotation=90,
    bins_y=8,
    fontsize=15, 
    legend=True,
    xlabel="Time",
    separator_update={"active":True,"week_count":2,"box_y_fraction":0.915}, 
    dpi=300,
    square_brackets=True,
    ylabel="Electricity demand [kWh]" 
)

In [ ]:
lwd=0.75
show_all_models_in_period(
    x_and_y,
    period=pd.date_range(
        start=pd.Timestamp(year=2022,month=12,day=12,hour=0,minute=0,tz="UTC"),
        end=pd.Timestamp(year=2022,month=12,day=25,hour=23,minute=0,tz="UTC"),
        freq="1h"
    ),
    lkb_in_period=True,
    figsize=(16,5.5), 
    styles={"labels":"-","lstm":"-","tft":"-","etsf":"-","smx":"-"},
    linewidths={"labels":lwd,"lstm":lwd,"tft":lwd,"etsf":lwd,"smx":lwd},
    ylims=(0,300),
    rotation=90,
    fontsize=15, 
    bins_y=8,
    xlabel="Time", 
    separator_update={"active":True,"box_y_fraction":0.915}, 
    dpi=300, 
    square_brackets=True, 
    stairplot=False, 
    ylabel="Electricity demand [kWh]" 
)

In [ ]:
show_all_models_in_period(
    x_and_y,
    period=pd.date_range(
        start=pd.Timestamp(year=2022,month=12,day=26,hour=0,minute=0,tz="UTC"),
        end=pd.Timestamp(year=2023,month=1,day=8,hour=23,minute=0,tz="UTC"),
        freq="1h"
    ),
    lkb_in_period=True,
    figsize=(16,5.5),
    styles={"labels":"-","lstm":"-","tft":"-","etsf":"-","smx":"-"},
    ylims=(0,300),
    rotation=90,
    fontsize=15,
    legend=True, 
    xlabel="Time", 
    bins_y=8,
    separator_update={"active":True,"week_count":3,"box_y_fraction":0.915},
    dpi=300,
    square_brackets=True,
    ylabel="Electricity demand [kWh]"
)

In [ ]:
show_all_models_in_period(
    x_and_y,
    period=pd.date_range(
        start=pd.Timestamp(year=2023,month=1,day=2,hour=0,minute=0,tz="UTC"),
        end=pd.Timestamp(year=2023,month=1,day=15,hour=23,minute=0,tz="UTC"),
        freq="1h"
    ),
    lkb_in_period=True,
    figsize=(16,5.5),
    styles={"labels":"-","lstm":"-","tft":"-","etsf":"-","smx":"-"},
    ylims=(0,300),
    rotation=90,
    fontsize=15,
    legend=True,
    xlabel="Time",
    bins_y=8,
    separator_update={"active":True,"week_count":4,"box_y_fraction":0.915}, 
    dpi=300,
    square_brackets=True,
    ylabel="Electricity demand [kWh]"
)